# What the knowledge can define: symmetry laws for concept grounding in neuro-symbolic learning (v2.1)

Companion notebook to `GROUNDING_SYMMETRY_proposal.md` (v2.1). The v1 SCOPE notebook is untouched.

> **If you already started the v2 pilot, keep it: every v2 unit is reused; v2.1 only adds conditions and changes the analysis.**

A neuro-symbolic (NeSy) model learns concepts (here: MNIST digits) only through a symbolic program $\beta$ applied to
them (here: a function of two digits). The **knowledge's symmetry group** $G=\mathrm{Aut}(\beta)$, the relabellings
$\pi$ of the concepts with $\beta(\pi a,\pi b)=\beta(a,b)$, together with the training support, anchors, prior and
architecture, decides what any such learner can and cannot get right. Everything below is computed from $\beta$ alone
**before training**:

| claim | prediction (no free parameters) | experiment |
|---|---|---|
| **Orbit law** | the law of the trained network is $G$-invariant; each seed's accuracy is a uniform draw from its own exact $G$-relabelling distribution; level $\sum_d \rho(d)/\lvert G\cdot d\rvert$ | E1 (13 programs, plain NeSy and re-grounding) |
| **Traps** | non-diagonal componentwise pairs with total frustration (Theorem B) create traps at exact losses $(m/n)^2\log 2$; programs without them reach the label optimum | E1 plain NeSy |
| **Architecture** | separate networks per slot obey the *componentwise* group (projected on one slot) | E2 |
| **Anchors** | $k$ concept labels leave the pointwise stabiliser $G_{(B)}$; same $k$, different outcome, decided by base structure | E3 |
| **Prior** | a known concept prior breaks $G$ to $G\cap\mathrm{Aut}(\rho)$; post-hoc fixing from $m$ unlabeled images within a Bhattacharyya bound | E4 |
| **Tied heads** | a head tied on an equitable partition stays tied (1-WL, Theorem A); label accuracy sandwiched between orbit-mode and $U_{\text{tie}}$; a zero head is dead iff the one-cell partition is equitable | E5 |
| **Support** | only labels in the training support constrain $G$; the observational equivalence set $E$ gives the identification capacity $\Lambda(E)$ (LP) | E6 |
| **Multi-task** | two programs on the same concepts leave the intersection of their groups | E7 |
| **Explanations** | the certified explanation of a prediction is its orbit; log-loss floor $\sum_g P(g)\log_2\lvert G\cdot g\rvert$ for equivariant read-outs | instrument I5 |

**How to run.** Set `SCALE` in the config cell and run all cells.
* `"smoke"`: 5-10 minutes on CPU (sklearn digits, tiny); checks that everything runs.
* `"pilot"`: 32 conditions, 610 seeded networks in 32 banks on MNIST, meant for a GPU (v2: 15 conditions, 285 networks, 15 banks, all reused).
* `"full"`: 70 conditions, 1,430 seeded networks in 76 banks. Units finished by the pilot are reused automatically.

**Two kinds of tests.**
* **Instrument checks** (I1-I6: pooled PIT and z, E5 orbit spread, Haar consistency, explanation floors, tied heads)
  verify that the code and the GPU arithmetic are equivariant. They follow from exchangeable initialisation plus an
  equivariant pipeline, so they can only fail through bugs or numerics, and they are **not** counted as evidence.
* **The primary family** (P1-P7, all on plain NeSy, Holm-Bonferroni at family-wise 0.05): achievability on programs
  without Theorem-B pairs, the trap contrast and the exact trap level, anchors under the stabiliser, multi-task, the
  prior and the support (with $\Lambda(E)$). These are the risky predictions. The re-grounding arm is reported
  separately as a method result, not as evidence for the theory.

**Pre-registration.** The plan cell computes every prediction; the next cell writes them with the decision rules to
`predictions_<scale>.json`, prints its SHA-256 and uploads it to the bucket **before** any network is trained.

**Self-test.** Before the run, the self-test cell trains one seed twice, the second time with its output units relabelled,
and checks that training commutes with the relabelling (PASS if the argmax agreement is at least 0.95). The orbit law
rests on this property; a CHECK line instead of PASS means the GPU numerics break it, and is worth a look before the long run.

**Checkpoints.** Seeds are trained in banks (one grouped network per bank, identical to separate runs). Every bank is
checkpointed after every epoch and every finished unit, log, table and figure is uploaded to the Hugging Face bucket
`APassionateResearchers/NeuroSym` under `grounding_symmetry_v2/<config hash>/`. A re-run (even on a fresh machine)
downloads that folder first and resumes: finished units are skipped, a half-trained bank continues from its last epoch.

In [ ]:
import subprocess, sys
_has_buckets = subprocess.run([sys.executable, "-c", "from huggingface_hub import HfApi; HfApi.batch_bucket_files"],
                              capture_output=True).returncode == 0
if not _has_buckets:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"], check=True)
HF_HUB_READY = True   # the checkpoint-store cell reads this, so it always runs after this install step
print("huggingface_hub with bucket support:", "already present" if _has_buckets else "installed")

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc, ctypes, json, time, hashlib, itertools, math, contextlib
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
try:
    import pandas as pd
except ImportError:
    pd = None
try:
    import psutil
except ImportError:
    psutil = None

RAM_SOFT_GB = 26.0   # stay well under a 30 GB host-RAM cap
VRAM_CAP_GB = 70.0   # cap PyTorch's allocator

def rss_gb():
    if psutil is not None:
        return psutil.Process().memory_info().rss / 2**30
    with open("/proc/self/status") as _f:
        for _line in _f:
            if _line.startswith("VmRSS:"):
                return int(_line.split()[1]) / 2**20
    return float("nan")

def trim_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except OSError:
        pass
    return rss_gb()

def gpu_setup():
    if not torch.cuda.is_available():
        return {"device": "cpu", "torch": torch.__version__}
    _free, _total = torch.cuda.mem_get_info()
    _total_gb = _total / 2**30
    _cap = min(VRAM_CAP_GB, 0.95 * _total_gb)
    torch.cuda.set_per_process_memory_fraction(min(1.0, _cap / _total_gb))
    torch.backends.cudnn.benchmark = True
    return {"device": "cuda", "name": torch.cuda.get_device_name(0), "total_gb": round(_total_gb, 1), "torch": torch.__version__}

def gpu_free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

def atomic_write(path, text):
    _tmp = f"{path}.tmp-{os.getpid()}"
    with open(_tmp, "w") as _f:
        _f.write(text)
        _f.flush()
        os.fsync(_f.fileno())
    os.replace(_tmp, path)

hw = gpu_setup()
hw["rss_gb"] = round(rss_gb(), 2)
print("numerics (logged, not changed):", dict(
    matmul_allow_tf32=torch.backends.cuda.matmul.allow_tf32, cudnn_allow_tf32=torch.backends.cudnn.allow_tf32,
    cudnn_benchmark=torch.backends.cudnn.benchmark, cudnn_deterministic=torch.backends.cudnn.deterministic,
    deterministic_algorithms=torch.are_deterministic_algorithms_enabled()))
hw

In [ ]:
SCALE = os.environ.get("NSYM_SCALE", "full")      # "smoke" (CPU check) | "pilot" (GPU, decides T1/T2) | "full"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_small = SCALE == "smoke"
CFG = dict(
    scale=SCALE,
    dataset=os.environ.get("NSYM_DATASET", "digits" if _small else "mnist"),   # mnist | digits (sklearn, offline)
    net="mlp" if _small else "cnn",       # per-image concept network (the CNN needs 28x28 MNIST)
    seed0=1000,                           # evaluation seeds start here (pilot experiments used seeds 0-7)
    bank=3 if _small else 20,             # seeds trained together in one grouped network
    smoke_seeds=3,
    epochs=3 if _small else 10,
    n_train_pairs=3000 if _small else 30000,
    n_test_pairs=1000 if _small else 5000,
    batch=256,
    lr=1e-3,
    reground_epochs=[1] if _small else [3, 6],      # exact discrete re-grounding (method gfrg) before these epochs
    reground_pairs=1000 if _small else 4000,
    lam_prior=1.0,                        # weight of KL(prior || mean prediction) for gf / gfrg
    lam_anchor=1.0,                       # weight of the anchor cross-entropy (E3)
    anchor_m=5,                           # labelled images per anchored digit (E3)
    bf16=False,                           # bf16 autocast on GPU (fp32 keeps exact G-fixed ties in E5)
    gauge_R=50 if _small else 200,        # resamples per m for post-hoc gauge-fixing (E4)
)
OUT_DIR = os.environ.get("NSYM_OUT", "grounding_symmetry_runs")
os.makedirs(OUT_DIR, exist_ok=True)

# ---- Hugging Face Storage Bucket: checkpoints and results are mirrored here during the run ----
HF_BUCKET = "APassionateResearchers/NeSy4"
HF_TOKEN = os.environ.get("HF_TOKEN") or "YOUR_HF_TOKEN_HERE"   # keep your current line
HF_PREFIX = os.environ.get("NSYM_HF_PREFIX", "grounding_symmetry_v2")   # folder inside the bucket
USE_HF = os.environ.get("NSYM_HF", "1") == "1"                          # set NSYM_HF=0 to run fully offline
print(DEVICE, CFG)

## 1. The symbolic core: groups and predictions (exact, CPU, seconds)

For each program $\beta$ the cell enumerates $G=\mathrm{Aut}(\beta)$, its orbits, pair orbits, twin classes and core,
the endomorphism count (structure theorem: $\alpha$ maps every twin class into $\tau(C)$ for a core automorphism $\tau$),
minimum and greedy bases, the slot group of separate networks, and the Bhattacharyya bound for frequency gauge-fixing.
v2.1 adds: the componentwise pairs with total frustration and their exact trap levels (Theorem B), an idealised
trap census, the coarsest equitable partition (1-WL) and the dead-head rule (Theorem A), the corrected tied-head
ceiling sandwich (Theorem D), and the observational equivalence set $E$ of a restricted support with its
identification capacity $\Lambda(E)$ (an LP). Unit tests at the end check the theory against brute force.

In [ ]:
N_CLASSES = 10
_A, _B = np.meshgrid(np.arange(N_CLASSES), np.arange(N_CLASSES), indexing="ij")
TASKS = {   # beta(a, b) for two MNIST digits a, b; full support D x D unless stated otherwise
    "add": _A + _B, "less": (_A < _B).astype(int), "max": np.maximum(_A, _B),          # rigid controls
    "summod10": (_A + _B) % 10, "absdiff": np.abs(_A - _B), "prodmod10": (_A * _B) % 10, # |G| = 2
    "summod5": (_A + _B) % 5, "summod4": (_A + _B) % 4, "summod3": (_A + _B) % 3,
    "eqmod3": ((_A % 3) == (_B % 3)).astype(int), "parity": (_A + _B) % 2, "equal": (_A == _B).astype(int),
}
_PET = np.zeros((10, 10), int)                       # Petersen graph adjacency: G = S5 acting on 10 vertices
for _i in range(5):
    for _u, _v in ((_i, (_i + 1) % 5), (5 + _i, 5 + (_i + 2) % 5), (_i, _i + 5)):
        _PET[_u, _v] = _PET[_v, _u] = 1
TASKS["petersen"] = _PET
for _name, (_t1, _t2) in {"mt_sum10_absdiff": ("summod10", "absdiff"), "mt_sum10_diff10": ("summod10", None),
                           "mt_parity_petersen": ("parity", "petersen")}.items():
    # multi-task: both labels are observed for every pair, so the program is the joint label (its group is the intersection)
    _j = TASKS[_t1] * 100 + (TASKS[_t2] if _t2 else (_A - _B) % 10)
    TASKS[_name] = np.unique(_j, return_inverse=True)[1].reshape(_j.shape)

RHOS = {"uniform": np.full(N_CLASSES, 1.0 / N_CLASSES),                 # concept priors used by the experiments
        "geom0.8": 0.8 ** np.arange(N_CLASSES) / np.sum(0.8 ** np.arange(N_CLASSES))}
SUPPORTS = {"evenodd": np.zeros((N_CLASSES, N_CLASSES), bool)}           # v1's restricted training support
for _a, _b in [(0, 6), (2, 8), (4, 6), (4, 8), (1, 5), (3, 7), (1, 9), (3, 9)]:
    SUPPORTS["evenodd"][_a, _b] = SUPPORTS["evenodd"][_b, _a] = True

def is_aut(B, p):
    return np.array_equal(B[np.ix_(p, p)], B)

def automorphisms(B, rho=None, cap=5_000_000):
    # all bijections p of D with B[p a, p b] == B[a, b] (and rho[p] == rho); rows of the returned int8 array are p
    n = B.shape[0]
    _gen = [np.r_[1, 0, np.arange(2, n)], np.r_[np.arange(1, n), 0]]       # (0 1) and the n-cycle generate S_n
    if rho is None and all(is_aut(B, g) for g in _gen):
        return np.array(list(itertools.permutations(range(n))), dtype=np.int8)
    part = np.zeros((1, 0), dtype=np.int8)
    for i in range(n):
        new = []
        for v in range(n):
            if rho is not None and not np.isclose(rho[v], rho[i]):
                continue
            ok = np.all(part != v, axis=1) & (B[v, v] == B[i, i])
            if i:
                P = part.astype(np.int64)
                ok &= np.all(B[v, P] == B[i, :i][None, :], axis=1) & np.all(B[P, v] == B[:i, i][None, :], axis=1)
            sel = part[ok]
            if len(sel):
                new.append(np.concatenate([sel, np.full((len(sel), 1), v, np.int8)], 1))
        part = np.concatenate(new, 0) if new else np.zeros((0, i + 1), np.int8)
        if len(part) > cap:
            raise RuntimeError("group too large to enumerate")
    return part

def twin_classes(B):
    # a ~ b iff replacing a by b in any slot never changes beta (identical rows AND columns for k = 2)
    n = B.shape[0]
    cls = -np.ones(n, int)
    k = 0
    for a in range(n):
        if cls[a] < 0:
            for b in range(a, n):
                if cls[b] < 0 and np.array_equal(B[a], B[b]) and np.array_equal(B[:, a], B[:, b]):
                    cls[b] = k
            k += 1
    return cls

def core_table(B):
    cls = twin_classes(B)
    reps = [int(np.flatnonzero(cls == c)[0]) for c in range(cls.max() + 1)]
    return B[np.ix_(reps, reps)], cls

def count_endomorphisms(B):
    # structure theorem: End = {alpha : alpha maps each twin class C into tau(C) for some tau in Aut(core)}
    Bc, cls = core_table(B)
    sizes = np.bincount(cls)
    return int(sum(np.prod(sizes[t].astype(float) ** sizes) for t in automorphisms(Bc).astype(int)))

def orbits(G, n=N_CLASSES):
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for d in range(n):
        for e in np.unique(G[:, d]):
            ra, rb = find(d), find(int(e))
            if ra != rb:
                parent[max(ra, rb)] = min(ra, rb)
    _, oid = np.unique([find(d) for d in range(n)], return_inverse=True)
    return oid

def pair_orbit_sizes(G, n=N_CLASSES):
    if len(G) == math.factorial(n):                                         # S_n: diagonal and off-diagonal orbits
        return np.where(np.eye(n, dtype=bool), n, n * (n - 1))
    G64 = G.astype(np.int64)
    return np.array([[len(np.unique(G64[:, a] * n + G64[:, b])) for b in range(n)] for a in range(n)])

def stabilizer(G, Bset):
    Bset = list(Bset)
    return G if not Bset else G[np.all(G[:, Bset] == np.array(Bset)[None, :], axis=1)]

def expected_acc(G, rho):
    oid = orbits(G)
    return float(np.sum(rho / np.bincount(oid)[oid]))

def predictions(G, rho):
    # Orbit law: E[acc] = sum_d rho(d)/|G.d|; conditional on landing in G (Haar):
    # Var = sum_{d,e} rho(d)rho(e)/|G.(d,e)| - E[acc]^2 ; naive explanation correct = sum_g P(g)/|G.g|
    oid = orbits(G)
    po = pair_orbit_sizes(G)
    m = expected_acc(G, rho)
    P2 = np.outer(rho, rho)
    return dict(order=int(len(G)), n_orbits=int(oid.max() + 1), orbit_id=oid.tolist(), acc_mean=m,
                acc_sd=float(np.sqrt(max(np.sum(P2 / po) - m * m, 0.0))),
                naive_expl=float(np.sum(P2 / po)), bits_unidentified=float(np.sum(P2 * np.log2(po))))

def min_base(G, n=N_CLASSES, max_order=200_000):
    if len(G) == 1:
        return []
    if len(G) > max_order:
        return None
    for k in range(1, n + 1):
        for Bset in itertools.combinations(range(n), k):
            if len(stabilizer(G, Bset)) == 1:
                return list(Bset)

def greedy_base(G, n=N_CLASSES):
    Bset, S = [], G
    while len(S) > 1:
        oid = orbits(S)
        d = int(np.argmax(np.bincount(oid)[oid]))
        Bset.append(d)
        S = stabilizer(G, Bset)
    return Bset

def base_lower_bound(order, n=N_CLASSES):
    # |G| <= n!/(n-b)!  (each base point has at most n, n-1, ... images)
    b, prod = 0, 1
    while prod < order:
        prod *= n - b
        b += 1
    return b

def trap_symmetries(B, chunk=200_000):
    # number of NON-diagonal componentwise automorphisms (s1, s2) of the twin-reduced core, s1 != s2:
    # B[s1 a, s2 b] == B[a, b]. They cannot be realised by one shared perception head; the head mixes them
    # and gets stuck at loss log 2 (frustrated local minima). Zero means the task should not trap.
    Bc, _ = core_table(B)
    m = Bc.shape[0]
    P = np.array(list(itertools.permutations(range(m))), dtype=np.int64)
    w = np.int64(Bc.max() + 1) ** np.arange(m, dtype=np.int64)
    row_codes = np.sort(Bc @ w)
    _, mult = np.unique(Bc @ w, return_counts=True)
    per_s2 = int(np.prod([math.factorial(int(c)) for c in mult]))
    total = 0
    for s in range(0, len(P), chunk):
        Q = P[s:s + chunk]
        Bp = Bc[:, Q].transpose(1, 0, 2)                         # (chunk, a, b) = Bc[a, Q[b]]: columns permuted
        codes = np.sort(Bp @ w, axis=1)
        total += int(np.all(codes == row_codes[None], axis=1).sum()) * per_s2
    return total - len(automorphisms(Bc))

def support_table(B, support):
    # The training loss sums over ALL concept pairs, so relabelling output units leaves it unchanged iff every label
    # fibre that occurs in the data is preserved. Labels that never occur merge into one wildcard value.
    seen = np.unique(B[support])
    return np.where(np.isin(B, seen), B, B.max() + 1)

def slot_group(B, slot, chunk=200_000):
    # Separate perception networks per slot: the symmetry is the componentwise group {(s1, s2): B[s1 a, s2 b] = B[a, b]};
    # slot `slot` only sees its projection. s2 is admissible iff the rows of B[:, s2] are a permutation of the rows of B.
    M = B if slot == 1 else B.T
    n = M.shape[0]
    w = np.int64(M.max() + 1) ** np.arange(n, dtype=np.int64)
    target = np.sort(M.astype(np.int64) @ w)
    P = np.array(list(itertools.permutations(range(n))), dtype=np.int8)
    keep = []
    for s in range(0, len(P), chunk):
        Q = P[s:s + chunk].astype(np.int64)
        codes = np.sort(M.astype(np.int64)[:, Q].transpose(1, 0, 2) @ w, axis=1)
        keep.append(Q[np.all(codes == target[None], axis=1)])
    return np.concatenate(keep).astype(np.int8)

def pair_orbit_ids(G, n=N_CLASSES):
    # orbit id of every concept pair (a, b) under the diagonal action of G
    if len(G) == math.factorial(n):
        return np.where(np.eye(n, dtype=bool), 0, 1)
    G64 = G.astype(np.int64)
    lab, k = -np.ones((n, n), int), 0
    for a in range(n):
        for b in range(n):
            if lab[a, b] < 0:
                c = np.unique(G64[:, a] * n + G64[:, b])
                lab[c // n, c % n] = k
                k += 1
    return lab

def haar_acc_values(C, G, rho, chunk=500_000):
    # accuracy of every G-relabelling of the confusion C: acc(pi) = sum_d rho(d) C[d, pi(d)].
    # Under the orbit law the observed accuracy is a uniform draw from this list (exact randomisation distribution).
    n = C.shape[0]
    return np.concatenate([(C[np.arange(n)[None, :], G[s:s + chunk].astype(np.int64)] * rho[None]).sum(1)
                           for s in range(0, len(G), chunk)])

def orbit_mode_acc(B, G, rho, cells=None):
    # label accuracy of the orbit-averaged posterior: both concepts known only up to their cell (default: G-orbit),
    # drawn independently and uniformly inside it, label = the block's most frequent value. A tied head can always
    # reach this, so it is the LOWER end of the tied-head ceiling (Theorem D); v2 wrongly called it the ceiling itself.
    n = len(rho)
    oid = orbits(G, n=n) if cells is None else np.asarray(cells)
    tot = 0.0
    for a in range(n):
        for b in range(n):
            Oa, Ob = np.flatnonzero(oid == oid[a]), np.flatnonzero(oid == oid[b])
            vals = B[np.ix_(Oa, Ob)].ravel()
            tot += rho[a] * rho[b] * np.bincount(vals).max() / len(vals)
    return float(tot)

label_ceiling = orbit_mode_acc   # v2 name kept so old code still runs; it is the lower bound of the sandwich below

def tie_ceiling_bounds(B, G, rho, cells=None):
    # Corrected G-fixed (tied-head) label ceiling (Theorem D, sandwich). A head tied on an equitable partition (default:
    # the G-orbits O_i) predicts argmax_y w_a' K_y w_b with K_y[i, j] = |R_y n O_i x O_j| / (|O_i||O_j|). Labels with
    # equal K_y tie identically for every code w, so at most 1/|[y]| of them can be right:
    #   orbit-mode (lower) <= best tied label accuracy <= U_tie = sum_ab rho_a rho_b / |[beta(a, b)]|.
    # Returns (lower, upper, lower == upper).
    n = len(rho)
    oid = orbits(G, n=n) if cells is None else np.asarray(cells)
    r, ny = int(oid.max()) + 1, int(B.max()) + 1
    K = np.zeros((ny, r, r))
    np.add.at(K, (B.ravel(), np.repeat(oid, n), np.tile(oid, n)), 1.0)
    K /= np.outer(np.bincount(oid), np.bincount(oid))[None]
    _, lab_cls = np.unique(K.reshape(ny, -1).round(12), axis=0, return_inverse=True)
    csize = np.bincount(lab_cls.ravel())[lab_cls.ravel()]
    upper = float(np.sum(np.outer(rho, rho) / csize[B]))
    lower = orbit_mode_acc(B, G, rho, oid)
    if r == 1:
        upper = lower                  # one cell: the code w is constant, the head always predicts the modal label
    return lower, upper, bool(abs(upper - lower) < 1e-12)

def is_equitable(B, cells):
    # every element of a cell has the same number of out- and in-neighbours of each label in every cell (Theorem A:
    # exactly these partitions keep a tied head tied under NeSy training)
    n, ny, cells = B.shape[0], int(B.max()) + 1, np.asarray(cells)
    k = int(cells.max()) + 1
    r, c = np.repeat(np.arange(n), n), np.tile(np.arange(n), n)
    out, inn = np.zeros((n, ny, k), int), np.zeros((n, ny, k), int)
    np.add.at(out, (r, B.ravel(), cells[c]), 1)
    np.add.at(inn, (c, B.ravel(), cells[r]), 1)
    return all(np.all(X[cells == q] == X[cells == q][0]) for q in range(k) for X in (out, inn))

def equitable_coarsest(B):
    # coarsest equitable partition of the coloured digraph (D; R_y): 1-WL colour refinement from one cell, refining on
    # both out- and in-label counts. Always coarser than (or equal to) the orbit partition of Aut(beta).
    n, ny = B.shape[0], int(B.max()) + 1
    r, c = np.repeat(np.arange(n), n), np.tile(np.arange(n), n)
    col = np.zeros(n, int)
    while True:
        k = int(col.max()) + 1
        out, inn = np.zeros((n, ny, k), int), np.zeros((n, ny, k), int)
        np.add.at(out, (r, B.ravel(), col[c]), 1)
        np.add.at(inn, (c, B.ravel(), col[r]), 1)
        sig = np.concatenate([col[:, None], out.reshape(n, -1), inn.reshape(n, -1)], 1)
        new = np.unique(sig, axis=0, return_inverse=True)[1].ravel()
        if new.max() == col.max():
            break
        col = new
    _, first = np.unique(col, return_index=True)            # canonical cell names: order of first appearance
    return np.argsort(np.argsort(first))[col]

def head_dead(B):
    # a fully tied head (e.g. the zero head) never moves under NeSy training iff the one-cell partition is equitable,
    # i.e. every row and every column of beta has the same label histogram (Theorem A, corollary 3)
    return is_equitable(B, np.zeros(B.shape[0], int))

def core_level_acc(G, B, rho):
    # predicted accuracy at the level of twin classes (is d mapped into its own twin class?): the orbits of the action
    # that G induces on the twin classes. With twins, labels cannot tell images apart inside a class, so this is the
    # level at which identification is attainable for image-dependent models.
    cls = twin_classes(B)
    m = int(cls.max()) + 1
    reps = np.array([int(np.flatnonzero(cls == c)[0]) for c in range(m)])
    Gc = np.unique(cls[G.astype(np.int64)[:, reps]], axis=0)
    oid = orbits(Gc, n=m)
    return float(np.sum(rho / np.bincount(oid)[oid][cls]))

def row_balanced(B):
    # every row (and column) of the label table has the same label histogram. Then a zero concept head, whose outputs
    # are uniform, receives exactly zero gradient on every example: NeSy training never leaves it (label acc = chance)
    ny = int(B.max()) + 1
    h = lambda M: np.array([np.bincount(r, minlength=ny) for r in M])
    return bool(np.all(h(B) == h(B)[0]) and np.all(h(B.T) == h(B.T)[0]))

def label_chance(B):
    # label accuracy of always predicting the most frequent label (uniform concepts)
    return float(np.bincount(B.ravel()).max() / B.size)

def core_classes_group(B):
    # twin classes q(d) and Aut(core) (the full core automorphism group, including the hidden symmetries
    # that only collapsing maps realise); used for the explanation that survives image-level twin splitting
    Bc, cls = core_table(B)
    return cls, automorphisms(Bc)

def bhattacharyya_bound(G, rho, ms):
    # P(frequency gauge-fixing from m unlabeled images picks a wrong element) <= sum_{pi != id} BC(rho, rho o pi)^m
    bc = np.sqrt(rho[None, :] * rho[G.astype(np.int64)]).sum(1)
    bc = bc[~np.all(G == np.arange(len(rho)), axis=1)]
    return np.minimum(1.0, (bc[None, :] ** np.asarray(ms, float)[:, None]).sum(1))

SYM_PERMS = {}
def perm_table(n=N_CLASSES):
    # all n! permutations of D as rows (int8), built by insertion and cached (36 MB for n = 10)
    if n not in SYM_PERMS:
        P = np.zeros((1, 0), np.int8)
        for k in range(n):
            P = np.concatenate([np.insert(P, i, k, axis=1) for i in range(k + 1)])
        SYM_PERMS[n] = P
    return SYM_PERMS[n]

def equivalence_set(B, support, fixed=(), rho=None, chunk=400_000):
    # observational equivalence set E = {sigma : beta(sigma a, sigma b) = beta(a, b) for every (a, b) in the training
    # support} (optionally within the pointwise stabiliser of `fixed` and Aut(rho)). Worlds sigma in E produce exactly
    # the same training data, so no learner can tell them apart. E contains the support group G_S but need not be a
    # group (accidental bijective shortcuts). Scans all n! permutations; meant for restricted-support conditions.
    a, b = np.nonzero(support)
    yab = B[a, b][None]
    P = perm_table(B.shape[0])
    keep = []
    for s in range(0, len(P), chunk):
        Q = P[s:s + chunk].astype(np.int64)
        ok = np.all(B[Q[:, a], Q[:, b]] == yab, axis=1)
        if len(fixed):
            ok &= np.all(Q[:, list(fixed)] == np.asarray(fixed)[None], axis=1)
        if rho is not None:
            ok &= np.all(np.isclose(rho[Q], rho[None]), axis=1)
        keep.append(Q[ok].astype(np.int8))
    return np.concatenate(keep)

def minimax_acc(E, rho):
    # identification capacity Lambda(E) = min_{mu in Delta(E)} sum_d rho(d) max_e mu{sigma(d) = e} (Theorem 1): no
    # learner of any kind has worst-case expected concept accuracy above it over the worlds in E. Solved as an LP:
    # variables mu (one per sigma) and t_d >= mu{sigma(d) = e} for every e; minimise sum_d rho(d) t_d.
    try:
        from scipy.optimize import linprog
        from scipy import sparse
    except ImportError:
        return float("nan")
    k, n = E.shape
    rows = (np.arange(n)[None, :] * n + E.astype(np.int64)).ravel()      # constraint (d, sigma(d)) of every sigma
    A_mu = sparse.csr_matrix((np.ones(k * n), (rows, np.repeat(np.arange(k), n))), shape=(n * n, k))
    A_t = sparse.csr_matrix((-np.ones(n * n), (np.arange(n * n), np.repeat(np.arange(n), n))), shape=(n * n, n))
    res = linprog(np.r_[np.zeros(k), rho], A_ub=sparse.hstack([A_mu, A_t]).tocsr(), b_ub=np.zeros(n * n),
                  A_eq=sparse.csr_matrix(np.r_[np.ones(k), np.zeros(n)][None]), b_eq=[1.0],
                  bounds=[(0, None)] * (k + n), method="highs")
    return float(res.fun) if res.status == 0 else float("nan")

def componentwise_pairs(B, nondiagonal_only=True, chunk=20_000):
    # componentwise automorphisms (s1, s2): B[s1 a, s2 b] == B[a, b] for every a, b. s2 is built column by column and a
    # prefix survives only while the truncated rows of B[:, s2] are a permutation of the truncated rows of B (multisets
    # compared through random 58-bit row hashes); s1 is then forced row by row. Every returned pair is verified exactly.
    n, ny = B.shape[0], int(B.max()) + 1
    off = B[~np.eye(n, dtype=bool)]
    if nondiagonal_only and n > 1 and np.all(off == off[0]) and np.all(np.diag(B) == B[0, 0]) and B[0, 0] != off[0]:
        return []                     # beta = [a = b]: [s1 a = s2 b] = [a = b] forces s1 = s2 (skips all n! pairs)
    H = np.random.default_rng(12345).integers(0, 2 ** 58, size=(n, ny), dtype=np.int64)
    tgt = np.cumsum(H[np.arange(n)[None, :], B], axis=1)                  # tgt[a, j]: hash of B[a, :j+1]
    tgt_sorted = np.sort(tgt, axis=0)
    part, hp = np.zeros((1, 0), np.int64), np.zeros((1, n), np.int64)
    for j in range(n):
        Hj = H[j][B.T]                                                    # Hj[v, a] = H[j, B[a, v]]
        newp, newh = [], []
        for s in range(0, len(part), chunk):
            p, h = part[s:s + chunk], hp[s:s + chunk]
            cand = h[:, None, :] + Hj[None]                               # (prefixes, next column v, rows a)
            ok = np.all(np.sort(cand, -1) == tgt_sorted[None, None, :, j], -1)
            if j:
                ok[np.arange(len(p))[:, None], p] = False                 # columns already used
            pi, vi = np.nonzero(ok)
            newp.append(np.concatenate([p[pi], vi[:, None]], 1))
            newh.append(cand[pi, vi])
        part, hp = np.concatenate(newp), np.concatenate(newh)
    hB = tgt[:, -1]
    if len(np.unique(hB)) == n:                                           # distinct rows: s1 is unique, vectorised
        S1 = np.empty_like(part)
        S1[:, np.argsort(hB)] = np.argsort(hp, axis=1)
        sel = np.any(S1 != part, axis=1) if nondiagonal_only else np.ones(len(part), bool)
        cands = [(S1[k], part[k]) for k in np.flatnonzero(sel)]
    else:
        cands = []
        for s2 in part:
            R = B[:, s2]
            opts = [np.flatnonzero(np.all(R == B[a][None], axis=1)) for a in range(n)]
            cands += [(np.array(s1), s2) for s1 in itertools.product(*opts)
                      if not (nondiagonal_only and tuple(s1) == tuple(s2))]
    return [(s1, s2) for s1, s2 in cands if len(set(s1.tolist())) == n and np.array_equal(B[np.ix_(s1, s2)], B)]

def frustrated_pairs(B):
    # Theorem B (frustrated componentwise pairs). On the twin-free core (size n), a non-diagonal componentwise pair
    # (s1, s2), F = {a : s1 a = s2 a}, m = n - |F|, is TOTALLY FRUSTRATED if beta(s_i a, s_i b) != beta(a, b) for all
    # a, b outside F and i = 1, 2. Then M = (P_s1 + P_s2)/2 is a KKT point of the idealised NeSy loss at the exact level
    # (m/n)^2 log 2, and a strict local minimum (a trap) iff the margin condition holds (every off-support score < 2n).
    # A shared head cannot realise s1 != s2, so it mixes them. Returns counts and the predicted trap loss levels.
    Bc, _ = core_table(B)
    n = Bc.shape[0]
    tf, trap, levels, nondiag = 0, 0, set(), 0
    for s1, s2 in componentwise_pairs(Bc):
        nondiag += 1
        Fset = s1 == s2
        Fc = np.flatnonzero(~Fset)
        if not all(np.all(Bc[np.ix_(si[Fc], si[Fc])] != Bc[np.ix_(Fc, Fc)]) for si in (s1, s2)):
            continue
        tf += 1
        margin_ok = True
        for a in range(n):
            for e in set(range(n)) - {int(s1[a]), int(s2[a])}:
                NF = int(np.sum(Bc[e, s1[Fset]] == Bc[a, Fset]))
                N1 = int(np.sum(Bc[e, s1[Fc]] == Bc[a, Fc]))
                N2 = int(np.sum(Bc[e, s2[Fc]] == Bc[a, Fc]))
                score = 2 * (NF + N1 + N2) if not Fset[a] else 2 * (NF + (N1 + N2) / 2)
                margin_ok &= score < 2 * n
        trap += int(margin_ok)
        levels.add(round(float((len(Fc) / n) ** 2 * np.log(2)), 4))
    return dict(count=tf, strict=trap, nondiagonal=nondiag, levels=sorted(levels), core_n=n,
                symmetric=bool(np.array_equal(Bc, Bc.T)))

def ideal_trap_census(B, inits=200, steps=2000, seed=0, lr=0.05, tol=1e-3):
    # Idealised landscape: one row-stochastic concept head M = softmax(theta) (perfect, image-independent perception),
    # loss L(M) = -mean_{a,b} log sum_{beta(e,f) = beta(a,b)} M[a,e] M[b,f] over the full support, Adam from
    # theta ~ N(0, 1). All inits are optimised together (batched, float64). Returns the fraction of inits that end
    # above `tol` and the distinct final loss levels (3 decimals) with their counts.
    n = B.shape[0]
    ny = int(B.max()) + 1
    R = F.one_hot(torch.as_tensor(B), ny).permute(2, 0, 1).double()            # R[y, e, f] = [beta(e, f) = y]
    yab = torch.as_tensor(B).reshape(1, 1, n, n)
    gen = torch.Generator().manual_seed(int(seed))
    th = torch.randn(inits, n, n, generator=gen, dtype=torch.float64).requires_grad_(True)
    opt = torch.optim.Adam([th], lr=lr)

    def losses():
        M = torch.softmax(th, -1)[:, None]                                    # (inits, 1, n, n)
        T = M @ R[None] @ M.transpose(-1, -2)                                 # T[i, y, a, b] = P(label y | a, b)
        P = T.gather(1, yab.expand(len(th), 1, n, n))[:, 0]
        return -torch.log(P.clamp_min(1e-300)).mean((1, 2))

    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        losses().sum().backward()
        opt.step()
        with torch.no_grad():
            th.clamp_(-25.0, 25.0)       # keeps every probability above ~1e-22: no float64 subnormals (100x slowdown)
    with torch.no_grad():
        L = losses().numpy()
    lv, cnt = np.unique(np.round(L[L > tol], 3), return_counts=True)
    return dict(frac_trapped=float(np.mean(L > tol)), levels=dict(zip(lv.tolist(), cnt.tolist())), inits=int(inits), steps=int(steps))

# ---- unit tests of the theory (exact, run in seconds) -------------------------------------------------
_rng = np.random.default_rng(0)
_Gs5 = automorphisms(TASKS["summod5"])
assert len(_Gs5) == 32 and orbits(_Gs5).max() + 1 == 5
_haar = (_Gs5 == np.arange(10)).mean(1)                          # accuracy of every gauge element
assert abs(_haar.mean() - 0.5) < 1e-12 and abs(_haar.std() - predictions(_Gs5, np.full(10, .1))["acc_sd"]) < 1e-12
_rho = _rng.dirichlet(np.ones(10))                               # Burnside-type identities for a non-uniform prior
_acc = (_Gs5 == np.arange(10)).astype(float) @ _rho
assert abs(_acc.mean() - expected_acc(_Gs5, _rho)) < 1e-12 and abs(_acc.std() - predictions(_Gs5, _rho)["acc_sd"]) < 1e-12
assert count_endomorphisms(TASKS["summod4"]) == 16848 and count_endomorphisms(TASKS["summod5"]) == 1024
for _ in range(30):                                              # structure theorem vs brute force on random tables
    _n = int(_rng.integers(3, 6)); _T = _rng.integers(2, size=(_n, _n)); _T = np.minimum(_T, _T.T)
    _brute = sum(all(_T[a[i], a[j]] == _T[i, j] for i in range(_n) for j in range(_n))
                 for a in itertools.product(range(_n), repeat=_n))
    assert _brute == count_endomorphisms(_T)
assert min_base(_Gs5) is not None and len(min_base(_Gs5)) == 5 and trap_symmetries(TASKS["summod5"]) == 4
assert len(slot_group(TASKS["summod10"], 0)) == 10 and orbits(slot_group(TASKS["summod10"], 0)).max() == 0
_Gp = automorphisms(TASKS["prodmod10"])
assert abs(label_ceiling(TASKS["parity"], automorphisms(TASKS["parity"]), np.full(10, .1)) - 0.5) < 1e-12
assert abs(label_ceiling(TASKS["summod10"], automorphisms(TASKS["summod10"]), np.full(10, .1)) - 0.5) < 1e-12
_C = _rng.dirichlet(np.ones(10), size=10)                         # any confusion: Haar mean = refined prediction
_oid = orbits(_Gs5)
_ref = sum(.1 * _C[d, _oid == _oid[d]].sum() / (_oid == _oid[d]).sum() for d in range(10))
assert abs(haar_acc_values(_C, _Gs5, np.full(10, .1)).mean() - _ref) < 1e-12
assert row_balanced(TASKS["summod10"]) and row_balanced(TASKS["parity"]) and not row_balanced(TASKS["prodmod10"])
# v2.1 additions
_Bx = np.array([[2, 2, 0, 2], [2, 2, 1, 0], [0, 1, 1, 1], [2, 0, 1, 2]])  # R3's program: a tied head beats orbit-mode
_lo, _up, _ = tie_ceiling_bounds(_Bx, automorphisms(_Bx), np.full(4, .25))
assert abs(_lo - 7 / 8) < 1e-12 and _up >= 15 / 16                  # (15/16 is reached by an explicit tied code)
for _t in ("summod10", "prodmod10", "parity", "summod5"):
    assert tie_ceiling_bounds(TASKS[_t], automorphisms(TASKS[_t]), np.full(10, .1))[2]
for _t, _B in TASKS.items():                                     # dead head <=> row/column balanced; orbits equitable
    assert head_dead(_B) == row_balanced(_B)
    if _t in ("summod10", "prodmod10", "absdiff", "petersen", "mt_parity_petersen"):
        _oid, _wl = orbits(automorphisms(_B)), equitable_coarsest(_B)
        assert is_equitable(_B, _oid) and is_equitable(_B, _wl)
        assert all(len(np.unique(_wl[_oid == _o])) == 1 for _o in np.unique(_oid))   # 1-WL is coarser than the orbits
_E = equivalence_set(TASKS["add"], SUPPORTS["evenodd"])          # accidental shortcut x -> x + 5 on the EvenOdd support
assert len(_E) == 2 and any(np.array_equal(_e, (np.arange(10) + 5) % 10) for _e in _E)
for _t in ("summod5", "petersen"):                               # for a group E, Lambda(E) is the Burnside level
    _G = automorphisms(TASKS[_t])
    _l = minimax_acc(_G, _rho)
    assert np.isnan(_l) or abs(_l - expected_acc(_G, _rho)) < 1e-7
assert frustrated_pairs(TASKS["summod10"])["levels"] == [round(math.log(2), 4)]
assert frustrated_pairs(TASKS["prodmod10"])["levels"] == [0.4436] and frustrated_pairs(TASKS["absdiff"])["count"] == 0
print("symbolic core: all unit tests passed")

In [ ]:
def load_data(name):
    if name == "mnist":
        try:
            from torchvision import datasets
            root = os.environ.get("MNIST_ROOT", "data_mnist")
            tr = datasets.MNIST(root, train=True, download=True)
            te = datasets.MNIST(root, train=False, download=True)
            f = lambda d: (d.data.numpy().astype(np.float32)[:, None] / 255.0 - 0.1307) / 0.3081
            return f(tr), tr.targets.numpy(), f(te), te.targets.numpy(), "mnist"
        except Exception as e:
            print(f"MNIST unavailable ({type(e).__name__}: {e}); falling back to sklearn digits")
    from sklearn.datasets import load_digits
    d = load_digits()
    X = (d.images.astype(np.float32) / 16.0 - 0.5)[:, None]
    perm = np.random.default_rng(0).permutation(len(d.target))
    cut = int(0.7 * len(perm))
    return X[perm[:cut]], d.target[perm[:cut]], X[perm[cut:]], d.target[perm[cut:]], "digits"

def make_pool(y):
    pools = [np.flatnonzero(y == c) for c in range(N_CLASSES)]
    sizes = torch.as_tensor([len(p) for p in pools])
    return dict(pools=pools, cat=torch.as_tensor(np.concatenate(pools)), sizes=sizes, starts=torch.cumsum(sizes, 0) - sizes)

_Xtr, _ytr, _Xte, _yte, DATASET = load_data(CFG["dataset"])
if DATASET != "mnist" and CFG["net"] == "cnn":
    raise RuntimeError("the CNN needs MNIST; set NSYM_DATASET=mnist or use SCALE='smoke'")
XTR, XTE = torch.as_tensor(_Xtr, device=DEVICE), torch.as_tensor(_Xte, device=DEVICE)
YTE_NP = _yte
PTR, PTE = make_pool(_ytr), make_pool(_yte)
print(DATASET, tuple(XTR.shape), tuple(XTE.shape))

In [ ]:
assert HF_HUB_READY
from huggingface_hub import HfApi, BucketFile
from huggingface_hub.errors import EntryNotFoundError, RemoteEntryNotFoundError

class BucketStore:
    # Local run folder mirrored to an HF Storage Bucket.
    # push(): uploads a snapshot of a file in a background thread (training never waits on the network).
    # pull(): downloads every file of this run that is missing locally (used to resume); checkpoints of units whose
    #         results already exist are skipped, so a resume only fetches what it needs.

    def __init__(self, local_dir, bucket, prefix, token, enabled=True):
        self.local, self.bucket, self.prefix, self.enabled = local_dir, bucket, prefix.strip("/"), enabled
        self.pool = ThreadPoolExecutor(max_workers=1)
        self.queued, self.errors = {}, []   # remote path -> latest upload future
        self.api = HfApi(token=token) if enabled else None
        if enabled:
            try:
                self.api.bucket_info(bucket)
            except Exception as e:
                raise RuntimeError(f"Cannot reach HF bucket {bucket!r} ({type(e).__name__}: {e}). "
                                   "Check the token / bucket name, or set NSYM_HF=0 to run offline.") from e

    def _remote(self, local_path):
        return f"{self.prefix}/{os.path.relpath(local_path, self.local)}".replace(os.sep, "/")

    def pull(self):
        if not self.enabled:
            return 0
        try:
            files = [f for f in self.api.list_bucket_tree(self.bucket, prefix=self.prefix, recursive=True)
                     if isinstance(f, BucketFile) and f.path.startswith(self.prefix + "/")]
        except (EntryNotFoundError, RemoteEntryNotFoundError):
            return 0   # first run of this configuration: nothing in the bucket yet
        rel = [f.path[len(self.prefix) + 1:] for f in files]
        finished = {r[len("units/"):-len(".json")] for r in rel if r.startswith("units/") and r.endswith(".json")}
        todo = []
        for f, r in zip(files, rel):
            if r.startswith("ckpt/") and r[len("ckpt/"):-len(".pt")] in finished:
                continue
            dst = os.path.join(self.local, *r.split("/"))
            if not os.path.exists(dst):
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                todo.append((f, dst))
        if todo:
            self.api.download_bucket_files(self.bucket, todo)
        return len(todo)

    def push(self, local_path):
        if not self.enabled:
            return
        with open(local_path, "rb") as fh:   # snapshot now, so later local writes cannot race the upload
            data = fh.read()
        dest = self._remote(local_path)
        older = self.queued.get(dest)
        if older is not None:
            older.cancel()   # a newer snapshot supersedes one still waiting: memory stays bounded on slow links
        self.queued[dest] = self.pool.submit(self._upload, data, dest)

    def _upload(self, data, dest):
        for attempt in range(4):
            try:
                self.api.batch_bucket_files(self.bucket, add=[(data, dest)])
                return
            except Exception as e:
                if attempt == 3:
                    self.errors.append(f"{dest}: {type(e).__name__}: {e}")
                    print(f"[hf] upload failed for {dest}: {e}")
                else:
                    time.sleep(2 ** (attempt + 1))

    def flush(self):
        for fut in self.queued.values():
            if not fut.cancelled():
                fut.result()
        self.queued = {}
        return list(self.errors)

# the folder is keyed by everything that changes a trained network, but not by SCALE: pilot units are reused by "full"
_train_keys = ["dataset", "net", "epochs", "n_train_pairs", "n_test_pairs", "batch", "lr", "reground_epochs",
               "reground_pairs", "lam_prior", "lam_anchor", "anchor_m", "bf16"]
cfg_hash = hashlib.sha1(json.dumps([{k: CFG[k] for k in _train_keys}, DATASET, "v2"], sort_keys=True).encode()).hexdigest()[:8]
RUN_DIR = os.path.join(OUT_DIR, cfg_hash)
os.makedirs(os.path.join(RUN_DIR, "ckpt"), exist_ok=True)
STORE = BucketStore(RUN_DIR, HF_BUCKET, HF_PREFIX + "/" + cfg_hash, HF_TOKEN, USE_HF)
print(f"run folder {RUN_DIR}; restored {STORE.pull()} file(s) from hf://buckets/{HF_BUCKET}/{HF_PREFIX}/{cfg_hash}")
atomic_write(os.path.join(RUN_DIR, f"config_{SCALE}.json"), json.dumps({"cfg": CFG, "dataset": DATASET}, indent=1))
STORE.push(os.path.join(RUN_DIR, f"config_{SCALE}.json"))

## 2. Plan and pre-registration

Each condition fixes a program, a training method and one symmetry-breaking factor. Methods:
`nesy` = exact NeSy likelihood; `gfrg` = the same plus prior matching and exact discrete **re-grounding**
(cluster the network's concept distributions, search all $10!$ cluster-to-symbol maps for the label-optimal ones, keep the
one closest to the current state). Re-grounding is equivariant, so it does not choose the gauge; it only removes the
frustrated traps that componentwise symmetries create. The primary tests use plain NeSy; re-grounding is the method arm.
v2.1 conditions (plain-NeSy E1/E3/E4, E6, E7) have their own hashed seed blocks, so they share no initialisations.

The predicted group per condition: $\mathrm{Aut}(\beta_S)$ (labels absent from the support merged)
$\cap\,\mathrm{Aut}(\rho_{\text{prior}})$ (only when a non-uniform prior is matched) $\cap$ the pointwise stabiliser of
the anchors; for separate networks, the slot projection of the componentwise group.

In [ ]:
def make_condition(exp, task, method="nesy", n_seeds=20, **kw):
    c = dict(exp=exp, task=task, method=method, arch="shared", rho_data="uniform", prior_target=None,
             anchors=None, head_init="random", support=None)
    c.update(kw)
    if method in ("gf", "gfrg") and c["prior_target"] is None:
        c["prior_target"] = c["rho_data"]                     # prior matching to the data's concept prior
    c["key"] = "_".join([exp, task, method] + (["sep"] if c["arch"] == "separate" else [])
                        + (["rho08"] if c["rho_data"] != "uniform" else [])
                        + (["anc" + "".join(map(str, c["anchors"]))] if c["anchors"] else [])
                        + ([c["head_init"]] if c["head_init"] != "random" else []) + ([c["support"]] if c["support"] else []))
    if c.get("seed_block") == "hash":
        # v2.1 conditions: every condition has its own seeds (stable hash of the condition key and the replicate), so
        # no two conditions start from the same initialisations; the v2 conditions keep seed0, seed0 + 1, ...
        c["seeds"] = [int(hashlib.sha1(f"{c['key']}|{r}".encode()).hexdigest()[:8], 16) for r in range(n_seeds)]
    else:
        c["seeds"] = list(range(CFG["seed0"], CFG["seed0"] + n_seeds))
    return c

def condition_group(c):
    # the symmetry group that the orbit law says governs this condition
    B = TASKS[c["task"]] if not c["support"] else support_table(TASKS[c["task"]], SUPPORTS[c["support"]])
    if c["arch"] == "separate":
        return slot_group(B, 0)                                       # slot-1 network: projection of the componentwise group
    pt = c["prior_target"] if c["method"] in ("gf", "gfrg") else None
    G = automorphisms(B, rho=None if pt in (None, "uniform") else RHOS[pt])
    return stabilizer(G, c["anchors"] or [])

def anchor_sets(G):
    # same label budget, different predicted outcome: a base (trivial stabiliser), the same number of anchors placed
    # where they break the least symmetry, and half a base
    base = greedy_base(G)
    worst = max(itertools.combinations(range(N_CLASSES), len(base)), key=lambda s: (len(stabilizer(G, s)), [-v for v in s]))
    return {"base": base, "nonbase": list(worst), "half": base[: (len(base) + 1) // 2]}

MT_TASKS = ["mt_sum10_absdiff", "mt_sum10_diff10", "mt_parity_petersen"]

def build_plan(scale):
    # Conditions that existed in the v2 pilot are listed first and unchanged (same keys, seeds and unit names), so a
    # started v2 pilot is reused. v2.1 conditions use their own hashed seed blocks (seed_block="hash"): the plain-NeSy
    # arm that carries the primary tests (E1, E3, E4, E6 with method nesy; E7) and the re-grounding E7 arm.
    F = make_condition
    H = dict(seed_block="hash")
    E1_all = ["add", "less", "max", "summod10", "absdiff", "prodmod10", "summod5", "summod4", "summod3", "eqmod3",
              "parity", "equal", "petersen"]
    if scale == "smoke":
        k = CFG["smoke_seeds"]
        conds = [F("E1", "add", n_seeds=k, **H), F("E1", "summod10", n_seeds=k, **H), F("E1", "summod10", "gfrg", n_seeds=k),
                 F("E1", "summod5", n_seeds=k, **H), F("E2", "summod10", arch="separate", n_seeds=k),
                 F("E3", "summod5", "gfrg", anchors=anchor_sets(automorphisms(TASKS["summod5"]))["base"], n_seeds=k),
                 F("E3", "summod5", "nesy", anchors=anchor_sets(automorphisms(TASKS["summod5"]))["half"], n_seeds=k, **H),
                 F("E4", "summod10", "gfrg", rho_data="geom0.8", n_seeds=k),
                 F("E5", "prodmod10", head_init="tied", n_seeds=k), F("E5", "summod10", head_init="zero", n_seeds=k),
                 F("E6", "absdiff", support="evenodd", n_seeds=k, **H), F("E7", "mt_sum10_diff10", "nesy", n_seeds=k, **H)]
        return conds
    n_e1 = lambda t: 60 if t == "petersen" else (5 if len(automorphisms(TASKS[t])) == 1 else (40 if len(automorphisms(TASKS[t])) == 2 else 20))
    pilot8 = ["add", "summod10", "absdiff", "prodmod10", "summod5", "summod4", "parity", "petersen"]
    _a = anchor_sets(automorphisms(TASKS["petersen"]))
    if scale == "pilot":
        conds = [F("E1", t, "gfrg", n_seeds=min(n_e1(t), 20)) for t in pilot8]                      # ---- v2 pilot
        conds += [F("E3", "petersen", "gfrg", anchors=_a["base"]), F("E3", "petersen", "gfrg", anchors=_a["nonbase"]),
                  F("E5", "summod10", head_init="tied"), F("E5", "prodmod10", head_init="tied"), F("E5", "summod10", head_init="zero"),
                  F("E2", "summod10", arch="separate"), F("E4", "summod10", "gfrg", rho_data="geom0.8")]
        conds += [F("E1", t, "nesy", n_seeds=min(n_e1(t), 20), **H) for t in pilot8]                # ---- v2.1: plain NeSy
        conds += [F("E3", "petersen", "nesy", anchors=_a["base"], **H), F("E3", "petersen", "nesy", anchors=_a["nonbase"], **H),
                  F("E4", "summod10", "nesy", rho_data="geom0.8", **H)]
        conds += [F("E7", t, m, **H) for m in ("nesy", "gfrg") for t in MT_TASKS]                     # ---- v2.1: multi-task
        return conds
    conds = [F("E1", t, m, n_seeds=n_e1(t), **(H if m == "nesy" else {})) for t in E1_all for m in ("nesy", "gfrg")]  # E1
    conds += [F("E2", t, arch="separate") for t in ["summod10", "prodmod10", "summod5"]]         # E2 architecture
    for t in ["summod5", "parity", "petersen"]:                                                   # E3 anchors in training
        conds += [F("E3", t, "gfrg", anchors=a) for a in anchor_sets(automorphisms(TASKS[t])).values()]
        conds += [F("E3", t, "nesy", anchors=a, **H) for a in anchor_sets(automorphisms(TASKS[t])).values()]
    for t in ["summod10", "absdiff", "summod5", "parity"]:                                        # E4 prior as symmetry breaker
        conds += [F("E4", t, "nesy", rho_data="geom0.8", **H), F("E4", t, "gfrg", rho_data="geom0.8")]
    conds += [F("E5", t, head_init="tied") for t in ["summod10", "prodmod10", "summod5", "parity"]]  # E5 G-fixed init
    conds += [F("E5", t, head_init="zero") for t in ["summod10", "prodmod10"]]                       # degenerate G-fixed
    conds += [F("E6", t, support="evenodd", **H) for t in ["add", "summod10", "absdiff"]]        # E6 restricted support
    conds += [F("E7", t, m, **H) for m in ("nesy", "gfrg") for t in MT_TASKS]                     # E7 multi-task
    return conds

CONDITIONS = build_plan(CFG["scale"])
GROUPS, PREDICTIONS = {}, {}
TRAP_INFO, IDEAL_TRAPS = {}, {}        # per program: Theorem-B frustrated pairs; idealised-landscape trap census
_census = dict(inits=50, steps=2000) if CFG["scale"] == "smoke" else dict(inits=200, steps=2000)
for _c in CONDITIONS:
    _G = condition_group(_c)
    GROUPS[_c["key"]] = _G
    _B = TASKS[_c["task"]]
    _rho = RHOS[_c["rho_data"]]
    _p = predictions(_G, _rho)
    _p["seeds"] = _c["seeds"]
    _p["label_chance"] = label_chance(_B)
    _p["twin_free"] = bool(twin_classes(_B).max() == N_CLASSES - 1)
    _p["core_acc_pred"] = core_level_acc(_G, _B, _rho) if _c["arch"] == "shared" else None
    _p["bits_bound_uniform"] = predictions(_G, RHOS["uniform"])["bits_unidentified"]
    _p["label_ceiling"] = _p["label_ceiling_upper"] = _p["ceiling_exact"] = _p["tie_cells"] = None
    if _c["head_init"] != "random":
        # tied head: stays on the G-orbit partition; zero head: on the coarsest equitable partition (1-WL, Theorem A)
        _cells = orbits(_G) if _c["head_init"] == "tied" else equitable_coarsest(_B)
        _lo, _up, _eq = tie_ceiling_bounds(_B, _G, RHOS["uniform"], _cells)
        _p.update(label_ceiling=_lo, label_ceiling_upper=_up, ceiling_exact=_eq, tie_cells=np.asarray(_cells).tolist())
    _p["zero_head_stationary"] = head_dead(_B) if _c["head_init"] == "zero" else None
    if _c["arch"] == "shared" and not _c["support"]:
        if _c["task"] not in TRAP_INFO:
            TRAP_INFO[_c["task"]] = frustrated_pairs(_B)
        _p["frustration"] = TRAP_INFO[_c["task"]]
        _p["trap_symmetries"] = TRAP_INFO[_c["task"]]["nondiagonal"]   # v2 column: non-diagonal componentwise pairs
        if _c["exp"] == "E1" and _c["method"] == "nesy":
            if _c["task"] not in IDEAL_TRAPS:
                IDEAL_TRAPS[_c["task"]] = ideal_trap_census(_B, **_census)
            _p["ideal_traps"] = IDEAL_TRAPS[_c["task"]]
    if _c["support"]:
        # observational equivalence set of the support (all worlds that give identical training data) and the
        # identification capacity Lambda(E): no learner beats it in the worst case; G_S is the loss symmetry
        _E = equivalence_set(_B, SUPPORTS[_c["support"]])
        _p.update(equiv_size=int(len(_E)), minimax_acc=minimax_acc(_E, _rho), burnside_support=_p["acc_mean"])
    if _c["exp"] == "E7":
        _p["component_orders"] = {t: int(len(automorphisms(TASKS[t]))) for t in (("summod10", "absdiff") if _c["task"] == "mt_sum10_absdiff"
                                  else ("summod10",) if _c["task"] == "mt_sum10_diff10" else ("parity", "petersen"))}
    PREDICTIONS[_c["key"]] = dict(condition={k: v for k, v in _c.items() if k != "seeds"}, **_p)
_rows = [(c["key"], PREDICTIONS[c["key"]]["order"], round(PREDICTIONS[c["key"]]["acc_mean"], 3), round(PREDICTIONS[c["key"]]["acc_sd"], 3),
          len(c["seeds"]), c.get("seed_block", "seed0")) for c in CONDITIONS]
N_UNITS = sum(math.ceil(len(c["seeds"]) / CFG["bank"]) for c in CONDITIONS)
_nh = [c for c in CONDITIONS if c.get("seed_block") == "hash"]
print(f"{len(CONDITIONS)} conditions, {sum(r[4] for r in _rows)} seeded runs, {N_UNITS} units (banks of up to {CFG['bank']}); "
      f"{len(CONDITIONS) - len(_nh)} conditions on seed0 blocks (v2), {len(_nh)} on their own hashed seed blocks "
      f"({sum(len(c['seeds']) for c in _nh)} runs)")
for _t, _f in TRAP_INFO.items():
    _ic = IDEAL_TRAPS.get(_t)
    print(f"  {_t:20s} Theorem-B frustrated pairs {_f['count']} (strict traps {_f['strict']}), levels {_f['levels']}"
          + (f"; idealised census: {_ic['frac_trapped']:.2f} of {_ic['inits']} inits trapped, levels {_ic['levels']}" if _ic else ""))
pd.DataFrame(_rows, columns=["condition", "|G|", "predicted acc", "predicted sd", "seeds", "seed block"]) if pd is not None else _rows

In [ ]:
PREREG = dict(
    study="What the knowledge can define: symmetry laws for concept grounding in NeSy learning (v2.1)",
    cfg=CFG, dataset=DATASET,
    definitions={
        "level": "digit level for twin-free programs; twin-class level (concepts compared through their twin classes) "
                 "for programs with twins, the only level the labels can identify.",
        "label_optimum": "program optimum = the highest label accuracy reached by any random-head, shared-network seed "
                         "of the program in this run; a seed is at the label optimum if its label accuracy >= optimum - 0.02.",
        "closure": "a seed is inside the (twin-class) closure of G if its argmax concept map, read through the twin "
                   "classes, equals cls o pi for some pi in G.",
        "included": "at the label optimum AND inside the closure of the condition's group (a G-invariant event).",
        "seed_class": "trap classification by loss, per seed of a random-head shared-network condition: 'optimum' if at "
                      "the label optimum; 'trapped' if not, and its excess loss (test label NLL minus the median NLL of "
                      "the program's optimal seeds) is within 10% of a Theorem B level (m/n)^2 log 2 of the program "
                      "(frustrated_pairs, computed before training) and positive; otherwise 'unclassified' (reported, "
                      "never dropped).",
        "haar_test": "exact randomisation test of the mean (level) concept accuracy of the included seeds: each "
                     "included seed independently draws one value uniformly from the accuracies of its own confusion "
                     "relabelled by every element of the condition's group; 20,000 Monte Carlo means give a two-sided "
                     "p-value and the central 99% interval. Vacuous when the group is trivial; not testable with fewer "
                     "than 3 included seeds.",
        "seeds": "conditions of the v2 pilot keep seeds seed0, seed0 + 1, ...; v2.1 conditions (plain-NeSy arm, E6, E7) "
                 "use their own hashed seed blocks, so no two of them share initialisations.",
    },
    instrument_checks={   # reported, never counted as evidence: they can only fail through bugs or numerics
        "I1_pooled_PIT": "pooled randomised PIT of every seed within its exact relabelling distribution; |G| > 1, "
                         "excluding E5 and seeds with tie rate > 0.5 (tie-decided PITs). KS p >= 0.01.",
        "I2_pooled_z": "pooled z of (acc - Haar mean) on the same seeds; |z| <= 2.58.",
        "I3_E5_spread": "E5: median orbit spread < 0.01. This checks that the GPU arithmetic preserves exact ties.",
        "I4_haar_consistency": "every condition with |G| > 1 (except E5): Haar test of its included seeds inside the "
                               "central 99% interval. Expected count outside if all code is right: 0.01 per condition.",
        "I5_explanation_floor": "uniform rho_data, twin-free programs, shared network, >= 2 included seeds: the "
                                "orbit-averaged read-out's log-loss is within 0.5 bits of the floor sum_g P(g) log2|G.g| "
                                "(the floor for equivariant read-outs), and the naive read-out is above the floor.",
        "I6_tied_heads": "E5: label accuracy <= U_tie + 0.02 (corrected sandwich: orbit-mode <= tied-head ceiling <= "
                         "U_tie; tied heads on the G-orbits, zero heads on the coarsest equitable partition); a zero head "
                         "on a program whose one-cell partition is equitable (head_dead) stays within 0.02 of chance.",
    },
    primary_family={   # all on plain NeSy (no re-grounding); Holm-Bonferroni over every p-value below at FWER 0.05
        "P1_achievability": "E1 nesy, pooled over programs WITHOUT Theorem-B frustrated pairs: fraction of included "
                            "seeds (twin-class level). H0 (theory): rate >= 0.8; one-sided exact binomial; the theory "
                            "FAILS if H0 is rejected.",
        "P2_trap_contrast": "E1 nesy: seeds NOT at the label optimum, programs with >= 1 Theorem-B pair vs programs "
                            "without; one-sided Fisher exact test (more with pairs). The theory PASSES if the contrast "
                            "is significant. (Counted on seeds off the label optimum, not on 'trapped', because "
                            "'trapped' is defined by the Theorem-B levels and would be 0 by construction without pairs.)",
        "P3_trap_level": "E1 nesy, programs with Theorem-B pairs, seeds off the label optimum (selected by label "
                         "accuracy, not by loss, so the test is not circular): excess loss / nearest Theorem-B level, "
                         "TOST of the mean against [0.9, 1.1] at 5%. PASSES if equivalence is shown; fewer than 3 such "
                         "seeds: 'not testable' (not a pass).",
        "P4_anchors": "E3 nesy: Haar test under the pointwise stabiliser of the anchors, twin-class level; FAILS if "
                      "rejected. Vacuous (not counted) when the stabiliser is trivial.",
        "P5_multitask": "E7 nesy: Haar test under the intersection group; FAILS if rejected; vacuous if trivial.",
        "P6_prior": "E4 nesy (skewed data prior, no prior matching): post-hoc frequency gauge-fixing from m unlabeled "
                    "images; pooled error count vs the Bhattacharyya bound + 0.05, one-sided exact binomial per m, "
                    "Bonferroni over m; FAILS if rejected.",
        "P7_support": "E6 (restricted support): mean concept accuracy vs the Burnside level of the support group G_S "
                      "(an exact upper bound for an equivariant learner); one-sided t-test, FAILS if the mean is "
                      "significantly above it. The identification capacity Lambda(E) of the observational "
                      "equivalence set is reported next to it.",
        "expected_failures": "If the theory is exactly true, P(any FAIL among P1, P4-P7) <= 0.05 (Holm). P2 and P3 "
                             "are confirmatory: their pass rate is a power, not computable from the theory.",
    },
    secondary_method_arm="Re-grounding (gfrg) conditions: achievability per E1 condition, anchors (E3), multi-task "
                         "(E7), prior matching (E4 acc >= 0.9). Reported separately; not evidence for the theory.",
    predictions=PREDICTIONS)
_txt = json.dumps(PREREG, sort_keys=True, indent=1, default=float)
PREREG_SHA256 = hashlib.sha256(_txt.encode()).hexdigest()
_path = os.path.join(RUN_DIR, f"predictions_{SCALE}.json")
if os.path.exists(_path):
    with open(_path) as _fh:
        _old = _fh.read()
    if hashlib.sha256(_old.encode()).hexdigest() != PREREG_SHA256:
        print("WARNING: the recomputed predictions differ from the pre-registered file; the original file is kept and "
              "the new one is saved as predictions_recomputed.json")
        print("(expected once when a v2 run folder is continued with v2.1: v2.1 adds conditions and tests; the v2.1 "
              "file is the *_recomputed.json one)")
        atomic_write(os.path.join(RUN_DIR, f"predictions_{SCALE}_recomputed.json"), _txt)
        STORE.push(os.path.join(RUN_DIR, f"predictions_{SCALE}_recomputed.json"))
    PREREG_SHA256 = hashlib.sha256(_old.encode()).hexdigest()
else:
    atomic_write(_path, _txt)
    STORE.push(_path)
    atomic_write(_path + ".sha256", f"{PREREG_SHA256}  {os.path.basename(_path)}  {time.strftime('%Y-%m-%d %H:%M:%S %Z')}\n")
    STORE.push(_path + ".sha256")
print("pre-registration SHA-256:", PREREG_SHA256)
print("uploads pending before training:", STORE.flush() or "none failed")

## 3. Training engine

`SeedBank` trains S seeds at once (grouped convolutions, batched heads); a bank equals S standalone runs exactly at
initialisation and never mixes parameters. All seeds of a bank see the same data stream (the theory only needs the data
to be independent of the network). The evaluation records, per seed, the concept confusion with random argmax
tie-breaking, label accuracy on fresh test pairs, the orbit spread of the outputs, and the log-loss (bits) of the true
concept pair under the naive read-out and under the orbit-averaged posterior.

In [ ]:
class RefNet(nn.Module):
    def __init__(self, cnn, in_dim):
        super().__init__()
        self.cnn = cnn
        if cnn:
            self.c1, self.c2 = nn.Conv2d(1, 32, 3), nn.Conv2d(32, 64, 3)
            self.fc1 = nn.Linear(64 * 5 * 5, 128)
        else:
            self.fc1 = nn.Linear(in_dim, 128)
        self.fc2 = nn.Linear(128, N_CLASSES)

class SeedBank(nn.Module):
    def __init__(self, seeds, cnn, in_dim, head_init="random", tie=None):
        super().__init__()
        self.S, self.cnn = len(seeds), cnn
        refs = []
        for s in seeds:
            torch.manual_seed(int(s))
            refs.append(RefNet(cnn, in_dim))
        S = self.S
        if cnn:
            self.c1 = nn.Conv2d(S, S * 32, 3, groups=S)
            self.c2 = nn.Conv2d(S * 32, S * 64, 3, groups=S)
            with torch.no_grad():
                self.c1.weight.copy_(torch.cat([r.c1.weight for r in refs])); self.c1.bias.copy_(torch.cat([r.c1.bias for r in refs]))
                self.c2.weight.copy_(torch.cat([r.c2.weight for r in refs])); self.c2.bias.copy_(torch.cat([r.c2.bias for r in refs]))
        self.W1 = nn.Parameter(torch.stack([r.fc1.weight.detach() for r in refs]))
        self.b1 = nn.Parameter(torch.stack([r.fc1.bias.detach() for r in refs]))
        self.W2 = nn.Parameter(torch.stack([r.fc2.weight.detach() for r in refs]))   # (S, n, 128): the concept head
        self.b2 = nn.Parameter(torch.stack([r.fc2.bias.detach() for r in refs]))
        if head_init == "zero":      # G-fixed and degenerate: every output unit starts identical
            with torch.no_grad():
                self.W2.zero_(); self.b2.zero_()
        elif head_init == "tied":    # G-fixed: output units in one G-orbit share their (random) weights
            rep = torch.as_tensor([int(np.flatnonzero(tie == tie[e])[0]) for e in range(N_CLASSES)])
            with torch.no_grad():
                self.W2.copy_(self.W2[:, rep]); self.b2.copy_(self.b2[:, rep])

    def forward(self, x):                       # x: (B, 1, H, W) -> logits (S, B, n)
        B = x.shape[0]
        if self.cnn:
            h = x.expand(B, self.S, x.shape[2], x.shape[3])
            h = F.max_pool2d(F.relu(self.c1(h)), 2)
            h = F.max_pool2d(F.relu(self.c2(h)), 2)
            h = h.reshape(B, self.S, -1).transpose(0, 1)
        else:
            h = x.reshape(1, B, -1).expand(self.S, B, -1)
        h = F.relu(torch.baddbmm(self.b1[:, None, :], h, self.W1.transpose(1, 2)))
        return torch.baddbmm(self.b2[:, None, :], h, self.W2.transpose(1, 2))

def make_bank(seeds, head_init="random", tie=None):
    return SeedBank(seeds, CFG["net"] == "cnn", int(np.prod(XTR.shape[1:])), head_init, tie).to(DEVICE)

PERMS_CACHE = {}
def all_perms_t(n, device):
    if (n, str(device)) not in PERMS_CACHE:
        PERMS_CACHE[(n, str(device))] = torch.as_tensor(np.array(list(itertools.permutations(range(n))), dtype=np.int64), device=device)
    return PERMS_CACHE[(n, str(device))]

def batched_kmeans(P, k, gen, iters=30):
    # P: (S, N, n) concept distributions; k-means++ init (distance based, so it ignores how output units are labelled)
    S, N, _ = P.shape
    C = P[torch.arange(S), torch.randint(N, (S,), generator=gen).to(P.device)][:, None, :]
    for _ in range(1, k):
        d2 = torch.cdist(P, C).min(-1).values ** 2 + 1e-12
        nxt = torch.multinomial((d2 / d2.sum(1, keepdim=True)).cpu(), 1, generator=gen).to(P.device)[:, 0]
        C = torch.cat([C, P[torch.arange(S), nxt][:, None, :]], 1)
    for _ in range(iters):
        z = torch.cdist(P, C).argmin(-1)                           # (S, N)
        oh = F.one_hot(z, k).float()
        C = torch.where(oh.sum(1)[..., None] > 0, (oh.transpose(1, 2) @ P) / oh.sum(1)[..., None].clamp_min(1), C)
    return z, C

def reground(bank, opt, X, B_t, i1, i2, y, seed_ids, salt, prior=None, anchors=None, margin=0.01):
    # Exact discrete re-grounding (escapes the frustrated log-2 traps). For each flagged seed: cluster its concept
    # distributions, find every cluster->symbol permutation that best explains the labels (a union of cosets of the
    # symmetry group), and among those prefer (1) agreement with the anchors, (2) the prior (if one is given), and
    # (3) the smallest change of the current predictions, so the step is equivariant and the gauge is still chosen by
    # the training dynamics, not by this function. Applied only where it helps. Every seed draws its randomness from
    # its own generator (seed id, salt), so seeds never influence each other and a relabelled copy of a seed makes the
    # relabelled choices up to floating-point effects (checked by the equivariance self-test cell).
    n = N_CLASSES
    ny = int(B_t.max()) + 1
    P = torch.cat([bank_probs(bank, X[i1]), bank_probs(bank, X[i2])], 1)                         # (S, 2m, n)
    PA = None if anchors is None else bank_probs(bank, X[anchors[0]])                               # (S, a, n)
    S, m2 = P.shape[0], P.shape[1] // 2
    c = P.argmax(-1)
    cons_cur = (B_t[c[:, :m2], c[:, m2:]] == y[None]).float().mean(1)
    hist = F.one_hot(c, n).float().mean(1)
    tgt = torch.full((n,), 1.0 / n, device=P.device) if prior is None else prior
    logt = torch.log(tgt)
    kl_cur = (tgt * (logt - torch.log(hist + 1e-3))).sum(1)
    flagged = torch.nonzero((cons_cur < 0.97) | (kl_cur > 0.05)).flatten().tolist()
    if not flagged:
        return 0
    Pm = all_perms_t(n, P.device)
    ar = torch.arange(n, device=P.device)
    targets = {}
    for s in flagged:
        gs = torch.Generator().manual_seed((int(salt) * 1_000_003 + int(seed_ids[s])) % (2 ** 62))
        z, cent = batched_kmeans(P[s:s + 1], n, gs)
        z, cent = z[0], cent[0]
        Nc = torch.zeros(n * n * ny, device=P.device)
        Nc.index_add_(0, (z[:m2] * n + z[m2:]) * ny + y, torch.ones(m2, device=P.device))
        best, cand = -1.0, None
        for a in range(0, len(Pm), 400_000):
            Q = Pm[a:a + 400_000]
            lab = B_t[Q[:, :, None], Q[:, None, :]]                                        # beta(sigma z1, sigma z2)
            sc = Nc[((ar[:, None] * n + ar[None, :]) * ny)[None] + lab].sum((1, 2))
            mx = float(sc.max())
            if mx > best + 0.5:
                best, cand = mx, Q[sc == sc.max()]
            elif abs(mx - best) <= 0.5:
                cand = torch.cat([cand, Q[sc == sc.max()]])
        cnt = torch.bincount(z, minlength=n).float()
        Cz = torch.zeros(n, n, device=P.device)
        Cz.index_put_((z, c[s]), torch.ones(2 * m2, device=P.device), accumulate=True)   # cluster -> current symbol
        sel = cand
        if anchors is not None:                                                            # (1) anchors
            za = torch.cdist(PA[s][None], cent[None])[0].argmin(-1)                      # anchor -> nearest cluster
            a_sc = (sel[:, za] == anchors[1][None]).float().sum(1)
            sel = sel[a_sc == a_sc.max()]
        if prior is not None:                                                              # (2) prior (ties if uniform)
            p_sc = (cnt[None, :] * logt[sel]).sum(1)
            sel = sel[p_sc >= p_sc.max() - 1e-6 * p_sc.max().abs()]
        ag = Cz[ar[None, :], sel].sum(1)                                                   # (3) least change
        top = sel[ag == ag.max()]
        sigma = top[torch.randint(len(top), (1,), generator=gs).item()]
        agree = float(Cz[ar, sigma].sum()) / (2 * m2)
        prior_gain = float((cnt * logt[sigma]).sum() - (hist[s] * 2 * m2 * logt).sum()) / (2 * m2)
        if best / m2 > float(cons_cur[s]) + margin or (agree < 0.97 and prior_gain > 0.01):
            targets[s] = sigma[z]
    if targets:
        distil_seeds(bank, opt, X, torch.cat([i1, i2]), targets, salt)
    return len(targets)

def distil_seeds(bank, opt, X, idx, targets, salt, steps=150, bs=256):
    # pseudo-label targets sigma(z) are distilled into the flagged seeds by a short supervised phase; every other
    # seed of the bank (and its Adam state) is restored afterwards, so the step never touches them
    params = list(bank.parameters())
    saved = [p.detach().clone() for p in params]
    saved_st = [{k: (v.clone() if torch.is_tensor(v) else v) for k, v in opt.state.get(p, {}).items()} for p in params]
    ss = torch.tensor(sorted(targets), device=idx.device)
    T = torch.stack([targets[s] for s in sorted(targets)])                             # (F, 2m)
    gen = torch.Generator().manual_seed(int(salt) % (2 ** 62) + 7)
    for _ in range(steps):
        b = torch.randint(len(idx), (bs,), generator=gen).to(idx.device)
        logits = bank(X[idx[b]])[ss]                                                    # (F, bs, n)
        loss = F.cross_entropy(logits.float().reshape(-1, N_CLASSES), T[:, b].reshape(-1), reduction="sum") / bs
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    keep = torch.ones(bank.S, dtype=torch.bool, device=idx.device)
    keep[ss] = False
    with torch.no_grad():
        for p, p0, st0 in zip(params, saved, saved_st):
            p.view(bank.S, -1)[keep] = p0.view(bank.S, -1)[keep]
            st = opt.state.get(p, {})
            for k, v0 in st0.items():
                if torch.is_tensor(v0) and v0.shape == p.shape:
                    st[k].view(bank.S, -1)[keep] = v0.view(bank.S, -1)[keep]
                else:
                    st[k] = v0                                                         # Adam step counter

def train_bank(cond, seeds, ckpt=None, store=None, epochs=None, init_hook=None):
    # cond: a condition of the plan (task, method nesy|gf|gfrg, arch shared|separate, rho_data, prior_target, anchors,
    #       head_init random|tied|zero, support); see the PLAN cell
    dev = DEVICE
    B_np = TASKS[cond["task"]]
    B_t = torch.as_tensor(B_np, device=dev)
    ny = int(B_np.max()) + 1
    mask = F.one_hot(B_t.reshape(-1), ny).float()                     # (n*n, ny)
    hi = cond.get("head_init", "random")
    tie = np.asarray(orbits(GROUPS[cond["key"]])) if hi == "tied" else None
    banks = [make_bank(seeds, hi, tie)]
    if cond.get("arch", "shared") == "separate":
        banks.append(make_bank([s + 1_000_003 for s in seeds], hi, tie))
    if init_hook is not None:
        init_hook(banks)
    params = [p for b in banks for p in b.parameters()]
    opt = torch.optim.Adam(params, lr=CFG["lr"])
    rho_d = RHOS[cond.get("rho_data", "uniform")]
    tgt_t = None
    if cond.get("prior_target") is not None and cond["method"] in ("gf", "gfrg"):
        tgt_t = torch.as_tensor(RHOS[cond["prior_target"]], dtype=torch.float32, device=dev)
    support = SUPPORTS[cond["support"]] if cond.get("support") else None
    gen = torch.Generator().manual_seed(int(hashlib.sha1(json.dumps([cond["key"], list(map(int, seeds))]).encode()).hexdigest()[:8], 16))
    start, n_rg = 0, 0
    if ckpt is not None and os.path.exists(ckpt):
        st = torch.load(ckpt, map_location=dev, weights_only=True)
        for b, sd in zip(banks, st["banks"]):
            b.load_state_dict(sd)
        opt.load_state_dict(st["opt"])
        gen.set_state(st["gen"].cpu())
        start, n_rg = int(st["epoch"]), int(st["n_rg"])
    anchor_idx = anchor_tgt = None
    if cond.get("anchors"):
        ag = torch.Generator().manual_seed(12345)
        anchor_idx = torch.cat([torch.as_tensor(PTR["pools"][v])[torch.randperm(len(PTR["pools"][v]), generator=ag)[:CFG["anchor_m"]]] for v in cond["anchors"]]).to(dev)
        anchor_tgt = torch.as_tensor(np.repeat(cond["anchors"], CFG["anchor_m"]), device=dev)
    amp = torch.autocast(device_type="cuda", dtype=torch.bfloat16) if (CFG["bf16"] and dev.type == "cuda") else contextlib.nullcontext()
    n_pairs = CFG["n_train_pairs"]
    for epoch in range(start, CFG["epochs"] if epochs is None else epochs):
        d1, d2 = sample_concepts(rho_d, n_pairs, gen, support)
        i1, i2 = sample_images(d1, PTR, gen), sample_images(d2, PTR, gen)
        yv = torch.as_tensor(B_np[d1, d2], device=dev)
        i1, i2 = i1.to(dev), i2.to(dev)
        if cond["method"] == "gfrg" and epoch in CFG["reground_epochs"] and len(banks) == 1:
            m = CFG["reground_pairs"]
            salt = int(hashlib.sha1(f"{cond['key']}|{epoch}".encode()).hexdigest()[:12], 16)
            n_rg += reground(banks[0], opt, XTR, B_t, i1[:m], i2[:m], yv[:m], seeds, salt, prior=tgt_t,
                             anchors=None if anchor_idx is None else (anchor_idx, anchor_tgt))
        order = torch.randperm(len(yv), generator=gen).to(dev)
        for s0 in range(0, len(yv), CFG["batch"]):
            b = order[s0:s0 + CFG["batch"]]
            with amp:
                if len(banks) == 1:
                    lg = banks[0](torch.cat([XTR[i1[b]], XTR[i2[b]]]))
                    lg1, lg2 = lg[:, :len(b)], lg[:, len(b):]
                else:
                    lg1, lg2 = banks[0](XTR[i1[b]]), banks[1](XTR[i2[b]])
            p1, p2 = F.softmax(lg1.float(), -1), F.softmax(lg2.float(), -1)
            joint = (p1[..., :, None] * p2[..., None, :]).reshape(p1.shape[0], len(b), -1)   # (S, B, n*n)
            py = (joint @ mask).gather(2, yv[b][None, :, None].expand(p1.shape[0], -1, 1))[..., 0]
            loss = -torch.log(py.clamp_min(1e-12)).mean(1).sum()
            if tgt_t is not None:
                for pa in ((p1, p2) if len(banks) == 2 else (torch.cat([p1, p2], 1),)):
                    pbar = pa.mean(1)
                    loss = loss + CFG["lam_prior"] * (tgt_t * (torch.log(tgt_t) - torch.log(pbar.clamp_min(1e-12)))).sum()
            if anchor_idx is not None:
                ab = anchor_idx[torch.randint(len(anchor_idx), (min(64, len(anchor_idx)),), generator=gen).to(dev)]
                at = anchor_tgt[(ab[:, None] == anchor_idx[None, :]).float().argmax(1)]
                for bk in banks:
                    with amp:
                        la = bk(XTR[ab]).float()
                    loss = loss + CFG["lam_anchor"] * F.cross_entropy(la.reshape(-1, N_CLASSES), at.repeat(bk.S), reduction="sum") / len(ab)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        if ckpt is not None:
            torch.save({"banks": [b.state_dict() for b in banks], "opt": opt.state_dict(), "gen": gen.get_state(),
                        "epoch": epoch + 1, "n_rg": n_rg}, ckpt + ".tmp")
            os.replace(ckpt + ".tmp", ckpt)
            if store is not None:
                store.push(ckpt)
    for b in banks:
        b.eval()
    return banks, n_rg

def sample_concepts(rho, n, gen, support=None):
    # i.i.d. concepts from rho, restricted to the training support if one is given (rejection sampling)
    out1, out2, have = [], [], 0
    pr = torch.as_tensor(rho, dtype=torch.float64)
    while have < n:
        a = torch.multinomial(pr, 2 * n, replacement=True, generator=gen).numpy()
        b = torch.multinomial(pr, 2 * n, replacement=True, generator=gen).numpy()
        if support is not None:
            keep = support[a, b]
            a, b = a[keep], b[keep]
        out1.append(a); out2.append(b); have += len(a)
    return np.concatenate(out1)[:n], np.concatenate(out2)[:n]

def sample_images(d, pool, gen):
    # one random image of each requested concept (pool from make_pool)
    d = torch.as_tensor(d)
    r = (torch.rand(len(d), generator=gen) * pool["sizes"][d]).long()
    return pool["cat"][pool["starts"][d] + r]

@torch.inference_mode()
def bank_probs(bank, X, bs=512):
    return torch.cat([F.softmax(bank(X[s:s + bs]).float(), -1) for s in range(0, len(X), bs)], 1)   # (S, N, n)

@torch.inference_mode()
def evaluate_bank(banks, cond, seeds, oid=None, pair_oid=None):
    # per seed: concept confusion on the test set (rows = true digit, row-normalised, random argmax tie-breaking),
    # tie rate, orbit spread (how far the outputs are from being constant on the orbits `oid`), label accuracy / NLL
    # on fresh test pairs, and the log-loss (bits) of the true concept pair under the naive read-out and under the
    # orbit-averaged posterior q(g) = P(g_hat in G.g) / |G.g| (pair orbits `pair_oid`).
    B_np = TASKS[cond["task"]]
    gen = torch.Generator().manual_seed(999)
    out = [dict(seed=int(s)) for s in seeds]
    probs = [bank_probs(b, XTE) for b in banks]
    for k, P in enumerate(probs):
        # uniform random tie-breaking: the noise lives in float64, far below any real float32 difference, so exact ties
        # (e.g. outputs constant on an orbit) are broken uniformly and nothing else changes
        noise = torch.rand(P.shape, generator=gen, dtype=torch.float64).to(P.device) * 1e-9
        pred = (P.double() + noise).argmax(-1).cpu().numpy()
        top2 = P.topk(2, -1).values
        ties = ((top2[..., 0] - top2[..., 1]) < 1e-6).float().mean(1).cpu().numpy()
        for j in range(len(seeds)):
            C = np.zeros((N_CLASSES, N_CLASSES))
            np.add.at(C, (YTE_NP, pred[j]), 1)
            out[j][f"C{k}"] = (C / C.sum(1, keepdims=True)).tolist()
            out[j][f"tie_rate{k}"] = float(ties[j])
            out[j][f"marg{k}"] = P[j].mean(0).tolist()
        if oid is not None and k == 0:
            spread = torch.zeros(P.shape[:2], device=P.device)
            for o in np.unique(oid):
                cols = torch.as_tensor(np.flatnonzero(oid == o), device=P.device)
                spread = torch.maximum(spread, P[..., cols].amax(-1) - P[..., cols].amin(-1))
            for j in range(len(seeds)):
                out[j]["orbit_spread"] = float(spread[j].mean())
    d1, d2 = sample_concepts(RHOS["uniform"], CFG["n_test_pairs"], gen)
    j1, j2 = sample_images(d1, PTE, gen).to(probs[0].device), sample_images(d2, PTE, gen).to(probs[0].device)
    P1, P2 = probs[0][:, j1], probs[-1][:, j2]
    ny = int(B_np.max()) + 1
    J = (P1[..., :, None] * P2[..., None, :]).reshape(P1.shape[0], P1.shape[1], -1)              # (S, pairs, n*n)
    M = F.one_hot(torch.as_tensor(B_np.reshape(-1), device=P1.device), ny).float()
    PY = J @ M                                                                                   # (S, pairs, ny)
    yt = torch.as_tensor(B_np[d1, d2], device=P1.device)
    acc = (PY.argmax(-1) == yt[None]).float().mean(1).cpu().numpy()
    nll = (-torch.log(PY.gather(2, yt[None, :, None].expand(PY.shape[0], -1, 1))[..., 0].clamp_min(1e-12))).mean(1).cpu().numpy()
    gt = torch.as_tensor(d1 * N_CLASSES + d2, device=P1.device)
    naive_bits = (-torch.log2(J.gather(2, gt[None, :, None].expand(J.shape[0], -1, 1))[..., 0].clamp_min(1e-12))).mean(1).cpu().numpy()
    if pair_oid is not None:
        po = torch.as_tensor(np.asarray(pair_oid).reshape(-1), device=P1.device)
        K = int(po.max()) + 1
        Mo = F.one_hot(po, K).float()
        size = Mo.sum(0)
        q = (J @ Mo).gather(2, po[gt][None, :, None].expand(J.shape[0], -1, 1))[..., 0] / size[po[gt]][None]
        orbit_bits = (-torch.log2(q.clamp_min(1e-12))).mean(1).cpu().numpy()
    for j in range(len(seeds)):
        out[j]["label_acc"], out[j]["label_nll"] = float(acc[j]), float(nll[j])
        out[j]["naive_bits"] = float(naive_bits[j])
        if pair_oid is not None:
            out[j]["orbit_bits"] = float(orbit_bits[j])
    return out

In [ ]:
def equivariance_selftest(task, method, epochs):
    G = automorphisms(TASKS[task])
    pi = G[~np.all(G == np.arange(N_CLASSES), axis=1)][0].astype(np.int64)
    inv = torch.as_tensor(np.argsort(pi))

    def mirror(banks):
        with torch.no_grad():
            for b in banks:
                for p in b.parameters():
                    p.view(b.S, -1)[1] = p.view(b.S, -1)[0]
                b.W2[1] = b.W2[0][inv.to(b.W2.device)]
                b.b2[1] = b.b2[0][inv.to(b.b2.device)]

    banks, nrg = train_bank(make_condition("SELFTEST", task, method), [7, 7], epochs=epochs, init_hook=mirror)
    P = bank_probs(banks[0], XTE[:2000])
    pit = torch.as_tensor(pi, device=P.device)
    return dict(task=task, method=method, epochs=epochs, re_groundings=nrg,
                max_prob_deviation=float((P[1][:, pit] - P[0]).abs().max()),
                argmax_agreement=float((pit[P[0].argmax(-1)] == P[1].argmax(-1)).float().mean()))

SELFTEST = [equivariance_selftest("summod10", "nesy", 2),
            equivariance_selftest("summod10", "gfrg", CFG["reground_epochs"][0] + 1)]
for _r in SELFTEST:
    print(("PASS " if _r["argmax_agreement"] >= 0.95 else "CHECK ") + str(_r))
gpu_free()
pd.DataFrame(SELFTEST) if pd is not None else SELFTEST

## 4. Run (resumable)

In [ ]:
os.makedirs(os.path.join(RUN_DIR, "units"), exist_ok=True)
unit_results = {}
_units = [(c, i, c["seeds"][i:i + CFG["bank"]]) for c in CONDITIONS for i in range(0, len(c["seeds"]), CFG["bank"])]
_t_run, _done_now = time.time(), 0
for _k, (_c, _i, _seeds) in enumerate(_units):
    _name = f"{_c['key']}_s{_seeds[0]}-{_seeds[-1]}"          # pilot units are reused by the full run
    _path = os.path.join(RUN_DIR, "units", _name + ".json")
    if os.path.exists(_path):
        with open(_path) as _fh:
            unit_results[_name] = json.load(_fh)
        continue
    _t0 = time.time()
    _ckpt = os.path.join(RUN_DIR, "ckpt", _name + ".pt")
    _banks, _nrg = train_bank(_c, _seeds, _ckpt, STORE)
    _G = GROUPS[_c["key"]]
    _ev = evaluate_bank(_banks, _c, _seeds, oid=np.array(orbits(_G)),
                        pair_oid=pair_orbit_ids(_G) if _c["arch"] == "shared" else None)
    _r = dict(key=_c["key"], unit=_name, seeds=_seeds, n_reground=int(_nrg), secs=time.time() - _t0, per_seed=_ev)
    atomic_write(_path, json.dumps(_r))
    STORE.push(_path)
    unit_results[_name] = _r
    if os.path.exists(_ckpt):
        os.remove(_ckpt)                                     # its final state is already in the bucket
    _done_now += 1
    _eta = (time.time() - _t_run) / _done_now * (len(_units) - _k - 1)
    _line = (f"[{_k + 1}/{len(_units)}] {_name}: {_r['secs']:.0f}s, label acc {np.mean([e['label_acc'] for e in _ev]):.3f}, "
             f"concept acc {np.mean([np.mean(np.diag(e['C0'])) for e in _ev]):.3f} (predicted {PREDICTIONS[_c['key']]['acc_mean']:.3f}), "
             f"re-groundings {_nrg}, ETA {_eta / 60:.0f} min, rss {rss_gb():.1f} GB")
    print(_line, flush=True)
    with open(os.path.join(RUN_DIR, "progress.log"), "a") as _fh:
        _fh.write(_line + "\n")
    STORE.push(os.path.join(RUN_DIR, "progress.log"))
    del _banks
    gpu_free()
    trim_ram()
print(f"{len(unit_results)}/{len(_units)} units available; upload errors: {STORE.flush() or 'none'}")

## 5. Analysis: instrument checks, primary family, method arm

* **Instrument checks (I1-I6)**, not evidence: I1/I2 pooled PIT (KS) and z of every seed within its own exact
  relabelling distribution (no E5, no tie-decided seeds); I3 E5 orbit spread (does the GPU keep exact ties?);
  I4 Haar consistency of each condition's included seeds (exact Monte Carlo null, central 99% interval);
  I5 explanation floors; I6 tied and zero heads under the ceiling sandwich and the 1-WL dead-head rule.
* **Primary family (P1-P7)**, plain NeSy, Holm-Bonferroni at family-wise 0.05: P1 achievability (programs without
  Theorem-B pairs), P2 trap contrast (Fisher), P3 exact trap level (TOST), P4 anchors, P5 multi-task, P6 prior,
  P7 support with $\Lambda(E)$. Seeds are classified by loss as optimum / trapped / unclassified (none dropped).
* **Secondary**: the re-grounding arm (achievability, anchors, multi-task, prior matching), not evidence for the theory.

In [ ]:
def ks_uniform(u):
    # one-sample Kolmogorov-Smirnov test against U(0,1), asymptotic p-value (Stephens' small-sample correction)
    u = np.sort(np.asarray(u, float))
    n = len(u)
    if n == 0:
        return float("nan"), float("nan")
    i = np.arange(1, n + 1)
    D = float(max(np.max(i / n - u), np.max(u - (i - 1) / n)))
    lam = (math.sqrt(n) + 0.12 + 0.11 / math.sqrt(n)) * D
    p = 2 * sum((-1) ** (k - 1) * math.exp(-2 * k * k * lam * lam) for k in range(1, 101))
    return D, float(min(1.0, max(0.0, p)))

def tost(x, target, margin):
    # (v2, kept for reference; no longer used for verdicts) 90% CI of the mean inside target +- margin
    x = np.asarray(x, float)
    if len(x) < 2:
        return False, float("nan")
    se = x.std(ddof=1) / math.sqrt(len(x))
    return bool(abs(x.mean() - target) + 1.645 * se < margin), float(se)

def binom_cdf(k, n, p):
    # P(X <= k), X ~ Bin(n, p), exact (log-space sum)
    if k < 0:
        return 0.0
    if k >= n or p <= 0:
        return 1.0
    if p >= 1:
        return 0.0
    lp = [math.lgamma(n + 1) - math.lgamma(i + 1) - math.lgamma(n - i + 1) + i * math.log(p) + (n - i) * math.log1p(-p)
          for i in range(k + 1)]
    mx = max(lp)
    return float(min(1.0, math.exp(mx) * sum(math.exp(v - mx) for v in lp)))

def fisher_greater(a, n1, b, n2):
    # one-sided Fisher exact test: is the rate a/n1 larger than b/n2? P(X >= a), X hypergeometric with the margins
    K, N = a + b, n1 + n2
    lc = lambda n, k: math.lgamma(n + 1) - math.lgamma(k + 1) - math.lgamma(n - k + 1)
    tot = lc(N, n1)
    return float(min(1.0, sum(math.exp(lc(K, x) + lc(N - K, n1 - x) - tot) for x in range(a, min(K, n1) + 1))))

def t_sf(t, df):
    # P(T >= t) for Student's t (scipy if present, else the normal approximation)
    try:
        from scipy.stats import t as _tdist
        return float(_tdist.sf(t, df))
    except ImportError:
        return float(0.5 * math.erfc(t / math.sqrt(2)))

def holm(pvals, alpha=0.05):
    # Holm-Bonferroni step-down: returns the set of rejected indices
    order = sorted(range(len(pvals)), key=lambda i: pvals[i])
    rej = set()
    for r, i in enumerate(order):
        if pvals[i] <= alpha / (len(pvals) - r):
            rej.add(i)
        else:
            break
    return rej

def explanation_soundness(C, pair_oid):
    # P(the predicted concept pair lies in the explanation set of the true pair), uniform test pairs:
    # naive = exact pair, orbit = same pair orbit
    K = np.kron(C, C)                                      # K[(d,e),(a,b)] = C[d,a] C[e,b]
    po = np.asarray(pair_oid).reshape(-1)
    return float(np.mean(np.diag(K))), float(np.mean((K * (po[:, None] == po[None, :])).sum(1)))

def class_expand(C, cls):
    # C_exp[d, f] = sum of C[d, e] over the twin class of f: accuracy at the twin-class level is rho . diag(C_exp)
    return C @ (cls[:, None] == cls[None, :]).astype(float)

def seed_metrics(c, r, rng):
    G, rho = GROUPS[c["key"]], RHOS[c["rho_data"]]
    C = np.array(r["C0"])
    acc = float(rho @ np.diag(C))
    hv = haar_acc_values(C, G, rho)                        # exact randomisation distribution under the orbit law
    pred, var = float(hv.mean()), float(hv.var())
    tie = np.abs(hv - acc) <= 1e-9
    u = float((np.sum((hv < acc) & ~tie) + rng.random() * np.sum(tie)) / len(hv))
    amap = C.argmax(1)
    G64 = G.astype(np.int64)
    in_G = bool(np.any(np.all(G64 == amap[None], axis=1)))
    cls = TWINS[c["task"]][0]
    in_closure = bool(np.any(np.all(cls[G64] == cls[amap][None], axis=1)))
    core_acc = float(sum(rho[d] * C[d, cls == cls[d]].sum() for d in range(N_CLASSES)))
    twin_free = bool(cls.max() == N_CLASSES - 1)
    m = dict(seed=r["seed"], acc=acc, pred=pred, var=var, u=u, in_G=in_G, in_closure=in_closure, label_acc=r["label_acc"],
             label_nll=r["label_nll"], tie_rate=r["tie_rate0"], alpha=amap.tolist(), core_acc=core_acc,
             lvl_acc=acc if twin_free else core_acc, included=in_G and r["label_acc"] >= 0.9,
             spread=r.get("orbit_spread", float("nan")), naive_bits=r["naive_bits"], orbit_bits=r.get("orbit_bits", float("nan")))
    if c["arch"] == "shared":
        m["naive_sound"], m["orbit_sound"] = explanation_soundness(C, PAIR_OIDS[c["key"]])
        m["core_sound"] = explanation_soundness(C, CORE_PAIR_OIDS[c["task"]])[1]
    return m

def freq_gauge_fixing(C, G, rho, ms, R, rng):
    # post-hoc symmetry breaking with m unlabeled images and the known concept prior: pick the element of G under
    # which the predicted symbols are most likely, P(error) = P(chosen element != the seed's own gauge)
    G64 = G.astype(np.int64)
    own = int(np.argmax((C[np.arange(N_CLASSES)[None, :], G64] * rho[None]).sum(1)))
    logr = np.log(rho)
    cdf = np.cumsum(C, 1)
    out = []
    for m in ms:
        err = 0
        for _ in range(R):
            d = rng.choice(N_CLASSES, size=m, p=rho)
            e = np.minimum((rng.random(m)[:, None] > cdf[d]).sum(1), N_CLASSES - 1)    # symbol ~ C[d, :]
            cnt = np.bincount(e, minlength=N_CLASSES)
            sc = cnt[G64] @ logr                                                      # sum_d n_{pi(d)} log rho(d)
            best = np.flatnonzero(sc >= sc.max() - 1e-9)
            err += int(rng.choice(best) != own)
        out.append(err / R)
    return np.array(out)

TWINS = {t: core_classes_group(TASKS[t]) for t in sorted({c["task"] for c in CONDITIONS})}
CORE_PAIR_OIDS = {}
for _t, (_cls, _H) in TWINS.items():
    _pc = pair_orbit_ids(_H, n=int(_cls.max()) + 1)
    CORE_PAIR_OIDS[_t] = _pc[np.ix_(_cls, _cls)]
PAIR_OIDS = {c["key"]: pair_orbit_ids(GROUPS[c["key"]]) for c in CONDITIONS if c["arch"] == "shared"}
COND = {c["key"]: c for c in CONDITIONS}

_rng = np.random.default_rng(2024)
SEED_TABLE, CONFUSIONS = {}, {}                    # condition key -> per-seed metric dicts / concept confusions
for _c in CONDITIONS:
    _per = [e for u in unit_results.values() if u["key"] == _c["key"] for e in u["per_seed"]]
    if _per:
        SEED_TABLE[_c["key"]] = [seed_metrics(_c, e, _rng) for e in _per]
        CONFUSIONS[_c["key"]] = [np.array(e["C0"]) for e in _per]

# ---- label optimum and trap classification (per program, from this run's seeds) ----
def plain_head(c):
    return c["arch"] == "shared" and c["head_init"] == "random" and not c["support"]

PROGRAM_OPT, PROGRAM_NLL = {}, {}
for _t in TWINS:
    _la = [m["label_acc"] for k, ms in SEED_TABLE.items() if COND[k]["task"] == _t and plain_head(COND[k]) for m in ms]
    if _la:
        PROGRAM_OPT[_t] = float(max(_la))
        _nl = [m["label_nll"] for k, ms in SEED_TABLE.items() if COND[k]["task"] == _t and plain_head(COND[k])
               for m in ms if m["label_acc"] >= PROGRAM_OPT[_t] - 0.02]
        PROGRAM_NLL[_t] = float(np.median(_nl))
for _k, _ms in SEED_TABLE.items():
    _c = COND[_k]
    _levels = TRAP_INFO.get(_c["task"], {}).get("levels", [])
    for _m in _ms:
        _opt = PROGRAM_OPT.get(_c["task"])
        _m["at_opt"] = bool(_opt is not None and _m["label_acc"] >= _opt - 0.02)
        _m["incl"] = _m["at_opt"] and _m["in_closure"]                 # inclusion for the v2.1 tests (closure level)
        _m["excess"] = _m["label_nll"] - PROGRAM_NLL[_c["task"]] if _c["task"] in PROGRAM_NLL else float("nan")
        _m["trap_ratio"] = (_m["excess"] / min(_levels, key=lambda L: abs(_m["excess"] - L))) if _levels else float("nan")
        if not plain_head(_c):
            _m["seed_class"] = "n/a"
        elif _m["at_opt"]:
            _m["seed_class"] = "optimum"
        elif _levels and _m["excess"] > 0 and abs(_m["trap_ratio"] - 1) <= 0.10:
            _m["seed_class"] = "trapped"
        else:
            _m["seed_class"] = "unclassified"

def haar_test(key, draws=20_000):
    # exact randomisation test of the mean level accuracy of the included seeds under the condition's group
    c, G, rho = COND[key], GROUPS[key], RHOS[COND[key]["rho_data"]]
    ms = SEED_TABLE.get(key, [])
    idx = [i for i, m in enumerate(ms) if m["incl"]]
    out = dict(order=int(len(G)), n=len(ms), n_incl=len(idx), mean=float(np.mean([ms[i]["lvl_acc"] for i in idx])) if idx else float("nan"),
               lo=float("nan"), hi=float("nan"), p=float("nan"), status="ok")
    if len(G) == 1:
        out["status"] = "vacuous (trivial group)"
        return out
    if len(idx) < 3:
        out["status"] = "not testable (< 3 included)"
        return out
    cls = TWINS[c["task"]][0]
    twin_free = cls.max() == N_CLASSES - 1
    rng = np.random.default_rng(int(hashlib.sha1(key.encode()).hexdigest()[:8], 16))
    tot = np.zeros(draws)
    for i in idx:
        C = CONFUSIONS[key][i]
        hv = haar_acc_values(C if twin_free else class_expand(C, cls), G, rho)
        tot += hv[rng.integers(len(hv), size=draws)]
    null = tot / len(idx)
    obs = out["mean"]
    p_lo, p_hi = (np.sum(null <= obs + 1e-12) + 1) / (draws + 1), (np.sum(null >= obs - 1e-12) + 1) / (draws + 1)
    out.update(lo=float(np.quantile(null, 0.005)), hi=float(np.quantile(null, 0.995)), p=float(min(1.0, 2 * min(p_lo, p_hi))))
    return out

def summarise_condition(c, ms):
    P = PREDICTIONS[c["key"]]
    a = lambda k: np.array([m[k] for m in ms], float)
    inc = [m for m in ms if m["included"]]
    res = a("acc") - a("pred")
    sc = [m["seed_class"] for m in ms]
    row = dict(condition=c["key"], exp=c["exp"], method=c["method"], order=P["order"], seeds=len(ms),
               label_acc=a("label_acc").mean(), off_optimum=float(np.mean([not m["at_opt"] for m in ms])), in_G=float(np.mean(a("in_G"))),
               n_optimum=sc.count("optimum"), n_trapped=sc.count("trapped"), n_unclassified=sc.count("unclassified"),
               acc=a("acc").mean(), acc_se=a("acc").std(ddof=1) / math.sqrt(len(ms)) if len(ms) > 1 else float("nan"),
               burnside=P["acc_mean"], burnside_sd=P["acc_sd"],
               acc_included=float(np.mean([m["acc"] for m in inc])) if inc else float("nan"), n_included=len(inc),
               n_incl=int(sum(m["incl"] for m in ms)), lvl_acc=a("lvl_acc").mean(),
               refined_pred=a("pred").mean(), residual=res.mean(),
               residual_z=float(res.sum() / math.sqrt(max(a("var").sum(), 1e-12))) if P["order"] > 1 else float("nan"),
               ks_p=ks_uniform(a("u"))[1] if P["order"] > 1 else float("nan"),
               core_acc=a("core_acc").mean(), core_pred=P["core_acc_pred"], tie_rate=a("tie_rate").mean(), spread=float(np.nanmedian(a("spread"))),
               label_ceiling=P["label_ceiling"], label_ceiling_upper=P.get("label_ceiling_upper"), bits_bound=P["bits_bound_uniform"],
               naive_bits=float(a("naive_bits").mean()), orbit_bits=float(np.nanmean([m["orbit_bits"] for m in inc])) if inc else float("nan"),
               trap_symmetries=P.get("trap_symmetries"), frustrated_pairs=P.get("frustration", {}).get("count"),
               minimax_acc=P.get("minimax_acc"), equiv_size=P.get("equiv_size"))
    if c["arch"] == "shared":
        row.update(naive_sound=a("naive_sound").mean(), orbit_sound=a("orbit_sound").mean(), core_sound=a("core_sound").mean())
    return row

SUMMARY = [summarise_condition(c, SEED_TABLE[c["key"]]) for c in CONDITIONS if c["key"] in SEED_TABLE]
_by = {r["condition"]: r for r in SUMMARY}
HAAR = {k: haar_test(k) for k in SEED_TABLE if COND[k]["head_init"] == "random"}

# E4: post-hoc frequency gauge-fixing on the prior-free runs with a skewed concept prior
GAUGE_MS = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
GAUGE_FIX = {}
for _c in CONDITIONS:
    if _c["exp"] == "E4" and _c["method"] == "nesy" and _c["key"] in SEED_TABLE:
        _inc = [C for m, C in zip(SEED_TABLE[_c["key"]], CONFUSIONS[_c["key"]]) if m["incl"]]
        if _inc:
            _err = np.mean([freq_gauge_fixing(C, GROUPS[_c["key"]], RHOS[_c["rho_data"]], GAUGE_MS, CFG["gauge_R"], _rng) for C in _inc], 0)
            GAUGE_FIX[_c["key"]] = dict(ms=GAUGE_MS, error=_err.tolist(), trials=len(_inc) * CFG["gauge_R"],
                                        bound=bhattacharyya_bound(GROUPS[_c["key"]], RHOS[_c["rho_data"]], GAUGE_MS).tolist(), n=len(_inc))

# ---- INSTRUMENT CHECKS (not evidence for the theory: they can only fail through bugs or numerics) ----
INSTRUMENT = []                                    # (name, ok, detail)
_pool = [m for k, ms in SEED_TABLE.items() if PREDICTIONS[k]["order"] > 1 and COND[k]["exp"] != "E5"
         for m in ms if m["tie_rate"] <= 0.5]
POOLED = dict(n=len(_pool), ks_D=ks_uniform([m["u"] for m in _pool])[0], ks_p=ks_uniform([m["u"] for m in _pool])[1],
              z=float(sum(m["acc"] - m["pred"] for m in _pool) / math.sqrt(max(sum(m["var"] for m in _pool), 1e-12))) if _pool else float("nan"))
INSTRUMENT.append(("I1 pooled PIT (KS; no E5, no tie-decided seeds)", POOLED["ks_p"] >= 0.01, f"n={POOLED['n']}, D={POOLED['ks_D']:.3f}, p={POOLED['ks_p']:.3g}"))
INSTRUMENT.append(("I2 pooled z", abs(POOLED["z"]) <= 2.58, f"z={POOLED['z']:+.2f}"))
for r in SUMMARY:
    c, P = COND[r["condition"]], PREDICTIONS[r["condition"]]
    if c["exp"] == "E5":
        INSTRUMENT.append((f"I3 orbit spread {r['condition']}", r["spread"] < 0.01, f"median spread {r['spread']:.2g} (GPU arithmetic keeps exact ties?)"))
        INSTRUMENT.append((f"I6 tied-head sandwich {r['condition']}", r["label_acc"] <= P["label_ceiling_upper"] + 0.02,
                           f"label acc {r['label_acc']:.3f}; orbit-mode {P['label_ceiling']:.3f} <= ceiling <= U_tie {P['label_ceiling_upper']:.3f}"))
        if P["zero_head_stationary"]:
            INSTRUMENT.append((f"I6 dead head {r['condition']}", abs(r["label_acc"] - P["label_chance"]) <= 0.02,
                               f"label acc {r['label_acc']:.3f} vs chance {P['label_chance']:.3f}"))
    if c["arch"] == "shared" and c["rho_data"] == "uniform" and P["twin_free"] and r["order"] > 1 and r["n_included"] >= 2 \
            and c["head_init"] == "random":
        INSTRUMENT.append((f"I5 explanation floor {r['condition']}", r["orbit_bits"] <= r["bits_bound"] + 0.5 and r["naive_bits"] >= r["bits_bound"],
                           f"orbit {r['orbit_bits']:.2f} / naive {r['naive_bits']:.2f} / floor {r['bits_bound']:.2f} bits"))
_i4 = [(k, h) for k, h in HAAR.items() if h["status"] == "ok"]
for _k, _h in _i4:
    INSTRUMENT.append((f"I4 Haar consistency {_k}", _h["lo"] <= _h["mean"] <= _h["hi"],
                       f"{_h['mean']:.3f} in [{_h['lo']:.3f}, {_h['hi']:.3f}] (n={_h['n_incl']}, p={_h['p']:.3g})"))

# ---- PRIMARY FAMILY (plain NeSy), Holm at FWER 0.05 ----
# each entry: name, p-value (or None), direction ("fail_if_rejected" | "pass_if_rejected"), detail
_PRIM = []
_e1n = [k for k in SEED_TABLE if COND[k]["exp"] == "E1" and COND[k]["method"] == "nesy"]
_with = [k for k in _e1n if TRAP_INFO[COND[k]["task"]]["count"] > 0]
_without = [k for k in _e1n if TRAP_INFO[COND[k]["task"]]["count"] == 0]
_n1 = sum(len(SEED_TABLE[k]) for k in _without)
_k1 = sum(m["incl"] for k in _without for m in SEED_TABLE[k])
_PRIM.append(("P1 achievability (no Theorem-B pairs, twin-class level)", binom_cdf(_k1, _n1, 0.8) if _n1 else None, "fail_if_rejected",
              f"{_k1}/{_n1} included ({', '.join(COND[k]['task'] for k in _without)}); H0 rate >= 0.8"))
_a1 = sum(not m["at_opt"] for k in _with for m in SEED_TABLE[k])
_m1 = sum(len(SEED_TABLE[k]) for k in _with)
_b1 = sum(not m["at_opt"] for k in _without for m in SEED_TABLE[k])
_PRIM.append(("P2 trap contrast (Fisher, off-optimum)", fisher_greater(_a1, _m1, _b1, _n1) if _m1 and _n1 else None, "pass_if_rejected",
              f"with pairs {_a1}/{_m1} off optimum ({sum(m['seed_class'] == 'trapped' for k in _with for m in SEED_TABLE[k])} trapped); "
              f"without {_b1}/{_n1}"))
_r = np.array([m["trap_ratio"] for k in _with for m in SEED_TABLE[k] if not m["at_opt"]], float)
if len(_r) >= 3:
    _se = max(_r.std(ddof=1) / math.sqrt(len(_r)), 1e-12)
    _p3 = max(t_sf((_r.mean() - 0.9) / _se, len(_r) - 1), t_sf((1.1 - _r.mean()) / _se, len(_r) - 1))
    _PRIM.append(("P3 trap level (TOST, excess loss / Theorem-B level in [0.9, 1.1])", _p3, "pass_if_rejected",
                  f"n={len(_r)}, mean ratio {_r.mean():.3f} (sd {_r.std(ddof=1):.3f})"))
else:
    _PRIM.append(("P3 trap level (TOST)", None, "pass_if_rejected", f"not testable: {len(_r)} off-optimum seeds on programs with pairs"))
for _k in SEED_TABLE:
    _c = COND[_k]
    if _c["method"] == "nesy" and _c["exp"] in ("E3", "E7") and _k in HAAR:
        _h = HAAR[_k]
        _nm = ("P4 anchors " if _c["exp"] == "E3" else "P5 multi-task ") + _k
        _PRIM.append((_nm, _h["p"] if _h["status"] == "ok" else None, "fail_if_rejected",
                      f"|G|={_h['order']}, {_h['n_incl']}/{_h['n']} included, mean {_h['mean']:.3f}, 99% [{_h['lo']:.3f}, {_h['hi']:.3f}]"
                      + ("" if _h["status"] == "ok" else f" -- {_h['status']}")))
for _k, _g in GAUGE_FIX.items():
    _pm = [1 - binom_cdf(int(round(e * _g["trials"])) - 1, _g["trials"], min(1.0, b + 0.05)) for e, b in zip(_g["error"], _g["bound"])]
    _PRIM.append((f"P6 prior (post-hoc gauge fixing) {_k}", min(1.0, len(_pm) * min(_pm)), "fail_if_rejected",
                  "error " + ", ".join(f"m={m}:{e:.2f} (bound {b:.2f})" for m, e, b in zip(_g["ms"], _g["error"], _g["bound"]) if m in (10, 100, 1000))))
for _k in SEED_TABLE:
    if COND[_k]["exp"] == "E6":
        _x = np.array([m["acc"] for m in SEED_TABLE[_k]])
        _B0 = PREDICTIONS[_k]["burnside_support"]
        _se = _x.std(ddof=1) / math.sqrt(len(_x)) if len(_x) > 1 else 0.0
        _p7 = t_sf((_x.mean() - _B0) / _se, len(_x) - 1) if _se > 0 else (1.0 if _x.mean() <= _B0 + 1e-9 else 0.0)
        _PRIM.append((f"P7 support {_k}", _p7 if len(_x) > 1 else None, "fail_if_rejected",
                      f"acc {_x.mean():.3f} vs Burnside(G_S) {_B0:.3f}; Lambda(E) = {PREDICTIONS[_k]['minimax_acc']:.3f} (|E| = {PREDICTIONS[_k]['equiv_size']})"))
_testable = [i for i, t in enumerate(_PRIM) if t[1] is not None]
_rej = {_testable[j] for j in holm([_PRIM[i][1] for i in _testable])}
PRIMARY = []
for _i, (_n, _p, _dir, _d) in enumerate(_PRIM):
    if _p is None:
        _v = "VACUOUS" if "vacuous" in _d else "NOT TESTABLE"
    elif _dir == "fail_if_rejected":
        _v = "FAIL" if _i in _rej else "PASS"
    else:
        _v = "PASS" if _i in _rej else "NOT CONFIRMED"
    PRIMARY.append(dict(test=_n, p=_p, direction=_dir, verdict=_v, detail=_d))

# ---- SECONDARY: the re-grounding method arm (not evidence for the theory) ----
SECONDARY = []
for _k in SEED_TABLE:
    _c, _h = COND[_k], HAAR.get(_k)
    if _c["method"] != "gfrg":
        continue
    if _c["exp"] == "E1":
        _n, _kk = len(SEED_TABLE[_k]), sum(m["incl"] for m in SEED_TABLE[_k])
        SECONDARY.append((f"achievability {_k}", _kk / _n >= 0.8, f"{_kk}/{_n} included (binomial p vs 0.8: {binom_cdf(_kk, _n, 0.8):.3g})"))
    if _c["exp"] in ("E3", "E7") and _h is not None:
        SECONDARY.append((f"Haar {_k}", _h["status"] != "ok" or _h["lo"] <= _h["mean"] <= _h["hi"],
                          f"|G|={_h['order']}, {_h['n_incl']}/{_h['n']} included, mean {_h['mean']:.3f}, 99% [{_h['lo']:.3f}, {_h['hi']:.3f}] {_h['status']}"))
    if _c["exp"] == "E4" and TWINS[_c["task"]][0].max() == N_CLASSES - 1:
        SECONDARY.append((f"prior matching {_k}", _by[_k]["acc"] >= 0.9, f"acc {_by[_k]['acc']:.3f}"))

VERDICTS = [(v["test"], v["verdict"], v["detail"]) for v in PRIMARY]          # v2 name, now the primary family
atomic_write(os.path.join(RUN_DIR, "summary.json"), json.dumps(dict(
    summary=SUMMARY, pooled=POOLED, gauge_fix=GAUGE_FIX, haar=HAAR, instrument=INSTRUMENT, primary=PRIMARY, secondary=SECONDARY,
    program_optimum=PROGRAM_OPT, program_nll=PROGRAM_NLL, seeds=SEED_TABLE, prereg_sha256=PREREG_SHA256), default=float))
STORE.push(os.path.join(RUN_DIR, "summary.json"))

if CFG["scale"] == "smoke":
    print("(smoke scale: tiny networks on 8x8 digits for 3 epochs; these verdicts only show that the pipeline runs, not evidence either way)")
print(f"INSTRUMENT CHECKS (not evidence; fail only through bugs or numerics): {sum(v[1] for v in INSTRUMENT)}/{len(INSTRUMENT)} ok; "
      f"I4 expected outside the 99% interval if everything is right: {0.01 * len(_i4):.2f} of {len(_i4)}")
for _v in INSTRUMENT:
    print(("  ok    " if _v[1] else "  CHECK ") + _v[0] + ": " + _v[2])
_nf = sum(1 for v in PRIMARY if v["direction"] == "fail_if_rejected" and v["p"] is not None)
print(f"PRIMARY FAMILY (plain NeSy; Holm-Bonferroni over {len(_testable)} p-values at FWER 0.05). If the theory is exactly "
      f"true, P(any FAIL among the {_nf} consistency tests) <= 0.05; P2/P3 are confirmatory (their pass rate is a power).")
for _v in PRIMARY:
    print(f"  {_v['verdict']:13s} {_v['test']}: " + (f"p={_v['p']:.3g}; " if _v["p"] is not None else "") + _v["detail"])
print("SECONDARY: re-grounding method arm (not evidence for the theory)")
for _v in SECONDARY:
    print(("  ok    " if _v[1] else "  miss  ") + _v[0] + ": " + _v[2])
print("SEED CLASSES (plain heads): " + "; ".join(f"{r['condition']}: {r['n_optimum']} optimum / {r['n_trapped']} trapped / "
                                                f"{r['n_unclassified']} unclassified" for r in SUMMARY if plain_head(COND[r['condition']])))
_tab = [dict(condition=k, G=h["order"], included=f"{h['n_incl']}/{h['n']}", mean_acc=round(h["mean"], 3),
             haar99=f"[{h['lo']:.3f}, {h['hi']:.3f}]" if h["status"] == "ok" else "", p=round(h["p"], 4) if h["status"] == "ok" else None,
             verdict=("inside" if h["lo"] <= h["mean"] <= h["hi"] else "OUTSIDE") if h["status"] == "ok" else h["status"])
        for k, h in HAAR.items()]
pd.DataFrame(_tab) if pd is not None else _tab

In [ ]:
def make_figures():
    cond = {c["key"]: c for c in CONDITIONS}
    by = {r["condition"]: r for r in SUMMARY}
    colors = {"E1": "C0", "E2": "C1", "E3": "C2", "E4": "C3", "E5": "C4", "E6": "C5", "E7": "C6"}
    fig, ax = plt.subplots(3, 3, figsize=(16, 14))
    ax = ax.ravel()
    # (a) parameter-free prediction vs measurement, seeds at the label optimum with an argmax map in G
    for r in SUMMARY:
        if r["n_included"]:
            inc = [m["acc"] for m in SEED_TABLE[r["condition"]] if m["included"]]
            se = np.std(inc, ddof=1) / math.sqrt(len(inc)) if len(inc) > 1 else 0
            ax[0].errorbar(r["burnside"], r["acc_included"], yerr=1.96 * se, fmt="o", color=colors[cond[r["condition"]]["exp"]], ms=4)
    ax[0].plot([0, 1], [0, 1], "k--", lw=1)
    ax[0].set(xlabel="predicted  sum_d rho(d) / |G.d|", ylabel="measured concept accuracy", title="(a) orbit law: Burnside level")
    for e, col in colors.items():
        ax[0].plot([], [], "o", color=col, label=e)
    ax[0].legend(fontsize=7)
    # (b) quality-adjusted prediction, all seeds (no selection)
    for r in SUMMARY:
        ax[1].errorbar(r["refined_pred"], r["acc"], yerr=1.96 * (r["acc_se"] if r["acc_se"] == r["acc_se"] else 0), fmt="o",
                       color=colors[cond[r["condition"]]["exp"]], ms=4)
    ax[1].plot([0, 1], [0, 1], "k--", lw=1)
    ax[1].set(xlabel="predicted from the seed's own G-invariant statistics", ylabel="measured", title="(b) orbit law: every seed")
    # (c) pooled randomised PIT
    u = [m["u"] for k, ms in SEED_TABLE.items() if PREDICTIONS[k]["order"] > 1 and cond[k]["exp"] != "E5" for m in ms if m["tie_rate"] <= 0.5]
    ax[2].hist(u, bins=10, range=(0, 1), color="C0", alpha=0.7, density=True)
    ax[2].axhline(1, color="k", ls="--", lw=1)
    ax[2].set(xlabel="PIT of each seed within its exact Haar distribution", title=f"(c) instrument I1: pooled PIT, KS p = {POOLED['ks_p']:.3g} (n = {len(u)})")
    # (d) per-seed accuracy vs exact Haar distribution for the largest non-trivial E1 condition
    big = [k for k in SEED_TABLE if cond[k]["exp"] == "E1" and PREDICTIONS[k]["order"] > 2]
    if big:
        k = max(big, key=lambda k: (len(SEED_TABLE[k]), -PREDICTIONS[k]["order"]))
        G, rho = GROUPS[k], RHOS[cond[k]["rho_data"]]
        haar = (G == np.arange(N_CLASSES)).astype(float) @ rho
        ax[3].hist(haar, bins=21, range=(-0.025, 1.025), density=True, alpha=0.4, color="gray", label="exact Haar law (perfect perception)")
        ax[3].hist([m["acc"] for m in SEED_TABLE[k]], bins=21, range=(-0.025, 1.025), density=True, alpha=0.6, color="C0", label="trained seeds")
        ax[3].legend(fontsize=7)
        ax[3].set(xlabel="concept accuracy", title=f"(d) {k}: per-seed accuracy")
    # (e) anchors: same number of labels, different predicted outcome
    e3 = [r for r in SUMMARY if cond[r["condition"]]["exp"] == "E3"]
    if e3:
        x = np.arange(len(e3))
        ax[4].bar(x - 0.2, [r["burnside"] for r in e3], 0.4, label="predicted", color="lightgray")
        ax[4].bar(x + 0.2, [r["acc"] for r in e3], 0.4, yerr=[1.96 * r["acc_se"] for r in e3], label="measured", color="C2")
        ax[4].set_xticks(x, [r["condition"].replace("E3_", "").replace("_gfrg", "") for r in e3], rotation=60, fontsize=6)
        ax[4].legend(fontsize=7)
        ax[4].set(ylabel="concept accuracy", title="(e) anchors during training (E3)")
    # (f) G-fixed initialisation: label accuracy capped by the orbit-averaged ceiling
    e5 = [r for r in SUMMARY if cond[r["condition"]]["exp"] == "E5"]
    if e5:
        x = np.arange(len(e5))
        ax[5].bar(x - 0.27, [r["label_ceiling"] for r in e5], 0.27, color="lightgray", label="orbit-mode (lower)")
        ax[5].bar(x, [r["label_ceiling_upper"] for r in e5], 0.27, color="darkgray", label="U_tie (upper)")
        ax[5].bar(x + 0.27, [r["label_acc"] for r in e5], 0.27, color="C4", label="measured label acc")
        for i, r in enumerate(e5):
            ax[5].text(i, 0.05, f"spread\n{r['spread']:.1e}", ha="center", fontsize=6)
        ax[5].set_xticks(x, [cond[r["condition"]]["task"] + "\n" + cond[r["condition"]]["head_init"] for r in e5], fontsize=7)
        ax[5].legend(fontsize=7)
        ax[5].set(title="(f) tied / zero concept head (E5): ceiling sandwich")
    # (g) explanations: naive read-out vs orbit vs core-orbit soundness
    e1 = [r for r in SUMMARY if cond[r["condition"]]["exp"] == "E1" and cond[r["condition"]]["method"] == "gfrg" and "orbit_sound" in r]
    if e1:
        x = np.arange(len(e1))
        for j, (k, lab) in enumerate([("naive_sound", "naive read-out"), ("orbit_sound", "orbit G.g"), ("core_sound", "core orbit H.q(g)")]):
            ax[6].bar(x + (j - 1) * 0.27, [r[k] for r in e1], 0.27, label=lab)
        ax[6].set_xticks(x, [cond[r["condition"]]["task"] for r in e1], rotation=60, fontsize=7)
        ax[6].legend(fontsize=7)
        ax[6].set(ylabel="P(explanation contains the truth)", title="(g) explanation soundness (E1, gfrg)")
    # (h) post-hoc frequency gauge-fixing
    for k, g in GAUGE_FIX.items():
        l = ax[7].plot(g["ms"], np.maximum(g["error"], 1e-3), "o-", label=f"{cond[k]['task']} measured")
        ax[7].plot(g["ms"], np.maximum(g["bound"], 1e-3), "--", color=l[0].get_color(), label=f"{cond[k]['task']} bound")
    ax[7].set(xscale="log", yscale="log", xlabel="unlabeled images m", ylabel="P(wrong gauge)", title="(h) symmetry breaking by the prior (E4)")
    if GAUGE_FIX:
        ax[7].legend(fontsize=6)
    # (i) traps: off-optimum fraction, plain NeSy vs re-grounding, against the number of unrealisable symmetries
    tasks = sorted({cond[k]["task"] for k in SEED_TABLE if cond[k]["exp"] == "E1"})
    if tasks:
        x = np.arange(len(tasks))
        for j, meth in enumerate(["nesy", "gfrg"]):
            vals = [by.get(f"E1_{t}_{meth}", {}).get("off_optimum", np.nan) for t in tasks]   # not at the label optimum
            ax[8].bar(x + (j - 0.5) * 0.4, vals, 0.4, label=meth)
        ax[8].set_xticks(x, [f"{t}\n({TRAP_INFO.get(t, {}).get('count')})" for t in tasks], rotation=60, fontsize=6)
        ax[8].legend(fontsize=7)
        ax[8].set(ylabel="fraction of seeds off the label optimum", title="(i) traps (Theorem-B frustrated pairs)")
    fig.tight_layout()
    return fig

def make_confusion_figure():
    # mean confusion of an E1 condition vs the predicted block structure E[C[d, e]] = 1/|G.d| on the orbit of d
    cand = [k for k in SEED_TABLE if k.startswith("E1_") and 2 < PREDICTIONS[k]["order"] <= 100000]
    if not cand:
        return None
    k = max(cand, key=lambda k: len(SEED_TABLE[k]))
    oid = np.array(PREDICTIONS[k]["orbit_id"])
    pred = (oid[:, None] == oid[None, :]) / np.bincount(oid)[oid][:, None]
    meas = np.mean([np.array(e["C0"]) for u in unit_results.values() if u["key"] == k for e in u["per_seed"]], 0)
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    for a, M, t in [(ax[0], pred, "predicted E[C]"), (ax[1], meas, f"measured mean C ({len(SEED_TABLE[k])} seeds)")]:
        im = a.imshow(M, vmin=0, vmax=1, cmap="viridis")
        a.set(title=f"{k}: {t}", xlabel="network symbol", ylabel="true digit")
    fig.colorbar(im, ax=ax, shrink=0.8)
    return fig

main_fig = make_figures()
main_fig.savefig(os.path.join(RUN_DIR, "figures.png"), dpi=140)
STORE.push(os.path.join(RUN_DIR, "figures.png"))
conf_fig = make_confusion_figure()
if conf_fig is not None:
    conf_fig.savefig(os.path.join(RUN_DIR, "confusion.png"), dpi=140)
    STORE.push(os.path.join(RUN_DIR, "confusion.png"))
    plt.close(conf_fig)
print("uploads finished:", STORE.flush() or "no errors")
plt.close(main_fig)
main_fig

## 6. Explanations as formulas

What can a trained model *certify* about its concepts? Exactly the $G$-invariant claims, which are the first-order
definable relations of the knowledge (Bodirsky-Pinsker-Tsankov 2013, Thm 4, for finite structures). The table turns the
orbit of a prediction into the shortest conjunction of invariant literals (arithmetic ones and the label literals
$\beta(a,b)=y$), and gives the orbit-AXp: the shortest certified reason that entails the label and is strictly stronger
than it, with the number of equally short reasons (minimal reasons are rarely unique). For parity and summod10 the
certified reasons need relational literals (about both digits jointly), which no per-concept explanation can express.
For programs with twins, every claim is certified only at twin-class level.

In [ ]:
def human_literals():
    A, Bv = np.meshgrid(np.arange(N_CLASSES), np.arange(N_CLASSES), indexing="ij")
    V = [("a=b", A == Bv), ("a≠b", A != Bv)]
    for m in (2, 3, 5, 10):
        for r in range(m):
            V += [(f"a≡{r} (mod {m})", A % m == r), (f"b≡{r} (mod {m})", Bv % m == r),
                  (f"a+b≡{r} (mod {m})", (A + Bv) % m == r), (f"a·b≡{r} (mod {m})", (A * Bv) % m == r),
                  (f"a²≡{r} (mod {m})", (A * A) % m == r), (f"b²≡{r} (mod {m})", (Bv * Bv) % m == r)]
        V += [(f"a≡b (mod {m})", (A - Bv) % m == 0), (f"a≢b (mod {m})", (A - Bv) % m != 0)]
    return V

def invariant_literals(G):
    G64 = G.astype(np.int64)
    sub = G64[np.random.default_rng(0).choice(len(G64), size=min(len(G64), 5000), replace=False)]
    out, seen = [], set()
    for name, M in human_literals():
        if np.all(M[sub[:, :, None], sub[:, None, :]] == M[None]) and M.tobytes() not in seen:
            seen.add(M.tobytes())
            out.append((name, M))
    return out

def shortest_conjunction(target, V, ok, maxlen=3):
    # shortest conjunctions of literals true on `target` that satisfy ok(); returns (the first one, how many there are
    # of that length): cardinality-minimal reasons are usually not unique, so the count is reported
    cand = [(n, M) for n, M in V if np.all(M[target])]
    for L in range(1, maxlen + 1):
        hits = [combo for combo in itertools.combinations(cand, L) if ok(np.logical_and.reduce([m for _, m in combo]))]
        if hits:
            return " ∧ ".join(n for n, _ in hits[0]), len(hits)
    return None, 0

def orbit_formula(task, a, b):
    B = TASKS[task]
    G = automorphisms(B)
    po = pair_orbit_ids(G)
    O = po == po[a, b]
    V = invariant_literals(G)
    seen = {M.tobytes() for _, M in V}
    for y in np.unique(B):                                   # label literals beta(a,b) = y are invariant for every G
        if (B == y).tobytes() not in seen:
            seen.add((B == y).tobytes())
            V.append((f"β(a,b)={y}", B == y))
    exact, _ = shortest_conjunction(O, V, lambda M: np.array_equal(M, O))
    lab = B == B[a, b]
    # orbit-AXp: shortest conjunction of certified (G-invariant) literals, true on the orbit, that on its own entails
    # the predicted label and is strictly stronger than it (a conjunction equivalent to the label only restates it).
    # Literals that the label implies stay allowed, because they can be part of such a reason (v2 banned them).
    why, n_why = shortest_conjunction(O, [(n, M) for n, M in V if not np.array_equal(M, lab)],
                                      lambda M: bool(np.all(lab[M])) and not np.array_equal(M, lab))
    return dict(task=task, prediction=f"({a},{b})", label=int(B[a, b]), orbit_size=int(O.sum()),
                twins="yes" if twin_classes(B).max() < N_CLASSES - 1 else "no",
                certified=exact or "(needs a counting literal)", why_label=why or "none beyond the label itself",
                n_minimal_reasons=n_why)

print("Certified explanations (G-invariant claims about the predicted concept pair) and orbit-AXps.\n"
      "For programs with twins these claims are certified only at twin-class level; image-level splitting can violate "
      "digit-level claims (e.g. a = b / a ≠ b, a ≡ r literals).")
# literals are G-invariance-checked on a random subset of G (exact for the groups here, which are small or S_n-like)
FORMULAS = [orbit_formula(t, a, b) for t, a, b in [("summod5", 3, 4), ("summod5", 2, 7), ("prodmod10", 3, 4),
                                                   ("prodmod10", 3, 5), ("parity", 2, 6), ("summod10", 3, 4), ("absdiff", 2, 7),
                                                   ("petersen", 0, 1)]]
pd.DataFrame(FORMULAS) if pd is not None else FORMULAS

## 7. How to read the results

* **Instrument checks** are implied by exchangeable initialisation and an equivariant pipeline. A CHECK line points to
  a bug or to numerics (for E5: non-deterministic GPU reductions breaking exact ties), not to a wrong theory.
* **Primary tests** are the evidence. FAIL on a consistency test (P1, P4-P7) contradicts a prediction; NOT CONFIRMED on
  a confirmatory test (P2, P3) means the predicted effect was not shown at this sample size; NOT TESTABLE / VACUOUS
  tests are reported and never counted as passes.
* **Twins**: for programs with twins every claim is made at twin-class level, the only level the labels identify;
  image-level splitting inside a twin class is allowed by the theory.

In [ ]:
from collections import Counter as _Counter
_pilot_progs = {"add", "summod10", "absdiff", "prodmod10", "summod5", "summod4", "parity", "petersen"}
for _k in sorted(SEED_TABLE):
    _c = COND[_k]
    if _c["exp"] != "E1" or _c["method"] != "nesy":
        continue
    _lv = sorted(IDEAL_TRAPS.get(_c["task"], {}).get("levels", {}))
    _fresh = set(_c["seeds"][20:]) if _c["task"] in _pilot_progs else set(_c["seeds"])
    for _m in SEED_TABLE[_k]:
        if _m["at_opt"]:
            continue
        _near = _lv[int(np.argmin([abs(_m["excess"] - _L) for _L in _lv]))] if _lv else float("nan")
        print(f"{_k:20s} {'fresh' if _m['seed'] in _fresh else 'pilot'} label acc {_m['label_acc']:.3f} "
              f"excess {_m['excess']:.3f} nearest level {_near:.3f} ratio {_m['excess'] / _near:.2f}")
for _k in sorted(SEED_TABLE):
    if COND[_k]["exp"] == "E6":
        print(_k, _Counter(tuple(_m["alpha"]) for _m in SEED_TABLE[_k]).most_common(5))

In [ ]:
print("PROGRAM_NLL petersen", round(PROGRAM_NLL.get("petersen", float("nan")), 4))
for _k in sorted(SEED_TABLE):
    _c = COND[_k]
    if _c["exp"] != "E1" or _c["method"] != "nesy":
        continue
    _n_opt = sum(_m["at_opt"] for _m in SEED_TABLE[_k])
    print(f"== {_k}: {_n_opt} at optimum, {len(SEED_TABLE[_k]) - _n_opt} off")
    for _m, _C in zip(SEED_TABLE[_k], CONFUSIONS[_k]):
        if _m["at_opt"]:
            continue
        _parts = []
        for _d in range(N_CLASSES):
            _top = [int(_e) for _e in np.argsort(-_C[_d]) if _C[_d, _e] >= 0.15]
            _parts.append(f"{_d}:" + "/".join(map(str, _top)))
        print(f"  seed {_m['seed']} label acc {_m['label_acc']:.3f} excess {_m['excess']:.3f} | " + " ".join(_parts))

In [ ]:
V22_SIGMA = {"hi": [5, 1, 6, 2, 9, 8, 4, 0, 7, 3], "lo": [8, 9, 7, 1, 3, 6, 2, 5, 0, 4]}
for _nm, _s in V22_SIGMA.items():
    TASKS[f"petersen_{_nm}"] = TASKS["petersen"][np.ix_(_s, _s)]          # beta_s(a, b) = beta(s a, s b)
V22_CONDS = [make_condition("E8", "petersen_hi", n_seeds=60, seed_block="hash"),
             make_condition("E8", "petersen_lo", n_seeds=60, seed_block="hash"),
             make_condition("E8mlp", "petersen", n_seeds=40, seed_block="hash")]
V22_CONDS[2]["net"] = "mlp"
V22_RESULTS = {}
_units22 = [(c, i, c["seeds"][i:i + CFG["bank"]]) for c in V22_CONDS for i in range(0, len(c["seeds"]), CFG["bank"])]
for _k, (_c, _i, _seeds) in enumerate(_units22):
    _name = f"{_c['key']}_s{_seeds[0]}-{_seeds[-1]}"
    _path = os.path.join(RUN_DIR, "units", _name + ".json")
    if os.path.exists(_path):
        with open(_path) as _fh:
            V22_RESULTS[_name] = json.load(_fh)
        continue
    _t0 = time.time()
    _ckpt = os.path.join(RUN_DIR, "ckpt", _name + ".pt")
    _net_was = CFG["net"]
    CFG["net"] = _c.get("net", _net_was)                                    # only the MLP arm changes the network
    try:
        _banks, _nrg = train_bank(_c, _seeds, _ckpt, STORE)
    finally:
        CFG["net"] = _net_was
    _ev = evaluate_bank(_banks, _c, _seeds)
    _r = dict(key=_c["key"], unit=_name, seeds=_seeds, net=_c.get("net", "cnn"), secs=time.time() - _t0, per_seed=_ev)
    atomic_write(_path, json.dumps(_r))
    STORE.push(_path)
    V22_RESULTS[_name] = _r
    if os.path.exists(_ckpt):
        os.remove(_ckpt)
    print(f"[{_k + 1}/{len(_units22)}] {_name}: {_r['secs']:.0f}s, label acc {np.mean([e['label_acc'] for e in _ev]):.3f}", flush=True)
    del _banks
    gpu_free()
    trim_ram()
print(f"{len(V22_RESULTS)}/{len(_units22)} v2.2 units available; upload errors: {STORE.flush() or 'none'}")

In [ ]:
from scipy.stats import fisher_exact as _fisher
V22_ESCAPE_ORDER = {
    "petersen_hi": [((2, 3), (4, 9), (5, 8)), ((0, 6), (4, 5), (8, 9)), ((0, 6), (2, 3), (4, 8), (5, 9)), ((0, 1), (2, 8), (3, 5)), ((0, 2), (1, 8), (4, 7)), ((0, 4), (2, 7), (5, 6)), ((0, 3), (1, 5), (7, 9)), ((0, 9), (3, 7), (6, 8)), ((1, 9), (2, 6), (5, 7)), ((0, 5), (2, 7), (4, 6), (8, 9)), ((0, 1), (2, 5), (3, 8), (4, 9)), ((1, 4), (3, 6), (7, 8)), ((1, 6), (2, 9), (3, 4)), ((0, 5), (1, 3), (2, 8), (7, 9)), ((1, 6), (2, 4), (3, 9), (5, 8)), ((0, 8), (3, 7), (4, 5), (6, 9)), ((0, 8), (1, 2), (3, 5), (4, 7)), ((0, 7), (1, 8), (2, 4), (5, 6)), ((1, 3), (2, 9), (4, 6), (7, 8)), ((1, 2), (3, 4), (5, 7), (6, 9)), ((0, 3), (1, 7), (2, 6), (5, 9)), ((0, 7), (1, 5), (3, 9), (6, 8)), ((0, 2), (1, 7), (3, 6), (4, 8)), ((0, 4), (1, 9), (2, 5), (6, 7)), ((0, 9), (1, 4), (3, 8), (6, 7))],
    "petersen_lo": [((0, 8), (1, 6), (3, 5), (4, 9)), ((1, 8), (2, 3), (5, 7)), ((2, 9), (4, 7), (6, 8)), ((0, 6), (2, 5), (3, 7)), ((0, 8), (3, 4), (5, 9)), ((1, 3), (2, 8), (6, 9)), ((0, 9), (1, 7), (5, 8)), ((1, 3), (2, 6), (4, 7), (8, 9)), ((0, 3), (4, 8), (6, 7)), ((0, 2), (1, 4), (5, 6)), ((1, 6), (3, 9), (4, 5)), ((0, 5), (1, 7), (3, 4), (8, 9)), ((0, 1), (2, 4), (7, 9)), ((0, 6), (1, 8), (2, 7), (3, 5)), ((0, 3), (2, 9), (4, 6), (7, 8)), ((0, 5), (1, 4), (2, 6), (3, 7)), ((1, 9), (2, 8), (3, 6), (4, 5)), ((0, 4), (1, 2), (5, 6), (7, 9)), ((0, 4), (3, 8), (5, 9), (6, 7)), ((1, 2), (3, 8), (5, 7), (6, 9)), ((0, 7), (2, 5), (3, 6), (4, 8)), ((0, 9), (1, 5), (2, 3), (7, 8)), ((0, 2), (1, 5), (3, 9), (4, 6)), ((0, 1), (2, 7), (4, 9), (6, 8)), ((0, 7), (1, 9), (2, 4), (5, 8))],
}
def v22_involutions(B):
    _A = automorphisms(B)
    return [p for p in _A if np.array_equal(p[p], np.arange(N_CLASSES)) and not np.array_equal(p, np.arange(N_CLASSES))], _A
def v22_cycles(p):
    return tuple((x, int(p[x])) for x in range(N_CLASSES) if p[x] > x)
def v22_decode(C, invs):
    # digits d and pi(d) share concepts (merged or mixed); a digit fixed by pi uses a concept no other digit uses
    sup = [set(int(e) for e in np.flatnonzero(C[d] >= 0.15)) | {int(np.argmax(C[d]))} for d in range(N_CLASSES)]
    from collections import Counter as _Ctr
    used = _Ctr(c for s in sup for c in s)
    ok = []
    for p in invs:
        cyc = v22_cycles(p)
        if all(sup[a] & sup[b] for a, b in cyc) and all(used[c] == 1 for x in range(N_CLASSES) if p[x] == x for c in sup[x]):
            ok.append(p)
    return min(ok, key=lambda p: len(v22_cycles(p))) if ok else None
V22_SEEDS = {}
for _c in V22_CONDS:
    _B = TASKS[_c["task"]]
    _invs, _A = v22_involutions(_B)
    _per = [e for u in V22_RESULTS.values() if u["key"] == _c["key"] for e in u["per_seed"]]
    _rows = []
    for _e in _per:
        _C = np.array(_e["C0"]); _amap = _C.argmax(1)
        _rows.append(dict(seed=_e["seed"], C=_C, nll=_e["label_nll"], acc=_e["label_acc"],
                          optimal=bool(np.any(np.all(_A.astype(np.int64) == _amap[None], 1)))))
    _base = float(np.median([r["nll"] for r in _rows if r["optimal"]])) if any(r["optimal"] for r in _rows) else float("nan")
    for _r in _rows:
        _r["excess"] = _r["nll"] - _base
        _r["pi"] = None if _r["optimal"] else v22_decode(_r["C"], _invs)
    V22_SEEDS[_c["key"]] = (_c, _B, _invs, _rows)
    _st = [r for r in _rows if not r["optimal"]]
    print(f"== {_c['key']} ({_c.get('net', 'cnn')}): {len(_rows)} seeds, {len(_st)} stuck, "
          f"{sum(r['pi'] is not None for r in _st)} decoded as involution states; optimum NLL {_base:.4f}")
    for _r in _st:
        _lv = np.log(2) * np.mean(_B[_r["pi"]] != _B) if _r["pi"] is not None else float("nan")
        print(f"   seed {_r['seed']}: acc {_r['acc']:.3f} excess {_r['excess']:.3f} -> "
              + (f"pi {v22_cycles(_r['pi'])} level {_lv:.4f} ratio {_r['excess'] / _lv:.2f}" if _r["pi"] is not None else "not an involution state"))
# T1 (primary): selection. Within-type rank percentile u = (rank + 0.5) / n_type of each decoded stuck CNN seed's
# involution in the frozen escape order (0 = slowest). Null (symmetry alone): uniform within type.
_u, _ntype = [], []
for _key in ("E8_petersen_hi_nesy", "E8_petersen_lo_nesy"):
    _c, _B, _invs, _rows = V22_SEEDS[_key]
    _order = [tuple(sorted(c)) for c in V22_ESCAPE_ORDER[_c["task"]]]
    for _r in _rows:
        if _r["pi"] is None: continue
        _cy = tuple(sorted(v22_cycles(_r["pi"])))
        _same = [o for o in _order if len(o) == len(_cy)]
        _u.append((_same.index(_cy) + 0.5) / len(_same)); _ntype.append(len(_same))
_rng22 = np.random.default_rng(22)
if _u:
    _null = np.mean([(_rng22.integers(0, n, 100_000) + 0.5) / n for n in _ntype], axis=0)
    _p1 = float(np.mean(_null <= np.mean(_u)))
    print(f"T1 selection: {len(_u)} decoded stuck CNN seeds, mean within-type escape percentile {np.mean(_u):.3f} (null 0.5), "
          f"p = {_p1:.2g} -> {'CONFIRMED' if (_p1 < 1e-3 and np.mean(_u) <= 0.25) else 'NOT CONFIRMED'} (rule: p < 0.001 and mean <= 0.25)")
else:
    print("T1 selection: no decoded stuck seeds -> NOT TESTABLE")
# T2: levels. excess / (ln2 * P(beta(pi a, b) != beta(a, b))) in [0.88, 1.05] for >= 80% of decoded stuck CNN seeds.
_ratios = []
for _key in ("E8_petersen_hi_nesy", "E8_petersen_lo_nesy"):
    _c, _B, _invs, _rows = V22_SEEDS[_key]
    _ratios += [r["excess"] / (np.log(2) * np.mean(_B[r["pi"]] != _B)) for r in _rows if r["pi"] is not None]
_in = float(np.mean([(0.88 <= q <= 1.05) for q in _ratios])) if _ratios else float("nan")
print(f"T2 levels: {len(_ratios)} seeds, fraction of ratios in [0.88, 1.05] = {_in:.2f}, median ratio {np.median(_ratios) if _ratios else float('nan'):.3f} "
      f"-> {'CONFIRMED' if _in >= 0.8 else 'NOT CONFIRMED'}")
# T3: decodability. >= 70% of stuck CNN seeds are involution states.
_st3 = [r for k in ("E8_petersen_hi_nesy", "E8_petersen_lo_nesy") for r in V22_SEEDS[k][3] if not r["optimal"]]
_f3 = float(np.mean([r["pi"] is not None for r in _st3])) if _st3 else float("nan")
print(f"T3 decodability: {sum(r['pi'] is not None for r in _st3)}/{len(_st3)} stuck CNN seeds decoded -> {'CONFIRMED' if _f3 >= 0.7 else 'NOT CONFIRMED'}")
# T4: MLP arm traps less than the CNN on the same program (CNN: E1_petersen_nesy of this run, same optimality rule).
_Ap = automorphisms(TASKS["petersen"]).astype(np.int64)
_cnn_stuck = sum(not np.any(np.all(_Ap == np.array(C).argmax(1)[None], 1)) for C in CONFUSIONS["E1_petersen_nesy"])
_cnn_n = len(CONFUSIONS["E1_petersen_nesy"])
_mlp = V22_SEEDS["E8mlp_petersen_nesy"][3]
_mlp_stuck = sum(not r["optimal"] for r in _mlp)
_p4 = _fisher([[_mlp_stuck, len(_mlp) - _mlp_stuck], [_cnn_stuck, _cnn_n - _cnn_stuck]], alternative="less")[1]
print(f"T4 MLP vs CNN: MLP stuck {_mlp_stuck}/{len(_mlp)}, CNN stuck {_cnn_stuck}/{_cnn_n}, one-sided Fisher p = {_p4:.2g} "
      f"-> {'CONFIRMED' if _p4 < 0.01 else 'NOT CONFIRMED'}")
# T5 (exploratory, not part of the confirmation): 'hi' (slowest escape 0.0025) traps more often than 'lo' (0.0048).
_h = V22_SEEDS["E8_petersen_hi_nesy"][3]; _l = V22_SEEDS["E8_petersen_lo_nesy"][3]
_hs, _ls = sum(not r["optimal"] for r in _h), sum(not r["optimal"] for r in _l)
print(f"T5 (exploratory) stuck: hi {_hs}/{len(_h)}, lo {_ls}/{len(_l)}, one-sided Fisher p = "
      f"{_fisher([[_hs, len(_h) - _hs], [_ls, len(_l) - _ls]], alternative='greater')[1]:.2g}")

In [ ]:
V23_DATASETS = [d for d in os.environ.get("NSYM_V23_DATASETS", "mnist,fashion,kmnist,svhn").split(",") if d]
V23_NETS = ["cnn", "mlp"]
V23_N = int(os.environ.get("NSYM_V23_SEEDS", "40"))
V23_DATA_ROOT = os.environ.get("NSYM_DATA_ROOT", "data_v23")
V23_DIR = os.path.join(RUN_DIR, "v23")
os.makedirs(V23_DIR, exist_ok=True)

def v23_graph(edges):
    _M = np.zeros((N_CLASSES, N_CLASSES), int)
    for _u, _v in edges:
        _M[_u, _v] = _M[_v, _u] = 1
    return _M
# three twin-free programs whose symmetry groups contain many involutions (adjacency of a 10-vertex graph)
TASKS["crown"] = v23_graph([(i, 5 + j) for i in range(5) for j in range(5) if i != j])    # K5,5 minus a matching, |G| = 240
TASKS["cycle10"] = v23_graph([(i, (i + 1) % 10) for i in range(10)])                       # 10-cycle, |G| = 20 (dihedral)
V23_PROGRAMS = ["petersen", "crown", "cycle10"]

def v23_load(name):
    # train / test images as (N, 1, 28, 28) float32, standardised with the training mean and sd; the test set is
    # subsampled (fixed seed) to the size of the MNIST test set so YTE_NP can be overwritten in place
    from torchvision import datasets as _ds
    _root = os.path.join(V23_DATA_ROOT, name)
    if name == "fashion":
        _tr, _te = _ds.FashionMNIST(_root, train=True, download=True), _ds.FashionMNIST(_root, train=False, download=True)
        _xtr, _xte = _tr.data.numpy().astype(np.float32)[:, None] / 255.0, _te.data.numpy().astype(np.float32)[:, None] / 255.0
        _ytr, _yte = _tr.targets.numpy(), _te.targets.numpy()
    elif name == "kmnist":
        _tr, _te = _ds.KMNIST(_root, train=True, download=True), _ds.KMNIST(_root, train=False, download=True)
        _xtr, _xte = _tr.data.numpy().astype(np.float32)[:, None] / 255.0, _te.data.numpy().astype(np.float32)[:, None] / 255.0
        _ytr, _yte = _tr.targets.numpy(), _te.targets.numpy()
    elif name == "svhn":                       # street-view house numbers: grey, resized 32 -> 28 (area averaging)
        _tr, _te = _ds.SVHN(_root, split="train", download=True), _ds.SVHN(_root, split="test", download=True)
        def _grey(a):
            _g = torch.as_tensor(a.astype(np.float32) / 255.0)
            _g = (0.299 * _g[:, 0] + 0.587 * _g[:, 1] + 0.114 * _g[:, 2])[:, None]
            return F.interpolate(_g, size=(28, 28), mode="area").numpy()
        _xtr, _xte, _ytr, _yte = _grey(_tr.data), _grey(_te.data), np.asarray(_tr.labels), np.asarray(_te.labels)
    else:
        raise ValueError(name)
    _mu, _sd = float(_xtr.mean()), float(_xtr.std())
    _keep = np.sort(np.random.default_rng(23).choice(len(_yte), len(YTE_NP), replace=len(_yte) < len(YTE_NP)))
    return dict(xtr=(_xtr - _mu) / _sd, ytr=_ytr.astype(np.int64), xte=((_xte - _mu) / _sd)[_keep], yte=_yte[_keep].astype(np.int64))

@contextlib.contextmanager
def v23_use_data(name):
    if name == "mnist":
        yield
        return
    _d = v23_load(name)
    _x0, _xe0 = torch.empty(0, device=XTR.device).set_(XTR), torch.empty(0, device=XTE.device).set_(XTE)
    _y0, _p0, _pe0 = YTE_NP.copy(), dict(PTR), dict(PTE)
    try:
        XTR.set_(torch.as_tensor(_d["xtr"], device=XTR.device))
        XTE.set_(torch.as_tensor(_d["xte"], device=XTE.device))
        YTE_NP[:] = _d["yte"]
        PTR.clear(); PTR.update(make_pool(_d["ytr"]))
        PTE.clear(); PTE.update(make_pool(_d["yte"]))
        del _d
        yield
    finally:
        XTR.set_(_x0); XTE.set_(_xe0); YTE_NP[:] = _y0
        PTR.clear(); PTR.update(_p0); PTE.clear(); PTE.update(_pe0)
        gpu_free(); trim_ram()

def v23_class_kernel(net, n_init=6, per=40):
    # class tangent kernel at initialisation: K[a, b] = mean over images x of class a, x' of class b and output units
    # of <d z(x) / d theta, d z(x') / d theta>, averaged over n_init initialisations (torch seeds 0..n_init-1, the same
    # constructor as every trained network). Uses the class means of the Jacobians, which gives the same numbers as
    # averaging the image-level kernel. Returned normalised to unit diagonal.
    from torch.func import vmap as _vmap, jacrev as _jacrev
    _rng = np.random.default_rng(0)
    _idx = [_rng.choice(PTR["pools"][c], per, replace=False) for c in range(N_CLASSES)]
    _K = np.zeros((N_CLASSES, N_CLASSES))
    for _s in range(n_init):
        torch.manual_seed(_s)
        _ref = RefNet(net == "cnn", int(np.prod(XTR.shape[1:])))
        _p = {k: v.detach().to(DEVICE) for k, v in _ref.named_parameters()}
        def _z(p, x):
            if net == "cnn":
                h = F.max_pool2d(F.relu(F.conv2d(x[None], p["c1.weight"], p["c1.bias"])), 2)
                h = F.max_pool2d(F.relu(F.conv2d(h, p["c2.weight"], p["c2.bias"])), 2).flatten(1)
            else:
                h = x[None].flatten(1)
            h = F.relu(h @ p["fc1.weight"].T + p["fc1.bias"])
            return (h @ p["fc2.weight"].T + p["fc2.bias"])[0]
        _means = []
        for _c in range(N_CLASSES):
            _J = _vmap(_jacrev(_z), in_dims=(None, 0))(_p, XTR[torch.as_tensor(_idx[_c], device=XTR.device)].float())
            _means.append(torch.cat([v.reshape(per, N_CLASSES, -1) for v in _J.values()], -1).mean(0).double())
            del _J
        _Mn = torch.stack(_means)                                                     # (class, output, params)
        _K += (torch.einsum("aop,bop->ab", _Mn, _Mn) / N_CLASSES).cpu().numpy()
        del _Mn, _means
        gpu_free()
    _K /= n_init
    _dg = np.sqrt(np.diag(_K))
    return _K / np.outer(_dg, _dg)

def v23_escape_rates(B, Kn, invs):
    # escape rate of the two-point state M = (I + P_pi) / 2 under kernel-coupled gradient flow on the logits: the top
    # eigenvalue of -(K (x) I) H, H the logit Hessian of the idealised NeSy loss at M (same code as the v2.2 freeze)
    _ny = int(B.max()) + 1
    _R = F.one_hot(torch.as_tensor(B), _ny).permute(2, 0, 1).double()
    _y = torch.as_tensor(B)
    def _L(z):
        _M = torch.softmax(z.reshape(N_CLASSES, N_CLASSES), -1)
        _T = torch.einsum("ac,ycd,bd->yab", _M, _R, _M)
        return -torch.log(_T.gather(0, _y[None])[0].clamp_min(1e-300)).mean()
    _KI = np.kron(Kn, np.eye(N_CLASSES))
    _out = []
    for _pi in invs:
        _M = 0.5 * (np.eye(N_CLASSES) + np.eye(N_CLASSES)[_pi])
        _Z = torch.log(torch.as_tensor(_M).clamp_min(1e-12)).reshape(-1)
        _H = torch.autograd.functional.hessian(_L, _Z).numpy()
        _out.append(float(np.max(np.linalg.eigvals(-_KI @ _H).real)))
    return _out

def v23_predict(ds, net):
    # computed once per family, before any of its networks is trained; never recomputed once the file exists
    _path = os.path.join(V23_DIR, f"predictions_{ds}_{net}.json")
    if os.path.exists(_path):
        with open(_path) as _fh:
            return json.load(_fh)
    _trained = [f for f in os.listdir(os.path.join(RUN_DIR, "units")) if f.startswith(f"E9{ds}{net}_")]
    _Kn = v23_class_kernel(net)
    _progs = {}
    for _t in V23_PROGRAMS:
        _B = TASKS[_t]
        _invs, _ = v22_involutions(_B)
        _r = v23_escape_rates(_B, _Kn, _invs)
        _st = [dict(cycles=[list(c) for c in v22_cycles(p)], ncyc=len(v22_cycles(p)), rate=r,
                    level=float(np.log(2) * np.mean(_B[p] != _B))) for p, r in zip(_invs, _r)]
        _progs[_t] = sorted(_st, key=lambda d: d["rate"])
    _pred = dict(dataset=ds, net=net, created=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                 units_trained_before=len(_trained), kernel=_Kn.tolist(),
                 kernel_eigs=np.sort(np.linalg.eigvalsh(_Kn))[::-1].tolist(), programs=_progs)
    _txt = json.dumps(_pred, indent=1)
    atomic_write(_path, _txt)
    STORE.push(_path)
    STORE.flush()                                   # the prediction is in the bucket before training starts
    print(f"   predictions {ds}/{net}: sha256 {hashlib.sha256(_txt.encode()).hexdigest()[:16]}, kernel eigs "
          f"{np.round(_pred['kernel_eigs'][:3], 3).tolist()}; slowest state per program: "
          + "; ".join(f"{t} {_progs[t][0]['cycles']} ({_progs[t][0]['rate']:.4f})" for t in V23_PROGRAMS), flush=True)
    return _pred

V23_CONDS = []
for _ds in V23_DATASETS:
    for _net in V23_NETS:
        for _t in V23_PROGRAMS:
            _c = make_condition(f"E9{_ds}{_net}", _t, n_seeds=V23_N, seed_block="hash")
            _c.update(net=_net, dataset=_ds)
            V23_CONDS.append(_c)
V23_RESULTS, V23_PRED, V23_STATUS = {}, {}, {}
_n_units = sum(math.ceil(len(c["seeds"]) / CFG["bank"]) for c in V23_CONDS)
_k23, _t23 = 0, time.time()
for _ds in V23_DATASETS:
    _todo = []
    for _c in [c for c in V23_CONDS if c["dataset"] == _ds]:
        for _i in range(0, len(_c["seeds"]), CFG["bank"]):
            _seeds = _c["seeds"][_i:_i + CFG["bank"]]
            _name = f"{_c['key']}_s{_seeds[0]}-{_seeds[-1]}"
            _path = os.path.join(RUN_DIR, "units", _name + ".json")
            if os.path.exists(_path):
                with open(_path) as _fh:
                    V23_RESULTS[_name] = json.load(_fh)
                _k23 += 1
            else:
                _todo.append((_c, _seeds, _name, _path))
    _need_pred = [n for n in V23_NETS if not os.path.exists(os.path.join(V23_DIR, f"predictions_{_ds}_{n}.json"))]
    if not _todo and not _need_pred:
        V23_STATUS[_ds] = "done"
        for _net in V23_NETS:
            V23_PRED[(_ds, _net)] = v23_predict(_ds, _net)
        continue
    try:
        with v23_use_data(_ds):
            print(f"== {_ds}: train {tuple(XTR.shape)}, test {tuple(XTE.shape)}, {len(_todo)} units to train", flush=True)
            for _net in V23_NETS:
                V23_PRED[(_ds, _net)] = v23_predict(_ds, _net)
            for _c, _seeds, _name, _path in _todo:
                _t0 = time.time()
                _ckpt = os.path.join(RUN_DIR, "ckpt", _name + ".pt")
                _net_was = CFG["net"]
                CFG["net"] = _c["net"]
                try:
                    _banks, _ = train_bank(_c, _seeds, _ckpt, STORE)
                finally:
                    CFG["net"] = _net_was
                _ev = evaluate_bank(_banks, _c, _seeds)
                _r = dict(key=_c["key"], unit=_name, seeds=_seeds, net=_c["net"], dataset=_ds, task=_c["task"],
                          secs=time.time() - _t0, per_seed=_ev)
                atomic_write(_path, json.dumps(_r))
                STORE.push(_path)
                V23_RESULTS[_name] = _r
                if os.path.exists(_ckpt):
                    os.remove(_ckpt)
                _k23 += 1
                _line = (f"[{_k23}/{_n_units}] {_name}: {_r['secs']:.0f}s, label acc {np.mean([e['label_acc'] for e in _ev]):.3f}, "
                         f"elapsed {(time.time() - _t23) / 60:.0f} min")
                print(_line, flush=True)
                with open(os.path.join(V23_DIR, "progress.log"), "a") as _fh:
                    _fh.write(_line + "\n")
                STORE.push(os.path.join(V23_DIR, "progress.log"))
                del _banks
                gpu_free()
                trim_ram()
        V23_STATUS[_ds] = "done"
    except Exception as _e:                             # a dataset that cannot be downloaded is reported, not hidden
        V23_STATUS[_ds] = f"FAILED: {type(_e).__name__}: {str(_e)[:200]}"
        print(f"== {_ds}: {V23_STATUS[_ds]}", flush=True)
atomic_write(os.path.join(V23_DIR, "status.json"), json.dumps(V23_STATUS, indent=1))
STORE.push(os.path.join(V23_DIR, "status.json"))
print(f"{len(V23_RESULTS)}/{_n_units} v2.3 units available; datasets {V23_STATUS}; upload errors: {STORE.flush() or 'none'}")

In [ ]:
from scipy.stats import chisquare as _chisq23, spearmanr as _spear23
V23_TAU_BASE, V23_TAU_MAX, V23_TAU_NONE = 0.15, 0.35, 0.25

def v23_decode(C, invs, tau):
    # two-point state of involution pi: the digits of each 2-cycle share concepts (mixed or merged), and different
    # orbits of pi use disjoint sets of concepts (a fixed digit keeps its own concept); fewest 2-cycles wins a tie
    _sup = [set(int(e) for e in np.flatnonzero(C[d] >= tau)) | {int(np.argmax(C[d]))} for d in range(N_CLASSES)]
    _ok = []
    for _p in invs:
        _cyc = v22_cycles(_p)
        if not all(_sup[a] & _sup[b] for a, b in _cyc):
            continue
        _orbs = list(_cyc) + [(x,) for x in range(N_CLASSES) if _p[x] == x]
        _U = [set().union(*(_sup[v] for v in o)) for o in _orbs]
        if sum(len(u) for u in _U) == len(set().union(*_U)):
            _ok.append(_p)
    return min(_ok, key=lambda p: len(v22_cycles(p))) if _ok else None

def v23_percentile(pred_states, cyc):
    # within-type rank percentile (0 = slowest escape) of a decoded involution in one family's frozen order
    _cy = tuple(sorted(tuple(c) for c in cyc))
    _same = [tuple(sorted(tuple(c) for c in s["cycles"])) for s in pred_states if s["ncyc"] == len(_cy)]
    return (_same.index(_cy) + 0.5) / len(_same), len(_same)

V23_ROWS = {}                                   # (dataset, net, program) -> list of per-seed rows
for _c in V23_CONDS:
    _per = [e for u in V23_RESULTS.values() if u["key"] == _c["key"] for e in u["per_seed"]]
    if not _per:
        continue
    _B = TASKS[_c["task"]]
    _invs, _A = v22_involutions(_B)
    _A64 = _A.astype(np.int64)
    _rows = []
    for _e in _per:
        _C = np.array(_e["C0"]); _amap = _C.argmax(1)
        _rows.append(dict(seed=_e["seed"], C=_C, nll=_e["label_nll"], acc=_e["label_acc"], amap=_amap,
                          optimal=bool(np.any(np.all(_A64 == _amap[None], 1)))))
    _opt = [r for r in _rows if r["optimal"]]
    _noise = [max(np.max(np.delete(r["C"][d], r["amap"][d])) for d in range(N_CLASSES)) for r in _opt]
    _tau = min(V23_TAU_MAX, V23_TAU_BASE + float(np.median(_noise))) if _opt else V23_TAU_NONE
    _base = float(np.median([r["nll"] for r in _opt])) if _opt else float("nan")
    for _r in _rows:
        _r["excess"] = _r["nll"] - _base
        _r["pi"] = None if _r["optimal"] else v23_decode(_r["C"], _invs, _tau)
        if _r["pi"] is not None:
            _r["cyc"] = v22_cycles(_r["pi"])
            _r["level"] = float(np.log(2) * np.mean(_B[_r["pi"]] != _B))
    V23_ROWS[(_c["dataset"], _c["net"], _c["task"])] = dict(rows=_rows, tau=_tau, base=_base, A=_A)

print(f"{'dataset':8s} {'net':4s} {'program':9s} {'n':>3s} {'stuck':>5s} {'decoded':>7s} {'tau':>5s} {'mean u':>6s} {'med ratio':>9s}  slowest predicted state, times chosen")
_U1, _NT1, _S3, _RAT, _CELLS = [], [], [], [], []
for (_ds, _net, _t), _v in V23_ROWS.items():
    _pr = V23_PRED[(_ds, _net)]["programs"][_t]
    _dec = [r for r in _v["rows"] if r["pi"] is not None]
    _uu = []
    for _r in _dec:
        _u, _nt = v23_percentile(_pr, _r["cyc"])
        _uu.append(_u); _U1.append(_u); _NT1.append(_nt)
        _RAT.append((_r["excess"] / _r["level"], _r["level"]))
        _oth = [v23_percentile(V23_PRED[k]["programs"][_t], _r["cyc"])[0] for k in V23_PRED if k != (_ds, _net)]
        if _oth:
            _S3.append(_u - float(np.mean(_oth)))
    _st = [r for r in _v["rows"] if not r["optimal"]]
    _slow = tuple(tuple(c) for c in _pr[0]["cycles"])
    _CELLS.append(dict(ds=_ds, net=_net, prog=_t, n=len(_v["rows"]), stuck=len(_st), dec=len(_dec),
                       mean_u=float(np.mean(_uu)) if _uu else float("nan"), slow_rate=_pr[0]["rate"]))
    print(f"{_ds:8s} {_net:4s} {_t:9s} {len(_v['rows']):3d} {len(_st):5d} {len(_dec):7d} {_v['tau']:5.2f} "
          f"{(np.mean(_uu) if _uu else float('nan')):6.3f} {(np.median([r['excess'] / r['level'] for r in _dec]) if _dec else float('nan')):9.3f}  "
          f"{_slow}, {sum(tuple(sorted(r['cyc'])) == tuple(sorted(_slow)) for r in _dec)}")

V23_TESTS = {}
# S1 (primary): selection. Pooled within-type escape percentile of every decoded stuck seed; null: uniform within type.
if _U1:
    _rng23 = np.random.default_rng(23)
    _null = np.mean([(_rng23.integers(0, n, 100_000) + 0.5) / n for n in _NT1], axis=0)
    _p = float(np.mean(_null <= np.mean(_U1)))
    V23_TESTS["S1"] = bool(_p < 1e-3 and np.mean(_U1) <= 0.25)
    print(f"S1 selection: {len(_U1)} decoded stuck seeds, mean percentile {np.mean(_U1):.3f} (null 0.5), p = {_p:.2g} "
          f"-> {'CONFIRMED' if V23_TESTS['S1'] else 'NOT CONFIRMED'} (rule: p < 0.001 and mean <= 0.25)")
else:
    V23_TESTS["S1"] = None
    print("S1 selection: no decoded stuck seeds -> NOT TESTABLE")
# S2: replication. Cells (dataset x net x program) with >= 5 decoded stuck seeds: >= 75% have mean percentile < 0.5.
_big = [c for c in _CELLS if c["dec"] >= 5]
if len(_big) >= 4:
    _f = float(np.mean([c["mean_u"] < 0.5 for c in _big]))
    V23_TESTS["S2"] = bool(_f >= 0.75)
    print(f"S2 replication: {sum(c['mean_u'] < 0.5 for c in _big)}/{len(_big)} cells with >= 5 decoded seeds have mean percentile < 0.5 "
          f"-> {'CONFIRMED' if V23_TESTS['S2'] else 'NOT CONFIRMED'} (rule: >= 75% of at least 4 cells)")
else:
    V23_TESTS["S2"] = None
    print(f"S2 replication: only {len(_big)} cells with >= 5 decoded seeds -> NOT TESTABLE (needs 4)")
# S3: specificity. A family's own kernel predicts its seeds better than the other families' kernels do.
if len(_S3) >= 10:
    _d = np.array(_S3)
    _rng23b = np.random.default_rng(230)
    _flip = np.mean((_rng23b.choice([-1.0, 1.0], (100_000, len(_d))) * _d[None]), 1)
    _p3 = float(np.mean(_flip <= _d.mean()))
    V23_TESTS["S3"] = bool(_d.mean() < 0 and _p3 < 0.01)
    _rho = [_spear23([s["rate"] for s in sorted(V23_PRED[a]["programs"][t], key=lambda s: str(s["cycles"]))],
                     [s["rate"] for s in sorted(V23_PRED[b]["programs"][t], key=lambda s: str(s["cycles"]))])[0]
            for t in V23_PROGRAMS for a, b in itertools.combinations(sorted(V23_PRED), 2)]
    print(f"S3 specificity: own minus other-kernel percentile {_d.mean():+.3f} over {len(_d)} seeds, sign-flip p = {_p3:.2g} "
          f"-> {'CONFIRMED' if V23_TESTS['S3'] else 'NOT CONFIRMED'} (rule: mean < 0 and p < 0.01); "
          f"median Spearman between families' escape orders {np.median(_rho):.2f}")
else:
    V23_TESTS["S3"] = None
    print(f"S3 specificity: {len(_S3)} seeds -> NOT TESTABLE (needs 10)")
# L: levels. excess / (ln2 * P(beta(pi a, b) != beta(a, b))) in [0.80, 1.10] for >= 75% of decoded stuck seeds whose
# predicted level is >= 0.10 (smaller levels are within the seed-to-seed spread of the optimum's NLL; reported only).
_RAT_small = [q for q, lv in _RAT if lv < 0.10]
_RAT = [q for q, lv in _RAT if lv >= 0.10]
if _RAT_small:
    print(f"L levels below 0.10 (reported, not tested): {len(_RAT_small)} seeds, median ratio {np.median(_RAT_small):.3f}")
if _RAT:
    _in = float(np.mean([(0.80 <= q <= 1.10) for q in _RAT]))
    V23_TESTS["L"] = bool(_in >= 0.75)
    print(f"L levels: {len(_RAT)} seeds, {_in:.2f} in [0.80, 1.10] ({np.mean([(0.88 <= q <= 1.05) for q in _RAT]):.2f} in the v2.2 band), "
          f"median ratio {np.median(_RAT):.3f} -> {'CONFIRMED' if V23_TESTS['L'] else 'NOT CONFIRMED'}")
else:
    V23_TESTS["L"] = None
    print("L levels: no decoded seeds -> NOT TESTABLE")
# D (secondary): >= 60% of stuck seeds decode as involution states.
_nst, _ndec = sum(c["stuck"] for c in _CELLS), sum(c["dec"] for c in _CELLS)
V23_TESTS["D"] = bool(_ndec / _nst >= 0.6) if _nst else None
print(f"D decodability (secondary): {_ndec}/{_nst} stuck seeds decoded -> "
      + ("NOT TESTABLE" if V23_TESTS["D"] is None else ("CONFIRMED" if V23_TESTS["D"] else "NOT CONFIRMED")))
# O (instrument): orbit law. Seeds at the optimum land on a Haar-uniform automorphism: the number of digits the map
# fixes follows its distribution over Aut(beta) (chi-square, bins with expected count < 5 merged). PASS if p > 0.01.
V23_TESTS["O"] = {}
for _t in V23_PROGRAMS:
    _maps = [r["amap"] for k, v in V23_ROWS.items() if k[2] == _t for r in v["rows"] if r["optimal"]]
    if len(_maps) < 20:
        print(f"O orbit law {_t}: {len(_maps)} optimal seeds -> NOT TESTABLE"); continue
    _A = next(v["A"] for k, v in V23_ROWS.items() if k[2] == _t)
    _haar = np.bincount((_A == np.arange(N_CLASSES)[None]).sum(1), minlength=N_CLASSES + 1) / len(_A)
    _obs = np.bincount([int(np.sum(m == np.arange(N_CLASSES))) for m in _maps], minlength=N_CLASSES + 1)
    _exp = _haar * len(_maps)
    _o, _e, _ao, _ae = [], [], 0.0, 0.0
    for _k in np.argsort(-_exp):
        if _exp[_k] == 0:
            _ao += _obs[_k]; continue
        _ao += _obs[_k]; _ae += _exp[_k]
        if _ae >= 5:
            _o.append(_ao); _e.append(_ae); _ao = _ae = 0.0
    if _ae > 0 and _e:
        _o[-1] += _ao; _e[-1] += _ae
    _po = float(_chisq23(_o, _e).pvalue) if len(_o) > 1 else float("nan")
    V23_TESTS["O"][_t] = bool(_po > 0.01)
    print(f"O orbit law {_t}: {len(_maps)} optimal seeds, mean fixed digits {np.mean([np.sum(m == np.arange(N_CLASSES)) for m in _maps]):.2f} "
          f"(Haar {np.dot(np.arange(N_CLASSES + 1), _haar):.2f}), chi-square p = {_po:.2g} -> {'PASS' if _po > 0.01 else 'CHECK'}")
# X1, X2 (exploratory): capture. Do families with slower escape trap more often? Does the MLP trap less than the CNN?
if len(_CELLS) >= 6:
    _rx = _spear23([c["slow_rate"] for c in _CELLS], [c["stuck"] / c["n"] for c in _CELLS])
    print(f"X1 (exploratory) Spearman(slowest escape rate, stuck fraction) over {len(_CELLS)} cells = {_rx[0]:+.2f} (p = {_rx[1]:.2g}); prediction: negative")
for _ds in V23_DATASETS:
    _cn = [c for c in _CELLS if c["ds"] == _ds and c["net"] == "cnn"]; _ml = [c for c in _CELLS if c["ds"] == _ds and c["net"] == "mlp"]
    if _cn and _ml:
        print(f"X2 (exploratory) {_ds}: stuck CNN {sum(c['stuck'] for c in _cn)}/{sum(c['n'] for c in _cn)}, "
              f"MLP {sum(c['stuck'] for c in _ml)}/{sum(c['n'] for c in _ml)}")
# R (robustness of v2.2): the v2.2 seeds re-decoded with this cell's stricter decoder (tau 0.15, disjoint orbits)
if "V22_SEEDS" in globals():
    _r22 = [(k, r) for k, v in V22_SEEDS.items() for r in v[3] if not r["optimal"]]
    _same = sum((v23_decode(r["C"], V22_SEEDS[k][2], 0.15) is not None) == (r["pi"] is not None) for k, r in _r22)
    print(f"R v2.2 seeds re-decoded with the v2.3 rule: {_same}/{len(_r22)} stuck seeds get the same decodable/undecodable verdict")
_core = [V23_TESTS.get(k) for k in ("S1", "S2", "S3", "L")]
_valid = all(V23_TESTS["O"].values()) if V23_TESTS["O"] else None
print("VERDICT:", "SELECTION AND LEVEL LAWS CONFIRMED ACROSS DATASETS, NETWORKS AND PROGRAMS" if all(x is not None and x for x in _core)
      else ("NOT CONFIRMED (see the failing tests above)" if any(x is not None and not x for x in _core) else "NOT TESTABLE YET (too few decoded seeds)"), "| instrument O:", "PASS" if _valid else ("CHECK" if _valid is False else "n/a"))
atomic_write(os.path.join(V23_DIR, "tests.json"), json.dumps(dict(tests={k: (v if not isinstance(v, dict) else v) for k, v in V23_TESTS.items()},
                                                                 cells=_CELLS, status=V23_STATUS), indent=1, default=str))
STORE.push(os.path.join(V23_DIR, "tests.json"))

In [ ]:
def v23f_cycles(p):
    _seen, _out = set(), []
    for _x in range(N_CLASSES):
        if _x in _seen:
            continue
        _c, _y = [_x], int(p[_x])
        _seen.add(_x)
        while _y != _x:
            _c.append(_y); _seen.add(_y); _y = int(p[_y])
        _out.append(tuple(_c))
    return _out

def v23f_decode(C, auts, tau):
    _sup = [set(int(e) for e in np.flatnonzero(C[d] >= tau)) | {int(np.argmax(C[d]))} for d in range(N_CLASSES)]
    from collections import Counter as _Ctr
    if max(_Ctr(c for s in _sup for c in s).values()) > 2:
        return None
    _ok = []
    for _p in auts:
        if np.array_equal(_p, np.arange(N_CLASSES)):
            continue
        _cy = v23f_cycles(_p)
        if not all(_sup[d] & _sup[int(_p[d])] for c in _cy if len(c) > 1 for d in c):
            continue
        _U = [set().union(*(_sup[v] for v in c)) for c in _cy]
        if sum(len(u) for u in _U) == len(set().union(*_U)):
            _ok.append(_p)
    return min(_ok, key=lambda p: sum(len(c) > 1 for c in v23f_cycles(p)) * 100 + len(v23f_cycles(p))) if _ok else None

def v23f_level(B, p):
    _M = 0.5 * (np.eye(N_CLASSES) + np.eye(N_CLASSES)[p])
    _y = B
    _Q = np.einsum("ac,bd,abcd->ab", _M, _M, (B[None, None, :, :] == _y[:, :, None, None]).astype(float))
    return float(-np.mean(np.log(_Q)))

V23F = {}
for (_ds, _net, _t), _v in V23_ROWS.items():
    if _ds == "svhn":
        continue
    _B = TASKS[_t]
    _A = _v["A"].astype(np.int64)
    _und = [r for r in _v["rows"] if not r["optimal"] and r["pi"] is None]
    if not _und:
        continue
    _Kn = np.array(V23_PRED[(_ds, _net)]["kernel"])
    _rates = dict(zip(map(tuple, _A.tolist()), v23_escape_rates(_B, _Kn, list(_A))))
    _types = {}
    for _p in _A:
        _types.setdefault(tuple(sorted(len(c) for c in v23f_cycles(_p))), []).append(tuple(_p.tolist()))
    print(f"== {_ds} {_net} {_t}: {len(_und)} undecoded stuck networks (optimum NLL {_v['base']:.4f})")
    for _r in _und:
        _p = v23f_decode(_r["C"], _A, _v["tau"])
        _parts = []
        for _d in range(N_CLASSES):
            _o = np.argsort(-_r["C"][_d])
            _parts.append(f"{_d}:{_o[0]}" + (f"/{_o[1]}" if _r["C"][_d, _o[1]] >= 0.15 else ""))
        _txt = f"   seed {_r['seed']} acc {_r['acc']:.3f} excess {_r['excess']:.3f} | " + " ".join(_parts)
        if _p is not None:
            _ty = tuple(sorted(len(c) for c in v23f_cycles(_p)))
            _same = sorted(_types[_ty], key=lambda q: _rates[q])
            _u = (_same.index(tuple(_p.tolist())) + 0.5) / len(_same)
            _lv = v23f_level(_B, _p)
            V23F.setdefault((_ds, _net, _t), []).append(dict(type=_ty, u=_u, ratio=_r["excess"] / _lv))
            _txt += (f" -> sigma {[c for c in v23f_cycles(_p) if len(c) > 1]} type {_ty} level {_lv:.3f} "
                     f"ratio {_r['excess'] / _lv:.2f} escape percentile {_u:.2f}")
        print(_txt)
print("summary (exploratory): newly decoded stuck networks per cell, cycle type counts, mean escape percentile, median level ratio")
for _k, _lst in V23F.items():
    from collections import Counter as _Ctr2
    print(f"   {_k}: {len(_lst)} decoded, types {dict(_Ctr2(d['type'] for d in _lst))}, mean u {np.mean([d['u'] for d in _lst]):.2f}, "
          f"median ratio {np.median([d['ratio'] for d in _lst]):.2f}")

In [ ]:
V24_SETS = [tuple(s.split(":")) for s in os.environ.get("NSYM_V24_SETS", "mnist:0123,fashion:0123,kmnist:0123,mnist:4589").split(",") if s]
V24_NETS = ["cnn", "mlp"]
V24_N = int(os.environ.get("NSYM_V24_SEEDS", "40"))
V24_STEPS = int(os.environ.get("NSYM_V24_STEPS", "1500"))
V24_PUZ = int(os.environ.get("NSYM_V24_PUZZLES", "1000"))         # valid boards in the training set (+ as many corrupted)
V24_BS = 32                                                        # boards per step (16 images each)
V24_DENSE = 300                                                    # probe every step up to here, then every 10 steps
V24_N50 = int(os.environ.get("NSYM_V24_N50", "10"))                # benchmark arm: 50 training puzzles, one split per network
V24_DIR = os.path.join(RUN_DIR, "v24")
for _d in ("units", "ckpt"):
    os.makedirs(os.path.join(V24_DIR, _d), exist_ok=True)

def v24_grids():
    _out, _g = [], [-1] * 16
    def _ok(i, v):
        _r, _c = divmod(i, 4)
        for _j in range(i):
            _rj, _cj = divmod(_j, 4)
            if _g[_j] == v and (_rj == _r or _cj == _c or (_rj // 2, _cj // 2) == (_r // 2, _c // 2)):
                return False
        return True
    def _rec(i):
        if i == 16:
            _out.append(list(_g))
            return
        for _v in range(4):
            if _ok(i, _v):
                _g[i] = _v
                _rec(i + 1)
        _g[i] = -1
    _rec(0)
    return np.array(_out, dtype=np.int64)
V24_VALID = v24_grids()
assert len(V24_VALID) == 288
V24_VSET = set(map(tuple, V24_VALID.tolist()))

def v24_boards(n_pos, rng, chance=0.5):
    # ViSudo-PC protocol; the corruption type (replacement or swap of two cells) is drawn per board, 50/50
    _pos = V24_VALID[rng.integers(0, 288, n_pos)]
    _neg = []
    while len(_neg) < n_pos:
        _g = V24_VALID[rng.integers(0, 288)].copy()
        _swap = rng.random() < 0.5
        while True:
            if _swap:
                _i, _j = rng.choice(16, 2, replace=False)
                _g[_i], _g[_j] = _g[_j], _g[_i]
            else:
                _i = rng.integers(16)
                _g[_i] = (_g[_i] + rng.integers(1, 4)) % 4
            if rng.random() >= chance:
                break
        if tuple(_g.tolist()) not in V24_VSET:
            _neg.append(_g)
    return np.concatenate([_pos, np.array(_neg)]), np.r_[np.ones(n_pos), np.zeros(n_pos)]

def v24_logvalid(lp):
    # lp: (..., 16, 4) log cell distributions -> (...) log P(board is valid), exact sum over the 288 valid grids
    _oh = F.one_hot(torch.as_tensor(V24_VALID, device=lp.device), 4).reshape(288, 64).to(lp.dtype)
    return torch.logsumexp(lp.reshape(*lp.shape[:-2], 64) @ _oh.T, -1)

def v24_nll(lp, y):
    _lv = v24_logvalid(lp)
    return -(y * _lv + (1 - y) * torch.log(-torch.expm1(_lv.clamp_max(-1e-7))))

def v24_partitions(s):
    if not s:
        yield []
        return
    for _p in v24_partitions(s[1:]):
        for _i in range(len(_p)):
            yield _p[:_i] + [[s[0]] + _p[_i]] + _p[_i + 1:]
        yield [[s[0]]] + _p
V24_PARTS = [sorted(sorted(b) for b in p) for p in v24_partitions([0, 1, 2, 3])]
V24_PAIRS = list(itertools.combinations(range(4), 2))

def v24_MP(p):
    _M = np.zeros((4, 4))
    for _b in p:
        _M[np.ix_(_b, _b)] = 1.0 / len(_b)
    return _M

def v24_load(name):
    # Fashion-MNIST or KMNIST via torchvision (self-contained, so this cell does not need cell D's globals)
    from torchvision import datasets as _ds
    _root = os.path.join(os.environ.get("NSYM_DATA_ROOT", "data_v23"), name)
    _cls = {"fashion": _ds.FashionMNIST, "kmnist": _ds.KMNIST}[name]
    _tr, _te = _cls(_root, train=True, download=True), _cls(_root, train=False, download=True)
    _xtr = _tr.data.numpy().astype(np.float32)[:, None] / 255.0
    _xte = _te.data.numpy().astype(np.float32)[:, None] / 255.0
    _mu, _sd = float(_xtr.mean()), float(_xtr.std())
    return dict(xtr=(_xtr - _mu) / _sd, ytr=_tr.targets.numpy(), xte=(_xte - _mu) / _sd, yte=_te.targets.numpy())

def v24_family_data(src, digits):
    # images of the four classes used (train / test pools as in the source split), standardised with the source's
    # training mean and sd (MNIST: the notebook's own tensors)
    _dg = [int(c) for c in digits]
    if src == "mnist":
        _ptr = [np.asarray(PTR["pools"][c]) for c in _dg]
        _xtr = torch.cat([XTR[torch.as_tensor(p, device=XTR.device)] for p in _ptr]).float()
        _te = [np.flatnonzero(YTE_NP == c) for c in _dg]
        _xte = torch.cat([XTE[torch.as_tensor(p, device=XTE.device)] for p in _te]).float()
    else:
        _d = v24_load(src)
        _ptr = [np.flatnonzero(_d["ytr"] == c) for c in _dg]
        _xtr = torch.as_tensor(np.concatenate([_d["xtr"][p] for p in _ptr]), device=DEVICE)
        _te = [np.flatnonzero(_d["yte"] == c) for c in _dg]
        _xte = torch.as_tensor(np.concatenate([_d["xte"][p] for p in _te]), device=DEVICE)
    _cut = np.cumsum([0] + [len(p) for p in _ptr])
    _cte = np.cumsum([0] + [len(p) for p in _te])
    return dict(xtr=_xtr.to(DEVICE), pools=[np.arange(_cut[i], _cut[i + 1]) for i in range(4)],
                xte=_xte.to(DEVICE), tpools=[np.arange(_cte[i], _cte[i + 1]) for i in range(4)],
                yte=np.repeat(np.arange(4), np.diff(_cte)))

def v24_seed(*parts):
    return int(hashlib.sha1("|".join(map(str, ("v24",) + parts)).encode()).hexdigest()[:8], 16)

def v24_board_images(Xb, pools, rng):
    return np.stack([[pools[s][rng.integers(len(pools[s]))] for s in g] for g in Xb]).astype(np.int64)

def v24_kernel(net, xtr, pools, n_init=6, per=40):
    # class tangent kernel at initialisation over the 4 output units the task uses (same constructor and seeds as
    # cell D: torch seeds 0..5, 40 images per class drawn with numpy seed 0), normalised to unit diagonal
    from torch.func import vmap as _vmap, jacrev as _jacrev
    _rng = np.random.default_rng(0)
    _idx = [_rng.choice(pools[c], per, replace=False) for c in range(4)]
    _K = np.zeros((4, 4))
    for _s in range(n_init):
        torch.manual_seed(_s)
        _ref = RefNet(net == "cnn", int(np.prod(xtr.shape[1:])))
        _p = {k: v.detach().to(DEVICE) for k, v in _ref.named_parameters()}
        def _z(p, x):
            if net == "cnn":
                h = F.max_pool2d(F.relu(F.conv2d(x[None], p["c1.weight"], p["c1.bias"])), 2)
                h = F.max_pool2d(F.relu(F.conv2d(h, p["c2.weight"], p["c2.bias"])), 2).flatten(1)
            else:
                h = x[None].flatten(1)
            h = F.relu(h @ p["fc1.weight"].T + p["fc1.bias"])
            return (h @ p["fc2.weight"][:4].T + p["fc2.bias"][:4])[0]
        _means = []
        for _c in range(4):
            _J = _vmap(_jacrev(_z), in_dims=(None, 0))(_p, xtr[torch.as_tensor(_idx[_c], device=xtr.device)].float())
            _means.append(torch.cat([v.reshape(per, 4, -1) for v in _J.values()], -1).mean(0).double())
            del _J
        _Mn = torch.stack(_means)
        _K += (torch.einsum("aop,bop->ab", _Mn, _Mn) / 4).cpu().numpy()
        del _Mn, _means
        gpu_free()
    _K /= n_init
    _dg = np.sqrt(np.diag(_K))
    return _K / np.outer(_dg, _dg)

def v24_states(Xb, yb):
    # level, Hessian (logits, CPU double) and negative eigenvalues of every partition state on these boards
    _X = torch.as_tensor(Xb)
    _y = torch.as_tensor(yb, dtype=torch.float64)
    def _L(z):
        return v24_nll(torch.log_softmax(z.reshape(4, 4), -1)[_X], _y).mean()
    _out = []
    for _p in V24_PARTS:
        _Z = torch.log(torch.as_tensor(v24_MP(_p)).clamp_min(1e-13)).reshape(-1)
        _H = torch.autograd.functional.hessian(_L, _Z).numpy()
        _ev = np.linalg.eigvalsh(_H)
        _out.append(dict(partition=_p, level=float(_L(_Z)), H=_H, n_neg=int((_ev < -1e-6).sum()),
                         index_formula=int(sum((len(b) - 1) ** 2 for b in _p))))
    return _out

def v24_predict(fam, net, data, states):
    # computed once per (family, net), before any of its networks is trained; never recomputed once the file exists
    _path = os.path.join(V24_DIR, f"predictions_{fam}_{net}.json")
    if os.path.exists(_path):
        with open(_path) as _fh:
            return json.load(_fh)
    _trained = [f for f in os.listdir(os.path.join(V24_DIR, "units")) if f.startswith(f"G{fam}{net}_")]
    _Kn = v24_kernel(net, data["xtr"], data["pools"])
    _KI = np.kron(_Kn, np.eye(4))
    _st = []
    for _s in states:
        _rate = float(np.max(np.linalg.eigvals(-_KI @ _s["H"]).real)) if 1 < len(_s["partition"]) < 4 else None
        _st.append(dict(partition=_s["partition"], level=_s["level"], n_neg=_s["n_neg"], index_formula=_s["index_formula"], rate=_rate))
    _pair_rate = {}
    for _s in _st:
        if len(_s["partition"]) == 3:
            _pair_rate["".join(map(str, next(b for b in _s["partition"] if len(b) == 2)))] = _s["rate"]
    _order = sorted(_pair_rate, key=_pair_rate.get)                    # slowest escape first = separates last
    _mu = np.stack([data["xtr"][torch.as_tensor(data["pools"][c][:500], device=data["xtr"].device)].mean(0).flatten().double().cpu().numpy()
                    for c in range(4)])                                 # baseline: cosine of the class-mean images
    _pix = _mu @ _mu.T / np.outer(np.linalg.norm(_mu, axis=1), np.linalg.norm(_mu, axis=1))
    _pred = dict(family=fam, net=net, created=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                 units_trained_before=len(_trained), kernel=_Kn.tolist(), states=_st, pair_rate=_pair_rate,
                 predicted_last_pair=_order[0], predicted_order_slow_to_fast=_order,
                 pixel_cosine=_pix.tolist(),
                 pixel_last_pair="".join(map(str, max(V24_PAIRS, key=lambda q: _pix[q]))))
    _txt = json.dumps(_pred, indent=1)
    atomic_write(_path, _txt)
    STORE.push(_path)
    STORE.flush()                                   # the prediction is in the bucket before training starts
    print(f"   predictions {fam}/{net}: sha256 {hashlib.sha256(_txt.encode()).hexdigest()[:16]}; predicted last pair "
          f"{_order[0]} (rate {_pair_rate[_order[0]]:.4f}); order slow->fast {_order}", flush=True)
    return _pred

def v24_bank(seeds, net):
    _was = CFG["net"]
    CFG["net"] = net
    try:
        _b = make_bank(seeds)
    finally:
        CFG["net"] = _was
    with torch.no_grad():                         # the task uses 4 output units: keep the first 4 rows of the head
        _b.W2 = nn.Parameter(_b.W2[:, :4].clone())
        _b.b2 = nn.Parameter(_b.b2[:, :4].clone())
    return _b

def v24_train_unit(name, seeds, net, data, I, yb, probe, test, gen_seed):
    # trains one bank on the boards (I: image indices (boards, 16), yb: validity); returns the record of cell H
    _ck = os.path.join(V24_DIR, "ckpt", name + ".pt")
    _bank = v24_bank(seeds, net)
    _opt = torch.optim.Adam(_bank.parameters(), lr=CFG["lr"])
    _g = torch.Generator().manual_seed(gen_seed)
    _It, _yt = torch.as_tensor(I, device=DEVICE), torch.as_tensor(yb, dtype=torch.float32, device=DEVICE)
    _start, _traj, _tsteps, _loss = 0, [], [], []
    if os.path.exists(_ck) and os.path.getsize(_ck) > 1000:
        _st = torch.load(_ck, map_location=DEVICE, weights_only=False)
        _bank.load_state_dict(_st["bank"]); _opt.load_state_dict(_st["opt"]); _g.set_state(_st["gen"].cpu())
        _start, _traj, _tsteps, _loss = _st["step"], _st["traj"], _st["tsteps"], _st["loss"]
    def _probe():
        with torch.no_grad():
            _p = F.softmax(_bank(probe[0]).float(), -1)
            return torch.stack([_p[:, probe[1] == c].mean(1) for c in range(4)], 1).cpu().numpy().astype(np.float16)
    for _s in range(_start, V24_STEPS):
        if _s < V24_DENSE or _s % 10 == 0:
            _traj.append(_probe()); _tsteps.append(_s)
        _b = torch.randint(0, len(_yt), (V24_BS,), generator=_g).to(DEVICE)
        _lg = _bank(data["xtr"][_It[_b].reshape(-1)])                                   # (S, bs * 16, 4)
        _lp = F.log_softmax(_lg.float(), -1).reshape(len(seeds), V24_BS, 16, 4)
        _l = v24_nll(_lp, _yt[_b][None]).mean(1)
        _opt.zero_grad(set_to_none=True)
        _l.sum().backward()
        _opt.step()
        _loss.append(_l.detach().cpu().numpy().astype(np.float16))
        if (_s + 1) % 500 == 0 and _s + 1 < V24_STEPS:
            torch.save(dict(bank=_bank.state_dict(), opt=_opt.state_dict(), gen=_g.get_state(), step=_s + 1,
                            traj=_traj, tsteps=_tsteps, loss=_loss), _ck)
            STORE.push(_ck)
    _traj.append(_probe()); _tsteps.append(V24_STEPS)
    with torch.no_grad():
        _pt = torch.cat([F.softmax(_bank(test["x"][i:i + 2000]).float(), -1) for i in range(0, len(test["x"]), 2000)], 1)
        _C = torch.stack([_pt[:, torch.as_tensor(test["y"] == c, device=DEVICE)].mean(1) for c in range(4)], 1).cpu().numpy()
        _hard = _pt.argmax(-1).cpu().numpy()                                                   # (S, n_test)
        _lpb = torch.log(torch.cat([F.softmax(_bank(test["bx"][i:i + 3200]).float(), -1) for i in range(0, len(test["bx"]), 3200)], 1).clamp_min(1e-30))
        _pv = v24_logvalid(_lpb.reshape(len(seeds), -1, 16, 4)).exp().cpu().numpy()             # (S, test boards)
    _perms = np.array(list(itertools.permutations(range(4))))
    _acc = np.array([[np.mean(_perms[k][test["y"]] == _hard[s]) for k in range(24)] for s in range(len(seeds))])
    def _auroc(sc, lab):
        _r = np.argsort(np.argsort(sc)) + 1.0
        _n1 = lab.sum()
        return float((_r[lab == 1].sum() - _n1 * (_n1 + 1) / 2) / (_n1 * (len(lab) - _n1)))
    _rec = dict(traj=np.stack(_traj, 1), tsteps=np.array(_tsteps), loss=np.stack(_loss, 1), C=_C,
                perm=_perms[_acc.argmax(1)], perm_acc=_acc.max(1), auroc=np.array([_auroc(_pv[s], test["by"]) for s in range(len(seeds))]),
                seeds=np.array(seeds))
    if os.path.exists(_ck):                       # the bucket keeps a tiny marker instead of the full checkpoint
        with open(_ck, "wb") as _fh:
            _fh.write(b"done")
        STORE.push(_ck)
    del _bank, _opt
    gpu_free()
    return _rec

V24_RES, V24_PRED, V24_STATUS = {}, {}, {}
_t24 = time.time()
for _src, _dig in V24_SETS:
    _fam = f"{_src}{_dig}"
    _names = [f"G{_fam}{n}_s{k}-{min(k + CFG['bank'], V24_N) - 1}" for n in V24_NETS for k in range(0, V24_N, CFG["bank"])]
    _names50 = [f"G{_fam}cnn50_split{k}" for k in range(V24_N50)] if _dig == "0123" else []
    _have = {n for n in _names + _names50 if os.path.exists(os.path.join(V24_DIR, "units", n + ".npz"))}
    _need_pred = [n for n in V24_NETS if not os.path.exists(os.path.join(V24_DIR, f"predictions_{_fam}_{n}.json"))]
    if len(_have) == len(_names) + len(_names50) and not _need_pred:
        for _n in _names + _names50:
            V24_RES[_n] = dict(np.load(os.path.join(V24_DIR, "units", _n + ".npz")))
        for _net in V24_NETS:
            with open(os.path.join(V24_DIR, f"predictions_{_fam}_{_net}.json")) as _fh:
                V24_PRED[(_fam, _net)] = json.load(_fh)
        V24_STATUS[_fam] = "done"
        continue
    try:
        _data = v24_family_data(_src, _dig)
        _rng = np.random.default_rng(v24_seed(_fam, "boards"))
        _Xb, _yb = v24_boards(V24_PUZ, _rng)
        _I = v24_board_images(_Xb, _data["pools"], _rng)
        _Xt, _ybt = v24_boards(100, np.random.default_rng(v24_seed(_fam, "test")))
        _It = v24_board_images(_Xt, _data["tpools"], np.random.default_rng(v24_seed(_fam, "test-img")))
        _test = dict(x=_data["xte"], y=_data["yte"], bx=_data["xte"][torch.as_tensor(_It.reshape(-1), device=DEVICE)], by=_ybt)
        _pr = np.random.default_rng(v24_seed(_fam, "probe"))
        _pidx = np.concatenate([_pr.choice(_data["tpools"][c], 100, replace=False) for c in range(4)])
        _probe = (_data["xte"][torch.as_tensor(_pidx, device=DEVICE)], torch.as_tensor(np.repeat(np.arange(4), 100), device=DEVICE))
        print(f"== {_fam}: {len(_data['xtr'])} train / {len(_data['xte'])} test images, {len(_yb)} training boards", flush=True)
        _states = v24_states(_Xb, _yb)
        print("   partition states (level, negative Hessian eigenvalues, formula sum (|b|-1)^2): "
              + "; ".join(f"{'|'.join(''.join(map(str, b)) for b in s['partition'])} {s['level']:.3f} {s['n_neg']}/{s['index_formula']}" for s in _states), flush=True)
        for _net in V24_NETS:
            V24_PRED[(_fam, _net)] = v24_predict(_fam, _net, _data, _states)
        _jobs = []
        for _net in V24_NETS:
            for _k in range(0, V24_N, CFG["bank"]):
                _sd = [v24_seed(_fam, _net, j) for j in range(_k, min(_k + CFG["bank"], V24_N))]
                _jobs.append((f"G{_fam}{_net}_s{_k}-{_k + len(_sd) - 1}", _sd, _net, _I, _yb))
        for _k in range(len(_names50)):                 # benchmark arm: its own 50-puzzle split per network
            _r50 = np.random.default_rng(v24_seed(_fam, "split", _k))
            _X50, _y50 = v24_boards(50, _r50)
            _jobs.append((_names50[_k], [v24_seed(_fam, "cnn50", _k)], "cnn", v24_board_images(_X50, _data["pools"], _r50), _y50))
        for _name, _sd, _net, _Ij, _yj in _jobs:
            _path = os.path.join(V24_DIR, "units", _name + ".npz")
            if os.path.exists(_path):
                V24_RES[_name] = dict(np.load(_path))
                continue
            _t0 = time.time()
            _rec = v24_train_unit(_name, _sd, _net, _data, _Ij, _yj, _probe, _test, v24_seed(_name, "gen"))
            _rec["net"] = np.array(_net)
            np.savez_compressed(_path[:-4], **_rec)
            STORE.push(_path)
            V24_RES[_name] = _rec
            _line = (f"[v24] {_name}: {time.time() - _t0:.0f}s, learned {np.mean(_rec['perm_acc'] >= 0.9):.2f}, "
                     f"symbol acc {np.mean(_rec['perm_acc']):.3f}, test AuROC {np.mean(_rec['auroc']):.3f}, "
                     f"elapsed {(time.time() - _t24) / 60:.0f} min")
            print(_line, flush=True)
            with open(os.path.join(V24_DIR, "progress.log"), "a") as _fh:
                _fh.write(_line + "\n")
            STORE.push(os.path.join(V24_DIR, "progress.log"))
        V24_STATUS[_fam] = "done"
        del _data, _test, _probe
        gpu_free()
        trim_ram()
    except Exception as _e:                             # a dataset that cannot be downloaded is reported, not hidden
        V24_STATUS[_fam] = f"FAILED: {type(_e).__name__}: {str(_e)[:200]}"
        print(f"== {_fam}: {V24_STATUS[_fam]}", flush=True)
atomic_write(os.path.join(V24_DIR, "status.json"), json.dumps(V24_STATUS, indent=1))
STORE.push(os.path.join(V24_DIR, "status.json"))
print(f"{len(V24_RES)} v2.4 units available; families {V24_STATUS}; upload errors: {STORE.flush() or 'none'}")

In [ ]:
from scipy.stats import chisquare as _chisq24, spearmanr as _spear24, binomtest as _binom24

V24_NEUPSL = {"mnist": 0.88, "fashion": 0.74, "kmnist": 0.65}   # Augustine et al. 2022, 4x4, 50 puzzles, AuROC

def v24_sep_times(T, tsteps, thr=0.5):
    # step after which digits a and b never again share more than half their probability mass (probe class means)
    _out = {}
    for _a, _b in V24_PAIRS:
        _ov = np.minimum(T[:, _a].astype(np.float64), T[:, _b].astype(np.float64)).sum(-1)
        _above = np.flatnonzero(_ov >= thr)
        _out[f"{_a}{_b}"] = int(tsteps[_above[-1]] + 1) if len(_above) else 0
    return _out

def v24_nearest_state(C):
    _best = (None, 9.0)
    for _p in V24_PARTS:
        for _s in itertools.permutations(range(4)):
            _d = float(np.abs(C - v24_MP(_p)[:, list(_s)]).sum(1).max() / 2)
            if _d < _best[1]:
                _best = (_p, _d)
    return _best

V24_SEEDROWS = {}                                  # (family, net) -> per-seed rows, main arm
V24_ARM50 = {}
for _name, _r in V24_RES.items():
    _fam = _name[1:].split("_")[0]
    if "cnn50" in _fam:
        V24_ARM50.setdefault(_fam.replace("cnn50", ""), []).extend(_r["auroc"].tolist())
        continue
    _net = str(_r["net"])
    _fam = _fam[:-len(_net)]
    for _i in range(len(_r["seeds"])):
        _st = v24_sep_times(_r["traj"][_i], _r["tsteps"])
        _m = max(_st.values())
        V24_SEEDROWS.setdefault((_fam, _net), []).append(dict(
            seed=int(_r["seeds"][_i]), learned=bool(_r["perm_acc"][_i] >= 0.90), acc=float(_r["perm_acc"][_i]),
            perm=tuple(int(v) for v in _r["perm"][_i]), auroc=float(_r["auroc"][_i]), sep=_st,
            last=[k for k, v in _st.items() if v == _m], C=_r["C"][_i]))

_rows_all = [(k, r) for k, rs in V24_SEEDROWS.items() for r in rs]
_learned = [(k, r) for k, r in _rows_all if r["learned"]]
print(f"main arm: {len(_rows_all)} networks in {len(V24_SEEDROWS)} families; learned (symbol accuracy >= 0.90 under the "
      f"best relabelling) {len(_learned)}")
V24_TESTS = {}

# ---- H0 (exact, CPU): the 15 partition states, their levels and Morse indices --------------------------------------
_any = next(iter(V24_PRED.values()), None)
if _any is not None:
    _ok_idx = all(s["n_neg"] == s["index_formula"] for p in V24_PRED.values() for s in p["states"])
    print("H0 partition states: Hessian index = sum over blocks (|b|-1)^2 in every family: " + str(_ok_idx))
    print("   levels (first family): " + "; ".join(f"{'|'.join(''.join(map(str, b)) for b in s['partition'])} {s['level']:.3f}" for s in _any["states"]))
    V24_TESTS["H0_index_formula"] = bool(_ok_idx)

# ---- H1 orbit law over S4 -----------------------------------------------------------------------------------------
_perm_ix = {p: i for i, p in enumerate(itertools.permutations(range(4)))}
if len(_learned) >= 48:
    _cnt = np.bincount([_perm_ix[r["perm"]] for _, r in _learned], minlength=24)
    _p1 = float(_chisq24(_cnt).pvalue)
    _pf = {}
    for _k, _rs in V24_SEEDROWS.items():
        _l = [r for r in _rs if r["learned"]]
        if len(_l) >= 10:
            _pf[f"{_k[0]}/{_k[1]}"] = float(_chisq24(np.bincount([_perm_ix[r["perm"]] for r in _l], minlength=24)).pvalue)
    _h1 = bool(_p1 >= 0.01 and all(v >= 0.01 / max(len(_pf), 1) for v in _pf.values()))
    print(f"H1 orbit law: pooled chi-square over the 24 relabellings p = {_p1:.3f} (n = {len(_learned)}); "
          f"identity labelling {_cnt[0]}/{len(_learned)} (1/24 = {len(_learned) / 24:.1f}); smallest family p "
          f"{min(_pf.values()) if _pf else float('nan'):.3f} -> {'PASS' if _h1 else 'FAIL'}")
    V24_TESTS["H1"] = dict(p=_p1, family_p=_pf, counts=_cnt.tolist(), passed=_h1)
else:
    print(f"H1 orbit law: NOT TESTABLE YET ({len(_learned)} learned networks, need 48)")
    V24_TESTS["H1"] = dict(passed=None)

# ---- H2 order of separation, every learned network ----------------------------------------------------------------
# per network: Spearman correlation over the 6 digit pairs between the predicted escape rate of the pair-merged state
# and the step at which the pair separated (the law predicts negative: slow escape -> late separation)
_rho_seed, _fam_rho = [], {}
for _k, _rs in sorted(V24_SEEDROWS.items()):
    _pr = V24_PRED[_k]["pair_rate"]
    _keys = sorted(_pr)
    _vals = []
    for _r in _rs:
        if not _r["learned"]:
            continue
        _v = _spear24([_pr[q] for q in _keys], [_r["sep"][q] for q in _keys]).correlation
        if np.isfinite(_v):
            _vals.append(float(_v))
    _rho_seed += _vals
    if len(_vals) >= 10:
        _fam_rho[f"{_k[0]}/{_k[1]}"] = float(np.mean(_vals))
if len(_rho_seed) >= 40 and len(_fam_rho) >= 2:
    _mr = float(np.mean(_rho_seed))
    _nneg = int(sum(v < 0 for v in _rho_seed))
    _npos = int(sum(v > 0 for v in _rho_seed))
    _psign = float(_binom24(_nneg, _nneg + _npos, 0.5, alternative="greater").pvalue)
    _fok = sum(v <= -0.3 for v in _fam_rho.values())
    _h2 = bool(_mr <= -0.4 and _psign < 1e-6 and _fok >= 0.75 * len(_fam_rho))
    print(f"H2 order: per-network Spearman(escape rate, separation step): mean {_mr:+.2f}, negative in {_nneg}/{_nneg + _npos} "
          f"(sign test p = {_psign:.1e}); family means {({k: round(v, 2) for k, v in _fam_rho.items()})}, <= -0.3 in "
          f"{_fok}/{len(_fam_rho)} -> {'PASS' if _h2 else 'FAIL'}")
    V24_TESTS["H2"] = dict(mean_rho=_mr, negative=_nneg, positive=_npos, p=_psign, family_rho=_fam_rho, passed=_h2)
    _pixr = []                                          # baseline (reported): 1 - cosine of class-mean images as the rate
    for _k, _rs in V24_SEEDROWS.items():
        _pc = np.array(V24_PRED[_k]["pixel_cosine"])
        for _r in _rs:
            if _r["learned"]:
                _v = _spear24([1 - _pc[int(q[0]), int(q[1])] for q in sorted(_r["sep"])], [_r["sep"][q] for q in sorted(_r["sep"])]).correlation
                if np.isfinite(_v):
                    _pixr.append(float(_v))
    print(f"   baseline (reported): same statistic with pixel cosine instead of the kernel escape rate: mean {np.mean(_pixr):+.2f}")
    V24_TESTS["H2"]["pixel_mean_rho"] = float(np.mean(_pixr))
else:
    print("H2 order: NOT TESTABLE YET")
    V24_TESTS["H2"] = dict(passed=None)

# ---- H3 the predicted pair separates last, in decisive families ----------------------------------------------------
# decisive: the slowest pair-merged state escapes slower than the next one by at least 0.20 (rate units of cell G).
# In families with a smaller gap the law predicts a near-tie, so they are reported, not tested.
def v24_hits(rows, pair):
    _strict = sum(1 for r in rows if r["last"] == [pair])            # ties count as misses (conservative)
    _frac = sum(1.0 / len(r["last"]) for r in rows if pair in r["last"])
    return _strict, _frac
from collections import Counter as _Ctr24
_dec_s, _dec_n, _h3rows = 0, 0, {}
for _k, _rs in sorted(V24_SEEDROWS.items()):
    _l = [r for r in _rs if r["learned"]]
    if not _l:
        continue
    _pr = V24_PRED[_k]["pair_rate"]
    _ord = sorted(_pr, key=_pr.get)
    _gap = _pr[_ord[1]] - _pr[_ord[0]]
    _pp = _ord[0]
    _s, _f = v24_hits(_l, _pp)
    _modal = _Ctr24(p for r in _l for p in r["last"]).most_common()
    _mtop = [p for p, c in _modal if c == _modal[0][1]]
    _pix = V24_PRED[_k]["pixel_last_pair"]
    _ok = (_k[0], "mlp" if _k[1] == "cnn" else "cnn")
    _other = V24_PRED[_ok]["predicted_last_pair"] if _ok in V24_PRED else None
    _dec = bool(_gap >= 0.20)
    if _dec:
        _dec_s += _s
        _dec_n += len(_l)
    _h3rows[f"{_k[0]}/{_k[1]}"] = dict(n=len(_l), predicted=_pp, gap=_gap, decisive=_dec, hits=_s, hits_with_ties=_f,
                                       modal=_mtop, pixel_pair=_pix, pixel_hits=v24_hits(_l, _pix)[0],
                                       other_net_pair=_other, other_net_hits=v24_hits(_l, _other)[0] if _other else None)
    print(f"   {_k[0]:>12s}/{_k[1]}: predicted last pair {_pp} (gap {_gap:.3f}, {'decisive' if _dec else 'near-tie'}); last in "
          f"{_s}/{len(_l)} (with ties {_f:.1f}); modal {_mtop}; pixel baseline {_pix}: {_h3rows[f'{_k[0]}/{_k[1]}']['pixel_hits']}/{len(_l)}"
          + (f"; other net's pair {_other}: {_h3rows[f'{_k[0]}/{_k[1]}']['other_net_hits']}/{len(_l)}" if _other else ""))
if _dec_n >= 20:
    _rate3 = _dec_s / _dec_n
    _p3 = float(_binom24(_dec_s, _dec_n, 1 / 6, alternative="greater").pvalue)
    _h3 = bool(_rate3 >= 0.50 and _p3 < 1e-3)
    print(f"H3 last pair (decisive families): {_dec_s}/{_dec_n} = {_rate3:.2f} (chance 1/6), binomial p = {_p3:.1e} -> "
          f"{'PASS' if _h3 else 'FAIL'}")
    V24_TESTS["H3"] = dict(rate=_rate3, p=_p3, rows=_h3rows, passed=_h3)
else:
    print(f"H3 last pair: NOT TESTABLE YET ({_dec_n} learned networks in decisive families, need 20)")
    V24_TESTS["H3"] = dict(rows=_h3rows, passed=None)

# ---- reported, not tested ----------------------------------------------------------------------------------------
# saddle passage: share of learned networks whose probe map came within 0.25 of the predicted pair-merged state
# (the predicted pair shares two symbols, every other digit has its own) at some step before the end
_near = {}
for _nm, _r in V24_RES.items():
    if "cnn50" in _nm:
        continue
    _net = str(_r["net"])
    _k = (_nm[1:].split("_")[0][:-len(_net)], _net)
    _pp = [int(c) for c in V24_PRED[_k]["predicted_last_pair"]]
    _P = sorted([_pp] + [[d] for d in range(4) if d not in _pp])
    _Ms = [v24_MP(_P)[:, list(s)] for s in itertools.permutations(range(4))]
    for _i in range(len(_r["seeds"])):
        if _r["perm_acc"][_i] < 0.90:
            continue
        _T = _r["traj"][_i].astype(np.float64)
        _d = min(float((np.abs(_T - _M[None]).sum(-1).max(-1) / 2).min()) for _M in _Ms)
        _near.setdefault(f"{_k[0]}/{_k[1]}", []).append(_d < 0.25)
print("saddle passage (reported): learned networks that passed within 0.25 of the predicted pair-merged state: "
      + ", ".join(f"{k} {np.mean(v):.2f}" for k, v in sorted(_near.items())))
V24_TESTS["saddle_passage"] = {k: float(np.mean(v)) for k, v in _near.items()}
print("stuck networks (not learned): nearest partition state, distance, its level")
_stuck = [(k, r) for k, r in _rows_all if not r["learned"]]
V24_TESTS["stuck"] = []
for _k, _r in _stuck:
    _p, _d = v24_nearest_state(_r["C"])
    _lv = next(s["level"] for s in V24_PRED[_k]["states"] if s["partition"] == _p)
    V24_TESTS["stuck"].append(dict(family=f"{_k[0]}/{_k[1]}", seed=_r["seed"], acc=_r["acc"], partition=_p, dist=_d, level=_lv))
    print(f"   {_k[0]}/{_k[1]} seed {_r['seed']}: acc {_r['acc']:.3f}, nearest {'|'.join(''.join(map(str, b)) for b in _p)} "
          f"(distance {_d:.2f}, level {_lv:.3f})")
if not _stuck:
    print("   none")
print("benchmark (ViSudo-PC protocol regenerated in-cell, so comparable in protocol, not in the exact files): test AuROC")
V24_TESTS["benchmark"] = {}
for _src in sorted({k[0] for k in V24_SEEDROWS} | set(V24_ARM50)):
    _base = next((v for s, v in V24_NEUPSL.items() if _src.startswith(s) and _src.endswith("0123")), None)
    _main = {n: float(np.mean([r["auroc"] for r in V24_SEEDROWS.get((_src, n), [])])) for n in V24_NETS if (_src, n) in V24_SEEDROWS}
    _a50 = V24_ARM50.get(_src)
    V24_TESTS["benchmark"][_src] = dict(main=_main, arm50=None if not _a50 else [float(np.mean(_a50)), float(np.std(_a50))], neupsl=_base)
    print(f"   {_src}: {V24_PUZ} puzzles " + ", ".join(f"{n} {v:.3f}" for n, v in _main.items())
          + (f"; 50 puzzles (CNN, {len(_a50)} splits) {np.mean(_a50):.3f} +- {np.std(_a50):.3f}" if _a50 else "")
          + (f"; NeuPSL (50 puzzles) {_base}" if _base else ""))
_prim = [V24_TESTS[k]["passed"] for k in ("H1", "H2", "H3")]
_verdict = ("CONFIRMED" if all(p is True for p in _prim) else
            "NOT TESTABLE YET" if any(p is None for p in _prim) else "NOT CONFIRMED")
V24_TESTS["verdict"] = _verdict
print(f"VERDICT (H1 and H2 and H3): {_verdict}")
atomic_write(os.path.join(V24_DIR, "tests.json"), json.dumps(V24_TESTS, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o)))
STORE.push(os.path.join(V24_DIR, "tests.json"))

In [ ]:
V25_SETS = [s for s in os.environ.get("NSYM_V25_SETS", "mnist,fashion,kmnist").split(",") if s]
V25_NETS = ["cnn", "mlp"]
V25_N = int(os.environ.get("NSYM_V25_SEEDS", "40"))
V25_STEPS = int(os.environ.get("NSYM_V25_STEPS", "1500"))
V25_PUZ = int(os.environ.get("NSYM_V25_PUZZLES", "1000"))
V25_BS = 8                                                          # boards per step (81 images each)
V25_DENSE = 300
V25_NB = int(os.environ.get("NSYM_V25_NBENCH", "10"))              # benchmark splits per source
V25_PRE_STEPS = 300                                                 # transfer arm: Fashion-MNIST steps (batch 128, Adam 1e-3)
V25_DIR = os.path.join(RUN_DIR, "v25")
for _d25 in ("units", "ckpt"):
    os.makedirs(os.path.join(V25_DIR, _d25), exist_ok=True)

V25_UNITS = np.array([[r * 9 + c for c in range(9)] for r in range(9)] + [[r * 9 + c for r in range(9)] for c in range(9)]
                     + [[(br * 3 + i) * 9 + bc * 3 + j for i in range(3) for j in range(3)] for br in range(3) for bc in range(3)])
V25_PAIRS = list(itertools.combinations(range(9), 2))

def v25_perm_units(P):
    # P: (..., 81, 9) cell distributions -> (..., 27) P(each unit's 9 cells are all different), Ryser's formula
    _sub = torch.as_tensor(np.array([[(m >> j) & 1 for j in range(9)] for m in range(512)]), dtype=P.dtype, device=P.device)
    _sgn = torch.as_tensor(np.array([(-1) ** (9 - bin(m).count("1")) for m in range(512)]), dtype=P.dtype, device=P.device)
    _A = P[..., torch.as_tensor(V25_UNITS, device=P.device), :]
    return ((_A @ _sub.T).prod(-2) * _sgn).sum(-1)

def v25_logvalid(P):
    return torch.log(v25_perm_units(P).clamp_min(1e-300)).sum(-1)

def v25_nll(P, y):
    _lv = v25_logvalid(P)
    return -(y * _lv + (1 - y) * torch.log(-torch.expm1(_lv.clamp_max(-1e-12))))

def v25_grid(rng):
    # a valid grid: base pattern, random band / stack / row / column shuffles, transpose, symbol relabelling
    _base = np.array([[(3 * (r % 3) + r // 3 + c) % 9 for c in range(9)] for r in range(9)])
    def _sh():
        return np.concatenate([b * 3 + rng.permutation(3) for b in rng.permutation(3)])
    _g = _base[_sh()][:, _sh()]
    if rng.random() < 0.5:
        _g = _g.T
    return rng.permutation(9)[_g].reshape(-1)

def v25_is_valid(g):
    return all(len(set(g[u].tolist())) == 9 for u in V25_UNITS)

def v25_boards(n_pos, rng, chance=0.5):
    _pos = np.array([v25_grid(rng) for _ in range(n_pos)])
    _neg = []
    while len(_neg) < n_pos:
        _g = v25_grid(rng)
        _swap = rng.random() < 0.5
        while True:
            if _swap:
                _i, _j = rng.choice(81, 2, replace=False)
                _g[_i], _g[_j] = _g[_j], _g[_i]
            else:
                _i = rng.integers(81)
                _g[_i] = (_g[_i] + rng.integers(1, 9)) % 9
            if rng.random() >= chance:
                break
        if not v25_is_valid(_g):
            _neg.append(_g)
    return np.concatenate([_pos, np.array(_neg)]), np.r_[np.ones(n_pos), np.zeros(n_pos)]

def v25_logits(p, x, cnn):
    # forward pass of one RefNet given its parameters as a dict; x: (B, 1, 28, 28) -> (B, 10)
    if cnn:
        h = F.max_pool2d(F.relu(F.conv2d(x, p["c1.weight"], p["c1.bias"])), 2)
        h = F.max_pool2d(F.relu(F.conv2d(h, p["c2.weight"], p["c2.bias"])), 2).flatten(1)
    else:
        h = x.flatten(1)
    h = F.relu(h @ p["fc1.weight"].T + p["fc1.bias"])
    return h @ p["fc2.weight"].T + p["fc2.bias"]

def v25_fresh_head(seed):
    torch.manual_seed(int(seed) + 100)
    _h = nn.Linear(128, N_CLASSES)
    return _h.weight.detach().to(DEVICE), _h.bias.detach().to(DEVICE)

def v25_pretrain(params, forward, pre, seed):
    # transfer arm: V25_PRE_STEPS Adam steps of 10-class cross-entropy on Fashion-MNIST, batch 128
    _x, _y = pre
    _g = torch.Generator().manual_seed(int(seed))
    _o = torch.optim.Adam(params, lr=1e-3)
    for _ in range(V25_PRE_STEPS):
        _b = torch.randint(0, len(_y), (128,), generator=_g).to(_x.device)
        _l = forward(_x[_b], _y[_b])
        _o.zero_grad(set_to_none=True)
        _l.backward()
        _o.step()

def v25_kernel(net, xtr, pools, n_init=6, per=30, pre=None):
    # class tangent kernel at initialisation over the 9 output units used (torch seeds 0..5, numpy seed 0); with pre,
    # at the Fashion-pretrained trunk with a fresh head (the transfer arm's starting point)
    from torch.func import vmap as _vmap, jacrev as _jacrev
    _rng = np.random.default_rng(0)
    _idx = [_rng.choice(pools[c], per, replace=False) for c in range(9)]
    _K = np.zeros((9, 9))
    for _s in range(n_init):
        torch.manual_seed(_s)
        _ref = RefNet(net == "cnn", int(np.prod(xtr.shape[1:]))).to(DEVICE)
        if pre is not None:
            _pp = dict(_ref.named_parameters())
            v25_pretrain(list(_pp.values()), lambda x, y: F.cross_entropy(v25_logits(_pp, x, net == "cnn"), y), pre, _s)
            _w, _bb = v25_fresh_head(_s)
            with torch.no_grad():
                _ref.fc2.weight.copy_(_w); _ref.fc2.bias.copy_(_bb)
        _p = {k: v.detach() for k, v in _ref.named_parameters()}
        def _z(p, x):
            return v25_logits(p, x[None], net == "cnn")[0, :9]
        _means = []
        for _c in range(9):
            _J = _vmap(_jacrev(_z), in_dims=(None, 0))(_p, xtr[torch.as_tensor(_idx[_c], device=xtr.device)].float())
            _means.append(torch.cat([v.reshape(per, 9, -1) for v in _J.values()], -1).mean(0).double())
            del _J
        _Mn = torch.stack(_means)
        _K += (torch.einsum("aop,bop->ab", _Mn, _Mn) / 9).cpu().numpy()
        del _Mn, _means
        gpu_free()
    _K /= n_init
    _dg = np.sqrt(np.diag(_K))
    return _K / np.outer(_dg, _dg)

def v25_predict(src, net, data, pre=None):
    # pair escape rate of a merged pair is |mu| (1 - K_ab), with mu the same for every pair (Theorem E), so the order is
    # fixed by 1 - K_ab; written once, before any network of the family trains
    _path = os.path.join(V25_DIR, f"predictions_{src}_{net}.json")
    if os.path.exists(_path):
        with open(_path) as _fh:
            return json.load(_fh)
    _trained = [f for f in os.listdir(os.path.join(V25_DIR, "units")) if f.startswith(f"I{src}{net}_")]
    _Kn = v25_kernel("cnn" if net == "cnnpre" else net, data["xtr"], data["pools"], pre=pre)
    _rate = {f"{a}{b}": float(1 - _Kn[a, b]) for a, b in V25_PAIRS}
    _order = sorted(_rate, key=_rate.get)
    _mu = np.stack([data["xtr"][torch.as_tensor(data["pools"][c][:500], device=data["xtr"].device)].mean(0).flatten().double().cpu().numpy()
                    for c in range(9)])
    _pix = _mu @ _mu.T / np.outer(np.linalg.norm(_mu, axis=1), np.linalg.norm(_mu, axis=1))
    _pred = dict(source=src, net=net, created=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                 units_trained_before=len(_trained), kernel=_Kn.tolist(), pair_rate=_rate,
                 predicted_order_slow_to_fast=_order, predicted_top3=_order[:3], pixel_cosine=_pix.tolist())
    _txt = json.dumps(_pred, indent=1)
    atomic_write(_path, _txt)
    STORE.push(_path)
    STORE.flush()
    print(f"   predictions {src}/{net}: sha256 {hashlib.sha256(_txt.encode()).hexdigest()[:16]}; predicted slowest pairs "
          f"{_order[:3]} (1-K {[round(_rate[q], 4) for q in _order[:3]]})", flush=True)
    return _pred

def v25_bank(seeds, net, pre=None):
    _was = CFG["net"]
    CFG["net"] = net
    try:
        _b = make_bank(seeds)
    finally:
        CFG["net"] = _was
    if pre is not None:                           # transfer arm: every seed's network trains on Fashion, then a fresh head
        v25_pretrain(list(_b.parameters()), lambda x, y: F.cross_entropy(_b(x).flatten(0, 1), y.repeat(_b.S)), pre, seeds[0])
        with torch.no_grad():
            for _i, _sd in enumerate(seeds):
                _w, _bb = v25_fresh_head(_sd)
                _b.W2[_i].copy_(_w); _b.b2[_i].copy_(_bb)
    with torch.no_grad():
        _b.W2 = nn.Parameter(_b.W2[:, :9].clone())
        _b.b2 = nn.Parameter(_b.b2[:, :9].clone())
    return _b

def v25_auroc(sc, lab):
    _r = np.argsort(np.argsort(sc)) + 1.0
    _n1 = lab.sum()
    return float((_r[lab == 1].sum() - _n1 * (_n1 + 1) / 2) / (_n1 * (len(lab) - _n1)))

def v25_board_scores(bank, x, nb):
    # log P(valid) of nb boards whose 81 images are x (nb * 81, ...), per seed of the bank
    with torch.no_grad():
        _out = []
        for _i in range(0, nb, 25):
            _xx = x[_i * 81:min(nb, _i + 25) * 81]
            _P = F.softmax(bank(_xx).double(), -1).reshape(bank.S, -1, 81, 9)
            _out.append(v25_logvalid(_P))
        return torch.cat(_out, 1).cpu().numpy()

def v25_train_unit(name, seeds, net, data, I, yb, probe, test, gen_seed, val=None, pre=None):
    _ck = os.path.join(V25_DIR, "ckpt", name + ".pt")
    _bank = v25_bank(seeds, net, pre)
    _opt = torch.optim.Adam(_bank.parameters(), lr=CFG["lr"])
    _g = torch.Generator().manual_seed(gen_seed)
    _It, _yt = torch.as_tensor(I, device=DEVICE), torch.as_tensor(yb, dtype=torch.float64, device=DEVICE)
    _start, _traj, _tsteps, _loss = 0, [], [], []
    if os.path.exists(_ck) and os.path.getsize(_ck) > 1000:
        _st = torch.load(_ck, map_location=DEVICE, weights_only=False)
        _bank.load_state_dict(_st["bank"]); _opt.load_state_dict(_st["opt"]); _g.set_state(_st["gen"].cpu())
        _start, _traj, _tsteps, _loss = _st["step"], _st["traj"], _st["tsteps"], _st["loss"]
    def _probe():
        with torch.no_grad():
            _p = F.softmax(_bank(probe[0]).float(), -1)
            return torch.stack([_p[:, probe[1] == c].mean(1) for c in range(9)], 1).cpu().numpy().astype(np.float16)
    for _s in range(_start, V25_STEPS):
        if probe is not None and (_s < V25_DENSE or _s % 10 == 0):
            _traj.append(_probe()); _tsteps.append(_s)
        _b = torch.randint(0, len(_yt), (V25_BS,), generator=_g).to(DEVICE)
        _P = F.softmax(_bank(data["xtr"][_It[_b].reshape(-1)]).double(), -1).reshape(len(seeds), V25_BS, 81, 9)
        _l = v25_nll(_P, _yt[_b][None]).mean(1)
        _opt.zero_grad(set_to_none=True)
        _l.sum().backward()
        _opt.step()
        _loss.append(_l.detach().cpu().numpy().astype(np.float16))
        if (_s + 1) % 500 == 0 and _s + 1 < V25_STEPS:
            torch.save(dict(bank=_bank.state_dict(), opt=_opt.state_dict(), gen=_g.get_state(), step=_s + 1,
                            traj=_traj, tsteps=_tsteps, loss=_loss), _ck)
            STORE.push(_ck)
    if probe is not None:
        _traj.append(_probe()); _tsteps.append(V25_STEPS)
    from scipy.optimize import linear_sum_assignment as _lsa25
    with torch.no_grad():
        _pt = torch.cat([F.softmax(_bank(test["x"][i:i + 2000]).float(), -1) for i in range(0, len(test["x"]), 2000)], 1)
        _C = torch.stack([_pt[:, torch.as_tensor(test["y"] == c, device=DEVICE)].mean(1) for c in range(9)], 1).cpu().numpy()
        _hard = _pt.argmax(-1).cpu().numpy()
    _perm, _pacc = [], []
    for _s in range(len(seeds)):
        _cnt = np.zeros((9, 9))
        np.add.at(_cnt, (test["y"], _hard[_s]), 1)
        _r, _c = _lsa25(-_cnt)
        _perm.append(_c[np.argsort(_r)]); _pacc.append(_cnt[_r, _c].sum() / len(test["y"]))
    _sc = v25_board_scores(_bank, test["bx"], len(test["by"]))
    _rec = dict(C=_C, perm=np.array(_perm), perm_acc=np.array(_pacc), seeds=np.array(seeds),
                auroc=np.array([v25_auroc(_sc[s], test["by"]) for s in range(len(seeds))]), loss=np.stack(_loss, 1))
    if probe is not None:
        _rec.update(traj=np.stack(_traj, 1), tsteps=np.array(_tsteps))
    if val is not None:                           # benchmark arm: accuracy at the validation-chosen threshold
        _sv = v25_board_scores(_bank, val["bx"], len(val["by"]))
        _thr = []
        for _s in range(len(seeds)):
            _cand = np.sort(_sv[_s])
            _thr.append(_cand[int(np.argmax([np.mean((_sv[_s] > c) == (val["by"] == 1)) for c in _cand]))])
        _rec["puzzle_acc"] = np.array([np.mean((_sc[s] > _thr[s]) == (test["by"] == 1)) for s in range(len(seeds))])
    if os.path.exists(_ck):
        with open(_ck, "wb") as _fh:
            _fh.write(b"done")
        STORE.push(_ck)
    del _bank, _opt
    gpu_free()
    return _rec

def v25_family_data(src):
    if src == "mnist":
        _ptr = [np.asarray(PTR["pools"][c]) for c in range(9)]
        _xtr = torch.cat([XTR[torch.as_tensor(p, device=XTR.device)] for p in _ptr]).float()
        _te = [np.flatnonzero(YTE_NP == c) for c in range(9)]
        _xte = torch.cat([XTE[torch.as_tensor(p, device=XTE.device)] for p in _te]).float()
    else:
        _d = v24_load(src)
        _ptr = [np.flatnonzero(_d["ytr"] == c) for c in range(9)]
        _xtr = torch.as_tensor(np.concatenate([_d["xtr"][p] for p in _ptr]), device=DEVICE)
        _te = [np.flatnonzero(_d["yte"] == c) for c in range(9)]
        _xte = torch.as_tensor(np.concatenate([_d["xte"][p] for p in _te]), device=DEVICE)
    _cut = np.cumsum([0] + [len(p) for p in _ptr])
    _cte = np.cumsum([0] + [len(p) for p in _te])
    return dict(xtr=_xtr.to(DEVICE), pools=[np.arange(_cut[i], _cut[i + 1]) for i in range(9)],
                xte=_xte.to(DEVICE), tpools=[np.arange(_cte[i], _cte[i + 1]) for i in range(9)],
                yte=np.repeat(np.arange(9), np.diff(_cte)))

def v25_images(Xb, pools, rng):
    return np.stack([[pools[s][rng.integers(len(pools[s]))] for s in g] for g in Xb]).astype(np.int64)

V25_RES, V25_PRED, V25_STATUS = {}, {}, {}
_t25 = time.time()
for _src in V25_SETS:
    _arms = V25_NETS + (["cnnpre"] if _src == "mnist" else [])
    _names = [f"I{_src}{n}_s{k}-{min(k + CFG['bank'], V25_N) - 1}" for n in _arms for k in range(0, V25_N, CFG["bank"])]
    _namesb = [f"I{_src}bench_split{k}" for k in range(V25_NB)]
    _all = _names + _namesb
    _have = [n for n in _all if os.path.exists(os.path.join(V25_DIR, "units", n + ".npz"))]
    _needp = [n for n in _arms if not os.path.exists(os.path.join(V25_DIR, f"predictions_{_src}_{n}.json"))]
    if len(_have) == len(_all) and not _needp:
        for _n in _all:
            V25_RES[_n] = dict(np.load(os.path.join(V25_DIR, "units", _n + ".npz")))
        for _net in _arms:
            V25_PRED[(_src, _net)] = v25_predict(_src, _net, None)
        V25_STATUS[_src] = "done"
        continue
    try:
        _data = v25_family_data(_src)
        _rng = np.random.default_rng(v24_seed("v25", _src, "boards"))
        _Xb, _yb = v25_boards(V25_PUZ, _rng)
        _I = v25_images(_Xb, _data["pools"], _rng)
        _trng = np.random.default_rng(v24_seed("v25", _src, "test"))
        _Xt, _ybt = v25_boards(100, _trng)
        _It = v25_images(_Xt, _data["tpools"], _trng)
        _test = dict(x=_data["xte"], y=_data["yte"], bx=_data["xte"][torch.as_tensor(_It.reshape(-1), device=DEVICE)], by=_ybt)
        _pr = np.random.default_rng(v24_seed("v25", _src, "probe"))
        _pidx = np.concatenate([_pr.choice(_data["tpools"][c], 50, replace=False) for c in range(9)])
        _probe = (_data["xte"][torch.as_tensor(_pidx, device=DEVICE)], torch.as_tensor(np.repeat(np.arange(9), 50), device=DEVICE))
        print(f"== {_src} 9x9: {len(_data['xtr'])} train / {len(_data['xte'])} test images, {len(_yb)} training boards", flush=True)
        _pre = None
        if "cnnpre" in _arms:
            _fd = v24_load("fashion")
            _pre = (torch.as_tensor(_fd["xtr"], device=DEVICE), torch.as_tensor(_fd["ytr"], device=DEVICE))
            del _fd
        for _net in _arms:
            V25_PRED[(_src, _net)] = v25_predict(_src, _net, _data, _pre if _net == "cnnpre" else None)
        _jobs = []
        for _net in _arms:
            for _k in range(0, V25_N, CFG["bank"]):
                _sd = [v24_seed("v25", _src, _net, j) for j in range(_k, min(_k + CFG["bank"], V25_N))]
                _jobs.append((f"I{_src}{_net}_s{_k}-{_k + len(_sd) - 1}", _sd, "mlp" if _net == "mlp" else "cnn", _I, _yb, _probe, None,
                              _pre if _net == "cnnpre" else None, _net))
        for _k in range(V25_NB):                     # A-NeSI protocol size: 100 + 100 training puzzles per split
            _r5 = np.random.default_rng(v24_seed("v25", _src, "split", _k))
            _Xs, _ys = v25_boards(100, _r5)
            _Is = v25_images(_Xs, _data["pools"], _r5)
            _used = set(_Is.reshape(-1).tolist())
            _vpool = [np.array([i for i in p if i not in _used]) for p in _data["pools"]]
            _Xv, _yv = v25_boards(100, _r5)
            _Iv = v25_images(_Xv, _vpool, _r5)
            _val = dict(bx=_data["xtr"][torch.as_tensor(_Iv.reshape(-1), device=DEVICE)], by=_yv)
            _jobs.append((_namesb[_k], [v24_seed("v25", _src, "bench", _k)], "cnn", _Is, _ys, None, _val, None, "cnn"))
        for _name, _sd, _net, _Ij, _yj, _pb, _val, _pj, _lab in _jobs:
            _path = os.path.join(V25_DIR, "units", _name + ".npz")
            if os.path.exists(_path):
                V25_RES[_name] = dict(np.load(_path))
                continue
            _t0 = time.time()
            _rec = v25_train_unit(_name, _sd, _net, _data, _Ij, _yj, _pb, _test, v24_seed(_name, "gen"), _val, _pj)
            _rec["net"] = np.array(_lab)
            np.savez_compressed(_path[:-4], **_rec)
            STORE.push(_path)
            V25_RES[_name] = _rec
            _line = (f"[v25] {_name}: {time.time() - _t0:.0f}s, learned {np.mean(_rec['perm_acc'] >= 0.9):.2f}, symbol acc "
                     f"{np.mean(_rec['perm_acc']):.3f}, test AuROC {np.mean(_rec['auroc']):.3f}"
                     + (f", puzzle accuracy {np.mean(_rec['puzzle_acc']):.3f}" if "puzzle_acc" in _rec else "")
                     + f", elapsed {(time.time() - _t25) / 60:.0f} min")
            print(_line, flush=True)
            with open(os.path.join(V25_DIR, "progress.log"), "a") as _fh:
                _fh.write(_line + "\n")
            STORE.push(os.path.join(V25_DIR, "progress.log"))
        V25_STATUS[_src] = "done"
        del _data, _test, _probe, _pre
        gpu_free()
        trim_ram()
    except Exception as _e:
        V25_STATUS[_src] = f"FAILED: {type(_e).__name__}: {str(_e)[:200]}"
        print(f"== {_src} 9x9: {V25_STATUS[_src]}", flush=True)
atomic_write(os.path.join(V25_DIR, "status.json"), json.dumps(V25_STATUS, indent=1))
STORE.push(os.path.join(V25_DIR, "status.json"))
print(f"{len(V25_RES)} v2.5 units available; sources {V25_STATUS}; upload errors: {STORE.flush() or 'none'}")

In [ ]:
from scipy.stats import chi2 as _chi2_25, spearmanr as _spear25, binomtest as _binom25
from collections import Counter as _Ctr25

V25_ANESI = {"mnist": 0.6215}       # best published 9x9 ViSudo-PC accuracy, 100 + 100 training puzzles (A-NeSI, 2023)

def v25_sep_times(T, tsteps, thr=0.5):
    # step after which digits a and b never again share more than half their probability mass (probe class means)
    _out = {}
    for _a, _b in V25_PAIRS:
        _ov = np.minimum(T[:, _a].astype(np.float64), T[:, _b].astype(np.float64)).sum(-1)
        _above = np.flatnonzero(_ov >= thr)
        _out[f"{_a}{_b}"] = int(tsteps[_above[-1]] + 1) if len(_above) else 0
    return _out

def v25_final_blocks(Tf, thr=0.5):
    # digits still merged at the end: connected components of the graph "share more than half their mass"
    _par = list(range(9))
    def _f(i):
        while _par[i] != i:
            i = _par[i]
        return i
    for _a, _b in V25_PAIRS:
        if np.minimum(Tf[_a], Tf[_b]).sum() >= thr:
            _par[_f(_a)] = _f(_b)
    _bl = {}
    for _d in range(9):
        _bl.setdefault(_f(_d), []).append(_d)
    return sorted(_bl.values())

V25_ROWS, V25_BENCH = {}, {}
for _name, _r in V25_RES.items():
    _fam = _name[1:].split("_")[0]
    if _fam.endswith("bench"):
        V25_BENCH.setdefault(_fam[:-5], []).append((float(_r["puzzle_acc"][0]), float(_r["auroc"][0]), float(_r["perm_acc"][0])))
        continue
    _net = str(_r["net"])
    _fam = _fam[:-len(_net)]
    for _i in range(len(_r["seeds"])):
        _T = _r["traj"][_i]
        _blocks = v25_final_blocks(_T[-1].astype(np.float64))
        _st = v25_sep_times(_T, _r["tsteps"])
        _m = max(_st.values())
        V25_ROWS.setdefault((_fam, _net), []).append(dict(
            seed=int(_r["seeds"][_i]), resolved=len(_blocks) == 9, blocks=_blocks, acc=float(_r["perm_acc"][_i]),
            perm=[int(v) for v in _r["perm"][_i]], auroc=float(_r["auroc"][_i]), sep=_st,
            last=[k for k, v in _st.items() if v == _m]))

V25_TRANSFER = {k: V25_ROWS.pop(k) for k in [k for k in V25_ROWS if k[1] == "cnnpre"]}
_rows25 = [(k, r) for k, rs in V25_ROWS.items() for r in rs]
_res25 = [(k, r) for k, r in _rows25 if r["resolved"]]
print(f"main arm: {len(_rows25)} networks in {len(V25_ROWS)} families; resolved (all 36 digit pairs separated at the end) "
      f"{len(_res25)}; symbol accuracy under the best relabelling: mean {np.mean([r['acc'] for _, r in _res25]) if _res25 else float('nan'):.3f}")
V25_TESTS = {}

# ---- J1 orbit law over S9 (362880 relabellings): the 9 x 9 digit-to-symbol count table is uniform ------------------
# Under the law each resolved network's relabelling is a uniform random permutation, so the centred table lies in the
# 64-dimensional space with zero row and column sums, with isotropic covariance there (std x std is irreducible).
# Hence (8/9) * Pearson X^2 -> chi-square with 64 degrees of freedom (checked by simulation: size 0.0103 at 0.01).
def v25_orbit_p(perms):
    _C = np.zeros((9, 9))
    for _p in perms:
        _C[np.arange(9), _p] += 1
    _E = len(perms) / 9
    return float(_chi2_25.sf(((_C - _E) ** 2 / _E).sum() * 8 / 9, 64)), _C

if len(_res25) >= 60:
    _p1, _C1 = v25_orbit_p([r["perm"] for _, r in _res25])
    _pf = {}
    for _k, _rs in V25_ROWS.items():
        _l = [r["perm"] for r in _rs if r["resolved"]]
        if len(_l) >= 15:
            _pf[f"{_k[0]}/{_k[1]}"] = v25_orbit_p(_l)[0]
    _j1 = bool(_p1 >= 0.01 and all(v >= 0.01 / max(len(_pf), 1) for v in _pf.values()))
    _idn = int(sum(1 for _, r in _res25 if r["perm"] == list(range(9))))
    print(f"J1 orbit law: pooled p = {_p1:.3f} (n = {len(_res25)}); digit d on symbol d in {int(np.trace(_C1))} of "
          f"{9 * len(_res25)} digit slots (1/9 = {len(_res25):.0f}); identity labelling {_idn}; smallest family p "
          f"{min(_pf.values()) if _pf else float('nan'):.3f} -> {'PASS' if _j1 else 'FAIL'}")
    V25_TESTS["J1"] = dict(p=_p1, family_p=_pf, table=_C1.tolist(), identity=_idn, passed=_j1)
else:
    print(f"J1 orbit law: NOT TESTABLE YET ({len(_res25)} resolved networks, need 60)")
    V25_TESTS["J1"] = dict(passed=None)

# ---- J2 order of separation over the 36 pairs, every resolved network -----------------------------------------------
def v25_rhos(rate_of, rows):
    _v = []
    for _r in rows:
        _keys = sorted(_r["sep"])
        _x = _spear25([rate_of(q) for q in _keys], [_r["sep"][q] for q in _keys]).correlation
        if np.isfinite(_x):
            _v.append(float(_x))
    return _v

_rho, _fam_rho, _pix_rho, _oth_rho = [], {}, [], []
for _k, _rs in sorted(V25_ROWS.items()):
    _l = [r for r in _rs if r["resolved"]]
    _pr = V25_PRED[_k]["pair_rate"]
    _pc = np.array(V25_PRED[_k]["pixel_cosine"])
    _v = v25_rhos(lambda q: _pr[q], _l)
    _rho += _v
    _pix_rho += v25_rhos(lambda q: 1 - _pc[int(q[0]), int(q[1])], _l)
    _ok = (_k[0], "mlp" if _k[1] == "cnn" else "cnn")
    if _ok in V25_PRED:
        _po = V25_PRED[_ok]["pair_rate"]
        _oth_rho += v25_rhos(lambda q: _po[q], _l)
    if len(_v) >= 10:
        _fam_rho[f"{_k[0]}/{_k[1]}"] = float(np.mean(_v))
if len(_rho) >= 40 and len(_fam_rho) >= 2:
    _mr = float(np.mean(_rho))
    _nn, _np_ = int(sum(v < 0 for v in _rho)), int(sum(v > 0 for v in _rho))
    _ps = float(_binom25(_nn, _nn + _np_, 0.5, alternative="greater").pvalue)
    _fok = sum(v <= -0.3 for v in _fam_rho.values())
    _j2 = bool(_mr <= -0.4 and _ps < 1e-6 and _fok >= 0.75 * len(_fam_rho))
    print(f"J2 order: per-network Spearman(escape rate, separation step) over 36 pairs: mean {_mr:+.2f}, negative in "
          f"{_nn}/{_nn + _np_} (sign test p = {_ps:.1e}); family means {({k: round(v, 2) for k, v in _fam_rho.items()})}, "
          f"<= -0.3 in {_fok}/{len(_fam_rho)} -> {'PASS' if _j2 else 'FAIL'}")
    print(f"   baselines (reported): pixel cosine of class-mean images {np.mean(_pix_rho):+.2f}; the other architecture's "
          f"kernel {np.mean(_oth_rho) if _oth_rho else float('nan'):+.2f}")
    V25_TESTS["J2"] = dict(mean_rho=_mr, negative=_nn, positive=_np_, p=_ps, family_rho=_fam_rho, pixel_mean_rho=float(np.mean(_pix_rho)),
                           other_net_mean_rho=float(np.mean(_oth_rho)) if _oth_rho else None, passed=_j2)
else:
    print("J2 order: NOT TESTABLE YET")
    V25_TESTS["J2"] = dict(passed=None)

# ---- J3 the last pair to separate is one of the 3 predicted slowest (chance 3/36) ---------------------------------
# tied last pairs share the credit: a network scores |last pairs in the predicted 3| / |last pairs|
_hit, _n3, _j3rows = 0.0, 0, {}
for _k, _rs in sorted(V25_ROWS.items()):
    _l = [r for r in _rs if r["resolved"]]
    if not _l:
        continue
    _top = V25_PRED[_k]["predicted_top3"]
    _pc = np.array(V25_PRED[_k]["pixel_cosine"])
    _ptop = sorted((f"{a}{b}" for a, b in V25_PAIRS), key=lambda q: 1 - _pc[int(q[0]), int(q[1])])[:3]
    _h = sum(len(set(r["last"]) & set(_top)) / len(r["last"]) for r in _l)
    _hp = sum(len(set(r["last"]) & set(_ptop)) / len(r["last"]) for r in _l)
    _modal = _Ctr25(p for r in _l for p in r["last"]).most_common(3)
    _hit += _h
    _n3 += len(_l)
    _j3rows[f"{_k[0]}/{_k[1]}"] = dict(n=len(_l), predicted=_top, hits=_h, pixel_top3=_ptop, pixel_hits=_hp, modal=_modal)
    print(f"   {_k[0]:>8s}/{_k[1]}: predicted slowest {_top}; last pair among them in {_h:.1f}/{len(_l)}; modal last pairs "
          f"{_modal}; pixel top 3 {_ptop}: {_hp:.1f}/{len(_l)}")
if _n3 >= 40:
    _rate3 = _hit / _n3
    _p3 = float(_binom25(int(np.floor(_hit)), _n3, 3 / 36, alternative="greater").pvalue)
    _j3 = bool(_rate3 >= 0.40 and _p3 < 1e-6)
    print(f"J3 last pair in the predicted 3: {_hit:.1f}/{_n3} = {_rate3:.2f} (chance 3/36 = 0.08), binomial p = {_p3:.1e} -> "
          f"{'PASS' if _j3 else 'FAIL'}")
    V25_TESTS["J3"] = dict(rate=_rate3, p=_p3, rows=_j3rows, passed=_j3)
else:
    print(f"J3: NOT TESTABLE YET ({_n3} resolved networks, need 40)")
    V25_TESTS["J3"] = dict(rows=_j3rows, passed=None)

# ---- reported, not tested ----------------------------------------------------------------------------------------
print("unresolved networks (some digits still share a symbol): merged blocks; are they the kernel's closest digits?")
V25_TESTS["unresolved"] = []
for _k, _r in _rows25:
    if _r["resolved"]:
        continue
    _pr = V25_PRED[_k]["pair_rate"]
    _mp = [f"{a}{b}" for bl in _r["blocks"] for a, b in itertools.combinations(bl, 2)]
    _rank = sorted(_pr, key=_pr.get)
    _rk = [_rank.index(q) + 1 for q in _mp]
    V25_TESTS["unresolved"].append(dict(family=f"{_k[0]}/{_k[1]}", seed=_r["seed"], acc=_r["acc"], blocks=_r["blocks"], pair_ranks=_rk))
    print(f"   {_k[0]}/{_k[1]} seed {_r['seed']}: acc {_r['acc']:.3f}, merged {[b for b in _r['blocks'] if len(b) > 1]}, "
          f"rank of merged pairs in the predicted slow-to-fast order (1 = slowest of 36) {_rk}")
if not V25_TESTS["unresolved"]:
    print("   none")
print("benchmark, A-NeSI protocol size (100 + 100 training puzzles, CNN, threshold set on validation puzzles, test on 100 + 100 "
      "puzzles from test images; protocol regenerated in-cell, so comparable in protocol, not in the exact files)")
V25_TESTS["benchmark"] = {}
for _src in sorted(set(V25_BENCH) | {k[0] for k in V25_ROWS}):
    _b = np.array(V25_BENCH.get(_src, []))
    _main = {n: float(np.mean([r["auroc"] for r in V25_ROWS.get((_src, n), [])])) for n in V25_NETS if (_src, n) in V25_ROWS}
    _ref = V25_ANESI.get(_src)
    V25_TESTS["benchmark"][_src] = dict(splits=len(_b), puzzle_acc=_b[:, 0].tolist() if len(_b) else [],
                                        auroc=_b[:, 1].tolist() if len(_b) else [], main_auroc=_main, anesi=_ref)
    print(f"   {_src}: " + (f"puzzle accuracy {_b[:, 0].mean():.3f} +- {_b[:, 0].std():.3f} over {len(_b)} splits (min {_b[:, 0].min():.3f}), "
                             f"AuROC {_b[:, 1].mean():.3f}, symbol accuracy {_b[:, 2].mean():.3f}" if len(_b) else "no benchmark units")
          + (f"; A-NeSI {_ref}" if _ref else "") + f"; {V25_PUZ}-puzzle arm AuROC " + ", ".join(f"{n} {v:.3f}" for n, v in _main.items()))
# ---- J4 transfer arm: pretraining moves the order the way it moves the kernel ------------------------------------
# MNIST CNNs whose trunk first trained on Fashion-MNIST see the same images as the main MNIST CNN family, so pixel
# similarity predicts no change in the order. The law predicts that pairs the pretraining made more kernel-similar
# separate later. Statistic: Spearman over the 36 pairs between the change in escape rate (transfer minus plain, both
# from cell I's predictions) and the change in mean separation rank (rank of a pair's separation step within its
# network, divided by 36). Null: networks shuffled between the two arms (10000 shuffles).
from scipy.stats import rankdata as _rankd25
_tr = [r for rs in V25_TRANSFER.values() for r in rs if r["resolved"]]
_pl = [r for r in V25_ROWS.get(("mnist", "cnn"), []) if r["resolved"]]
if ("mnist", "cnnpre") in V25_PRED and ("mnist", "cnn") in V25_PRED and len(_tr) >= 20 and len(_pl) >= 20:
    _kp, _k0 = V25_PRED[("mnist", "cnnpre")], V25_PRED[("mnist", "cnn")]
    _keys = sorted(_kp["pair_rate"])
    _drate = np.array([_kp["pair_rate"][q] - _k0["pair_rate"][q] for q in _keys])
    _R = np.array([_rankd25([r["sep"][q] for q in _keys]) / 36 for r in _tr + _pl])
    _ntr = len(_tr)
    def _stat(ix):
        return _spear25(_drate, _R[ix[:_ntr]].mean(0) - _R[ix[_ntr:]].mean(0)).correlation
    _obs = float(_stat(np.arange(len(_R))))
    _rg = np.random.default_rng(25)
    _null = np.array([_stat(_rg.permutation(len(_R))) for _ in range(10000)])
    _p4 = float((1 + np.sum(_null <= _obs)) / (1 + len(_null)))
    _j4 = bool(_obs <= -0.25 and _p4 < 0.01)
    _pc = np.array(_kp["pixel_cosine"])
    _rk = v25_rhos(lambda q: _kp["pair_rate"][q], _tr)
    _rp = v25_rhos(lambda q: 1 - _pc[int(q[0]), int(q[1])], _tr)
    _rz = v25_rhos(lambda q: _k0["pair_rate"][q], _tr)
    _h4 = sum(len(set(r["last"]) & set(_kp["predicted_top3"])) / len(r["last"]) for r in _tr)
    print(f"J4 transfer arm ({len(_tr)} resolved transfer vs {len(_pl)} plain MNIST CNNs): Spearman(change in escape rate, change "
          f"in separation rank) = {_obs:+.2f}, shuffle p = {_p4:.1e} -> {'PASS' if _j4 else 'FAIL'}")
    print(f"   reported: on the transfer networks, mean rho with the transfer kernel {np.mean(_rk):+.2f}, plain CNN kernel "
          f"{np.mean(_rz):+.2f}, pixel {np.mean(_rp):+.2f}; last pair in the transfer kernel's top 3 {_h4:.1f}/{len(_tr)} "
          f"(predicted {_kp['predicted_top3']})")
    V25_TESTS["J4"] = dict(n_transfer=len(_tr), n_plain=len(_pl), spearman=_obs, p=_p4, rho_transfer_kernel=float(np.mean(_rk)),
                           rho_plain_kernel=float(np.mean(_rz)), rho_pixel=float(np.mean(_rp)), last_hits=_h4, passed=_j4)
else:
    print(f"J4 transfer arm: NOT TESTABLE YET ({len(_tr)} resolved transfer, {len(_pl)} plain networks, need 20 each)")
    V25_TESTS["J4"] = dict(passed=None)

_prim = [V25_TESTS[k]["passed"] for k in ("J1", "J2", "J3")]
_v25 = ("CONFIRMED" if all(p is True for p in _prim) else "NOT TESTABLE YET" if any(p is None for p in _prim) else "NOT CONFIRMED")
V25_TESTS["verdict"] = _v25
print(f"VERDICT (J1 and J2 and J3): {_v25}")
print(f"VERDICT J4 (pretraining moves the order as it moves the kernel): {'CONFIRMED' if V25_TESTS['J4']['passed'] is True else 'NOT TESTABLE YET' if V25_TESTS['J4']['passed'] is None else 'NOT CONFIRMED'}")
atomic_write(os.path.join(V25_DIR, "tests.json"), json.dumps(V25_TESTS, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o)))
STORE.push(os.path.join(V25_DIR, "tests.json"))

In [ ]:
V26_SETS = [s for s in os.environ.get("NSYM_V26_SETS", "mnist,kmnist,fashion").split(",") if s]
V26_NETS = ["cnn", "mlp"]
V26_ARMS = {"plain": None, "frozen0": 0.0, "r06": 0.6, "r10": 1.0, "r15": 1.5}    # start of lam / lam_1(net)
V26_N = int(os.environ.get("NSYM_V26_SEEDS", "30"))
V26_TAU = int(os.environ.get("NSYM_V26_TAU", "2000"))
V26_STEPS = int(os.environ.get("NSYM_V26_STEPS", "1170"))
V26_BS = 256
V26_PROBE_EVERY = 25
V26_DIR = os.path.join(RUN_DIR, "v26")
for _d26 in ("units", "ckpt"):
    os.makedirs(os.path.join(V26_DIR, _d26), exist_ok=True)

V26_B = np.zeros((10, 10), dtype=np.int64)
for _i26 in range(10):
    V26_B[_i26, (_i26 + 1) % 10] = V26_B[(_i26 + 1) % 10, _i26] = 1
V26_AUT = np.array([[(s * i + r) % 10 for i in range(10)] for s in (1, -1) for r in range(10)])

def v26_blocks():
    # Hessians at the uniform point (zero logits): pair loss -log P(y | two images) per label y, and per-image entropy
    _out = {}
    for _y in (0, 1):
        _R = torch.as_tensor((V26_B == _y).astype(np.float64))
        def _l(z):
            return -torch.log(torch.softmax(z[:10], -1) @ _R @ torch.softmax(z[10:], -1))
        _H = torch.autograd.functional.hessian(_l, torch.zeros(20, dtype=torch.float64)).numpy()
        _out[_y] = (_H[:10, :10], _H[10:, 10:], _H[:10, 10:])
    _He = torch.autograd.functional.hessian(lambda z: -(torch.softmax(z, -1) * torch.log_softmax(z, -1)).sum(),
                                            torch.zeros(10, dtype=torch.float64)).numpy()
    return _out, _He

def v26_lamc(h, cls, k=4):
    # critical lams of the network, largest first: generalised eigenvalues of (loss Hessian, entropy Hessian) over the
    # head weights at the uniform point; h: (N, d) features of images with digit labels cls. With one-hot features this
    # returns the free-logit values 0.5, 0.4045, ...
    _pb, _he = v26_blocks()
    _ha = np.concatenate([h, np.ones((len(h), 1))], 1)
    _hb = np.stack([_ha[cls == c].mean(0) for c in range(10)])
    _m2 = np.stack([_ha[cls == c].T @ _ha[cls == c] / (cls == c).sum() for c in range(10)])
    _d = _ha.shape[1]
    _HL, _HH = np.zeros((10 * _d, 10 * _d)), np.zeros((10 * _d, 10 * _d))
    for _a in range(10):
        _HH += np.kron(_m2[_a], _he) / 10
        for _b in range(10):
            _H1, _H2, _C = _pb[V26_B[_a, _b]]
            _HL += (np.kron(_m2[_a], _H1) + np.kron(_m2[_b], _H2) + np.kron(np.outer(_hb[_a], _hb[_b]), _C)
                    + np.kron(np.outer(_hb[_b], _hb[_a]), _C.T)) / 100
    _e, _V = np.linalg.eigh(-_HH)
    _W = _V[:, _e > 1e-9 * _e.max()] / np.sqrt(_e[_e > 1e-9 * _e.max()])
    _G = _W.T @ (-_HL) @ _W
    return np.sort(np.linalg.eigvalsh((_G + _G.T) / 2))[::-1][:k]

def v26_feats_ref(ref, x, cnn):
    with torch.no_grad():
        if cnn:
            x = F.max_pool2d(F.relu(ref.c2(F.max_pool2d(F.relu(ref.c1(x)), 2))), 2)
        return F.relu(ref.fc1(x.flatten(1))).double().cpu().numpy()

def v26_predict(src, net, data):
    _path = os.path.join(V26_DIR, f"lamc_{src}_{net}.json")
    if os.path.exists(_path):
        with open(_path) as _fh:
            return json.load(_fh)
    _rng = np.random.default_rng(0)
    _idx = np.concatenate([_rng.choice(data["pools"][c], 100, replace=False) for c in range(10)])
    _cls = np.repeat(np.arange(10), 100)
    _vals = []
    for _s in range(6):
        torch.manual_seed(_s)
        _ref = RefNet(net == "cnn", 784).to(DEVICE)
        _vals.append(v26_lamc(v26_feats_ref(_ref, data["xtr"][torch.as_tensor(_idx, device=DEVICE)], net == "cnn"), _cls).tolist())
    _l1 = float(np.mean([v[0] for v in _vals]))
    _free = v26_lamc(np.eye(10)[_cls], _cls).tolist()
    _pred = dict(source=src, net=net, created=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), lamc_per_init=_vals,
                 lam1=_l1, lam0={a: (m or 0.0) * _l1 for a, m in V26_ARMS.items()}, free_logit_lamc=_free)
    _txt = json.dumps(_pred, indent=1)
    atomic_write(_path, _txt)
    STORE.push(_path)
    STORE.flush()
    print(f"   {src}/{net}: network lam_1 = {_l1:.4f} (free logits {_free[0]:.4f}); anneal starts "
          + ", ".join(f"{a} {v:.4f}" for a, v in _pred["lam0"].items() if V26_ARMS[a]) + f"; sha256 {hashlib.sha256(_txt.encode()).hexdigest()[:16]}", flush=True)
    return _pred

def v26_bank_feats(bank, x):
    # the bank's hidden features (S, B, 128) and logits (S, B, 10), as in SeedBank.forward
    _B = x.shape[0]
    if bank.cnn:
        _h = x.expand(_B, bank.S, x.shape[2], x.shape[3])
        _h = F.max_pool2d(F.relu(bank.c1(_h)), 2)
        _h = F.max_pool2d(F.relu(bank.c2(_h)), 2)
        _h = _h.reshape(_B, bank.S, -1).transpose(0, 1)
    else:
        _h = x.reshape(1, _B, -1).expand(bank.S, _B, -1)
    _h = F.relu(torch.baddbmm(bank.b1[:, None, :], _h, bank.W1.transpose(1, 2)))
    return _h, torch.baddbmm(bank.b2[:, None, :], _h, bank.W2.transpose(1, 2))

def v26_train_unit(name, seeds, net, data, lam0, frozen, probe, gen_seed):
    _ck = os.path.join(V26_DIR, "ckpt", name + ".pt")
    _was = CFG["net"]
    CFG["net"] = net
    try:
        _bank = make_bank(seeds)
    finally:
        CFG["net"] = _was
    _opt = torch.optim.Adam(_bank.parameters(), lr=1e-3)
    _opth = torch.optim.Adam([_bank.W2, _bank.b2], lr=1e-3)            # head only, used while the trunk is frozen
    _g = torch.Generator().manual_seed(gen_seed)
    _tau = V26_TAU if frozen else 0
    _total = _tau + V26_STEPS
    _Bt = torch.as_tensor(V26_B, device=DEVICE)
    _mask = F.one_hot(_Bt.reshape(-1), 2).float()
    _st0 = torch.as_tensor([int(p[0]) for p in data["pools"]])         # each class is one contiguous block of xtr
    _sz = torch.as_tensor([len(p) for p in data["pools"]])
    _start, _traj, _tsteps = 0, [], []
    if os.path.exists(_ck) and os.path.getsize(_ck) > 1000:
        _st = torch.load(_ck, map_location=DEVICE, weights_only=False)
        _bank.load_state_dict(_st["bank"]); _opt.load_state_dict(_st["opt"]); _opth.load_state_dict(_st["opth"]); _g.set_state(_st["gen"].cpu())
        _start, _traj, _tsteps = _st["step"], _st["traj"], _st["tsteps"]
    def _probe():
        with torch.no_grad():
            _p = F.softmax(_bank(probe[0]).float(), -1)
            return torch.stack([_p[:, probe[1] == c].mean(1) for c in range(10)], 1).cpu().numpy().astype(np.float16)
    for _s in range(_start, _total):
        _lam = lam0 * max(0.0, 1 - _s / _tau) if _tau else 0.0
        if _s % V26_PROBE_EVERY == 0 and _s < _tau + 300:
            _traj.append(_probe()); _tsteps.append(_s)
        _a = torch.randint(0, 10, (V26_BS,), generator=_g)
        _b = torch.randint(0, 10, (V26_BS,), generator=_g)
        _u = torch.rand(2, V26_BS, generator=_g)
        _ia = _st0[_a] + (_u[0] * _sz[_a]).long().clamp_max(_sz[_a] - 1)
        _ib = _st0[_b] + (_u[1] * _sz[_b]).long().clamp_max(_sz[_b] - 1)
        _frozen = _s < _tau
        if _frozen:
            with torch.no_grad():
                _h, _ = v26_bank_feats(_bank, data["xtr"][torch.cat([_ia, _ib]).to(DEVICE)])
            _z = torch.baddbmm(_bank.b2[:, None, :], _h, _bank.W2.transpose(1, 2))
        else:
            _h, _z = v26_bank_feats(_bank, data["xtr"][torch.cat([_ia, _ib]).to(DEVICE)])
        _lp = F.log_softmax(_z.float(), -1)
        _p1, _p2 = _lp[:, :V26_BS].exp(), _lp[:, V26_BS:].exp()
        _J = (_p1[..., :, None] * _p2[..., None, :]).reshape(len(seeds), V26_BS, 100)
        _y = _Bt[_a.to(DEVICE), _b.to(DEVICE)]
        _py = (_J @ _mask).gather(2, _y[None, :, None].expand(len(seeds), V26_BS, 1))[..., 0]
        _l = -torch.log(_py.clamp_min(1e-12)).mean(1)
        if _lam > 0:                                           # mean per-image entropy of the head's output
            _l = _l - _lam * (-(_lp.exp() * _lp).sum(-1).mean(1))
        _o = _opth if _frozen else _opt
        _o.zero_grad(set_to_none=True)
        _l.sum().backward()
        _o.step()
        if (_s + 1) % 500 == 0 and _s + 1 < _total:
            torch.save(dict(bank=_bank.state_dict(), opt=_opt.state_dict(), opth=_opth.state_dict(), gen=_g.get_state(), step=_s + 1,
                            traj=_traj, tsteps=_tsteps), _ck)
            STORE.push(_ck)
    with torch.no_grad():
        _pt = torch.cat([F.softmax(_bank(data["xte"][i:i + 2000]).float(), -1) for i in range(0, len(data["xte"]), 2000)], 1)
        _C = torch.stack([_pt[:, torch.as_tensor(data["yte"] == c, device=DEVICE)].mean(1) for c in range(10)], 1).cpu().numpy()
    _opt_flag = np.array([any(np.array_equal(_C[s].argmax(1), q) for q in V26_AUT) for s in range(len(seeds))])
    _strict = np.array([_opt_flag[s] and _C[s].max(1).min() > 0.5 for s in range(len(seeds))])
    if os.path.exists(_ck):
        with open(_ck, "wb") as _fh:
            _fh.write(b"done")
        STORE.push(_ck)
    _rec = dict(C=_C, optimum=_opt_flag, optimum_strict=_strict, seeds=np.array(seeds), lam0=np.array(lam0), traj=np.stack(_traj, 1), tsteps=np.array(_tsteps), tau=np.array(_tau))
    del _bank, _opt
    gpu_free()
    return _rec

def v26_family_data(src):
    if src == "mnist":
        _ytr = np.concatenate([np.full(len(PTR["pools"][c]), c) for c in range(10)])
        _xtr = torch.cat([XTR[torch.as_tensor(np.asarray(PTR["pools"][c]), device=XTR.device)] for c in range(10)]).float()
        _xte, _yte = XTE.float(), np.asarray(YTE_NP)
    else:
        _d = v24_load(src)
        _o = np.argsort(_d["ytr"], kind="stable")
        _xtr, _ytr = torch.as_tensor(_d["xtr"][_o]), _d["ytr"][_o]
        _xte, _yte = torch.as_tensor(_d["xte"]), _d["yte"]
    _pools = [np.flatnonzero(_ytr == c) for c in range(10)]
    assert all(np.array_equal(p, np.arange(p[0], p[0] + len(p))) for p in _pools)
    return dict(xtr=_xtr.to(DEVICE), pools=_pools, xte=_xte.to(DEVICE), yte=np.asarray(_yte))

V26_RES, V26_PRED, V26_STATUS = {}, {}, {}
_t26 = time.time()
for _src in V26_SETS:
    _names = [f"K{_src}{n}{a}_s{k}-{min(k + CFG['bank'], V26_N) - 1}" for n in V26_NETS for a in V26_ARMS for k in range(0, V26_N, CFG["bank"])]
    _have = [n for n in _names if os.path.exists(os.path.join(V26_DIR, "units", n + ".npz"))]
    _needp = [n for n in V26_NETS if not os.path.exists(os.path.join(V26_DIR, f"lamc_{_src}_{n}.json"))]
    if len(_have) == len(_names) and not _needp:
        for _n in _names:
            V26_RES[_n] = dict(np.load(os.path.join(V26_DIR, "units", _n + ".npz")))
        for _net in V26_NETS:
            V26_PRED[(_src, _net)] = v26_predict(_src, _net, None)
        V26_STATUS[_src] = "done"
        continue
    try:
        _data = v26_family_data(_src)
        _pr = np.random.default_rng(v24_seed("v26", _src, "probe"))
        _pidx = np.concatenate([_pr.choice(np.flatnonzero(_data["yte"] == c), 50, replace=False) for c in range(10)])
        _probe = (_data["xte"][torch.as_tensor(_pidx, device=DEVICE)], torch.as_tensor(np.repeat(np.arange(10), 50), device=DEVICE))
        print(f"== {_src} cycle10 annealing: {len(_data['xtr'])} train / {len(_data['xte'])} test images", flush=True)
        for _net in V26_NETS:
            V26_PRED[(_src, _net)] = v26_predict(_src, _net, _data)
        for _net in V26_NETS:
            for _arm in V26_ARMS:
                for _k in range(0, V26_N, CFG["bank"]):
                    _name = f"K{_src}{_net}{_arm}_s{_k}-{min(_k + CFG['bank'], V26_N) - 1}"
                    _path = os.path.join(V26_DIR, "units", _name + ".npz")
                    if os.path.exists(_path):
                        V26_RES[_name] = dict(np.load(_path))
                        continue
                    _sd = [v24_seed("v26", _src, _net, j) for j in range(_k, min(_k + CFG["bank"], V26_N))]   # same seeds in every arm
                    _t0 = time.time()
                    _rec = v26_train_unit(_name, _sd, _net, _data, V26_PRED[(_src, _net)]["lam0"][_arm], V26_ARMS[_arm] is not None,
                                          _probe, v24_seed(_name, "gen"))
                    _rec["net"], _rec["arm"] = np.array(_net), np.array(_arm)
                    np.savez_compressed(_path[:-4], **_rec)
                    STORE.push(_path)
                    V26_RES[_name] = _rec
                    _line = (f"[v26] {_name}: {time.time() - _t0:.0f}s, optimum {int(_rec['optimum'].sum())}/{len(_sd)}, "
                             f"elapsed {(time.time() - _t26) / 60:.0f} min")
                    print(_line, flush=True)
                    with open(os.path.join(V26_DIR, "progress.log"), "a") as _fh:
                        _fh.write(_line + "\n")
                    STORE.push(os.path.join(V26_DIR, "progress.log"))
        V26_STATUS[_src] = "done"
        del _data, _probe
        gpu_free()
        trim_ram()
    except Exception as _e:
        V26_STATUS[_src] = f"FAILED: {type(_e).__name__}: {str(_e)[:200]}"
        print(f"== {_src} cycle10 annealing: {V26_STATUS[_src]}", flush=True)
atomic_write(os.path.join(V26_DIR, "status.json"), json.dumps(V26_STATUS, indent=1))
STORE.push(os.path.join(V26_DIR, "status.json"))
print(f"{len(V26_RES)} v2.6 units available; sources {V26_STATUS}; upload errors: {STORE.flush() or 'none'}")

In [ ]:
from scipy.stats import binomtest as _binom26

V26_ROWS = {}                                     # (src, net) -> {arm: {seed: row}}
for _name, _r in V26_RES.items():
    _net, _arm = str(_r["net"]), str(_r["arm"])
    _src = _name[1:].split("_")[0][:-len(_net) - len(_arm)]
    for _i, _sd in enumerate(_r["seeds"]):
        V26_ROWS.setdefault((_src, _net), {}).setdefault(_arm, {})[int(_sd)] = dict(
            opt=bool(_r["optimum"][_i]), strict=bool(_r["optimum_strict"][_i]), C=_r["C"][_i], traj=_r["traj"][_i].astype(np.float64), tsteps=_r["tsteps"],
            lam0=float(_r["lam0"]), tau=int(_r["tau"]))

def v26_paired(a1, a2):
    # discordant pairs over all families: a1 optimum & a2 not, and the reverse
    _b = _c = 0
    for _fam in V26_ROWS.values():
        for _sd, _r in _fam.get(a1, {}).items():
            if _sd in _fam.get(a2, {}):
                _b += _r["opt"] and not _fam[a2][_sd]["opt"]
                _c += _fam[a2][_sd]["opt"] and not _r["opt"]
    return int(_b), int(_c)

def v26_rate(arm, fam=None):
    _v = [r["opt"] for k, f in V26_ROWS.items() if fam in (None, k) for r in f.get(arm, {}).values()]
    return float(np.mean(_v)) if _v else float("nan"), len(_v)

V26_TESTS = {}
print("optimum rate per family (" + " / ".join(V26_ARMS) + "):")
for _k in sorted(V26_ROWS):
    _rs = {a: v26_rate(a, _k) for a in V26_ARMS}
    print(f"   {_k[0]:>8s}/{_k[1]}: " + " / ".join(f"{_rs[a][0]:.2f}" for a in V26_ARMS) + f" (n = {_rs['plain'][1]}); lam_1(net) {V26_PRED[_k]['lam1']:.3f}")
_rate26 = {a: v26_rate(a) for a in V26_ARMS}
print("   pooled: " + ", ".join(f"{a} {v[0]:.2f}" for a, v in _rate26.items()) + "; reported, strict optimum (also > 0.5 on every "
      "digit's concept): " + ", ".join(f"{a} {np.mean([r['strict'] for f in V26_ROWS.values() for r in f.get(a, {}).values()]):.2f}" for a in V26_ARMS))
_rn, _ra = _rate26["plain"], _rate26["r15"]
_ready = min(v[1] for v in _rate26.values()) >= 60

# ---- K1 annealing from above removes fold minima ------------------------------------------------------------------
if _ready:
    _b, _c = v26_paired("r15", "plain")
    _p1 = float(_binom26(_b, _b + _c, 0.5, alternative="greater").pvalue) if _b + _c else 1.0
    _fw = sum(v26_rate("r15", k)[0] >= v26_rate("plain", k)[0] for k in V26_ROWS)
    _k1 = bool(_p1 < 1e-6 and _ra[0] - _rn[0] >= 0.30 and _fw >= len(V26_ROWS) - 1)
    print(f"K1 anneal from 1.5 x lam_1(net) vs plain: optimum {_ra[0]:.2f} vs {_rn[0]:.2f}; paired: rescued {_b}, lost {_c} (p = {_p1:.1e}); "
          f"above >= none in {_fw}/{len(V26_ROWS)} families -> {'PASS' if _k1 else 'FAIL'}")
    V26_TESTS["K1"] = dict(r15=_ra, plain=_rn, rescued=_b, lost=_c, p=_p1, families_ok=int(_fw), passed=_k1)
else:
    print("K1: NOT TESTABLE YET"); V26_TESTS["K1"] = dict(passed=None)

# ---- K2 the entropy term at the computed scale does it: dose-response in lam_0 / lam_1(net) --------------------------
# (a) freezing alone: frozen0 <= plain + 0.10 (pooled); (b) Cochran-Armitage trend over frozen0, r06, r10, r15 with dose
# 0, 0.6, 1.0, 1.5: one-sided p < 1e-6; (c) r15 >= r06 pooled.
if _ready:
    _arms2 = ["frozen0", "r06", "r10", "r15"]
    _x = np.array([0.0, 0.6, 1.0, 1.5])
    _n = np.array([_rate26[a][1] for a in _arms2], float)
    _y = np.array([_rate26[a][0] * _rate26[a][1] for a in _arms2])
    _pb = _y.sum() / _n.sum()
    _T = np.sum(_x * (_y - _n * _pb))
    _V = _pb * (1 - _pb) * (np.sum(_n * _x ** 2) - np.sum(_n * _x) ** 2 / _n.sum())
    from scipy.stats import norm as _norm26
    _ptr = float(_norm26.sf(_T / np.sqrt(_V))) if _V > 0 else 1.0
    _k2a = bool(_rate26["frozen0"][0] <= _rn[0] + 0.10)
    _k2 = bool(_k2a and _ptr < 1e-6 and _rate26["r15"][0] >= _rate26["r06"][0])
    print(f"K2 dose-response: frozen0 {_rate26['frozen0'][0]:.2f} vs plain {_rn[0]:.2f} (freezing alone must not help: "
          f"{'ok' if _k2a else 'violated'}); rates by start/lam_1 0, 0.6, 1.0, 1.5: "
          + ", ".join(f"{_rate26[a][0]:.2f}" for a in _arms2) + f"; trend p = {_ptr:.1e} -> {'PASS' if _k2 else 'FAIL'}")
    V26_TESTS["K2"] = dict(rates={a: _rate26[a][0] for a in _arms2}, plain=_rn[0], trend_p=_ptr, freezing_alone_ok=_k2a, passed=_k2)
else:
    print("K2: NOT TESTABLE YET"); V26_TESTS["K2"] = dict(passed=None)

# ---- K3 the first broken mode is the even/odd split -----------------------------------------------------------------
def v26_first_break(T, ts, lo=0.02, hi=0.05):
    # first probe after the map has been uniform (max deviation < lo) where it deviates by >= hi; returns (step, share)
    _dev = np.abs(T - 0.1).max((1, 2))
    _u = np.flatnonzero(_dev < lo)
    if not len(_u):
        return None
    _after = np.flatnonzero((np.arange(len(_dev)) > _u[0]) & (_dev >= hi))
    if not len(_after):
        return None
    _M = T[_after[0]]
    _Mc = _M - _M.mean(0) - _M.mean(1, keepdims=True) + _M.mean()
    _F2 = np.abs(np.fft.fft2(_Mc)) ** 2
    return int(ts[_after[0]]), float(_F2[5, 5] / _F2.sum())
_shares, _lams, _brk_opt = [], [], []
for _k, _f in V26_ROWS.items():
    for _r in [r for a in ("r10", "r15") for r in _f.get(a, {}).values()]:
        _fb = v26_first_break(_r["traj"], _r["tsteps"])
        if _fb is not None:
            _shares.append(_fb[1])
            _lams.append(_r["lam0"] * max(0.0, 1 - _fb[0] / _r["tau"]) / V26_PRED[_k]["lam1"])
            _brk_opt.append(bool(_r["opt"]))
if len(_shares) >= 60:
    _frac = float(np.mean(np.array(_shares) > 0.3))
    _med = float(np.median(_shares))
    _k3 = bool(_frac >= 0.70 and _med > 0.4)
    print(f"K3 first broken mode: even/odd share of the first deviation > 0.3 in {_frac:.2f} of {len(_shares)} annealed networks, "
          f"median share {_med:.2f} (a random direction gives about 1/81) -> {'PASS' if _k3 else 'FAIL'}")
    _so, _st = np.array(_shares)[np.array(_brk_opt)], np.array(_shares)[~np.array(_brk_opt)]
    if len(_so) and len(_st):
        _auc = float(np.mean([(a > b) + 0.5 * (a == b) for a in _so for b in _st]))
        print(f"   reported: even/odd share at the break, optimum vs trapped networks: median {np.median(_so):.2f} vs "
              f"{np.median(_st):.2f}, AUROC {_auc:.2f} (n = {len(_so)} / {len(_st)})")
    print(f"   reported: lam at the break / predicted lam_1(net): median {np.median(_lams):.2f} (IQR {np.percentile(_lams, 25):.2f}-"
          f"{np.percentile(_lams, 75):.2f}); below 1 means a delayed break, as in slow passage through a pitchfork")
    V26_TESTS["K3"] = dict(frac=_frac, median=_med, n=len(_shares), lam_ratio_median=float(np.median(_lams)), passed=_k3)
else:
    print(f"K3: NOT TESTABLE YET ({len(_shares)} annealed networks with a detectable break)"); V26_TESTS["K3"] = dict(passed=None)

_prim = [V26_TESTS[k]["passed"] for k in ("K1", "K2", "K3")]
_v26 = "CONFIRMED" if all(p is True for p in _prim) else "NOT TESTABLE YET" if any(p is None for p in _prim) else "NOT CONFIRMED"
V26_TESTS["verdict"] = _v26
print(f"VERDICT (K1 and K2 and K3): {_v26}")
atomic_write(os.path.join(V26_DIR, "tests.json"), json.dumps(V26_TESTS, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o)))
STORE.push(os.path.join(V26_DIR, "tests.json"))

In [ ]:
import zipfile as _zf27
import urllib.request as _ur27
V27_URL = "https://linqs-data.soe.ucsc.edu/public/datasets/ViSudo-PC/v01/ViSudo-PC_dimension::{d}_datasets::{s}_strategy::simple.zip"
V27_JOBS = [tuple(j.split(":")) for j in os.environ.get("NSYM_V27_JOBS", "4:mnist,4:fmnist,4:kmnist,9:mnist").split(",") if j]
V27_NTRAIN = {4: [int(x) for x in os.environ.get("NSYM_V27_NTRAIN4", "50").split(",")],
              9: [int(x) for x in os.environ.get("NSYM_V27_NTRAIN9", "100").split(",")]}
V27_SPLITS = [f"{k:02d}" for k in range(1, int(os.environ.get("NSYM_V27_NSPLITS", "10")) + 1)]   # 01-10 are scored
V27_STEPS = {4: int(os.environ.get("NSYM_V24_STEPS", "1500")), 9: int(os.environ.get("NSYM_V25_STEPS", "1500"))}
V27_BS = {4: 32, 9: 8}                                              # boards per step, as in cells G and I
V27_DATA = os.path.join(os.environ.get("NSYM_DATA_ROOT", "data_v23"), "visudo_pc")
V27_DIR = os.path.join(RUN_DIR, "v27")
for _d27 in ("units", "ckpt"):
    os.makedirs(os.path.join(V27_DIR, _d27), exist_ok=True)
def v27_guard(gb, what):
    # a readable error instead of a host-RAM kill (30 GB cap; stop at 26)
    if rss_gb() + gb > 26.0:
        raise MemoryError(f"{what}: would need {rss_gb():.1f} + {gb:.1f} GB host RAM (cap 26 of 30 GB)")

V27_PUBLISHED = {(4, "mnist"): "NeuPSL AuROC 0.88 (50 puzzles)", (4, "fmnist"): "NeuPSL AuROC 0.74 (50 puzzles)",
                 (4, "kmnist"): "NeuPSL AuROC 0.65 (50 puzzles)",
                 (9, "mnist"): "accuracy: A-NeSI 62.15 / 62.25, NeuPSL 51.5, CNN 51.2 (100 puzzles)"}

def v27_fetch(d, src):
    # download (once, streamed to disk) and unpack one released archive; returns the folder or None
    _dst = os.path.join(V27_DATA, f"d{d}_{src}")
    if os.path.exists(os.path.join(_dst, ".complete")):
        return _dst
    os.makedirs(V27_DATA, exist_ok=True)
    _zip = _dst + ".zip"
    _url = V27_URL.format(d=d, s=src)
    for _attempt in range(4):
        try:
            with _ur27.urlopen(_url, timeout=120) as _r, open(_zip + ".part", "wb") as _fh:
                while True:
                    _chunk = _r.read(1 << 22)
                    if not _chunk:
                        break
                    _fh.write(_chunk)
            os.replace(_zip + ".part", _zip)
            break
        except Exception as _e:
            print(f"   download attempt {_attempt + 1} failed for {_url}: {type(_e).__name__}: {_e}")
            time.sleep(2 ** (_attempt + 1))
    else:
        return None
    with _zf27.ZipFile(_zip) as _z:
        _z.extractall(_dst)
    os.remove(_zip)
    open(os.path.join(_dst, ".complete"), "w").close()
    return _dst

def v27_configs(root):
    # every released split folder (it holds options.json) -> dict of its options and path
    _out = []
    for _dp, _dn, _fn in os.walk(root):
        if "options.json" in _fn:
            with open(os.path.join(_dp, "options.json")) as _fh:
                _o = json.load(_fh)
            _o["path"] = _dp
            _out.append(_o)
    return sorted(_out, key=lambda o: (o["numTrain"], o.get("overlap", 0.0), str(o["splitId"])))

def v27_read(path, prefix, d):
    # one released partition: images (n * d * d, 1, 28, 28) in [0, 1], puzzle validity (n,), cell label names
    with open(os.path.join(path, f"{prefix}_puzzle_pixels.txt")) as _fh:
        _px = np.stack([np.array(_l.rstrip("\n").split("\t"), dtype=np.float32) for _l in _fh if _l.strip()])
    with open(os.path.join(path, f"{prefix}_puzzle_labels.txt")) as _fh:
        _lab = np.array([[int(float(v)) for v in _l.split("\t")] for _l in _fh if _l.strip()])
    with open(os.path.join(path, f"{prefix}_cell_labels.txt")) as _fh:
        _cells = [_l.rstrip("\n").split("\t") for _l in _fh if _l.strip()]
    assert _px.shape[1] == d * d * 784, (_px.shape, d)
    return _px.reshape(len(_px) * d * d, 1, 28, 28), (_lab[:, 0] == 1).astype(np.float64), np.array(_cells)

def v27_train_unit(name, d, seed, tr, va, te):
    # one CNN on one released split, trained as the benchmark arms of cells G (4x4) and I (9x9)
    _ck = os.path.join(V27_DIR, "ckpt", name + ".pt")
    _bank = v24_bank([seed], "cnn") if d == 4 else v25_bank([seed], "cnn")
    _opt = torch.optim.Adam(_bank.parameters(), lr=CFG["lr"])
    _g = torch.Generator().manual_seed(v24_seed(name, "gen"))
    _y = torch.as_tensor(tr["by"], dtype=torch.float64, device=DEVICE)
    _start, _loss = 0, []
    if os.path.exists(_ck) and os.path.getsize(_ck) > 1000:
        _st = torch.load(_ck, map_location=DEVICE, weights_only=False)
        _bank.load_state_dict(_st["bank"]); _opt.load_state_dict(_st["opt"]); _g.set_state(_st["gen"].cpu())
        _start, _loss = _st["step"], _st["loss"]
    _nc = d * d
    for _s in range(_start, V27_STEPS[d]):
        _b = torch.randint(0, len(_y), (V27_BS[d],), generator=_g).to(DEVICE)
        _x = tr["x"].reshape(-1, _nc, 1, 28, 28)[_b].reshape(-1, 1, 28, 28)
        if d == 4:
            _l = v24_nll(F.log_softmax(_bank(_x).float(), -1).reshape(1, V27_BS[d], 16, 4), _y[_b].float()[None]).mean(1)
        else:
            _l = v25_nll(F.softmax(_bank(_x).double(), -1).reshape(1, V27_BS[d], 81, 9), _y[_b][None]).mean(1)
        _opt.zero_grad(set_to_none=True)
        _l.sum().backward()
        _opt.step()
        _loss.append(float(_l.detach()))
        if (_s + 1) % 500 == 0 and _s + 1 < V27_STEPS[d]:
            torch.save(dict(bank=_bank.state_dict(), opt=_opt.state_dict(), gen=_g.get_state(), step=_s + 1, loss=_loss), _ck)
            STORE.push(_ck)
    def _scores(part):
        with torch.no_grad():
            _out = []
            for _i in range(0, len(part["by"]), 25):
                _xx = part["x"][_i * _nc:min(len(part["by"]), _i + 25) * _nc]
                if d == 4:
                    _out.append(v24_logvalid(F.log_softmax(_bank(_xx).float(), -1).reshape(-1, 16, 4)))
                else:
                    _out.append(v25_logvalid(F.softmax(_bank(_xx).double(), -1).reshape(1, -1, 81, 9))[0])
            return torch.cat(_out).double().cpu().numpy()
    _sv, _st_ = _scores(va), _scores(te)
    _cand = np.sort(_sv)
    _thr = _cand[int(np.argmax([np.mean((_sv > c) == (va["by"] == 1)) for c in _cand]))]
    with torch.no_grad():
        _hard = torch.cat([_bank(te["x"][i:i + 4000]).argmax(-1)[0] for i in range(0, len(te["x"]), 4000)]).cpu().numpy()
    from scipy.optimize import linear_sum_assignment as _lsa27
    _cnt = np.zeros((d, d))
    np.add.at(_cnt, (te["cell"], _hard), 1)
    _r, _c = _lsa27(-_cnt)
    _rec = dict(puzzle_acc=float(np.mean((_st_ > _thr) == (te["by"] == 1))), auroc=v25_auroc(_st_, te["by"]),
                symbol_acc=float(_cnt[_r, _c].sum() / _cnt.sum()), threshold=float(_thr), loss=np.array(_loss, np.float32),
                n_train=len(tr["by"]), n_valid=len(va["by"]), n_test=len(te["by"]))
    if os.path.exists(_ck):
        with open(_ck, "wb") as _fh:
            _fh.write(b"done")
        STORE.push(_ck)
    del _bank, _opt
    gpu_free()
    return _rec

V27_RES, V27_STATUS = {}, {}
_t27 = time.time()
for _d, _src in V27_JOBS:
    _d = int(_d)
    _key = f"{_d}x{_d}/{_src}"
    try:
        _root = v27_fetch(_d, _src)
        if _root is None:
            V27_STATUS[_key] = "download failed (see the lines above); nothing trained"
            continue
        _cfgs = v27_configs(_root)
        print(f"== ViSudo-PC {_key}: released configurations (numTrain, overlap, split): "
              + ", ".join(sorted({f"({o['numTrain']}, {o.get('overlap', 0.0)})" for o in _cfgs})))
        _use = [o for o in _cfgs if o["numTrain"] in V27_NTRAIN[_d] and float(o.get("overlap", 0.0)) == 0.0
                and str(o["splitId"]).zfill(2) in V27_SPLITS]
        if not _use:
            V27_STATUS[_key] = f"no released split with numTrain in {V27_NTRAIN[_d]} and no overlap"
            continue
        for _o in _use:
            _name = f"V{_d}{_src}_n{_o['numTrain']}_split{str(_o['splitId']).zfill(2)}"
            _path = os.path.join(V27_DIR, "units", _name + ".json")
            if os.path.exists(_path):
                with open(_path) as _fh:
                    V27_RES[_name] = json.load(_fh)
                continue
            _t0 = time.time()
            _parts = {}
            for _p in ("train", "valid", "test"):
                if _p == "valid" and not os.path.exists(os.path.join(_o["path"], "valid_puzzle_pixels.txt")):
                    continue
                _x, _by, _cl = v27_read(_o["path"], _p, _d)
                v27_guard(_x.nbytes / 2**30, f"{_name} {_p} images")
                _parts[_p] = dict(x=_x, by=_by, cl=_cl)
            _names = sorted({c for p in _parts.values() for c in p["cl"].reshape(-1)}, key=lambda s: (len(s), s))
            assert len(_names) == _d, f"expected {_d} symbols, found {_names}"
            _mu, _sd = float(_parts["train"]["x"].mean()), float(_parts["train"]["x"].std())
            _dev = {}
            for _p, _v in _parts.items():
                _dev[_p] = dict(x=torch.as_tensor((_v["x"] - _mu) / _sd, device=DEVICE), by=_v["by"],
                                cell=np.array([_names.index(c) for c in _v["cl"].reshape(-1)]))
            if "valid" not in _dev:                     # no released validation puzzles: pick the threshold on train
                _dev["valid"] = _dev["train"]
            del _parts
            _rec = v27_train_unit(_name, _d, v24_seed("v27", _d, _src, _o["numTrain"], _o["splitId"]), _dev["train"], _dev["valid"], _dev["test"])
            _rec.update(dim=_d, source=_src, numTrain=int(_o["numTrain"]), split=str(_o["splitId"]), secs=time.time() - _t0)
            del _dev
            _rec["loss_last"] = float(np.mean(_rec.pop("loss")[-50:]))
            atomic_write(_path, json.dumps(_rec))
            STORE.push(_path)
            V27_RES[_name] = _rec
            _line = (f"[v27] {_name}: {_rec['secs']:.0f}s, puzzle accuracy {_rec['puzzle_acc']:.3f}, AuROC {_rec['auroc']:.3f}, "
                     f"symbol accuracy (best relabelling) {_rec['symbol_acc']:.3f}, elapsed {(time.time() - _t27) / 60:.0f} min")
            print(_line, flush=True)
            with open(os.path.join(V27_DIR, "progress.log"), "a") as _fh:
                _fh.write(_line + "\n")
            STORE.push(os.path.join(V27_DIR, "progress.log"))
            gpu_free()
            trim_ram()
        V27_STATUS[_key] = "done"
    except Exception as _e:
        V27_STATUS[_key] = f"FAILED: {type(_e).__name__}: {str(_e)[:200]}"
        print(f"== ViSudo-PC {_key}: {V27_STATUS[_key]}", flush=True)

V27_TABLE = []
for _d, _src in V27_JOBS:
    _d = int(_d)
    for _nt in V27_NTRAIN[_d]:
        _rs = [r for r in V27_RES.values() if r["dim"] == _d and r["source"] == _src and r["numTrain"] == _nt]
        if _rs:
            _pa, _au = np.array([r["puzzle_acc"] for r in _rs]), np.array([r["auroc"] for r in _rs])
            V27_TABLE.append(dict(task=f"{_d}x{_d} {_src}", numTrain=_nt, splits=len(_rs),
                                  puzzle_acc=f"{_pa.mean():.3f} +- {_pa.std(ddof=1) if len(_pa) > 1 else 0:.3f} (min {_pa.min():.3f})",
                                  auroc=f"{_au.mean():.3f} +- {_au.std(ddof=1) if len(_au) > 1 else 0:.3f}",
                                  published=V27_PUBLISHED.get((_d, _src), "")))
print("RELEASED ViSudo-PC FILES (file-matched benchmark):")
for _row in V27_TABLE:
    print(f"   {_row['task']}, {_row['numTrain']} train puzzles, {_row['splits']} splits: puzzle accuracy {_row['puzzle_acc']}, "
          f"AuROC {_row['auroc']} | published: {_row['published']}")
atomic_write(os.path.join(V27_DIR, "tests.json"), json.dumps(dict(status=V27_STATUS, table=V27_TABLE), indent=1))
STORE.push(os.path.join(V27_DIR, "tests.json"))
print(f"{len(V27_RES)} v2.7 units; status {V27_STATUS}; upload errors: {STORE.flush() or 'none'}; host RAM {rss_gb():.1f} GB")

In [ ]:
V21S_TOKEN = "YOUR_HF_TOKEN_HERE"   # hard-coded at the owner's request (wins over any HF_TOKEN in the environment)
V21S_BUCKET = os.environ.get("NSYM_HF_BUCKET", "APassionateResearchers/NeSy4")
V21S_PREFIX = os.environ.get("NSYM_HF_PREFIX", "grounding_symmetry_v2")
V21S_LOCAL = os.environ.get("NSYM_PAPER_DIR", "paper_tables")

def v21s_find_run(api):
    # the run folder (prefix/<config hash>) that holds the v2.1 summary; prefers the one with the newest parts (v27)
    _files = [f for f in api.list_bucket_tree(V21S_BUCKET, prefix=V21S_PREFIX, recursive=True) if isinstance(f, BucketFile)]
    _runs = {}
    for _f in _files:
        _parts = _f.path[len(V21S_PREFIX) + 1:].split("/")
        if len(_parts) >= 2:
            _runs.setdefault(_parts[0], set()).add("/".join(_parts[1:]))
    _ok = [h for h, rel in _runs.items() if "summary.json" in rel]
    if not _ok:
        raise RuntimeError(f"no summary.json under hf://buckets/{V21S_BUCKET}/{V21S_PREFIX}/ (was the v2.1 ANALYSIS cell ever run?)")
    _best = max(_ok, key=lambda h: ("v27/tests.json" in _runs[h], any(r.startswith("v27/units/") for r in _runs[h]), len(_runs[h])))
    return _best, _runs[_best], _files

def v21s_get(api, run, files, rels):
    # download the listed files of the run (always fresh copies); returns {rel: local path} for those that exist
    _out, _todo = {}, []
    for _f in files:
        _rel = _f.path[len(V21S_PREFIX) + len(run) + 2:]
        if _f.path.startswith(f"{V21S_PREFIX}/{run}/") and _rel in rels:
            _dst = os.path.join(V21S_LOCAL, run, *_rel.split("/"))
            os.makedirs(os.path.dirname(_dst), exist_ok=True)
            _todo.append((_f, _dst))
            _out[_rel] = _dst
    if _todo:
        api.download_bucket_files(V21S_BUCKET, _todo)
    return _out

def binom_cdf_s(k, n, p):
    # P(X <= k), X ~ Bin(n, p), exact (log-space sum); same as the ANALYSIS cell
    if k < 0:
        return 0.0
    if k >= n or p <= 0:
        return 1.0
    if p >= 1:
        return 0.0
    _lp = [math.lgamma(n + 1) - math.lgamma(i + 1) - math.lgamma(n - i + 1) + i * math.log(p) + (n - i) * math.log1p(-p)
           for i in range(k + 1)]
    _mx = max(_lp)
    return float(min(1.0, math.exp(_mx) * sum(math.exp(v - _mx) for v in _lp)))

def t_sf_s(t, df):
    try:
        from scipy.stats import t as _tdist
        return float(_tdist.sf(t, df))
    except ImportError:
        return float(0.5 * math.erfc(t / math.sqrt(2)))

def holm_s(pvals, alpha=0.05):
    _order = sorted(range(len(pvals)), key=lambda i: pvals[i])
    _rej = set()
    for _r, _i in enumerate(_order):
        if pvals[_i] <= alpha / (len(pvals) - _r):
            _rej.add(_i)
        else:
            break
    return _rej

V21S_API = HfApi(token=V21S_TOKEN)
try:                                       # fail early with a clear message instead of a 401 deep inside a download
    V21S_API.bucket_info(V21S_BUCKET)
except Exception as _e:
    raise RuntimeError(f"cannot open hf://buckets/{V21S_BUCKET} with the hard-coded token ({type(_e).__name__}: {_e}). "
                       "Check that the token has read access to this bucket.") from _e
print(f"bucket {V21S_BUCKET} reachable with the hard-coded token")
V21S_RUN, _rels, _files = v21s_find_run(V21S_API)
_cands = [f"predictions_{s}{r}.json" for s in (os.environ.get("NSYM_SCALE", "full"), "full", "pilot", "smoke") for r in ("_recomputed", "")]
_pred = next((c for c in _cands if c in _rels), "predictions_full.json")   # the v2.1 file (*_recomputed) when there is one
_got = v21s_get(V21S_API, V21S_RUN, _files, {"summary.json", _pred})
if _pred not in _got:
    raise RuntimeError(f"run {V21S_RUN} has no {_pred}; found: {sorted(r for r in _rels if r.startswith('predictions'))}")
with open(_got["summary.json"]) as _fh:
    _summ = json.load(_fh)
with open(_got[_pred]) as _fh:
    _preds = json.load(_fh)["predictions"]
V21S_DIR = os.path.join(V21S_LOCAL, V21S_RUN)
print(f"run hf://buckets/{V21S_BUCKET}/{V21S_PREFIX}/{V21S_RUN}: summary.json + {_pred} downloaded")
SEED_TABLE_S = _summ["seeds"]
PRIMARY_S = _summ["primary"]
HAAR_S = _summ["haar"]
COND_S = {k: dict(v["condition"], seeds=v["seeds"]) for k, v in _preds.items()}
TRAP_INFO_S = {v["condition"]["task"]: v["frustration"] for v in _preds.values() if v.get("frustration")}

V21A_LEVELS = {"summod10": [0.693], "summod5": [0.693], "summod4": [0.693], "summod3": [0.693], "prodmod10": [0.444],
               "absdiff": [(0.44, 0.70)], "add": [0.796, 0.978, 1.025, 1.136], "petersen": [(0.195, 0.202)],
               "max": [], "equal": [], "eqmod3": [], "parity": []}             # idealised critical levels (nats)
V21A_OLD = {"summod10", "absdiff", "prodmod10"}
V21A_NEW = {"less", "max", "summod3", "eqmod3", "equal"}

def v21a_match(task, excess):
    for _L in V21A_LEVELS.get(task, []):
        if isinstance(_L, tuple):
            _lo, _hi = (_L if task == "absdiff" else (0.9 * _L[0], 1.1 * _L[1]))
            if _lo <= excess <= _hi:
                return True
        elif excess > 0 and abs(excess / _L - 1) <= 0.10:
            return True
    return False

V21A_FRESH = []
for _k, _ms in SEED_TABLE_S.items():
    _c = COND_S[_k]
    if _c["exp"] != "E1" or _c["method"] != "nesy" or _c["task"] not in V21A_OLD | V21A_NEW:
        continue
    for _m in _ms:
        _rep = _c["seeds"].index(_m["seed"])
        if _c["task"] in V21A_NEW or _rep >= 20:
            V21A_FRESH.append(dict(task=_c["task"], rep=_rep, at_opt=bool(_m["at_opt"]), incl=bool(_m["incl"]),
                                   excess=float(_m["excess"]), trap_ratio=float(_m["trap_ratio"])))
_bin_ge = lambda k, n, p: 1.0 - binom_cdf_s(k - 1, n, p)                     # P(X >= k)
_off = [r for r in V21A_FRESH if not r["at_opt"] and r["task"] != "less"]   # less is exploratory (below resolution)
_a1k = sum(v21a_match(r["task"], r["excess"]) for r in _off)
_a1p = _bin_ge(_a1k, len(_off), 0.8) if _off else None
_r2 = np.array([r["trap_ratio"] for r in V21A_FRESH if not r["at_opt"] and TRAP_INFO_S[r["task"]]["count"] > 0
                and np.isfinite(r["trap_ratio"])], float)
if len(_r2) >= 3:
    _se2 = max(_r2.std(ddof=1) / math.sqrt(len(_r2)), 1e-12)
    _a2p = max(t_sf_s((_r2.mean() - 0.9) / _se2, len(_r2) - 1), t_sf_s((1.1 - _r2.mean()) / _se2, len(_r2) - 1))
else:
    _a2p = None
_a3 = [r for r in V21A_FRESH if r["task"] in ("max", "equal", "eqmod3", "parity")]
_a3k = sum(r["incl"] for r in _a3)
_a3p = _bin_ge(_a3k, len(_a3), 0.8) if _a3 else None
V21A_TESTS = [("A1 off-optimum fresh seeds at an idealised critical level (>= 80%)", _a1p, "pass_if_rejected",
               f"{_a1k}/{len(_off)} match; misses: " + ", ".join(f"{r['task']} {r['excess']:.3f}" for r in _off if not v21a_match(r["task"], r["excess"]))),
              ("A2 Theorem-B level on fresh seeds (TOST, ratio in [0.9, 1.1])", _a2p, "pass_if_rejected",
               f"n={len(_r2)}, mean ratio {_r2.mean() if len(_r2) else float('nan'):.3f} (sd {_r2.std(ddof=1) if len(_r2) > 1 else float('nan'):.3f})"),
              ("A3 achievability, fresh seeds of programs with no idealised traps (>= 80%)", _a3p, "pass_if_rejected",
               f"{_a3k}/{len(_a3)} reach the optimum inside the twin-class closure ("
               + ", ".join(f"{t} {sum(r['incl'] for r in _a3 if r['task'] == t)}/{sum(r['task'] == t for r in _a3)}" for t in ("max", "equal", "eqmod3", "parity") if any(r["task"] == t for r in _a3)) + ")")]
_fam = V21A_TESTS + [(v["test"], v["p"], v["direction"], v["detail"]) for v in PRIMARY_S if v["test"].startswith(("P5", "P6", "P7"))]
_tst = [i for i, t in enumerate(_fam) if t[1] is not None]
_rj = {_tst[j] for j in holm_s([_fam[i][1] for i in _tst])}
V21A_VERDICTS = []
for _i, (_n, _p, _dir, _d) in enumerate(_fam):
    _v = "NOT TESTABLE" if _p is None else (("FAIL" if _i in _rj else "PASS") if _dir == "fail_if_rejected" else ("PASS" if _i in _rj else "NOT CONFIRMED"))
    V21A_VERDICTS.append(dict(test=_n, p=_p, verdict=_v, detail=_d))
print(f"FULL-RUN AMENDMENTS (fresh seeds only; Holm over {len(_tst)} tests):")
for _v in V21A_VERDICTS:
    print(f"  {_v['verdict']:13s} {_v['test']}: " + (f"p={_v['p']:.3g}; " if _v["p"] is not None else "") + _v["detail"])
_txt = json.dumps(dict(fresh=V21A_FRESH, verdicts=V21A_VERDICTS), default=float)
with open(os.path.join(V21S_DIR, "amendments.json"), "w") as _fh:
    _fh.write(_txt)
try:
    V21S_API.batch_bucket_files(V21S_BUCKET, add=[(_txt.encode(), f"{V21S_PREFIX}/{V21S_RUN}/amendments.json")])
    print("amendments.json uploaded to the bucket")
except Exception as _e:
    print(f"upload of amendments.json failed ({type(_e).__name__}: {_e}); it is saved locally in {V21S_DIR}")

V21A_HAAR = []
for _k, _h in HAAR_S.items():
    if _h["status"] != "ok":
        _v = _h["status"]
    elif _h["hi"] - _h["lo"] < 1e-9:
        _v = "vacuous (point band: the group only swaps twin digits)"
    else:
        _v = "inside" if _h["lo"] - 1e-9 <= _h["mean"] <= _h["hi"] + 1e-9 else "OUTSIDE"
    V21A_HAAR.append(dict(condition=_k, G=_h["order"], included=f"{_h['n_incl']}/{_h['n']}", mean_acc=round(_h["mean"], 3),
                          haar99=f"[{_h['lo']:.3f}, {_h['hi']:.3f}]" if _h["status"] == "ok" else "", verdict=_v))
print("HAAR TABLE (amendment A5 applied):")
for _r in V21A_HAAR:
    print(f"  {_r['verdict'][:12]:12s} {_r['condition']}: |G|={_r['G']}, {_r['included']} included, mean {_r['mean_acc']}, 99% {_r['haar99']}")
print("OUTSIDE rows that remain:", [r["condition"] for r in V21A_HAAR if r["verdict"] == "OUTSIDE"] or "none")

In [ ]:
V99S_TOKEN = "YOUR_HF_TOKEN_HERE"   # hard-coded at the owner's request (wins over any HF_TOKEN in the environment)
V99S_BUCKET = os.environ.get("NSYM_HF_BUCKET", "APassionateResearchers/NeSy4")
V99S_PREFIX = os.environ.get("NSYM_HF_PREFIX", "grounding_symmetry_v2")
V99S_LOCAL = os.environ.get("NSYM_PAPER_DIR", "paper_tables")
V99S_PUBLISHED = {(4, "mnist"): "NeuPSL AuROC 0.88 (50 puzzles)", (4, "fmnist"): "NeuPSL AuROC 0.74 (50 puzzles)",
                  (4, "kmnist"): "NeuPSL AuROC 0.65 (50 puzzles)",
                  (9, "mnist"): "accuracy: A-NeSI 62.15 / 62.25, NeuPSL 51.5, CNN 51.2 (100 puzzles)"}
V99S_API = HfApi(token=V99S_TOKEN)
try:                                       # fail early with a clear message instead of a 401 deep inside a download
    V99S_API.bucket_info(V99S_BUCKET)
except Exception as _e:
    raise RuntimeError(f"cannot open hf://buckets/{V99S_BUCKET} with the hard-coded token ({type(_e).__name__}: {_e}). "
                       "Check that the token has read access to this bucket.") from _e
print(f"bucket {V99S_BUCKET} reachable with the hard-coded token")

def v99s_fetch():
    # pick the run folder (as standalone P does) and download every small result file of it; returns (run, {rel: path})
    _files = [f for f in V99S_API.list_bucket_tree(V99S_BUCKET, prefix=V99S_PREFIX, recursive=True) if isinstance(f, BucketFile)]
    _runs = {}
    for _f in _files:
        _parts = _f.path[len(V99S_PREFIX) + 1:].split("/")
        if len(_parts) >= 2:
            _runs.setdefault(_parts[0], {})["/".join(_parts[1:])] = _f
    _ok = [h for h, rel in _runs.items() if "summary.json" in rel]
    if not _ok:
        raise RuntimeError(f"no summary.json under hf://buckets/{V99S_BUCKET}/{V99S_PREFIX}/")
    _run = max(_ok, key=lambda h: ("v27/tests.json" in _runs[h], any(r.startswith("v27/units/") for r in _runs[h]), len(_runs[h])))
    _want = {r: f for r, f in _runs[_run].items()
             if r in ("summary.json", "amendments.json") or (r.startswith("v2") and r.endswith("/tests.json"))
             or (r.startswith("v27/units/") and r.endswith(".json"))}
    _todo, _out = [], {}
    for _r, _f in _want.items():
        _dst = os.path.join(V99S_LOCAL, _run, *_r.split("/"))
        os.makedirs(os.path.dirname(_dst), exist_ok=True)
        _todo.append((_f, _dst))
        _out[_r] = _dst
    if _todo:
        V99S_API.download_bucket_files(V99S_BUCKET, _todo)
    return _run, _out, sum(r.startswith("units/") for r in _runs[_run])

def v99s_json(paths, rel):
    if rel not in paths or not os.path.exists(paths[rel]):
        return None
    with open(paths[rel]) as _fh:
        return json.load(_fh)

def v99s_v27_table(paths):
    # the v2.7 table from the per-split unit files (used when v27/tests.json is missing from the bucket)
    _res = [v99s_json(paths, r) for r in paths if r.startswith("v27/units/")]
    _rows = []
    for (_d, _src), _pub in V99S_PUBLISHED.items():
        _rs = [r for r in _res if r and r.get("dim") == _d and r.get("source") == _src]
        for _nt in sorted({r["numTrain"] for r in _rs}):
            _sel = [r for r in _rs if r["numTrain"] == _nt]
            _pa, _au = np.array([r["puzzle_acc"] for r in _sel]), np.array([r["auroc"] for r in _sel])
            _rows.append(dict(task=f"{_d}x{_d} {_src}", numTrain=_nt, splits=len(_sel),
                              puzzle_acc=f"{_pa.mean():.3f} +- {_pa.std(ddof=1) if len(_pa) > 1 else 0:.3f} (min {_pa.min():.3f})",
                              auroc=f"{_au.mean():.3f} +- {_au.std(ddof=1) if len(_au) > 1 else 0:.3f}", published=_pub))
    return dict(status="rebuilt from v27/units (tests.json was not in the bucket)", table=_rows)

def v99s_lines(run, paths, n_units):
    _L = [f"PAPER SUMMARY (standalone)  hf://buckets/{V99S_BUCKET}/{V99S_PREFIX}/{run}  {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}"]
    _s = v99s_json(paths, "summary.json")
    _L.append("v2.1 laws (CNN, MNIST, plain NeSy; notebook-registered family, Holm 5%):")
    for _v in _s["primary"]:
        _L.append(f"   {_v['verdict']:13s} {_v['test']}" + (f" p={_v['p']:.3g}" if _v["p"] is not None else "") + f" | {_v['detail']}")
    _L.append(f"   instrument checks {sum(bool(v[1]) for v in _s['instrument'])}/{len(_s['instrument'])} ok")
    _out = []
    for _k, _h in _s["haar"].items():
        if _h["status"] == "ok" and _h["hi"] - _h["lo"] >= 1e-9 and not (_h["lo"] - 1e-9 <= _h["mean"] <= _h["hi"] + 1e-9):
            _out.append(_k)
    _L.append("v2.1 Haar rows still OUTSIDE after amendment A5: " + str(_out or "none"))
    _a = v99s_json(paths, "amendments.json")
    if _a is None:
        _L.append("v2.1 full-run amendments: no amendments.json yet (run standalone cell P first)")
    else:
        _L.append("v2.1 full-run amendments (fresh seeds):")
        for _v in _a["verdicts"]:
            _L.append(f"   {_v['verdict']:13s} {_v['test']}" + (f" p={_v['p']:.3g}" if _v["p"] is not None else "") + f" | {_v['detail']}")
    _L.append(f"v2.2 relabelled Petersen + MLP: T1-T5 verdicts were printed by cell C (CONFIRMED on GPU, 2026-10-12); "
              f"{n_units} unit files in the run folder")
    for _tag, _rel, _keys in (("v2.3 robustness (4 datasets x 2 nets x 3 programs)", "v23/tests.json", ("tests", "status")),
                              ("v2.4 4x4 visual Sudoku", "v24/tests.json", ("H0_index_formula", "H1", "H2", "H3", "verdict", "benchmark")),
                              ("v2.5 9x9 visual Sudoku + transfer", "v25/tests.json", ("J1", "J2", "J3", "J4", "verdict", "benchmark")),
                              ("v2.6 fold minima (entropy anneal)", "v26/tests.json", ("K1", "K2", "K3", "verdict")),
                              ("v2.7 released ViSudo-PC files", "v27/tests.json", ("status", "table"))):
        _t = v99s_json(paths, _rel)
        if _t is None and _rel == "v27/tests.json" and any(r.startswith("v27/units/") for r in paths):
            _t = v99s_v27_table(paths)
        if _t is None:
            _L.append(f"{_tag}: no results in the bucket")
            continue
        _L.append(f"{_tag}:")
        for _k in _keys:
            if _k not in _t:
                continue
            _v = _t[_k]
            if isinstance(_v, list) and _v and isinstance(_v[0], dict):
                for _row in _v:
                    _L.append(f"   {_k}: " + json.dumps(_row, default=str))
            else:
                _L.append(f"   {_k}: " + json.dumps(_v, default=lambda o: round(o, 4) if isinstance(o, float) else str(o))[:600])
    return _L

V99S_RUN, V99S_PATHS, _nu = v99s_fetch()
V99S_SUMMARY = v99s_lines(V99S_RUN, V99S_PATHS, _nu)
_txt = "\n".join(V99S_SUMMARY) + "\n"
with open(os.path.join(V99S_LOCAL, V99S_RUN, "PAPER_SUMMARY.txt"), "w") as _fh:
    _fh.write(_txt)
try:
    V99S_API.batch_bucket_files(V99S_BUCKET, add=[(_txt.encode(), f"{V99S_PREFIX}/{V99S_RUN}/PAPER_SUMMARY.txt")])
    _up = "uploaded to the bucket"
except Exception as _e:
    _up = f"upload failed ({type(_e).__name__}: {_e}); saved locally"
print(_txt + f"PAPER_SUMMARY.txt {_up}")

In [ ]:
import zipfile
from collections import Counter
# from huggingface_hub import HfApi, BucketFile

V97_TOKEN = "YOUR_HF_TOKEN_HERE"   # hard-coded at the owner's request (wins over any HF_TOKEN in the environment)
V97_BUCKET = os.environ.get("NSYM_HF_BUCKET", "APassionateResearchers/NeSy4")
V97_PREFIXES = [p for p in os.environ.get("NSYM_HF_PREFIXES", "grounding_symmetry_v2,scope_runs").split(",") if p]
V97_DIR = os.environ.get("NSYM_RESULTS_DIR", "all_results")
V97_WITH_CKPT = os.environ.get("NSYM_WITH_CKPT", "0") == "1"
V97_API = HfApi(token=V97_TOKEN)
try:                                       # fail early with a clear message instead of a 401 deep inside a download
    V97_API.bucket_info(V97_BUCKET)
except Exception as _e:
    raise RuntimeError(f"cannot open hf://buckets/{V97_BUCKET} with the hard-coded token ({type(_e).__name__}: {_e}). "
                       "Check that the token has read access to this bucket.") from _e
print(f"bucket {V97_BUCKET} reachable with the hard-coded token")

def v97_list():
    # every file under the chosen bucket folders; a folder that does not exist is reported, not an error
    _out = []
    for _p in V97_PREFIXES:
        try:
            _fs = [f for f in V97_API.list_bucket_tree(V97_BUCKET, prefix=_p, recursive=True) if isinstance(f, BucketFile)]
        except Exception as _e:
            print(f"   {_p}/: not listed ({type(_e).__name__}: {str(_e)[:120]})")
            continue
        print(f"   {_p}/: {len(_fs)} files in the bucket")
        _out += _fs
    return _out

def v97_fetch(files):
    # download in batches of 200 with 4 retries per batch; returns (downloaded, skipped, failed)
    _todo, _skip = [], 0
    for _f in files:
        if _f.path.endswith(".pt") and not V97_WITH_CKPT:
            _skip += 1
            continue
        _dst = os.path.join(V97_DIR, *_f.path.split("/"))
        _sz = getattr(_f, "size", None)
        if os.path.exists(_dst) and (_sz is None or os.path.getsize(_dst) == _sz):
            continue
        os.makedirs(os.path.dirname(_dst), exist_ok=True)
        _todo.append((_f, _dst))
    print(f"   {len(_todo)} files to download, {_skip} checkpoints skipped, {len(files) - len(_todo) - _skip} already local")
    _done, _failed, _t0 = 0, [], time.time()
    for _i in range(0, len(_todo), 200):
        _batch = _todo[_i:_i + 200]
        for _attempt in range(4):
            try:
                V97_API.download_bucket_files(V97_BUCKET, _batch)
                _done += len(_batch)
                break
            except Exception as _e:
                if _attempt == 3:
                    _failed += [f.path for f, _ in _batch]
                    print(f"   batch {_i // 200 + 1} failed: {type(_e).__name__}: {str(_e)[:160]}")
                else:
                    time.sleep(2 ** (_attempt + 1))
        print(f"   [fetch] {_done + len(_failed)}/{len(_todo)} files, {(time.time() - _t0) / 60:.1f} min", flush=True)
    return _done, _skip, _failed

print(f"Listing hf://buckets/{V97_BUCKET}/ ...")
V97_FILES = v97_list()
V97_DONE, V97_SKIPPED, V97_FAILED = v97_fetch(V97_FILES)

# inventory: per run folder and part (v23, v24, ..., units), how many result files are now local
_inv = Counter()
for _f in V97_FILES:
    _parts = _f.path.split("/")
    if _f.path.endswith(".pt") and not V97_WITH_CKPT:
        continue
    if os.path.exists(os.path.join(V97_DIR, *_parts)):
        _inv["/".join(_parts[:3]) if len(_parts) > 3 else "/".join(_parts[:2]) + "/(top level)"] += 1
print("LOCAL INVENTORY (result files per folder):")
for _k in sorted(_inv):
    print(f"   {_inv[_k]:5d}  {_k}")
_keys = ["summary.json", "amendments.json", "v23/tests.json", "v24/tests.json", "v25/tests.json", "v26/tests.json",
         "v27/tests.json", "PAPER_SUMMARY.txt"]
for _f in V97_FILES:
    for _k in _keys:
        if _f.path.endswith("/" + _k):
            print(f"   key result present: {_f.path}")

# one zip of every local result file, to download to your own computer
V97_ZIP = V97_DIR.rstrip("/") + ".zip"
with zipfile.ZipFile(V97_ZIP, "w", zipfile.ZIP_DEFLATED) as _z:
    for _dp, _dn, _fn in os.walk(V97_DIR):
        for _n in _fn:
            if _n.endswith(".pt") and not V97_WITH_CKPT:
                continue
            _z.write(os.path.join(_dp, _n), os.path.relpath(os.path.join(_dp, _n), os.path.dirname(V97_DIR.rstrip("/")) or "."))
print(f"{V97_DONE} downloaded, {len(V97_FAILED)} failed{(': ' + ', '.join(V97_FAILED[:10])) if V97_FAILED else ''}; "
      f"everything is in {os.path.abspath(V97_DIR)} and zipped as {os.path.abspath(V97_ZIP)} ({os.path.getsize(V97_ZIP) / 2**20:.1f} MB)")

In [ ]:
V96_TOKEN = "YOUR_HF_TOKEN_HERE"   # hard-coded at the owner's request (wins over any HF_TOKEN in the environment)
V96_BUCKET = os.environ.get("NSYM_HF_BUCKET", "APassionateResearchers/NeSy4")
V96_PREFIX = os.environ.get("NSYM_HF_PREFIX", "grounding_symmetry_v2")
V96_DIR = os.environ.get("NSYM_RESULTS_DIR", "all_results")
V96_TEXT = (".json", ".txt", ".md", ".csv", ".log", ".sha256")
V96_OUT = []

def v96_p(*a):
    _s = " ".join(str(x) for x in a)
    V96_OUT.append(_s)
    print(_s, flush=True)

def v96_short(v):
    # full value, except long numeric lists: n / first / last / mean
    if isinstance(v, list) and len(v) > 24 and all(isinstance(x, (int, float)) or x is None for x in v):
        _f = [x for x in v if isinstance(x, (int, float)) and not (isinstance(x, float) and math.isnan(x))]
        return f"[{len(v)} numbers: first {v[0]}, last {v[-1]}, mean {sum(_f) / len(_f) if _f else float('nan'):.4g}]"
    if isinstance(v, list):
        return [v96_short(x) for x in v]
    if isinstance(v, dict):
        return {k: v96_short(x) for k, x in v.items()}
    if isinstance(v, float):
        return round(v, 5)
    return v

def v96_cell(v):
    _v = v96_short(v)
    return _v if isinstance(_v, str) else json.dumps(_v, default=str)

def v96_table(rows, title):
    # rows: list of dicts (or dict of dicts); prints every row and every column
    if isinstance(rows, dict):
        rows = [{"key": k, **(r if isinstance(r, dict) else {"value": r})} for k, r in rows.items()]
    v96_p(f"\n### {title}  ({len(rows)} rows)")
    if not rows:
        return
    _cols = []
    for _r in rows:
        for _k in (_r if isinstance(_r, dict) else {"value": _r}):
            if _k not in _cols:
                _cols.append(_k)
    v96_p(" | ".join(_cols))
    for _r in rows:
        _r = _r if isinstance(_r, dict) else {"value": _r}
        v96_p(" | ".join(v96_cell(_r.get(c, "")) for c in _cols))

def v96_json(obj, title):
    v96_p(f"\n### {title}")
    for _line in json.dumps(v96_short(obj), indent=1, default=str).splitlines():
        v96_p(_line)

def v96_load(path):
    with open(path) as _fh:
        return json.load(_fh)

v96_p(f"[S] opening hf://buckets/{V96_BUCKET} ...")
V96_API = HfApi(token=V96_TOKEN)
try:
    V96_API.bucket_info(V96_BUCKET)
except Exception as _e:
    raise RuntimeError(f"cannot open hf://buckets/{V96_BUCKET} with the hard-coded token ({type(_e).__name__}: {_e})") from _e
v96_p(f"[S] bucket reachable; listing {V96_PREFIX}/ (this can take a minute for thousands of files) ...")
_all = [f for f in V96_API.list_bucket_tree(V96_BUCKET, prefix=V96_PREFIX, recursive=True) if isinstance(f, BucketFile)]
_runs = {}
for _f in _all:
    _parts = _f.path[len(V96_PREFIX) + 1:].split("/")
    if len(_parts) >= 2:
        _runs.setdefault(_parts[0], []).append(_f)
for _h, _fs in _runs.items():
    v96_p(f"[S] run folder {_h}: {len(_fs)} files ({sum(f.path.endswith('.pt') for f in _fs)} checkpoints)")
_ok = [h for h, fs in _runs.items() if any(f.path.endswith(f"{h}/summary.json") for f in fs)]
if not _ok:
    raise RuntimeError(f"no run folder with summary.json under {V96_PREFIX}/; folders found: {sorted(_runs)}")
V96_RUN = max(_ok, key=lambda h: (any("/v27/" in f.path for f in _runs[h]), len(_runs[h])))
V96_ROOT = os.path.join(V96_DIR, V96_PREFIX, V96_RUN)
v96_p(f"[S] using run folder {V96_RUN}")

_todo = []
for _f in _runs[V96_RUN]:
    if not _f.path.endswith(V96_TEXT):
        continue
    _dst = os.path.join(V96_DIR, *_f.path.split("/"))
    _sz = getattr(_f, "size", None)
    if os.path.exists(_dst) and (_sz is None or os.path.getsize(_dst) == _sz):
        continue
    os.makedirs(os.path.dirname(_dst), exist_ok=True)
    _todo.append((_f, _dst))
v96_p(f"[S] {len(_todo)} result files to download (the rest are already local)")
for _i in range(0, len(_todo), 100):
    for _attempt in range(4):
        try:
            V96_API.download_bucket_files(V96_BUCKET, _todo[_i:_i + 100])
            break
        except Exception as _e:
            if _attempt == 3:
                v96_p(f"[S] batch {_i // 100 + 1} failed: {type(_e).__name__}: {str(_e)[:200]}")
            time.sleep(2 ** (_attempt + 1))
    v96_p(f"[S] downloaded {min(_i + 100, len(_todo))}/{len(_todo)}")

_files = sorted(os.path.relpath(os.path.join(dp, n), V96_ROOT).replace(os.sep, "/")
                for dp, _dn, fn in os.walk(V96_ROOT) for n in fn if n.endswith(V96_TEXT))
v96_p(f"\n==================== ALL RESULTS: run {V96_RUN}, {len(_files)} result files ====================")

# ---- v2.1
if "summary.json" in _files:
    _s = v96_load(os.path.join(V96_ROOT, "summary.json"))
    v96_p("\n==================== v2.1 (CNN, MNIST, 70 conditions) ====================")
    v96_table(_s.get("primary", []), "v2.1 PRIMARY family (Holm 5%)")
    v96_table([dict(name=x[0], ok=x[1], detail=x[2]) if isinstance(x, list) else x for x in _s.get("secondary", [])], "v2.1 SECONDARY (method arm)")
    v96_table([dict(name=x[0], ok=x[1], detail=x[2]) if isinstance(x, list) else x for x in _s.get("instrument", [])], "v2.1 INSTRUMENT checks")
    v96_json(_s.get("pooled", {}), "v2.1 pooled PIT")
    v96_table(_s.get("haar", {}), "v2.1 Haar tests (raw; amendment A5 is applied in the amendments section)")
    v96_table(_s.get("summary", []), "v2.1 per-condition summary")
    v96_json(_s.get("gauge_fix", {}), "v2.1 frequency gauge fixing")
    v96_json(dict(program_optimum=_s.get("program_optimum"), program_nll=_s.get("program_nll"), prereg_sha256=_s.get("prereg_sha256")), "v2.1 program optima")
    _seeds = [{"condition": k, **m} for k, ms in _s.get("seeds", {}).items() for m in ms]
    v96_table(_seeds, "v2.1 per-seed table (every network)")
for _n in _files:
    if _n.startswith("predictions_") and _n.endswith(".json"):
        _pr = v96_load(os.path.join(V96_ROOT, _n)).get("predictions", {})
        v96_table({k: {kk: vv for kk, vv in v.items() if kk not in ("seeds", "tie_cells")} for k, v in _pr.items()}, f"v2.1 pre-registered predictions ({_n})")
if "amendments.json" in _files:
    _a = v96_load(os.path.join(V96_ROOT, "amendments.json"))
    v96_table(_a.get("verdicts", []), "v2.1 full-run amendments A1-A3 + P5-P7 (Holm)")
    v96_table(_a.get("fresh", []), "v2.1 amendments: fresh seeds")

# ---- v2.3 - v2.7: every tests.json, then every other json in those folders
for _part in ("v23", "v24", "v25", "v26", "v27"):
    _in = [n for n in _files if n.startswith(_part + "/") and n.endswith(".json") and "/units/" not in "/" + n]
    if not _in:
        v96_p(f"\n==================== {_part}: no files in the bucket ====================")
        continue
    v96_p(f"\n==================== {_part} ====================")
    for _n in sorted(_in, key=lambda n: (not n.endswith("tests.json"), n)):
        _o = v96_load(os.path.join(V96_ROOT, _n))
        if isinstance(_o, list) and _o and isinstance(_o[0], dict):
            v96_table(_o, _n)
        else:
            v96_json(_o, _n)

# ---- every unit file of every part: one row each
_units = {}
for _n in _files:
    if "units/" in _n and _n.endswith(".json"):
        _units.setdefault(_n.rsplit("units/", 1)[0] or "(v2.1 and v2.2)", []).append(_n)
for _grp, _ns in sorted(_units.items()):
    _rows = []
    for _n in sorted(_ns):
        try:
            _o = v96_load(os.path.join(V96_ROOT, _n))
        except Exception as _e:
            _rows.append({"unit_file": _n, "error": str(_e)})
            continue
        if isinstance(_o, dict):
            _rows.append({"unit_file": _n.rsplit("/", 1)[-1][:-5], **{k: v for k, v in _o.items() if not isinstance(v, (dict, list)) or len(json.dumps(v, default=str)) < 400 or (isinstance(v, list) and len(v) > 24)}})
        else:
            _rows.append({"unit_file": _n.rsplit("/", 1)[-1][:-5], "value": _o})
    v96_table(_rows, f"unit files {_grp}units/")

# ---- every log / text / markdown file in full
for _n in _files:
    if _n.endswith((".txt", ".md", ".log", ".csv")):
        v96_p(f"\n### {_n}")
        with open(os.path.join(V96_ROOT, _n), errors="replace") as _fh:
            for _line in _fh.read().splitlines():
                v96_p(_line)

_txt = "\n".join(V96_OUT) + "\n"
with open(os.path.join(V96_ROOT, "ALL_RESULTS.txt"), "w") as _fh:
    _fh.write(_txt)
try:
    V96_API.batch_bucket_files(V96_BUCKET, add=[(_txt.encode(), f"{V96_PREFIX}/{V96_RUN}/ALL_RESULTS.txt")])
    _up = "uploaded to the bucket"
except Exception as _e:
    _up = f"upload failed ({type(_e).__name__}); saved locally"
print(f"\n[S] done: {len(V96_OUT)} lines, also saved to {os.path.abspath(os.path.join(V96_ROOT, 'ALL_RESULTS.txt'))} and {_up}", flush=True)

In [ ]:
V95_TOKEN = "YOUR_HF_TOKEN_HERE"   # hard-coded at the owner's request
V95_MAIN = "APassionateResearchers/NeSy4"
V95_OTHER = ["APassionateResearchers/NeuroSym"]
V95_PREFIX = "grounding_symmetry_v2"
V95_API = HfApi(token=V95_TOKEN)

def v95_runs(bucket):
    try:
        _fs = [f.path for f in V95_API.list_bucket_tree(bucket, prefix=V95_PREFIX, recursive=True) if isinstance(f, BucketFile)]
    except Exception as _e:
        print(f"[T] {bucket}: cannot list ({type(_e).__name__}: {str(_e)[:150]})", flush=True)
        return {}
    _runs = {}
    for _p in _fs:
        _parts = _p[len(V95_PREFIX) + 1:].split("/")
        if len(_parts) >= 2:
            _runs.setdefault(_parts[0], Counter())[_parts[1] if _parts[1].startswith("v2") and len(_parts) > 2 else "(v2.1/v2.2)"] += 1
    for _h, _c in sorted(_runs.items()):
        print(f"[T] {bucket} / {_h}: " + ", ".join(f"{k} {v}" for k, v in sorted(_c.items())), flush=True)
    if not _runs:
        print(f"[T] {bucket}: no run folders under {V95_PREFIX}/", flush=True)
    return {h: set() for h in _runs} | {"_files": set(_fs)}

print("[T] buckets:", flush=True)
V95_REMOTE = v95_runs(V95_MAIN)
for _b in V95_OTHER:
    v95_runs(_b)

print("[T] this machine's disk:", flush=True)
V95_LOCAL = []
for _dp, _dn, _fn in os.walk(os.getcwd()):
    if _dp.count(os.sep) - os.getcwd().count(os.sep) > 4:
        _dn[:] = []
        continue
    if "summary.json" in _fn or "v26" in _dn or "v27" in _dn:
        _c = {p: sum(len(f) for _, _, f in os.walk(os.path.join(_dp, p))) for p in ("v24", "v25", "v26", "v27") if os.path.isdir(os.path.join(_dp, p))}
        print(f"[T] local run folder {_dp}: " + (", ".join(f"{k} {v} files" for k, v in _c.items()) or "no v24-v27 folders"), flush=True)
        V95_LOCAL.append(_dp)
if not V95_LOCAL:
    print("[T] no run folder on this disk (a new machine, or the session was reset)", flush=True)

_done = 0
for _dp in V95_LOCAL:
    _run = os.path.basename(_dp.rstrip(os.sep))
    for _part in ("v26", "v27"):
        for _d, _, _fn in os.walk(os.path.join(_dp, _part)):
            for _n in sorted(_fn):
                if not _n.endswith((".json", ".txt", ".log", ".md")):
                    continue
                _loc = os.path.join(_d, _n)
                _dest = f"{V95_PREFIX}/{_run}/" + os.path.relpath(_loc, _dp).replace(os.sep, "/")
                if _dest in V95_REMOTE.get("_files", set()):
                    continue
                with open(_loc, "rb") as _fh:
                    _data = _fh.read()
                for _attempt in range(4):
                    try:
                        V95_API.batch_bucket_files(V95_MAIN, add=[(_data, _dest)])
                        _done += 1
                        print(f"[T] uploaded {_dest}", flush=True)
                        break
                    except Exception as _e:
                        if _attempt == 3:
                            print(f"[T] FAILED {_dest}: {type(_e).__name__}: {str(_e)[:150]}", flush=True)
                        time.sleep(2 ** (_attempt + 1))
print(f"[T] done: {_done} v2.6/v2.7 files copied into {V95_MAIN}. Re-run standalone cell N afterwards.", flush=True)

In [ ]:
V26L_TOKEN = "YOUR_HF_TOKEN_HERE"   # hard-coded at the owner's request
V26L_BUCKET = os.environ.get("NSYM_HF_BUCKET", "APassionateResearchers/NeSy4")
V26L_PREFIX = os.environ.get("NSYM_HF_PREFIX", "grounding_symmetry_v2")
V26L_API = HfApi(token=V26L_TOKEN)
V26L_API.bucket_info(V26L_BUCKET)
print(f"[L] bucket {V26L_BUCKET} reachable; listing ...", flush=True)
_all26 = [f for f in V26L_API.list_bucket_tree(V26L_BUCKET, prefix=V26L_PREFIX, recursive=True) if isinstance(f, BucketFile)]
_byrun26 = {}
for _f in _all26:
    _p = _f.path[len(V26L_PREFIX) + 1:].split("/")
    if len(_p) >= 3 and _p[1] == "v26" and (_p[-1].endswith(".npz") or _p[-1].startswith("lamc_")):
        _byrun26.setdefault(_p[0], []).append(_f)
if not _byrun26:
    raise RuntimeError("no v26 unit files (.npz) or lamc files in the bucket: cell K's uploads never arrived; cell K must be re-run (about 20 min)")
V26L_RUN = max(_byrun26, key=lambda h: len(_byrun26[h]))
V26_DIR = os.path.join("all_results", V26L_PREFIX, V26L_RUN, "v26")
_todo26 = []
for _f in _byrun26[V26L_RUN]:
    _dst = os.path.join("all_results", *_f.path.split("/"))
    os.makedirs(os.path.dirname(_dst), exist_ok=True)
    if not os.path.exists(_dst):
        _todo26.append((_f, _dst))
print(f"[L] run {V26L_RUN}: {sum(f.path.endswith('.npz') for f in _byrun26[V26L_RUN])} unit files, downloading {len(_todo26)} ...", flush=True)
for _i in range(0, len(_todo26), 50):
    V26L_API.download_bucket_files(V26L_BUCKET, _todo26[_i:_i + 50])
    print(f"[L] downloaded {min(_i + 50, len(_todo26))}/{len(_todo26)}", flush=True)

V26_ARMS = {"plain": None, "frozen0": 0.0, "r06": 0.6, "r10": 1.0, "r15": 1.5}    # as in cell K
V26_RES, V26_PRED = {}, {}
for _n in sorted(os.listdir(os.path.join(V26_DIR, "units"))):
    if _n.endswith(".npz"):
        V26_RES[_n[:-4]] = dict(np.load(os.path.join(V26_DIR, "units", _n)))
for _n in os.listdir(V26_DIR):
    if _n.startswith("lamc_") and _n.endswith(".json"):
        with open(os.path.join(V26_DIR, _n)) as _fh:
            _o = json.load(_fh)
        V26_PRED[(_o["source"], _o["net"])] = _o
print(f"[L] {len(V26_RES)} units ({sum(len(r['seeds']) for r in V26_RES.values())} networks), lam_1 for {sorted(V26_PRED)}", flush=True)

# reported only: the most common final concept map (argmax per digit) of each family and arm
_cyc26 = {tuple((s * i + r) % 10 for i in range(10)) for s in (1, -1) for r in range(10)}
_maps26 = {}
for _name, _r in V26_RES.items():
    _net, _arm = str(_r["net"]), str(_r["arm"])
    _src = _name[1:].split("_")[0][:-len(_net) - len(_arm)]
    for _C in _r["C"]:
        _maps26.setdefault((_src, _net, _arm), Counter())[tuple(int(x) for x in np.asarray(_C).argmax(1))] += 1
print("reported: most common final maps (digit -> concept; * = one of the 20 optimal cycle maps)")
for _k in sorted(_maps26):
    print(f"   {_k[0]:>7s}/{_k[1]}/{_k[2]:7s}: " + "; ".join(f"{''.join(map(str, m))}{'*' if m in _cyc26 else ''} x{c}" for m, c in _maps26[_k].most_common(3))
          + f"; distinct maps {len(_maps26[_k])}", flush=True)

from scipy.stats import binomtest as _binom26

V26_ROWS = {}                                     # (src, net) -> {arm: {seed: row}}
for _name, _r in V26_RES.items():
    _net, _arm = str(_r["net"]), str(_r["arm"])
    _src = _name[1:].split("_")[0][:-len(_net) - len(_arm)]
    for _i, _sd in enumerate(_r["seeds"]):
        V26_ROWS.setdefault((_src, _net), {}).setdefault(_arm, {})[int(_sd)] = dict(
            opt=bool(_r["optimum"][_i]), strict=bool(_r["optimum_strict"][_i]), C=_r["C"][_i], traj=_r["traj"][_i].astype(np.float64), tsteps=_r["tsteps"],
            lam0=float(_r["lam0"]), tau=int(_r["tau"]))

def v26_paired(a1, a2):
    # discordant pairs over all families: a1 optimum & a2 not, and the reverse
    _b = _c = 0
    for _fam in V26_ROWS.values():
        for _sd, _r in _fam.get(a1, {}).items():
            if _sd in _fam.get(a2, {}):
                _b += _r["opt"] and not _fam[a2][_sd]["opt"]
                _c += _fam[a2][_sd]["opt"] and not _r["opt"]
    return int(_b), int(_c)

def v26_rate(arm, fam=None):
    _v = [r["opt"] for k, f in V26_ROWS.items() if fam in (None, k) for r in f.get(arm, {}).values()]
    return float(np.mean(_v)) if _v else float("nan"), len(_v)

V26_TESTS = {}
print("optimum rate per family (" + " / ".join(V26_ARMS) + "):")
for _k in sorted(V26_ROWS):
    _rs = {a: v26_rate(a, _k) for a in V26_ARMS}
    print(f"   {_k[0]:>8s}/{_k[1]}: " + " / ".join(f"{_rs[a][0]:.2f}" for a in V26_ARMS) + f" (n = {_rs['plain'][1]}); lam_1(net) {V26_PRED[_k]['lam1']:.3f}")
_rate26 = {a: v26_rate(a) for a in V26_ARMS}
print("   pooled: " + ", ".join(f"{a} {v[0]:.2f}" for a, v in _rate26.items()) + "; reported, strict optimum (also > 0.5 on every "
      "digit's concept): " + ", ".join(f"{a} {np.mean([r['strict'] for f in V26_ROWS.values() for r in f.get(a, {}).values()]):.2f}" for a in V26_ARMS))
_rn, _ra = _rate26["plain"], _rate26["r15"]
_ready = min(v[1] for v in _rate26.values()) >= 60

# ---- K1 annealing from above removes fold minima ------------------------------------------------------------------
if _ready:
    _b, _c = v26_paired("r15", "plain")
    _p1 = float(_binom26(_b, _b + _c, 0.5, alternative="greater").pvalue) if _b + _c else 1.0
    _fw = sum(v26_rate("r15", k)[0] >= v26_rate("plain", k)[0] for k in V26_ROWS)
    _k1 = bool(_p1 < 1e-6 and _ra[0] - _rn[0] >= 0.30 and _fw >= len(V26_ROWS) - 1)
    print(f"K1 anneal from 1.5 x lam_1(net) vs plain: optimum {_ra[0]:.2f} vs {_rn[0]:.2f}; paired: rescued {_b}, lost {_c} (p = {_p1:.1e}); "
          f"above >= none in {_fw}/{len(V26_ROWS)} families -> {'PASS' if _k1 else 'FAIL'}")
    V26_TESTS["K1"] = dict(r15=_ra, plain=_rn, rescued=_b, lost=_c, p=_p1, families_ok=int(_fw), passed=_k1)
else:
    print("K1: NOT TESTABLE YET"); V26_TESTS["K1"] = dict(passed=None)

# ---- K2 the entropy term at the computed scale does it: dose-response in lam_0 / lam_1(net) --------------------------
# (a) freezing alone: frozen0 <= plain + 0.10 (pooled); (b) Cochran-Armitage trend over frozen0, r06, r10, r15 with dose
# 0, 0.6, 1.0, 1.5: one-sided p < 1e-6; (c) r15 >= r06 pooled.
if _ready:
    _arms2 = ["frozen0", "r06", "r10", "r15"]
    _x = np.array([0.0, 0.6, 1.0, 1.5])
    _n = np.array([_rate26[a][1] for a in _arms2], float)
    _y = np.array([_rate26[a][0] * _rate26[a][1] for a in _arms2])
    _pb = _y.sum() / _n.sum()
    _T = np.sum(_x * (_y - _n * _pb))
    _V = _pb * (1 - _pb) * (np.sum(_n * _x ** 2) - np.sum(_n * _x) ** 2 / _n.sum())
    from scipy.stats import norm as _norm26
    _ptr = float(_norm26.sf(_T / np.sqrt(_V))) if _V > 0 else 1.0
    _k2a = bool(_rate26["frozen0"][0] <= _rn[0] + 0.10)
    _k2 = bool(_k2a and _ptr < 1e-6 and _rate26["r15"][0] >= _rate26["r06"][0])
    print(f"K2 dose-response: frozen0 {_rate26['frozen0'][0]:.2f} vs plain {_rn[0]:.2f} (freezing alone must not help: "
          f"{'ok' if _k2a else 'violated'}); rates by start/lam_1 0, 0.6, 1.0, 1.5: "
          + ", ".join(f"{_rate26[a][0]:.2f}" for a in _arms2) + f"; trend p = {_ptr:.1e} -> {'PASS' if _k2 else 'FAIL'}")
    V26_TESTS["K2"] = dict(rates={a: _rate26[a][0] for a in _arms2}, plain=_rn[0], trend_p=_ptr, freezing_alone_ok=_k2a, passed=_k2)
else:
    print("K2: NOT TESTABLE YET"); V26_TESTS["K2"] = dict(passed=None)

# ---- K3 the first broken mode is the even/odd split -----------------------------------------------------------------
def v26_first_break(T, ts, lo=0.02, hi=0.05):
    # first probe after the map has been uniform (max deviation < lo) where it deviates by >= hi; returns (step, share)
    _dev = np.abs(T - 0.1).max((1, 2))
    _u = np.flatnonzero(_dev < lo)
    if not len(_u):
        return None
    _after = np.flatnonzero((np.arange(len(_dev)) > _u[0]) & (_dev >= hi))
    if not len(_after):
        return None
    _M = T[_after[0]]
    _Mc = _M - _M.mean(0) - _M.mean(1, keepdims=True) + _M.mean()
    _F2 = np.abs(np.fft.fft2(_Mc)) ** 2
    return int(ts[_after[0]]), float(_F2[5, 5] / _F2.sum())
_shares, _lams, _brk_opt = [], [], []
for _k, _f in V26_ROWS.items():
    for _r in [r for a in ("r10", "r15") for r in _f.get(a, {}).values()]:
        _fb = v26_first_break(_r["traj"], _r["tsteps"])
        if _fb is not None:
            _shares.append(_fb[1])
            _lams.append(_r["lam0"] * max(0.0, 1 - _fb[0] / _r["tau"]) / V26_PRED[_k]["lam1"])
            _brk_opt.append(bool(_r["opt"]))
if len(_shares) >= 60:
    _frac = float(np.mean(np.array(_shares) > 0.3))
    _med = float(np.median(_shares))
    _k3 = bool(_frac >= 0.70 and _med > 0.4)
    print(f"K3 first broken mode: even/odd share of the first deviation > 0.3 in {_frac:.2f} of {len(_shares)} annealed networks, "
          f"median share {_med:.2f} (a random direction gives about 1/81) -> {'PASS' if _k3 else 'FAIL'}")
    _so, _st = np.array(_shares)[np.array(_brk_opt)], np.array(_shares)[~np.array(_brk_opt)]
    if len(_so) and len(_st):
        _auc = float(np.mean([(a > b) + 0.5 * (a == b) for a in _so for b in _st]))
        print(f"   reported: even/odd share at the break, optimum vs trapped networks: median {np.median(_so):.2f} vs "
              f"{np.median(_st):.2f}, AUROC {_auc:.2f} (n = {len(_so)} / {len(_st)})")
    print(f"   reported: lam at the break / predicted lam_1(net): median {np.median(_lams):.2f} (IQR {np.percentile(_lams, 25):.2f}-"
          f"{np.percentile(_lams, 75):.2f}); below 1 means a delayed break, as in slow passage through a pitchfork")
    V26_TESTS["K3"] = dict(frac=_frac, median=_med, n=len(_shares), lam_ratio_median=float(np.median(_lams)), passed=_k3)
else:
    print(f"K3: NOT TESTABLE YET ({len(_shares)} annealed networks with a detectable break)"); V26_TESTS["K3"] = dict(passed=None)

_prim = [V26_TESTS[k]["passed"] for k in ("K1", "K2", "K3")]
_v26 = "CONFIRMED" if all(p is True for p in _prim) else "NOT TESTABLE YET" if any(p is None for p in _prim) else "NOT CONFIRMED"
V26_TESTS["verdict"] = _v26
print(f"VERDICT (K1 and K2 and K3): {_v26}")
_txt26 = json.dumps(V26_TESTS, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
with open(os.path.join(V26_DIR, "tests.json"), "w") as _fh:
    _fh.write(_txt26)
try:
    V26L_API.batch_bucket_files(V26L_BUCKET, add=[(_txt26.encode(), f"{V26L_PREFIX}/{V26L_RUN}/v26/tests.json")])
    print("v26/tests.json uploaded to the bucket", flush=True)
except Exception as _e:
    print(f"upload of v26/tests.json failed ({type(_e).__name__}: {_e}); saved in {V26_DIR}", flush=True)

In [ ]:
import traceback  # added: referenced below but never imported in the original
from matplotlib.figure import Figure
from matplotlib import rcParams
from scipy.stats import spearmanr, rankdata

VU_TOKEN = "YOUR_HF_TOKEN_HERE"   # hard-coded at the owner's request (wins over any HF_TOKEN in the environment)
VU_BUCKET = os.environ.get("NSYM_HF_BUCKET", "APassionateResearchers/NeSy4")
VU_PREFIX = os.environ.get("NSYM_HF_PREFIX", "grounding_symmetry_v2")
VU_DIR = os.environ.get("NSYM_PAPER_DATA", "paper_data")
VU_OUT = os.environ.get("NSYM_PAPER_FIGS", "paper_figures")
VU_LOCAL = os.environ.get("NSYM_FIG_LOCAL", "")      # testing only: a local folder laid out like the bucket
os.makedirs(VU_OUT, exist_ok=True)

rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans"], "font.size": 6.5, "axes.titlesize": 7,
                 "axes.labelsize": 6.5, "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 5.8,
                 "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.major.size": 2.5,
                 "ytick.major.size": 2.5, "axes.spines.top": False, "axes.spines.right": False, "pdf.fonttype": 42,
                 "legend.frameon": False, "lines.linewidth": 1.2})
VU_C = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", sky="#56B4E9",
            grey="#8C8C8C", light="#C8C8C8", ink="#222222")
VU_CM = 1 / 2.54                                       # inches per cm
VU_W = 6.5                                             # TMLR text width, inches


# ------------------------------------------------------------------ 1. data from the bucket
def vu_fetch():
    # returns the local root of the chosen run folder
    if VU_LOCAL:
        _runs = [d for d in os.listdir(os.path.join(VU_LOCAL, VU_PREFIX)) if os.path.exists(os.path.join(VU_LOCAL, VU_PREFIX, d, "summary.json"))]
        if not _runs:
            raise RuntimeError(f"no run folder with summary.json under {VU_LOCAL}/{VU_PREFIX}")
        print(f"[U] local stand-in bucket {VU_LOCAL}, run folder {_runs[0]}")
        return os.path.join(VU_LOCAL, VU_PREFIX, _runs[0])
    from huggingface_hub import HfApi, BucketFile
    _api = HfApi(token=VU_TOKEN)
    try:
        _api.bucket_info(VU_BUCKET)
    except Exception as _e:
        raise RuntimeError(f"cannot open hf://buckets/{VU_BUCKET} with the hard-coded token ({type(_e).__name__}: {_e})") from _e
    print(f"[U] bucket {VU_BUCKET} reachable; listing {VU_PREFIX}/ ...", flush=True)
    _files = [f for f in _api.list_bucket_tree(VU_BUCKET, prefix=VU_PREFIX, recursive=True) if isinstance(f, BucketFile)]
    _runs = {}
    for _f in _files:
        _parts = _f.path[len(VU_PREFIX) + 1:].split("/")
        if len(_parts) >= 2:
            _runs.setdefault(_parts[0], {})["/".join(_parts[1:])] = _f
    _ok = [h for h, rel in _runs.items() if "summary.json" in rel]
    if not _ok:
        raise RuntimeError(f"no summary.json under hf://buckets/{VU_BUCKET}/{VU_PREFIX}/; folders: {sorted(_runs)}")
    _run = max(_ok, key=lambda h: (any(r.startswith("v26/units/") for r in _runs[h]), len(_runs[h])))
    def _want(r):
        if r.endswith(".pt") or "/ckpt/" in "/" + r:
            return False
        return (r in ("summary.json", "amendments.json") or (r.startswith("units/") and r.endswith(".json"))
                or r.endswith("/tests.json") or (r.startswith("v2") and "/predictions_" in r)
                or r.startswith("v26/lamc_") or (r.split("/")[0] in ("v24", "v25", "v26") and r.endswith(".npz")))
    _todo = []
    for _r, _f in _runs[_run].items():
        if not _want(_r):
            continue
        _dst = os.path.join(VU_DIR, VU_PREFIX, _run, *_r.split("/"))
        _sz = getattr(_f, "size", None)
        if os.path.exists(_dst) and (_sz is None or os.path.getsize(_dst) == _sz):
            continue
        os.makedirs(os.path.dirname(_dst), exist_ok=True)
        _todo.append((_f, _dst))
    print(f"[U] run folder {_run}: {len(_todo)} files to download (the rest are local or not needed)", flush=True)
    for _i in range(0, len(_todo), 200):
        for _attempt in range(4):
            try:
                _api.download_bucket_files(VU_BUCKET, _todo[_i:_i + 200])
                break
            except Exception as _e:
                if _attempt == 3:
                    raise RuntimeError(f"download failed: {type(_e).__name__}: {_e}") from _e
                time.sleep(2 ** (_attempt + 1))
        print(f"[U] downloaded {min(_i + 200, len(_todo))}/{len(_todo)}", flush=True)
    return os.path.join(VU_DIR, VU_PREFIX, _run)

def vu_json(root, rel):
    _p = os.path.join(root, *rel.split("/"))
    if not os.path.exists(_p):
        return None
    with open(_p) as _fh:
        return json.load(_fh)

def vu_units_json(root):
    # every v2.1-v2.3 unit file: key -> list of per-seed records
    _out = {}
    _d = os.path.join(root, "units")
    for _n in sorted(os.listdir(_d)) if os.path.isdir(_d) else []:
        if _n.endswith(".json"):
            with open(os.path.join(_d, _n)) as _fh:
                _u = json.load(_fh)
            _out.setdefault(_u["key"], []).extend(_u["per_seed"])
    return _out

def vu_npz(root, part):
    # unit name -> dict of arrays, for v24 / v25 / v26
    _out = {}
    _d = os.path.join(root, part, "units")
    for _n in sorted(os.listdir(_d)) if os.path.isdir(_d) else []:
        if _n.endswith(".npz"):
            with np.load(os.path.join(_d, _n), allow_pickle=False) as _z:
                _out[_n[:-4]] = {k: _z[k] for k in _z.files}
    return _out


# ------------------------------------------------------------------ 2. programs, groups, decoders (as in cells A-E)
def vu_programs():
    _P = np.zeros((10, 10), int)
    for _i in range(5):
        for _u, _v in ((_i, (_i + 1) % 5), (5 + _i, 5 + (_i + 2) % 5), (_i, _i + 5)):
            _P[_u, _v] = _P[_v, _u] = 1
    def _g(edges):
        _M = np.zeros((10, 10), int)
        for _u, _v in edges:
            _M[_u, _v] = _M[_v, _u] = 1
        return _M
    _T = {"petersen": _P, "crown": _g([(i, 5 + j) for i in range(5) for j in range(5) if i != j]),
          "cycle10": _g([(i, (i + 1) % 10) for i in range(10)])}
    for _nm, _s in {"hi": [5, 1, 6, 2, 9, 8, 4, 0, 7, 3], "lo": [8, 9, 7, 1, 3, 6, 2, 5, 0, 4]}.items():
        _T[f"petersen_{_nm}"] = _P[np.ix_(_s, _s)]
    return _T

def vu_automorphisms(B):
    # all bijections p with B[p a, p b] == B[a, b] (backtracking, as notebook cell 5)
    n = B.shape[0]
    part = np.zeros((1, 0), dtype=np.int64)
    for i in range(n):
        new = []
        for v in range(n):
            ok = np.all(part != v, axis=1) & (B[v, v] == B[i, i])
            if i:
                ok &= np.all(B[v, part] == B[i, :i][None, :], axis=1) & np.all(B[part, v] == B[:i, i][None, :], axis=1)
            sel = part[ok]
            if len(sel):
                new.append(np.concatenate([sel, np.full((len(sel), 1), v)], 1))
        part = np.concatenate(new, 0) if new else np.zeros((0, i + 1), np.int64)
    return part

def vu_cycles(p):
    return tuple((x, int(p[x])) for x in range(len(p)) if p[x] > x)

def vu_involutions(B):
    _A = vu_automorphisms(B)
    _I = np.arange(B.shape[0])
    return [p for p in _A if np.array_equal(p[p], _I) and not np.array_equal(p, _I)], _A

def vu_decode22(C, invs):
    # cell C rule: the digits of each 2-cycle share a concept (>= 0.15); a fixed digit uses a concept no other digit uses
    sup = [set(int(e) for e in np.flatnonzero(C[d] >= 0.15)) | {int(np.argmax(C[d]))} for d in range(len(C))]
    used = Counter(c for s in sup for c in s)
    ok = [p for p in invs if all(sup[a] & sup[b] for a, b in vu_cycles(p))
          and all(used[c] == 1 for x in range(len(C)) if p[x] == x for c in sup[x])]
    return min(ok, key=lambda p: len(vu_cycles(p))) if ok else None

def vu_decode23(C, invs, tau):
    # cell E rule: 2-cycles share concepts, different orbits of pi use disjoint concept sets
    _sup = [set(int(e) for e in np.flatnonzero(C[d] >= tau)) | {int(np.argmax(C[d]))} for d in range(len(C))]
    _ok = []
    for _p in invs:
        _cyc = vu_cycles(_p)
        if not all(_sup[a] & _sup[b] for a, b in _cyc):
            continue
        _orbs = list(_cyc) + [(x,) for x in range(len(C)) if _p[x] == x]
        _U = [set().union(*(_sup[v] for v in o)) for o in _orbs]
        if sum(len(u) for u in _U) == len(set().union(*_U)):
            _ok.append(_p)
    return min(_ok, key=lambda p: len(vu_cycles(p))) if _ok else None

def vu_level(B, p):
    return float(np.log(2) * np.mean(B[p] != B))

def vu_task_of(key):
    _t = key.split("_")
    _i = next(i for i in range(2, len(_t) + 1) if i == len(_t) or _t[i] in ("nesy", "gf", "gfrg"))
    return "_".join(_t[1:_i])

def vu_wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    _p = k / n
    _d = 1 + z * z / n
    _c = (_p + z * z / (2 * n)) / _d
    _h = z * math.sqrt(_p * (1 - _p) / n + z * z / (4 * n * n)) / _d
    return max(0.0, _c - _h), min(1.0, _c + _h)


# ------------------------------------------------------------------ 3. exact class-level Hessians (Theorems 4 and 5)
def vu_softmax(Z):
    _Z = Z - Z.max(1, keepdims=True)
    _E = np.exp(_Z)
    return _E / _E.sum(1, keepdims=True)

def vu_hessians(B, Z):
    # exact logit Hessians of L(M) = mean_{a,b} -log sum_{c,c'} M_ac M_bc' [B(c,c') = B(a,b)] and of mean row entropy
    n = B.shape[0]
    M = vu_softmax(Z)
    gM, HM = np.zeros((n, n)), np.zeros((n, n, n, n))
    for a in range(n):
        for b in range(n):
            R = (B == B[a, b]).astype(float)
            P = M[a] @ R @ M[b]
            ga, gb = R @ M[b], R.T @ M[a]
            gM[a] -= ga / P
            gM[b] -= gb / P
            HM[a, :, a, :] += np.outer(ga, ga) / P ** 2
            HM[b, :, b, :] += np.outer(gb, gb) / P ** 2
            HM[a, :, b, :] += np.outer(ga, gb) / P ** 2 - R / P
            HM[b, :, a, :] += np.outer(gb, ga) / P ** 2 - R.T / P
    gM /= n * n
    HM /= n * n
    J = np.stack([np.diag(M[a]) - np.outer(M[a], M[a]) for a in range(n)])
    H = np.einsum("aij,aibk,bkl->ajbl", J, HM, J)
    HE = np.zeros((n * n, n * n))
    for a in range(n):
        m = M[a]
        for g, tgt in ((gM[a], "L"), (-(np.log(np.clip(m, 1e-300, None)) + 1), "E")):
            gb_ = g @ m                                    # sum_c g_c d2 m_c / dz_j dz_k
            T = np.diag(m * g) - np.outer(m * g, m) - np.outer(m, m * g) + 2 * gb_ * np.outer(m, m) - gb_ * np.diag(m)
            if tgt == "L":
                H[a, :, a, :] += T
            else:
                HE[a * n:(a + 1) * n, a * n:(a + 1) * n] = J[a] @ np.diag(-1 / np.clip(m, 1e-300, None)) @ J[a] + T
    return H.reshape(n * n, n * n), HE / n

def vu_set_partitions(s):
    if not s:
        yield []
        return
    f, rest = s[0], s[1:]
    for p in vu_set_partitions(rest):
        for i in range(len(p)):
            yield p[:i] + [[f] + p[i]] + p[i + 1:]
        yield [[f]] + p

def vu_critical(B):
    # critical lambdas of F = L - lam * entropy at the uniform map (largest first) and the generalised eigenvectors
    n = B.shape[0]
    H, HE = vu_hessians(B, np.zeros((n, n)))
    w, V = np.linalg.eigh(-HE)
    W = V[:, w > 1e-10] / np.sqrt(w[w > 1e-10])
    lam, Y = np.linalg.eigh(-(W.T @ H @ W))
    _o = np.argsort(-lam)
    return lam[_o], (W @ Y[:, _o])

def vu_closed_form(B):
    n, d = B.shape[0], int(B.sum(1)[0])
    mu = np.linalg.eigvalsh(B.astype(float))
    mu = np.delete(mu, np.argmin(np.abs(mu - d)))           # drop the constant eigenvector once
    return np.sort([2 * a * b / (d * (n - d)) for a in mu for b in mu])[::-1]

def vu_graphs():
    def _circ(n, steps):
        _M = np.zeros((n, n), int)
        for i in range(n):
            for s in steps:
                _M[i, (i + s) % n] = _M[(i + s) % n, i] = 1
        return _M
    _G = {f"C{n}": _circ(n, [1]) for n in (6, 7, 8, 10, 12, 14, 16, 20)}
    _G["Petersen"] = vu_programs()["petersen"]
    _G["K3,3"] = np.kron(np.array([[0, 1], [1, 0]]), np.ones((3, 3), int))
    _G["3-cube"] = np.array([[int(bin(i ^ j).count("1") == 1) for j in range(8)] for i in range(8)])
    _G["C9(1,3)"] = _circ(9, [1, 3])
    _rng = np.random.default_rng(12)                       # random 3-regular graph on 12 vertices (pairing model)
    while True:
        _st = np.repeat(np.arange(12), 3)
        _rng.shuffle(_st)
        _M = np.zeros((12, 12), int)
        _ok = True
        for _u, _v in _st.reshape(-1, 2):
            if _u == _v or _M[_u, _v]:
                _ok = False
                break
            _M[_u, _v] = _M[_v, _u] = 1
        if _ok:
            break
    _G["rand 3-reg"] = _M
    return _G


# ------------------------------------------------------------------ 4. per-study tables
def vu_study1(root, units):
    # Study I: every network of summary.json, its class and (for pair states) its predicted level
    _s = vu_json(root, "summary.json")
    _B = vu_programs()["petersen"]
    _invs, _ = vu_involutions(_B)
    _rows = []
    for _k, _ms in _s["seeds"].items():
        _task = vu_task_of(_k)
        _C = {int(e["seed"]): np.array(e["C0"]) for e in units.get(_k, [])}
        for _m in _ms:
            _x = _m.get("excess")
            if _x is None or not np.isfinite(_x):
                continue
            _cl, _lv = "other", None
            if _m.get("at_opt"):
                _cl = "optimal"
            elif _m.get("seed_class", "n/a") != "n/a" and _m.get("trap_ratio") is not None and np.isfinite(_m["trap_ratio"]) and _m["trap_ratio"] != 0:
                _cl, _lv = "pair", _x / _m["trap_ratio"]
            elif _task == "petersen" and int(_m["seed"]) in _C:
                _p = vu_decode22(_C[int(_m["seed"])], _invs)
                if _p is not None:
                    _cl, _lv = "involution", vu_level(_B, _p)
            _rows.append(dict(cond=_k, task=_task, seed=int(_m["seed"]), excess=float(_x), cls=_cl, level=_lv))
    return _rows

def vu_study2(units):
    _T = vu_programs()
    _out = {}
    for _arm in ("hi", "lo"):
        _key = f"E8_petersen_{_arm}_nesy"
        _B = _T[f"petersen_{_arm}"]
        _invs, _A = vu_involutions(_B)
        _rows = []
        for _e in units.get(_key, []):
            _C = np.array(_e["C0"])
            _rows.append(dict(seed=_e["seed"], C=_C, nll=_e["label_nll"], optimal=bool(np.any(np.all(_A == _C.argmax(1)[None], 1)))))
        _opt = [r["nll"] for r in _rows if r["optimal"]]
        _base = float(np.median(_opt)) if _opt else None
        for _r in _rows:
            _r["pi"] = None if (_r["optimal"] or _base is None) else vu_decode22(_r["C"], _invs)
            if _r["pi"] is not None:
                _r["level"], _r["excess"], _r["cyc"] = vu_level(_B, _r["pi"]), _r["nll"] - _base, tuple(sorted(vu_cycles(_r["pi"])))
        _out[_arm] = _rows
    return _out

VU_HI_ORDER = [((2, 3), (4, 9), (5, 8)), ((0, 6), (4, 5), (8, 9)), ((0, 6), (2, 3), (4, 8), (5, 9)), ((0, 1), (2, 8), (3, 5)), ((0, 2), (1, 8), (4, 7)), ((0, 4), (2, 7), (5, 6)), ((0, 3), (1, 5), (7, 9)), ((0, 9), (3, 7), (6, 8)), ((1, 9), (2, 6), (5, 7)), ((0, 5), (2, 7), (4, 6), (8, 9)), ((0, 1), (2, 5), (3, 8), (4, 9)), ((1, 4), (3, 6), (7, 8)), ((1, 6), (2, 9), (3, 4)), ((0, 5), (1, 3), (2, 8), (7, 9)), ((1, 6), (2, 4), (3, 9), (5, 8)), ((0, 8), (3, 7), (4, 5), (6, 9)), ((0, 8), (1, 2), (3, 5), (4, 7)), ((0, 7), (1, 8), (2, 4), (5, 6)), ((1, 3), (2, 9), (4, 6), (7, 8)), ((1, 2), (3, 4), (5, 7), (6, 9)), ((0, 3), (1, 7), (2, 6), (5, 9)), ((0, 7), (1, 5), (3, 9), (6, 8)), ((0, 2), (1, 7), (3, 6), (4, 8)), ((0, 4), (1, 9), (2, 5), (6, 7)), ((0, 9), (1, 4), (3, 8), (6, 7))]   # frozen escape order of cell C (slowest first)

def vu_study3(root, units):
    _T = vu_programs()
    _pred = {}
    for _ds in ("mnist", "fashion", "kmnist", "svhn"):
        for _net in ("cnn", "mlp"):
            _p = vu_json(root, f"v23/predictions_{_ds}_{_net}.json")
            if _p is not None:
                _pred[(_ds, _net)] = _p
    def _pct(states, cyc):
        _same = [tuple(sorted(tuple(c) for c in s["cycles"])) for s in states if s["ncyc"] == len(cyc)]
        return (_same.index(cyc) + 0.5) / len(_same)
    _cells, _dec = {}, []
    for (_ds, _net) in _pred:
        for _t in ("petersen", "crown", "cycle10"):
            _per = units.get(f"E9{_ds}{_net}_{_t}_nesy", [])
            _B = _T[_t]
            _invs, _A = vu_involutions(_B)
            _rows = []
            for _e in _per:
                _C = np.array(_e["C0"])
                _rows.append(dict(C=_C, nll=_e["label_nll"], amap=_C.argmax(1), optimal=bool(np.any(np.all(_A == _C.argmax(1)[None], 1)))))
            _opt = [r for r in _rows if r["optimal"]]
            _noise = [max(np.max(np.delete(r["C"][d], r["amap"][d])) for d in range(10)) for r in _opt]
            _tau = min(0.35, 0.15 + float(np.median(_noise))) if _opt else 0.25
            _base = float(np.median([r["nll"] for r in _opt])) if _opt else None
            _us = []
            for _r in _rows:
                if _r["optimal"] or _base is None:
                    continue
                _p = vu_decode23(_r["C"], _invs, _tau)
                if _p is None:
                    continue
                _cy = tuple(sorted(vu_cycles(_p)))
                _u = _pct(_pred[(_ds, _net)]["programs"][_t], _cy)
                _uo = [_pct(_pred[k]["programs"][_t], _cy) for k in _pred if k != (_ds, _net)]
                _d = dict(ds=_ds, net=_net, prog=_t, level=vu_level(_B, _p), excess=_r["nll"] - _base, u=_u,
                          u_other=float(np.mean(_uo)) if _uo else None)
                _dec.append(_d)
                _us.append(_u)
            _cells[(_ds, _net, _t)] = dict(n=len(_rows), stuck=sum(not r["optimal"] for r in _rows), dec=len(_us),
                                           mean_u=float(np.mean(_us)) if _us else None,
                                           learned=bool(_opt) or any(r["nll"] < 0.5 for r in _rows))
    return _dec, _cells, _pred

def vu_sep_times(T, tsteps, pairs, thr=0.5):
    _out = {}
    for _a, _b in pairs:
        _ov = np.minimum(T[:, _a].astype(np.float64), T[:, _b].astype(np.float64)).sum(-1)
        _ab = np.flatnonzero(_ov >= thr)
        _out[f"{_a}{_b}"] = int(tsteps[_ab[-1]] + 1) if len(_ab) else 0
    return _out

def vu_rho(rate_of, sep):
    _k = sorted(sep)
    _x = spearmanr([rate_of(q) for q in _k], [sep[q] for q in _k]).correlation
    return float(_x) if np.isfinite(_x) else None

def vu_study4(root):
    _res = vu_npz(root, "v24")
    _pairs = list(itertools.combinations(range(4), 2))
    _pred, _rows = {}, {}
    for _name, _r in _res.items():
        if "cnn50" in _name:
            continue
        _net = str(_r["net"])
        _fam = _name[1:].split("_")[0][:-len(_net)]
        if (_fam, _net) not in _pred:
            _pred[(_fam, _net)] = vu_json(root, f"v24/predictions_{_fam}_{_net}.json")
        _pr = _pred[(_fam, _net)]
        _pc = np.array(_pr["pixel_cosine"])
        for _i in range(len(_r["seeds"])):
            _st = vu_sep_times(_r["traj"][_i], _r["tsteps"], _pairs)
            _m = max(_st.values())
            _learned = bool(_r["perm_acc"][_i] >= 0.90)
            _rows.setdefault((_fam, _net), []).append(dict(
                learned=_learned, sep=_st, last=[k for k, v in _st.items() if v == _m], traj=_r["traj"][_i], tsteps=_r["tsteps"],
                rho=vu_rho(lambda q: _pr["pair_rate"][q], _st) if _learned else None,
                rho_pix=vu_rho(lambda q: 1 - _pc[int(q[0]), int(q[1])], _st) if _learned else None))
    return _rows, _pred

def vu_study5(root):
    _res = vu_npz(root, "v25")
    _pairs = list(itertools.combinations(range(9), 2))
    _pred, _rows = {}, {}
    for _name, _r in _res.items():
        _fam = _name[1:].split("_")[0]
        if _fam.endswith("bench") or "traj" not in _r:
            continue
        _net = str(_r["net"])
        _fam = _fam[:-len(_net)]
        if (_fam, _net) not in _pred:
            _pred[(_fam, _net)] = vu_json(root, f"v25/predictions_{_fam}_{_net}.json")
        _pr = _pred[(_fam, _net)]
        _pc = np.array(_pr["pixel_cosine"])
        for _i in range(len(_r["seeds"])):
            _T = _r["traj"][_i].astype(np.float64)
            _Tf = _T[-1]
            _res9 = all(np.minimum(_Tf[a], _Tf[b]).sum() < 0.5 for a, b in _pairs)    # all 36 pairs separated at the end
            _st = vu_sep_times(_T, _r["tsteps"], _pairs)
            _rows.setdefault((_fam, _net), []).append(dict(
                resolved=_res9, sep=_st, rho=vu_rho(lambda q: _pr["pair_rate"][q], _st) if _res9 else None,
                rho_pix=vu_rho(lambda q: 1 - _pc[int(q[0]), int(q[1])], _st) if _res9 else None))
    return _rows, _pred

def vu_first_break(T, ts, lo=0.02, hi=0.05):
    _dev = np.abs(T - 0.1).max((1, 2))
    _u = np.flatnonzero(_dev < lo)
    if not len(_u):
        return None
    _after = np.flatnonzero((np.arange(len(_dev)) > _u[0]) & (_dev >= hi))
    if not len(_after):
        return None
    _M = T[_after[0]]
    _Mc = _M - _M.mean(0) - _M.mean(1, keepdims=True) + _M.mean()
    _F2 = np.abs(np.fft.fft2(_Mc)) ** 2
    return int(ts[_after[0]]), float(_F2[5, 5] / _F2.sum())

def vu_study6(root):
    _res = vu_npz(root, "v26")
    _lam1, _rows = {}, []
    for _name, _r in _res.items():
        _net, _arm = str(_r["net"]), str(_r["arm"])
        _src = _name[1:].split("_")[0][:-len(_net) - len(_arm)]
        if (_src, _net) not in _lam1:
            _lam1[(_src, _net)] = vu_json(root, f"v26/lamc_{_src}_{_net}.json")["lam1"]
        for _i, _sd in enumerate(_r["seeds"]):
            _row = dict(src=_src, net=_net, arm=_arm, seed=int(_sd), opt=bool(_r["optimum"][_i]), C=_r["C"][_i],
                        lam0=float(_r["lam0"]), tau=int(_r["tau"]), brk=None, lam_ratio=None)
            if _arm in ("r10", "r15"):
                _fb = vu_first_break(_r["traj"][_i].astype(np.float64), _r["tsteps"])
                if _fb is not None:
                    _row["brk"] = _fb[1]
                    _row["lam_ratio"] = _row["lam0"] * max(0.0, 1 - _fb[0] / _row["tau"]) / _lam1[(_src, _net)]
            _rows.append(_row)
    return _rows, _lam1


# ------------------------------------------------------------------ 5. figures
VU_DONE, VU_FAILED = [], []

def vu_save(fig, name):
    for _ext in ("pdf", "png"):
        fig.savefig(os.path.join(VU_OUT, f"{name}.{_ext}"), dpi=300, bbox_inches="tight", pad_inches=0.01)
    VU_DONE.append(name)
    print(f"[U] saved {name}.pdf / .png", flush=True)

def vu_label(ax, s):
    ax.text(-0.02, 1.02, s, transform=ax.transAxes, fontsize=7.5, fontweight="bold", va="bottom", ha="right")

def vu_fig1c(s1):
    _fig = Figure(figsize=(2.05, 4.4 * VU_CM))
    _ax = _fig.add_axes([0.2, 0.22, 0.77, 0.68])
    _bins = np.linspace(-0.05, 0.8, 35)
    _sty = [("optimal", "optimum", VU_C["grey"]), ("pair", "pair state (Thm 1)", VU_C["blue"]),
            ("involution", "involution saddle (Thm 2)", VU_C["orange"]), ("other", "other off-optimum", VU_C["light"])]
    _data = [np.clip([r["excess"] for r in s1 if r["cls"] == c], -0.05, 0.8) for c, _, _ in _sty]
    _ax.hist(_data, bins=_bins, stacked=True, color=[c for _, _, c in _sty], label=[f"{l} ({len(d)})" for (_, l, _), d in zip(_sty, _data)],
             edgecolor="white", linewidth=0.3)
    _ax.set_yscale("log")
    _lvs = sorted({round(r["level"], 2) for r in s1 if r["cls"] == "pair" and r["level"] is not None})
    for _lv in _lvs:
        _ax.axvline(_lv, color=VU_C["ink"], lw=0.5, ls=(0, (2, 2)))
    _ax.set_xlabel("excess test NLL (nats)")
    _ax.set_ylabel("networks")
    _ax.set_xlim(-0.05, 0.8)
    _ax.set_ylim(0.7, None)
    _ax.legend(loc="upper right", handlelength=1, borderaxespad=0.1, frameon=True, framealpha=0.92, edgecolor="none", facecolor="white")
    print(f"   fig1c: {len(s1)} Study-I networks; " + ", ".join(f"{c} {len(d)}" for (c, _, _), d in zip(_sty, _data))
          + f"; dashed lines at pair-state levels {_lvs}")
    vu_save(_fig, "fig1c_levels")

def vu_fig2a(pred4):
    _p = pred4.get(("mnist0123", "cnn"))
    if _p is None:
        raise RuntimeError("v24/predictions_mnist0123_cnn.json missing")
    _st = {tuple(tuple(b) for b in sorted(s["partition"])): s for s in _p["states"]}
    _rows = {k: sorted([p for p in _st if len(p) == k], key=lambda p: (sorted(map(len, p), reverse=True), p)) for k in (1, 2, 3, 4)}
    _pos = {}
    for _k, _ps in _rows.items():
        for _i, _q in enumerate(_ps):
            _pos[_q] = ((_i + 0.5) / len(_ps), 1 - (_k - 1) / 3)
    def _covers(p, q):              # q splits one block of p into two
        if len(q) != len(p) + 1:
            return False
        return all(any(set(b) <= set(c) for c in p) for b in q)
    _last = tuple(int(c) for c in _p["predicted_last_pair"])
    _s3 = next(q for q in _rows[3] if _last in q)
    _cand2 = [q for q in _rows[2] if _covers(q, _s3)]
    _s2 = min(_cand2, key=lambda q: _st[q]["rate"] if _st[q]["rate"] is not None else 9)
    _path = {(_rows[1][0], _s2), (_s2, _s3), (_s3, _rows[4][0])}
    _rates = [s["rate"] for s in _st.values() if s["rate"] is not None]
    _lo, _hi = min(_rates), max(_rates)
    _fig = Figure(figsize=(2.55, 3.9 * VU_CM))
    _ax = _fig.add_axes([0.0, 0.0, 1.0, 0.93])
    _ax.axis("off")
    for _q in _pos:
        for _r in _pos:
            if _covers(_q, _r):
                _on = (_q, _r) in _path
                _ax.plot([_pos[_q][0], _pos[_r][0]], [_pos[_q][1], _pos[_r][1]], color=VU_C["red"] if _on else VU_C["light"],
                         lw=1.6 if _on else 0.5, zorder=1)
    from matplotlib import colormaps
    _cmap = colormaps["Blues"]
    for _q, (_x, _y) in _pos.items():
        _s = _st[_q]
        _fc = "#EEEEEE" if _s["rate"] is None else _cmap(0.25 + 0.7 * (1 - (_s["rate"] - _lo) / max(_hi - _lo, 1e-12)))
        _txt = "|".join("".join(map(str, b)) for b in _q)
        _ax.text(_x, _y, f"{_txt}\n{_s['n_neg']}", ha="center", va="center", fontsize=4.6, zorder=3, color=VU_C["ink"],
                 bbox=dict(boxstyle="round,pad=0.25", fc=_fc, ec=VU_C["red"] if any(_q in e for e in _path) else "none", lw=0.6))
    _ax.text(0.0, -0.11, "number: Morse index; fill: escape rate (darker = slower); red: predicted route", fontsize=4.4, color=VU_C["grey"])
    _ax.set_xlim(-0.02, 1.02)
    _ax.set_ylim(-0.14, 1.12)
    vu_label(_ax, "a")
    print(f"   fig2a: MNIST 0123 CNN; predicted last pair {_last}; path {[ '|'.join(''.join(map(str, b)) for b in q) for q in (_rows[1][0], _s2, _s3)]}; "
          f"index = formula for all 15: {all(s['n_neg'] == s['index_formula'] for s in _st.values())}; rates {_lo:.4f}-{_hi:.4f}")
    vu_save(_fig, "fig2a_lattice")

def vu_fig2b():
    _B = vu_programs()["cycle10"]
    _lam, _V = vu_critical(_B)
    _vals = []
    for _l in _lam[_lam > 1e-9]:
        if not _vals or abs(_vals[-1][0] - _l) > 1e-6:
            _vals.append([_l, 0])
        _vals[-1][1] += 1
    _fig = Figure(figsize=(1.95, 3.9 * VU_CM))
    _ax = _fig.add_axes([0.14, 0.25, 0.83, 0.36])
    for _l, _m in _vals:
        _ax.vlines(_l, 0, _m, color=VU_C["blue"], lw=1.4)
        _ax.plot([_l], [_m], "o", ms=2.5, color=VU_C["blue"])
    _ax.set_xlim(0.56, -0.01)
    _ax.set_ylim(0, max(m for _, m in _vals) + 1.5)
    _ax.set_xlabel(r"entropy weight $\lambda$ (annealed $\rightarrow$)")
    _ax.set_ylabel("modes", labelpad=1)
    _ax.set_xticks([0.5, 0.4045, 0.3273, 0.1545, 0.125, 0.0477])
    _ax.set_xticklabels(["0.5", ".40", ".33", ".15", "", ".05"], fontsize=5.2)
    _shown = 0
    _x10 = np.arange(10)
    _modes = [np.outer((-1.0) ** _x10, (-1.0) ** _x10),                                  # mu = -2 x -2: even/odd
              np.outer((-1.0) ** _x10, np.cos(2 * np.pi * 4 * _x10 / 10)),               # mu = -2 x -1.618
              np.cos(2 * np.pi * (_x10[:, None] - _x10[None, :]) / 10)]                  # winding-one, mu = 1.618 x 1.618
    _names = ["even/odd", "even/odd × k=4", "winding 1"]
    for _k, (_l, _m) in enumerate(_vals[:3]):
        _M = _modes[_k]
        _ia = _fig.add_axes([0.16 + _k * 0.29, 0.68, 0.21, 0.22])
        _ia.imshow(_M / np.abs(_M).max(), cmap="RdBu_r", vmin=-1, vmax=1)
        _ia.set_xticks([]); _ia.set_yticks([])
        for _sp in _ia.spines.values():
            _sp.set_visible(True); _sp.set_linewidth(0.4)
        _ia.set_title(f"{_l:.3f}\n{_names[_k]}", fontsize=4.4, pad=1.2, linespacing=1.0)
        if _k == 0:
            _ia.set_ylabel("digit", fontsize=4.5, labelpad=1); _ia.set_xlabel("symbol", fontsize=4.5, labelpad=1)
        _shown += 1
    vu_label(_ax, "b")
    _cf = vu_closed_form(_B)
    print(f"   fig2b: 10-cycle critical values {[(round(float(l), 4), m) for l, m in _vals]}; closed form max error "
          f"{np.abs(np.sort(_lam[_lam > 1e-9])[::-1][:len(_cf[_cf > 1e-9])] - _cf[_cf > 1e-9]).max():.1e}")
    vu_save(_fig, "fig2b_cascade")

def vu_fig3(s1, s2, s3dec):
    _fig = Figure(figsize=(VU_W, 4.0 * VU_CM))
    _a = _fig.add_axes([0.06, 0.24, 0.27, 0.68])
    _p1 = [r for r in s1 if r["cls"] == "pair"]
    _a.plot([0, 1], [0, 1], color=VU_C["ink"], lw=0.6, zorder=1)
    _xx = np.linspace(0, 1, 50)
    _a.fill_between(_xx, 0.8 * _xx, 1.1 * _xx, color=VU_C["light"], alpha=0.5, lw=0, zorder=0)
    _a.scatter([r["level"] for r in _p1], [r["excess"] for r in _p1], s=9, marker="o", color=VU_C["blue"], lw=0, label=f"I pair states ({len(_p1)})", zorder=3)
    _n2 = 0
    for _arm, _fill in (("hi", True), ("lo", False)):
        _d = [r for r in s2.get(_arm, []) if r.get("pi") is not None]
        _n2 += len(_d)
        _a.scatter([r["level"] for r in _d], [r["excess"] for r in _d], s=10, marker="^", lw=0.7, zorder=3,
                   facecolor=VU_C["orange"] if _fill else "none", edgecolor=VU_C["orange"], label=f"II {_arm} ({len(_d)})")
    _big = [r for r in s3dec if r["level"] >= 0.10]
    _small = [r for r in s3dec if r["level"] < 0.10]
    _a.scatter([r["level"] for r in _big], [r["excess"] for r in _big], s=8, marker="s", color=VU_C["green"], lw=0, label=f"III ({len(_big)} + {len(_small)} faded)", zorder=2)
    _a.scatter([r["level"] for r in _small], [r["excess"] for r in _small], s=8, marker="s", color=VU_C["green"], alpha=0.3, lw=0, zorder=2)
    _top = max([0.8] + [r["level"] for r in _p1] + [r["excess"] for r in _p1]) * 1.05
    _a.set_xlim(0, _top); _a.set_ylim(min(0, min([r["excess"] for r in s3dec], default=0)) - 0.02, _top)
    _a.set_xlabel("predicted level of decoded state (nats)")
    _a.set_ylabel("measured excess NLL (nats)")
    _a.legend(loc="lower right", handlelength=1, borderaxespad=0.1, labelspacing=0.25)
    vu_label(_a, "a")
    _b = _fig.add_axes([0.41, 0.24, 0.25, 0.68])
    _u = np.array([r["u"] for r in s3dec])
    _uo = np.array([r["u_other"] for r in s3dec if r["u_other"] is not None])
    _bins = np.linspace(0, 1, 11)
    _b.hist(_u, bins=_bins, color=VU_C["blue"], edgecolor="white", lw=0.3, label=f"own kernel (mean {_u.mean():.2f})")
    _b.hist(_uo, bins=_bins, histtype="step", color=VU_C["red"], lw=1.0, label=f"other families (mean {_uo.mean():.2f})")
    _b.axvline(0.5, color=VU_C["ink"], lw=0.6, ls=(0, (2, 2)))
    _b.set_xlabel("escape percentile $u$ (0 = slowest)")
    _b.set_ylabel("decoded networks (III)")
    _b.legend(loc="upper right", handlelength=1, borderaxespad=0.1)
    vu_label(_b, "b")
    _c = _fig.add_axes([0.74, 0.24, 0.25, 0.68])
    _cnt = Counter(r["cyc"] for r in s2.get("hi", []) if r.get("pi") is not None)
    _h = [_cnt.get(tuple(sorted(o)), 0) for o in VU_HI_ORDER]
    _c.bar(np.arange(1, 26), _h, color=[VU_C["orange"]] + [VU_C["grey"]] * 24, width=0.75)
    _c.set_xlabel("involution rank (slow → fast escape)")
    _c.set_ylabel("decoded networks (II, hi)")
    _c.set_xticks([1, 5, 10, 15, 20, 25])
    _c.text(0.12, 0.95, f"slowest state: {_h[0]} of {sum(_h)}", fontsize=6, va="top", transform=_c.transAxes)
    vu_label(_c, "c")
    print(f"   fig3: (a) {len(_p1)} pair states, {_n2} Study-II involution states, {len(s3dec)} Study-III decoded ({len(_small)} below 0.10); "
          f"(b) mean u own {_u.mean():.3f}, others {_uo.mean():.3f} (n {len(_u)}); (c) hi: slowest state {_h[0]} of {sum(_h)} decoded")
    vu_save(_fig, "fig3")

def vu_fig4(r4, p4, r5, p5):
    _fig = Figure(figsize=(VU_W, 3.8 * VU_CM))
    _a = _fig.add_axes([0.055, 0.25, 0.25, 0.66])
    _k = ("mnist0123", "cnn")
    _pr = p4[_k]
    _pp = _pr["predicted_last_pair"]
    _net = next((r for r in r4[_k] if r["learned"] and r["last"] == [_pp]), next(r for r in r4[_k] if r["learned"]))
    _rates = _pr["pair_rate"]
    _ordr = sorted(_rates, key=_rates.get)
    from matplotlib import colormaps
    _cm = colormaps["viridis"]
    _ts = np.asarray(_net["tsteps"]) + 1
    for _j, _q in enumerate(_ordr):
        _a_, _b_ = int(_q[0]), int(_q[1])
        _ov = np.minimum(_net["traj"][:, _a_].astype(float), _net["traj"][:, _b_].astype(float)).sum(-1)
        _a.plot(_ts, _ov, color=_cm(0.05 + 0.85 * _j / 5), lw=1.0, label=f"({_q[0]},{_q[1]})" + (" pred. last" if _j == 0 else ""))
    _a.axhline(0.5, color=VU_C["ink"], lw=0.5, ls=(0, (2, 2)))
    _a.set_xscale("log")
    _a.set_xlim(1, _ts[-1] * 1.1)
    _a.legend(loc="lower left", fontsize=4.6, handlelength=1, borderaxespad=0.1, labelspacing=0.15)
    _a.set_ylim(0, 1.02)
    _a.set_xlabel("training step")
    _a.set_ylabel("class overlap of digit pair")
    vu_label(_a, "a")
    _b = _fig.add_axes([0.385, 0.25, 0.32, 0.66])
    _fams = [("IV", k, r4) for k in sorted(r4)] + [("V", k, r5) for k in sorted(r5) if k[1] != "cnnpre"]
    _ab = {"mnist": "M", "fashion": "F", "kmnist": "K"}
    _lab, _kv, _pv = [], [], []
    for _st, _kk, _rr in _fams:
        _kr = [x["rho"] for x in _rr[_kk] if x.get("rho") is not None]
        _pr_ = [x["rho_pix"] for x in _rr[_kk] if x.get("rho_pix") is not None]
        if len(_kr) < 2:
            continue
        _src = next((v for k, v in _ab.items() if _kk[0].startswith(k)), _kk[0][:1].upper())
        _lab.append(f"{_src}{chr(8242) if _kk[0].endswith('4589') else ''}\n{_kk[1][0].upper()}")
        _kv.append(_kr); _pv.append(_pr_)
    _x = np.arange(len(_kv))
    _vk = _b.violinplot(_kv, positions=_x - 0.17, widths=0.36, showextrema=False, showmedians=True)
    for _bd in _vk["bodies"]:
        _bd.set_facecolor(VU_C["blue"]); _bd.set_alpha(0.75); _bd.set_edgecolor("none")
    _vk["cmedians"].set_color("white"); _vk["cmedians"].set_linewidth(0.8)
    _vp = _b.violinplot(_pv, positions=_x + 0.17, widths=0.36, showextrema=False, showmedians=True)
    for _bd in _vp["bodies"]:
        _bd.set_facecolor("none"); _bd.set_edgecolor(VU_C["red"]); _bd.set_linewidth(0.7); _bd.set_alpha(1)
    _vp["cmedians"].set_color(VU_C["red"]); _vp["cmedians"].set_linewidth(0.8)
    _n4 = sum(1 for s, k, _ in _fams if s == "IV" and k in r4)
    _b.axvline(_n4 - 0.5, color=VU_C["light"], lw=0.6)
    _b.text(_n4 / 2 - 0.5, 1.05, "4×4 (IV)", ha="center", fontsize=6)
    _b.text(_n4 + (len(_kv) - _n4) / 2 - 0.5, 1.05, "9×9 (V)", ha="center", fontsize=6)
    _b.axhline(0, color=VU_C["ink"], lw=0.5)
    _b.set_xticks(_x); _b.set_xticklabels(_lab, fontsize=5)
    _b.set_ylim(-1.05, 1.15)
    _b.set_ylabel(r"per-network Spearman $\rho$")
    from matplotlib.patches import Patch
    _b.legend(handles=[Patch(fc=VU_C["blue"], alpha=0.75, label="kernel"), Patch(fc="none", ec=VU_C["red"], label="pixel cosine")],
              loc="lower right", ncol=2, handlelength=1, borderaxespad=0.1, frameon=True, framealpha=0.9, edgecolor="none", facecolor="white")
    vu_label(_b, "b")
    _c = _fig.add_axes([0.785, 0.25, 0.205, 0.66])
    _txt = "J4 data missing"
    if ("mnist", "cnnpre") in p5 and ("mnist", "cnn") in p5:
        _kp, _k0 = p5[("mnist", "cnnpre")], p5[("mnist", "cnn")]
        _keys = sorted(_kp["pair_rate"])
        _dr = np.array([_kp["pair_rate"][q] - _k0["pair_rate"][q] for q in _keys])
        _tr = [r for r in r5.get(("mnist", "cnnpre"), []) if r["resolved"]]
        _pl = [r for r in r5.get(("mnist", "cnn"), []) if r["resolved"]]
        _R = np.array([rankdata([r["sep"][q] for q in _keys]) / 36 for r in _tr + _pl])
        _nt = len(_tr)
        _dR = _R[:_nt].mean(0) - _R[_nt:].mean(0)
        _obs = float(spearmanr(_dr, _dR).correlation)
        _rg = np.random.default_rng(25)
        _null = np.array([spearmanr(_dr, _R[ix[:_nt]].mean(0) - _R[ix[_nt:]].mean(0)).correlation
                          for ix in (_rg.permutation(len(_R)) for _ in range(10000))])
        _p = float((1 + np.sum(_null <= _obs)) / (1 + len(_null)))
        _c.scatter(_dr, _dR, s=7, color=VU_C["blue"], lw=0)
        _co = np.polyfit(_dr, _dR, 1)
        _xx = np.linspace(_dr.min(), _dr.max(), 10)
        _c.plot(_xx, np.polyval(_co, _xx), color=VU_C["red"], lw=0.9)
        _c.axhline(0, color=VU_C["light"], lw=0.5); _c.axvline(0, color=VU_C["light"], lw=0.5)
        _txt = f"$\\rho$ = {_obs:+.2f}, shuffle p = {_p:.3f}\nfrozen threshold 0.01: not met" if _p >= 0.01 else f"$\\rho$ = {_obs:+.2f}, shuffle p = {_p:.3f}"
        _c.set_title(_txt, fontsize=5.3, pad=2)
        print(f"   fig4c (J4): {_nt} transfer vs {len(_pl)} plain resolved CNNs; Spearman {_obs:+.3f}, shuffle p {_p:.4f}")
    _c.set_xlabel(r"$\Delta$ predicted escape rate")
    _c.set_ylabel(r"$\Delta$ mean separation rank")
    vu_label(_c, "c")
    print(f"   fig4: (a) MNIST 0123 CNN example, predicted last pair {_pp}, its last pairs {_net['last']}; (b) "
          + "; ".join(f"{l.replace(chr(10), '/')} kernel {np.mean(k):+.2f} pixel {np.mean(p):+.2f} (n {len(k)})" for l, k, p in zip(_lab, _kv, _pv)))
    vu_save(_fig, "fig4")

def vu_ring(ax, C, title, B):
    _s = C.argmax(1)
    _ang = np.pi / 2 - 2 * np.pi * np.arange(10) / 10
    _x, _y = np.cos(_ang), np.sin(_ang)
    _walls = 0
    for _d in range(10):
        _e = (_d + 1) % 10
        _ok = B[_s[_d], _s[_e]] == 1
        _walls += not _ok
        ax.plot([_x[_d], _x[_e]], [_y[_d], _y[_e]], color=VU_C["light"] if _ok else VU_C["red"], lw=0.8 if _ok else 1.8, zorder=1)
    for _d in range(10):
        ax.scatter(_x[_d], _y[_d], s=34, color="white", edgecolor=VU_C["ink"], lw=0.5, zorder=2)
        ax.text(_x[_d], _y[_d], str(_s[_d]), ha="center", va="center", fontsize=4.8, zorder=3)
        ax.text(1.38 * _x[_d], 1.38 * _y[_d], str(_d), ha="center", va="center", fontsize=4, color=VU_C["grey"])
    ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.6, 1.6); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=4.8, pad=0, linespacing=1.0)
    return "".join(map(str, _s)), _walls

def vu_fig5(r6, lam1):
    _fig = Figure(figsize=(VU_W, 3.7 * VU_CM))
    _a = _fig.add_axes([0.055, 0.25, 0.27, 0.66])
    _dose = {"frozen0": 0.0, "r06": 0.6, "r10": 1.0, "r15": 1.5}
    _col = {"mnist": VU_C["blue"], "fashion": VU_C["orange"], "kmnist": VU_C["red"]}
    _lines = []
    for (_src, _net) in sorted(lam1):
        _xs, _ys, _lo, _hi = [], [], [], []
        for _arm, _x in _dose.items():
            _o = [r["opt"] for r in r6 if r["src"] == _src and r["net"] == _net and r["arm"] == _arm]
            if not _o:
                continue
            _w = vu_wilson(sum(_o), len(_o))
            _xs.append(_x + (0.03 if _net == "mlp" else -0.03)); _ys.append(np.mean(_o)); _lo.append(np.mean(_o) - _w[0]); _hi.append(_w[1] - np.mean(_o))
        _c = _col.get(_src, VU_C["grey"])
        _a.errorbar(_xs, _ys, yerr=[_lo, _hi], color=_c, ls="-" if _net == "cnn" else "--", lw=0.9, marker="o" if _net == "cnn" else "s",
                    ms=2.5, elinewidth=0.5, capsize=0, label=f"{_src} {_net.upper()}")
        _pl = [r["opt"] for r in r6 if r["src"] == _src and r["net"] == _net and r["arm"] == "plain"]
        if _pl:
            _a.plot([-0.32, -0.2], [np.mean(_pl)] * 2, color=_c, lw=1.4, ls="-" if _net == "cnn" else "--")
        _lines.append(f"{_src}/{_net} plain {np.mean(_pl) if _pl else float('nan'):.2f} -> " + ", ".join(f"{y:.2f}" for y in _ys))
    _a.set_xticks([-0.26, 0, 0.6, 1.0, 1.5]); _a.set_xticklabels(["plain", "0", "0.6", "1.0", "1.5"])
    _a.set_xlim(-0.4, 1.62); _a.set_ylim(-0.03, 1.05)
    _a.set_xlabel(r"anneal start $\lambda_0/\lambda_1^{\mathrm{net}}$")
    _a.set_ylabel("optimum rate")
    _a.legend(loc="upper left", ncol=2, handlelength=1.4, columnspacing=0.6, borderaxespad=0.1, fontsize=4.8)
    vu_label(_a, "a")
    _b = _fig.add_axes([0.40, 0.25, 0.22, 0.66])
    _so = np.array([r["brk"] for r in r6 if r["brk"] is not None and r["opt"]])
    _st = np.array([r["brk"] for r in r6 if r["brk"] is not None and not r["opt"]])
    _bins = np.linspace(0, 1, 21)
    _b.hist(_so, bins=_bins, color=VU_C["grey"], edgecolor="white", lw=0.3, label=f"optimum ({len(_so)})")
    _b.hist(_st, bins=_bins, histtype="step", color=VU_C["ink"], lw=0.9, label=f"trapped ({len(_st)})")
    _b.axvline(0.3, color=VU_C["red"], lw=0.7, ls=(0, (2, 2)))
    _auc = float(np.mean([(a > b) + 0.5 * (a == b) for a in _so for b in _st])) if len(_so) and len(_st) else float("nan")
    _b.text(0.97, 0.55, f"AUROC {_auc:.2f}", transform=_b.transAxes, fontsize=5.8, va="top", ha="right")
    _b.set_xlabel("even/odd share of first deviation")
    _b.set_ylabel("annealed networks")
    _b.legend(loc="upper right", handlelength=1, borderaxespad=0.1)
    vu_label(_b, "b")
    _B = vu_programs()["cycle10"]
    _ex = [("plain", None, False, "plain\ntraining"), ("r15", "mnist", True, "MNIST\nannealed"),
           ("r15", "kmnist", False, "KMNIST\nannealed"), ("r15", "kmnist", False, "KMNIST\nannealed")]
    _used, _maps = set(), []
    for _j, (_arm, _src, _opt, _tt) in enumerate(_ex):
        _cand = [r for r in r6 if r["arm"] == _arm and (_src is None or r["src"] == _src) and r["opt"] == _opt and (r["src"], r["net"], r["seed"], r["arm"]) not in _used]
        _ax = _fig.add_axes([0.655 + _j * 0.087, 0.12, 0.085, 0.80])
        if not _cand:
            _ax.axis("off"); _ax.text(0.5, 0.5, "no such\nnetwork", ha="center", fontsize=5, transform=_ax.transAxes)
            continue
        _r = sorted(_cand, key=lambda r: (r["net"] != "cnn", r["seed"]))[0]
        _used.add((_r["src"], _r["net"], _r["seed"], _r["arm"]))
        _m, _w = vu_ring(_ax, _r["C"], f"{_tt}, {_r['net'].upper()}", _B)
        _ax.text(0.5, -0.02, f"{_m}\n{_w} wall{'s' if _w != 1 else ''}", ha="center", va="top", fontsize=4.6, transform=_ax.transAxes, family="monospace")
        _maps.append(f"{_tt} {_r['src']}/{_r['net']} seed {_r['seed']}: {_m} ({_w} walls)")
        if _j == 0:
            vu_label(_ax, "c")
    print(f"   fig5: (a) " + "; ".join(_lines) + f"; (b) optimum {len(_so)} vs trapped {len(_st)}, AUROC {_auc:.3f}; (c) " + "; ".join(_maps))
    vu_save(_fig, "fig5")

def vu_figS1(p4):
    _fig = Figure(figsize=(VU_W, 4.2 * VU_CM))
    _a = _fig.add_axes([0.055, 0.24, 0.26, 0.68])
    _rng = np.random.default_rng(0)
    _xs = [(s["index_formula"], s["n_neg"]) for p in p4.values() for s in p["states"]]
    _a.scatter([x + _rng.uniform(-0.15, 0.15) for x, _ in _xs], [y + _rng.uniform(-0.15, 0.15) for _, y in _xs], s=5, color=VU_C["blue"],
               lw=0, alpha=0.6, label=f"4×4 Sudoku, {len(p4)} families ({len(_xs)})")
    _eq, _bad = [], 0
    for _n in (4, 5, 6):
        _B = np.eye(_n, dtype=int)
        _seen = {}
        for _p in vu_set_partitions(list(range(_n))):
            _ty = tuple(sorted(map(len, _p), reverse=True))
            if _ty in _seen:
                continue
            _Z = np.full((_n, _n), -30.0)
            for _b in _p:
                for _d in _b:
                    _Z[_d, _b] = 0.0
            _ev = np.linalg.eigvalsh(vu_hessians(_B, _Z)[0])
            _seen[_ty] = (sum((len(b) - 1) ** 2 for b in _p), int((_ev < -1e-9).sum()))
        _eq += [(_n, t, f, k) for t, (f, k) in _seen.items()]
    _bad = sum(f != k for _, _, f, k in _eq) + sum(x != y for x, y in _xs)
    _a.scatter([f for _, _, f, _ in _eq], [k for _, _, _, k in _eq], s=16, marker="D", facecolor="none", edgecolor=VU_C["red"], lw=0.7,
               label=f"$\\beta=[a=b]$, n=4,5,6, all types ({len(_eq)})")
    _m = max([x for x, _ in _xs] + [f for _, _, f, _ in _eq]) + 1
    _a.plot([0, _m], [0, _m], color=VU_C["ink"], lw=0.5, zorder=0)
    _a.set_xlabel(r"$\sum_b(|b|-1)^2$")
    _a.set_ylabel("negative Hessian eigenvalues")
    _a.legend(loc="upper left", handlelength=1, borderaxespad=0.1)
    vu_label(_a, "a")
    _b = _fig.add_axes([0.39, 0.24, 0.26, 0.68])
    _err, _ng = 0.0, 0
    from matplotlib import colormaps
    _G = vu_graphs()
    _cm = colormaps["tab20"]
    for _j, (_nm, _A) in enumerate(_G.items()):
        _lam, _ = vu_critical(_A)
        _lam = _lam[_lam > 1e-9]
        _cf = vu_closed_form(_A)
        _cf = _cf[_cf > 1e-9]
        if len(_cf) != len(_lam):
            raise RuntimeError(f"{_nm}: {len(_lam)} numerical vs {len(_cf)} closed-form critical values")
        _err = max(_err, float(np.abs(_lam - _cf).max()))
        _b.scatter(_cf, _lam, s=5, color=_cm(_j / 20), lw=0, label=_nm)
        _ng += 1
    _b.plot([0, 1.05], [0, 1.05], color=VU_C["ink"], lw=0.5, zorder=0)
    _b.set_xlabel(r"closed form $2\mu_i\mu_j/(d(n-d))$")
    _b.set_ylabel("critical value from exact Hessian")
    _b.text(0.97, 0.04, f"{_ng} graphs, max |error| {_err:.0e}", transform=_b.transAxes, ha="right", fontsize=5.6)
    _b.legend(loc="upper left", ncol=2, fontsize=4.2, handlelength=0.6, columnspacing=0.4, borderaxespad=0.05, labelspacing=0.15)
    vu_label(_b, "b")
    _c = _fig.add_axes([0.73, 0.24, 0.26, 0.68])
    _l0 = np.array([0.30, 0.32, 0.34, 0.37, 0.39, 0.41, 0.43])       # Appendix B CPU sweep (free logits, Adam, 2,000-step
    _rt = np.array([0.623, 0.665, 0.730, 0.895, 0.968, 1.000, 1.000])  # anneal, 400 starts each); not in the bucket
    _ci = np.array([vu_wilson(round(r * 400), 400) for r in _rt])
    _c.errorbar(_l0, _rt, yerr=[_rt - _ci[:, 0], _ci[:, 1] - _rt], color=VU_C["blue"], marker="o", ms=2.5, lw=0.9, elinewidth=0.6, capsize=0)
    for _v, _t in ((0.4045, r"$\lambda_2$"), (0.3273, r"$\lambda_3$")):
        _c.axvline(_v, color=VU_C["red"], lw=0.6, ls=(0, (2, 2)))
        _c.text(_v + 0.002, 0.58, _t, fontsize=6, color=VU_C["red"])
    _c.set_xlabel(r"anneal start $\lambda_0$ (free logits, 10-cycle)")
    _c.set_ylabel("optimum rate (400 starts)")
    _c.set_ylim(0.55, 1.03)
    vu_label(_c, "c")
    print(f"   figS1: (a) 4x4 states {len(_xs)}, 1[a=b] types {len(_eq)}, mismatches with the formula {_bad}; (b) {_ng} graphs, max |error| {_err:.1e}; "
          f"(c) embedded Appendix-B sweep")
    vu_save(_fig, "figS1")

def vu_figS2(cells3, r6, lam1):
    _fig = Figure(figsize=(VU_W, 4.0 * VU_CM))
    _a = _fig.add_axes([0.10, 0.27, 0.40, 0.62])
    _ds = ["mnist", "fashion", "kmnist", "svhn"]
    _cols = [(n, t) for n in ("cnn", "mlp") for t in ("petersen", "crown", "cycle10")]
    _M = np.full((len(_ds), len(_cols)), np.nan)
    from matplotlib import colormaps
    _cm = colormaps["Blues_r"].copy()
    _cm.set_bad("#F2F2F2")
    for _i, _d in enumerate(_ds):
        for _j, (_n, _t) in enumerate(_cols):
            _c = cells3.get((_d, _n, _t))
            if _c is None:
                _s = "no run"
            elif _c["dec"] == 0:
                _s = "none\ndecoded" if _c["learned"] else "did not\nlearn"
            else:
                _M[_i, _j] = _c["mean_u"]
                _s = f"{_c['mean_u']:.2f}\nn={_c['dec']}"
            _a.text(_j, _i, _s, ha="center", va="center", fontsize=4.5,
                    color="white" if (not np.isnan(_M[_i, _j]) and _M[_i, _j] < 0.3) else VU_C["ink"])
    _im = _a.imshow(np.ma.masked_invalid(_M), cmap=_cm, vmin=0, vmax=1, aspect="auto")
    _a.set_xticks(range(len(_cols))); _a.set_xticklabels([f"{t if t != 'cycle10' else 'cycle'}\n{n.upper()}" for n, t in _cols], fontsize=5)
    _a.set_yticks(range(len(_ds))); _a.set_yticklabels(["MNIST", "Fashion", "KMNIST", "SVHN"], fontsize=5.5)
    for _sp in _a.spines.values():
        _sp.set_visible(False)
    _a.tick_params(length=0)
    _cb = _fig.colorbar(_im, ax=_a, fraction=0.04, pad=0.01)
    _cb.set_label("mean escape percentile $u$", fontsize=5.5); _cb.ax.tick_params(labelsize=5)
    vu_label(_a, "a")
    _b = _fig.add_axes([0.62, 0.27, 0.36, 0.62])
    _col = {"mnist": VU_C["blue"], "fashion": VU_C["orange"], "kmnist": VU_C["red"]}
    _lr = {s: [r["lam_ratio"] for r in r6 if r["src"] == s and r["lam_ratio"] is not None] for s in sorted({r["src"] for r in r6})}
    _all = np.array([x for v in _lr.values() for x in v])
    _bins = np.linspace(0, 1.25, 26)
    _b.hist([_lr[s] for s in _lr], bins=_bins, stacked=True, color=[_col.get(s, VU_C["grey"]) for s in _lr], edgecolor="white", lw=0.3,
            label=[f"{s} ({len(v)})" for s, v in _lr.items()])
    _q1, _md, _q3 = np.percentile(_all, [25, 50, 75])
    _b.axvspan(_q1, _q3, color=VU_C["light"], alpha=0.35, lw=0, zorder=0)
    _b.axvline(_md, color=VU_C["ink"], lw=0.7, ls=(0, (2, 2)))
    _b.axvline(1.0, ymax=0.55, color=VU_C["ink"], lw=0.5)
    _b.set_xlabel(r"$\lambda$ at first break / $\lambda_1^{\mathrm{net}}$")
    _b.set_ylabel("annealed networks")
    _b.set_ylim(0, _b.get_ylim()[1] * 1.7)
    _b.set_xlim(0, 1.25)
    _b.legend(loc="upper left", handlelength=1, borderaxespad=0.1)
    _ia = _fig.add_axes([0.865, 0.62, 0.11, 0.24])
    _xr = np.array([0.45, 0.6, 0.75, 0.9, 1.2])                       # Appendix B constant-lambda MLP runs (CPU); not in the bucket
    _yd = np.array([0.69, 0.45, 0.16, 0.025, 0.0])
    _ia.errorbar(_xr, _yd, yerr=[[0, 0, 0, 0.005, 0], [0, 0, 0, 0.005, 0]], color=VU_C["ink"], marker="o", ms=2, lw=0.7, elinewidth=0.6, capsize=0)
    _ia.axvline(1.0, color=VU_C["red"], lw=0.5, ls=(0, (2, 2)))
    _ia.set_xticks([0.5, 1.0]); _ia.set_yticks([0, 0.5])
    _ia.tick_params(labelsize=4.5, length=1.5)
    _ia.set_title(r"fixed $\lambda$: deviation", fontsize=4.6, pad=1)
    vu_label(_b, "b")
    print(f"   figS2: (a) {sum(1 for c in cells3.values() if c['dec'])} cells with decoded networks; (b) {len(_all)} annealed networks with a break, "
          f"median {_md:.2f}, IQR {_q1:.2f}-{_q3:.2f}")
    vu_save(_fig, "figS2")


# ------------------------------------------------------------------ 6. run
def vu_try(name, fn, *args):
    try:
        fn(*args)
    except Exception as _e:
        VU_FAILED.append(name)
        print(f"[U] {name} FAILED: {type(_e).__name__}: {_e}")
        traceback.print_exc(limit=3)

VU_ROOT = vu_fetch()
print("[U] reading results ...", flush=True)
VU_UNITS = vu_units_json(VU_ROOT)
VU_S1 = vu_study1(VU_ROOT, VU_UNITS)
VU_S2 = vu_study2(VU_UNITS)
VU_S3DEC, VU_S3CELLS, _ = vu_study3(VU_ROOT, VU_UNITS)
VU_R4, VU_P4 = vu_study4(VU_ROOT)
VU_R5, VU_P5 = vu_study5(VU_ROOT)
VU_R6, VU_LAM1 = vu_study6(VU_ROOT)
print(f"[U] Study I {len(VU_S1)} networks, II {sum(len(v) for v in VU_S2.values())}, III cells {len(VU_S3CELLS)}, "
      f"IV {sum(len(v) for v in VU_R4.values())}, V {sum(len(v) for v in VU_R5.values())}, VI {len(VU_R6)}", flush=True)
vu_try("fig1c_levels", vu_fig1c, VU_S1)
vu_try("fig2a_lattice", vu_fig2a, VU_P4)
vu_try("fig2b_cascade", vu_fig2b)
vu_try("fig3", vu_fig3, VU_S1, VU_S2, VU_S3DEC)
vu_try("fig4", vu_fig4, VU_R4, VU_P4, VU_R5, VU_P5)
vu_try("fig5", vu_fig5, VU_R6, VU_LAM1)
vu_try("figS1", vu_figS1, VU_P4)
vu_try("figS2", vu_figS2, VU_S3CELLS, VU_R6, VU_LAM1)

VU_ZIP = VU_OUT.rstrip("/") + ".zip"
with zipfile.ZipFile(VU_ZIP, "w", zipfile.ZIP_DEFLATED) as _z:
    for _n in sorted(os.listdir(VU_OUT)):
        _z.write(os.path.join(VU_OUT, _n), os.path.join("figures", _n))
if not VU_LOCAL and VU_DONE:
    try:
        # from huggingface_hub import HfApi
        _run = os.path.basename(VU_ROOT.rstrip("/"))
        HfApi(token=VU_TOKEN).batch_bucket_files(VU_BUCKET, add=[(open(os.path.join(VU_OUT, n), "rb").read(), f"{VU_PREFIX}/{_run}/figures/{n}") for n in sorted(os.listdir(VU_OUT))])
        print(f"[U] figures uploaded to hf://buckets/{VU_BUCKET}/{VU_PREFIX}/{_run}/figures/")
    except Exception as _e:
        print(f"[U] upload failed ({type(_e).__name__}: {_e}); the figures are saved locally")
print(f"[U] done: {len(VU_DONE)} figures in {os.path.abspath(VU_OUT)} (zip: {os.path.abspath(VU_ZIP)}); "
      f"failed: {VU_FAILED or 'none'}. Copy the PDFs to the paper's figures/ folder.")

In [ ]:
def _cv_main():
    print(f"[V] run folder used by cell U: {VU_ROOT}")
    # 1. Unit files per key: several files for one key, or one seed in several files, means duplicated or re-run data.
    _d = os.path.join(VU_ROOT, "units")
    _files = {}
    for _n in sorted(os.listdir(_d)) if os.path.isdir(_d) else []:
        if _n.endswith(".json"):
            with open(os.path.join(_d, _n)) as _fh:
                _u = json.load(_fh)
            _files.setdefault(_u["key"], []).append((_n, len(_u["per_seed"])))
    for _k in sorted(_files):
        if not (_k.startswith("E8_petersen") or _k.startswith("E9")):
            continue
        _seeds = Counter(int(e["seed"]) for e in VU_UNITS.get(_k, []))
        _dup = sum(c - 1 for c in _seeds.values() if c > 1)
        print(f"   {_k}: {len(_files[_k])} files, {sum(n for _, n in _files[_k])} records, "
              f"{len(_seeds)} distinct seeds, {_dup} duplicate records"
              + ("" if len(_files[_k]) == 1 else "  files: " + ", ".join(f"{n}({m})" for n, m in _files[_k])))
    # 2. Frozen test outputs stored next to the units, if any.
    for _root, _dirs, _fs in os.walk(VU_ROOT):
        for _f in _fs:
            if _f in ("tests.json", "amendments.json"):
                _p = os.path.join(_root, _f)
                with open(_p) as _fh:
                    _t = json.load(_fh)
                _txt = json.dumps(_t)
                print(f"[V] {os.path.relpath(_p, VU_ROOT)} ({len(_txt)} chars): {_txt[:600]}")
    # 3. The counts cell U computed, by arm and cell.
    for _arm in ("hi", "lo"):
        _rows = VU_S2.get(_arm, [])
        print(f"[V] Study II {_arm}: {len(_rows)} networks, {sum(not r['optimal'] for r in _rows)} stuck, "
              f"{sum(r.get('pi') is not None for r in _rows)} decoded (paper: hi 36, lo 19)")
    for _k, _c in sorted(VU_S3CELLS.items()):
        if _c["dec"]:
            print(f"[V] Study III {'/'.join(_k)}: n {_c['n']}, stuck {_c['stuck']}, decoded {_c['dec']}, mean u {_c['mean_u']:.3f}")
    print(f"[V] Study III decoded total {len(VU_S3DEC)} (paper 115)")

_cv_main()